In [1]:
!pip install -q -U transformers datasets accelerate evaluate

import os
import re
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
import datasets

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
PyTorch version: 2.11.0+cu128
Transformers version: 5.14.1
Datasets version: 5.0.0
CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os

DATA_DIR = "/content/drive/MyDrive/Task2_FinalShot/data"
OUTPUT_DIR = "/content/drive/MyDrive/Task2_FinalShot/outputs"
MODEL_DIR = "/content/drive/MyDrive/Task2_FinalShot/paragraph_qa_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.jsonl")
VAL_PATH = os.path.join(DATA_DIR, "val.jsonl")
TEST_PATH = os.path.join(DATA_DIR, "test.jsonl")
SAMPLE_PATH = os.path.join(DATA_DIR, "sample_solution.csv")

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH, SAMPLE_PATH]:
    print(os.path.basename(path), os.path.exists(path))

train.jsonl True
val.jsonl True
test.jsonl True
sample_solution.csv True


In [12]:
import json
import pandas as pd


def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number}: {path}"
                ) from error

    return pd.DataFrame(records)


train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Sample solution shape:", sample_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nSample solution columns:")
print(sample_df.columns.tolist())

display(train_df.head(2))
display(sample_df.head())

Train shape: (3200, 14)
Validation shape: (400, 14)
Test shape: (400, 10)
Sample solution shape: (400, 2)

Train columns:
['uuid', 'postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags']

Validation columns:
['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags', 'id']

Test columns:
['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'id']

Sample solution columns:
['id', 'spoiler']


,uuid,postId,postText,postPlatform,targetParagraphs,targetTitle,targetDescription,targetKeywords,targetMedia,targetUrl,provenance,spoiler,spoilerPositions,tags
0,0af11f6b-c889-4520-9372-66ba25cb7657,532quh,"[Wes Welker Wanted Dinner With Tom Brady, But ...",reddit,[It’ll be just like old times this weekend for...,"Wes Welker Wanted Dinner With Tom Brady, But P...",It'll be just like old times this weekend for ...,"new england patriots, ricky doyle, top stories,","[http://pixel.wp.com/b.gif?v=noscript, http://...",http://nesn.com/2016/09/wes-welker-wanted-dinn...,"{'source': 'anonymized', 'humanSpoiler': 'They...",[how about that morning we go throw?],"[[[3, 151], [3, 186]]]",[passage]
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,411701128456593408,[NASA sets date for full recovery of ozone hole],Twitter,[2070 is shaping up to be a great year for Mot...,Hole In Ozone Layer Expected To Make Full Reco...,2070 is shaping up to be a great year for Moth...,"ozone layer,ozone hole determined by weather,M...",[http://s.m.huffpost.com/assets/Logo_Huffingto...,http://huff.to/1cH672Z,"{'source': 'anonymized', 'humanSpoiler': '2070...",[2070],"[[[0, 0], [0, 4]]]",[phrase]


,id,spoiler
0,0,"Woman Buys Doormat Off Amazon, Here’s What She..."
1,1,Something Unexpected Gave This Paralyzed Man H...
2,2,"In the Digital Age, is shopping America's new ..."
3,3,Snowshoer Who Fell Into Tree Well Survives Aft...
4,4,This Rock Seems Out Of Place. When He Moves It...


## validate before building teh QA dataset

In [13]:
import re
import pandas as pd


def normalize_text(value):
    """
    Normalize text only for comparison.
    Character offsets will still use the original paragraph text.
    """
    if value is None:
        return ""

    value = str(value)
    value = re.sub(r"\s+", " ", value).strip().lower()

    return value


def get_spoiler_type(value):
    if isinstance(value, list) and len(value) > 0:
        return str(value[0])

    return str(value)


def validate_spoiler_positions(dataframe, split_name):
    """
    Check whether each spoilerPosition points to valid text inside
    targetParagraphs.

    The end character is treated as exclusive:
        paragraph[start_character:end_character]
    """
    records = []

    for row_number, row in dataframe.iterrows():
        paragraphs = row.get("targetParagraphs", [])
        positions = row.get("spoilerPositions", [])
        gold_spoilers = row.get("spoiler", [])

        if not isinstance(paragraphs, list):
            paragraphs = []

        if not isinstance(positions, list):
            positions = []

        if not isinstance(gold_spoilers, list):
            gold_spoilers = [gold_spoilers]

        article_id = row.get("uuid", row.get("id", row_number))
        spoiler_type = get_spoiler_type(row.get("tags", ""))

        normalized_gold = [
            normalize_text(spoiler)
            for spoiler in gold_spoilers
        ]

        for span_number, span in enumerate(positions):
            result = {
                "split": split_name,
                "row_number": row_number,
                "article_id": article_id,
                "spoiler_type": spoiler_type,
                "span_number": span_number,
                "position": span,
                "status": None,
                "paragraph_index": None,
                "answer_start": None,
                "answer_end": None,
                "extracted_text": None,
                "exact_gold_match": False,
                "partial_gold_match": False
            }

            if not isinstance(span, list) or len(span) != 2:
                result["status"] = "malformed_span"
                records.append(result)
                continue

            start_position, end_position = span

            if (
                not isinstance(start_position, list)
                or not isinstance(end_position, list)
                or len(start_position) != 2
                or len(end_position) != 2
            ):
                result["status"] = "malformed_position"
                records.append(result)
                continue

            try:
                start_paragraph = int(start_position[0])
                start_character = int(start_position[1])
                end_paragraph = int(end_position[0])
                end_character = int(end_position[1])
            except (TypeError, ValueError):
                result["status"] = "non_integer_position"
                records.append(result)
                continue

            result["paragraph_index"] = start_paragraph
            result["answer_start"] = start_character
            result["answer_end"] = end_character

            if start_paragraph != end_paragraph:
                result["status"] = "cross_paragraph"
                records.append(result)
                continue

            if start_paragraph < 0:
                result["status"] = "negative_paragraph_index"
                records.append(result)
                continue

            if start_paragraph >= len(paragraphs):
                result["status"] = "paragraph_index_out_of_range"
                records.append(result)
                continue

            context = str(paragraphs[start_paragraph])

            if (
                start_character < 0
                or end_character <= start_character
                or end_character > len(context)
            ):
                result["status"] = "character_offset_out_of_range"
                records.append(result)
                continue

            extracted_text = context[start_character:end_character]
            normalized_extracted = normalize_text(extracted_text)

            result["extracted_text"] = extracted_text
            result["exact_gold_match"] = (
                normalized_extracted in normalized_gold
            )

            result["partial_gold_match"] = any(
                normalized_extracted in gold
                or gold in normalized_extracted
                for gold in normalized_gold
                if gold and normalized_extracted
            )

            result["status"] = "valid"
            records.append(result)

    return pd.DataFrame(records)


train_span_check = validate_spoiler_positions(
    train_df,
    split_name="train"
)

val_span_check = validate_spoiler_positions(
    val_df,
    split_name="validation"
)

all_span_check = pd.concat(
    [train_span_check, val_span_check],
    ignore_index=True
)

print("Total annotated spans:", len(all_span_check))

print("\nStatus counts:")
print(all_span_check["status"].value_counts())

valid_spans = all_span_check[
    all_span_check["status"] == "valid"
]

print("\nValid spans:", len(valid_spans))
print(
    "Exact gold matches:",
    valid_spans["exact_gold_match"].sum()
)
print(
    "Exact or partial gold matches:",
    (
        valid_spans["exact_gold_match"]
        | valid_spans["partial_gold_match"]
    ).sum()
)

print("\nValid spans by spoiler type:")
print(
    pd.crosstab(
        valid_spans["spoiler_type"],
        valid_spans["split"],
        margins=True
    )
)

print("\nExample valid spans:")
display(
    valid_spans[
        [
            "split",
            "article_id",
            "spoiler_type",
            "paragraph_index",
            "answer_start",
            "answer_end",
            "extracted_text",
            "exact_gold_match"
        ]
    ].head(12)
)

problem_spans = all_span_check[
    all_span_check["status"] != "valid"
]

print("\nProblem spans:")
display(problem_spans.head(20))

Total annotated spans: 5234

Status counts:
status
valid                            5070
negative_paragraph_index           83
cross_paragraph                    71
character_offset_out_of_range       9
malformed_span                      1
Name: count, dtype: int64

Valid spans: 5070
Exact gold matches: 5059
Exact or partial gold matches: 5067

Valid spans by spoiler type:
split         train  validation   All
spoiler_type                         
multi          1931         293  2224
passage        1222         147  1369
phrase         1321         156  1477
All            4474         596  5070

Example valid spans:


,split,article_id,spoiler_type,paragraph_index,answer_start,answer_end,extracted_text,exact_gold_match
0,train,0af11f6b-c889-4520-9372-66ba25cb7657,passage,3.0,151.0,186.0,how about that morning we go throw?,True
1,train,b1a1f63d-8853-4a11-89e8-6b2952a393ec,phrase,0.0,0.0,4.0,2070,True
2,train,008b7b19-0445-4e16-8f9e-075b73f80ca4,phrase,1.0,186.0,210.0,intellectual stimulation,True
3,train,31ecf93c-3e21-4c80-949b-aa549a046b93,multi,11.0,25.0,101.0,Purpose connects us to something bigger and in...,True
4,train,31ecf93c-3e21-4c80-949b-aa549a046b93,multi,17.0,56.0,85.0,"be ruthless with your ""No’s.""",True
5,train,31ecf93c-3e21-4c80-949b-aa549a046b93,multi,23.0,240.0,306.0,Practice means greatness is doable ... one tin...,True
6,train,31ecf93c-3e21-4c80-949b-aa549a046b93,multi,28.0,65.0,120.0,planning of the SMART goal and number-crunchin...,True
7,train,31ecf93c-3e21-4c80-949b-aa549a046b93,multi,37.0,106.0,163.0,Objectivity — the ability to see the world as ...,True
8,train,31b108a3-c828-421a-a4b9-cf651e9ac859,phrase,5.0,60.0,76.0,in a rice cooker,True
9,train,12e3034e-a98f-4cd2-8773-3f7974161c45,passage,4.0,0.0,110.0,"Apple says that if AirPods are lost or stolen,...",True



Problem spans:


,split,row_number,article_id,spoiler_type,span_number,position,status,paragraph_index,answer_start,answer_end,extracted_text,exact_gold_match,partial_gold_match
26,train,18,5127c4b6-a988-448b-a1b1-b19fbd5a6702,passage,0,"[[8, 709], [9, 815]]",cross_paragraph,8.0,709.0,815.0,None,False,False
32,train,23,172c7ce1-e2cb-4855-8a99-f260820de812,passage,0,"[[-1, 0], [-1, 40]]",negative_paragraph_index,-1.0,0.0,40.0,None,False,False
78,train,60,53266840-6498-4607-9e1d-013db3c512da,passage,0,"[[-1, 38], [-1, 102]]",negative_paragraph_index,-1.0,38.0,102.0,None,False,False
152,train,106,0772c47c-9b37-432e-8c44-0d2b3493cfb5,phrase,0,"[[-1, 47], [-1, 75]]",negative_paragraph_index,-1.0,47.0,75.0,None,False,False
155,train,109,68563e28-ccea-4b85-ad28-77fa6bc4442e,phrase,0,"[[-1, 34], [-1, 45]]",negative_paragraph_index,-1.0,34.0,45.0,None,False,False
169,train,121,e920b3d9-eb55-4968-a125-e9444a23cf7d,multi,2,"[[0, 1], [1, 12]]",cross_paragraph,0.0,1.0,12.0,None,False,False
278,train,203,796a48cb-e0f0-4b86-ad05-ef1e42da7b20,passage,0,"[[-1, 0], [-1, 67]]",negative_paragraph_index,-1.0,0.0,67.0,None,False,False
280,train,205,19db34ff-d0f1-4c8d-b2ed-bea67b302939,phrase,0,"[[-1, 35], [-1, 54]]",negative_paragraph_index,-1.0,35.0,54.0,None,False,False
362,train,267,ad548ee0-107c-4d40-b57f-569b0adcfa58,phrase,0,"[[-1, 42], [-1, 65]]",negative_paragraph_index,-1.0,42.0,65.0,None,False,False
378,train,276,d3061110-3e28-4414-95f5-e43422dce83a,phrase,0,"[[-1, 67], [-1, 85]]",negative_paragraph_index,-1.0,67.0,85.0,None,False,False


In [14]:
## build positive QA examples

from collections import Counter


def clean_text(value):
    """
    Convert list-based or scalar text into one normalized string.

    This is used for the question only. Context text must remain
    unchanged because answer offsets depend on exact characters.
    """
    if value is None:
        return ""

    if isinstance(value, list):
        value = " ".join(
            str(item)
            for item in value
            if item is not None
        )

    return re.sub(r"\s+", " ", str(value)).strip()


def get_article_id(row, dataframe):
    """
    Training uses uuid, while validation and test use id.
    """
    if "uuid" in dataframe.columns:
        return row["uuid"]

    return row["id"]


def build_positive_qa_examples(dataframe, split_name):
    """
    Convert valid spoiler positions into extractive QA examples.

    Source index:
        -1 = targetTitle
         0+ = targetParagraphs[index]

    Cross-context spoilers are skipped because one QA example
    must use one continuous context.
    """
    records = []
    skipped = Counter()

    for row_number, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"Building {split_name} positives"
    ):
        article_id = get_article_id(row, dataframe)
        question = clean_text(row.get("postText", ""))
        spoiler_type = get_spoiler_type(row.get("tags", ""))

        title = str(row.get("targetTitle", "") or "")

        paragraphs = row.get("targetParagraphs", [])
        if not isinstance(paragraphs, list):
            paragraphs = []

        positions = row.get("spoilerPositions", [])
        if not isinstance(positions, list):
            positions = []

        for span_number, span in enumerate(positions):
            if not isinstance(span, list) or len(span) != 2:
                skipped["malformed_span"] += 1
                continue

            start_position, end_position = span

            if (
                not isinstance(start_position, list)
                or not isinstance(end_position, list)
                or len(start_position) != 2
                or len(end_position) != 2
            ):
                skipped["malformed_position"] += 1
                continue

            try:
                start_source = int(start_position[0])
                start_character = int(start_position[1])
                end_source = int(end_position[0])
                end_character = int(end_position[1])
            except (TypeError, ValueError):
                skipped["non_integer_position"] += 1
                continue

            if start_source != end_source:
                skipped["cross_context"] += 1
                continue

            if start_source == -1:
                context = title
                source_kind = "title"
                source_index = -1

            elif 0 <= start_source < len(paragraphs):
                context = str(paragraphs[start_source])
                source_kind = "paragraph"
                source_index = start_source

            else:
                skipped["invalid_source_index"] += 1
                continue

            if (
                start_character < 0
                or end_character <= start_character
                or end_character > len(context)
            ):
                skipped["invalid_character_offsets"] += 1
                continue

            answer_text = context[
                start_character:end_character
            ]

            if not answer_text.strip():
                skipped["empty_answer"] += 1
                continue

            records.append({
                "id": (
                    f"{split_name}_{row_number}"
                    f"_positive_{span_number}"
                ),
                "split": split_name,
                "row_number": row_number,
                "article_id": article_id,
                "spoiler_type": spoiler_type,
                "question": question,
                "context": context,
                "source_kind": source_kind,
                "source_index": source_index,
                "answer_text": answer_text,
                "answer_start": start_character,
                "answers": {
                    "text": [answer_text],
                    "answer_start": [start_character]
                },
                "is_answerable": 1
            })

    return pd.DataFrame(records), skipped


train_positive_qa, train_positive_skipped = (
    build_positive_qa_examples(
        train_df,
        split_name="train"
    )
)

val_positive_qa, val_positive_skipped = (
    build_positive_qa_examples(
        val_df,
        split_name="validation"
    )
)

print("Training positive QA shape:", train_positive_qa.shape)
print("Validation positive QA shape:", val_positive_qa.shape)

print("\nTraining skipped:")
print(train_positive_skipped)

print("\nValidation skipped:")
print(val_positive_skipped)

print("\nTraining positives by source:")
print(train_positive_qa["source_kind"].value_counts())

print("\nValidation positives by source:")
print(val_positive_qa["source_kind"].value_counts())

print("\nTraining positives by spoiler type:")
print(train_positive_qa["spoiler_type"].value_counts())

display(
    train_positive_qa[
        [
            "spoiler_type",
            "source_kind",
            "source_index",
            "question",
            "context",
            "answer_text",
            "answer_start"
        ]
    ].head(10)
)

Building train positives:   0%|          | 0/3200 [00:00<?, ?it/s]

Building validation positives:   0%|          | 0/400 [00:00<?, ?it/s]

Training positive QA shape: (4546, 13)
Validation positive QA shape: (607, 13)

Training skipped:
Counter({'cross_context': 67, 'invalid_character_offsets': 8, 'malformed_span': 1})

Validation skipped:
Counter({'cross_context': 4, 'invalid_character_offsets': 1})

Training positives by source:
source_kind
paragraph    4474
title          72
Name: count, dtype: int64

Validation positives by source:
source_kind
paragraph    596
title         11
Name: count, dtype: int64

Training positives by spoiler type:
spoiler_type
multi      1936
phrase     1366
passage    1244
Name: count, dtype: int64


,spoiler_type,source_kind,source_index,question,context,answer_text,answer_start
0,passage,paragraph,3,"Wes Welker Wanted Dinner With Tom Brady, But P...","""I hit him up to do dinner Saturday night. He’...",how about that morning we go throw?,151
1,phrase,paragraph,0,NASA sets date for full recovery of ozone hole,2070 is shaping up to be a great year for Moth...,2070,0
2,phrase,paragraph,1,This is what makes employees happy -- and it's...,A study by hiring software provider Cangrade r...,intellectual stimulation,186
3,multi,paragraph,11,Passion is overrated — 7 work habits you need ...,Passion makes us bigger. Purpose connects us t...,Purpose connects us to something bigger and in...,25
4,multi,paragraph,17,Passion is overrated — 7 work habits you need ...,Sleep on it. Reach out. The sun will rise tomo...,"be ruthless with your ""No’s.""",56
5,multi,paragraph,23,Passion is overrated — 7 work habits you need ...,Easy: while the quality group held back — labo...,Practice means greatness is doable ... one tin...,240
6,multi,paragraph,28,Passion is overrated — 7 work habits you need ...,"Where passion disconnects us from reality, pla...",planning of the SMART goal and number-crunchin...,65
7,multi,paragraph,37,Passion is overrated — 7 work habits you need ...,Passion makes us myopic. We become so focused ...,Objectivity — the ability to see the world as ...,106
8,phrase,paragraph,5,The perfect way to cook rice so that it's perf...,Several commenters agreed that the best way to...,in a rice cooker,60
9,passage,paragraph,4,What happens if your new AirPods get lost or s...,"Apple says that if AirPods are lost or stolen,...","Apple says that if AirPods are lost or stolen,...",0


In [16]:
## add hard no-answer examples

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


NEGATIVES_PER_ARTICLE = 2


def normalize_for_matching(text):
    """
    Normalize text for checking whether an answer occurs inside
    a candidate negative context.
    """
    text = clean_text(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_article_contexts(row):
    """
    Return the title and every non-empty target paragraph.
    """
    contexts = []

    title = str(row.get("targetTitle", "") or "")

    if title.strip():
        contexts.append({
            "source_kind": "title",
            "source_index": -1,
            "context": title
        })

    paragraphs = row.get("targetParagraphs", [])

    if not isinstance(paragraphs, list):
        paragraphs = []

    for paragraph_index, paragraph in enumerate(paragraphs):
        paragraph = str(paragraph)

        if paragraph.strip():
            contexts.append({
                "source_kind": "paragraph",
                "source_index": paragraph_index,
                "context": paragraph
            })

    return contexts


def context_contains_answer(context, answer_texts):
    """
    Return True when a candidate context contains any known answer.

    This prevents duplicate answer mentions in different paragraphs
    from being mislabeled as no-answer.
    """
    normalized_context = normalize_for_matching(context)

    for answer in answer_texts:
        normalized_answer = normalize_for_matching(answer)

        if not normalized_answer:
            continue

        # Avoid unreliable matching for one-character annotations.
        if len(normalized_answer) < 2:
            continue

        if normalized_answer in normalized_context:
            return True

    return False


def select_hard_negative_contexts(
    question,
    candidate_contexts,
    positive_source_indices,
    known_answer_texts,
    number_to_select=2
):
    """
    Select relevant contexts that:
      1. are not annotated positive sources;
      2. do not contain any known gold answer text.
    """
    available = []

    for candidate in candidate_contexts:
        if candidate["source_index"] in positive_source_indices:
            continue

        if context_contains_answer(
            candidate["context"],
            known_answer_texts
        ):
            continue

        available.append(candidate)

    if not available:
        return []

    context_texts = [
        clean_text(candidate["context"])
        for candidate in available
    ]

    try:
        documents = [clean_text(question)] + context_texts

        vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            stop_words="english"
        )

        matrix = vectorizer.fit_transform(documents)

        similarities = cosine_similarity(
            matrix[0:1],
            matrix[1:]
        ).ravel()

        ranked_positions = np.argsort(-similarities)

        selected = [
            available[position]
            for position in ranked_positions[:number_to_select]
        ]

    except ValueError:
        selected = available[:number_to_select]

    return selected


def build_qa_training_table(
    source_dataframe,
    positive_dataframe,
    negatives_per_article=2
):
    """
    Combine positive QA examples with verified hard negatives.
    """
    qa_records = []

    positives_by_row = {
        row_number: group.copy()
        for row_number, group
        in positive_dataframe.groupby("row_number")
    }

    articles_without_valid_positive = 0
    articles_with_fewer_negatives = 0

    for row_number, row in tqdm(
        source_dataframe.iterrows(),
        total=len(source_dataframe),
        desc="Building corrected QA table"
    ):
        if row_number not in positives_by_row:
            articles_without_valid_positive += 1
            continue

        article_positives = positives_by_row[row_number]

        # Add answerable examples.
        for _, positive in article_positives.iterrows():
            qa_records.append(positive.to_dict())

        positive_source_indices = set(
            article_positives["source_index"].astype(int)
        )

        # Include every annotated answer recovered for this article.
        known_answer_texts = (
            article_positives["answer_text"]
            .dropna()
            .astype(str)
            .tolist()
        )

        # Also include the original gold spoiler strings.
        gold_spoilers = row.get("spoiler", [])

        if not isinstance(gold_spoilers, list):
            gold_spoilers = [gold_spoilers]

        known_answer_texts.extend(
            str(spoiler)
            for spoiler in gold_spoilers
            if spoiler is not None
        )

        candidate_contexts = get_article_contexts(row)

        selected_negatives = select_hard_negative_contexts(
            question=clean_text(row.get("postText", "")),
            candidate_contexts=candidate_contexts,
            positive_source_indices=positive_source_indices,
            known_answer_texts=known_answer_texts,
            number_to_select=negatives_per_article
        )

        if len(selected_negatives) < negatives_per_article:
            articles_with_fewer_negatives += 1

        article_id = get_article_id(
            row,
            source_dataframe
        )

        spoiler_type = get_spoiler_type(
            row.get("tags", "")
        )

        for negative_number, negative in enumerate(
            selected_negatives
        ):
            qa_records.append({
                "id": (
                    f"train_{row_number}"
                    f"_negative_{negative_number}"
                ),
                "split": "train",
                "row_number": row_number,
                "article_id": article_id,
                "spoiler_type": spoiler_type,
                "question": clean_text(
                    row.get("postText", "")
                ),
                "context": negative["context"],
                "source_kind": negative["source_kind"],
                "source_index": negative["source_index"],
                "answer_text": "",
                "answer_start": -1,
                "answers": {
                    "text": [],
                    "answer_start": []
                },
                "is_answerable": 0
            })

    qa_dataframe = pd.DataFrame(qa_records)

    return (
        qa_dataframe,
        articles_without_valid_positive,
        articles_with_fewer_negatives
    )


(
    train_qa_df,
    missing_positive_articles,
    fewer_negative_articles
) = build_qa_training_table(
    source_dataframe=train_df,
    positive_dataframe=train_positive_qa,
    negatives_per_article=NEGATIVES_PER_ARTICLE
)

print("Corrected QA training shape:", train_qa_df.shape)

print("\nAnswerability counts:")
print(train_qa_df["is_answerable"].value_counts())

print("\nAnswerability percentages:")
print(
    train_qa_df["is_answerable"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(
    "\nArticles without a usable positive:",
    missing_positive_articles
)

print(
    "Articles with fewer than two safe negatives:",
    fewer_negative_articles
)

print("\nCorrected hard-negative examples:")
display(
    train_qa_df[
        train_qa_df["is_answerable"] == 0
    ][
        [
            "spoiler_type",
            "source_kind",
            "question",
            "context"
        ]
    ].head(12)
)

Building corrected QA table:   0%|          | 0/3200 [00:00<?, ?it/s]

Corrected QA training shape: (10773, 13)

Answerability counts:
is_answerable
0    6227
1    4546
Name: count, dtype: int64

Answerability percentages:
is_answerable
0    57.8
1    42.2
Name: proportion, dtype: float64

Articles without a usable positive: 40
Articles with fewer than two safe negatives: 84

Corrected hard-negative examples:


,spoiler_type,source_kind,question,context
1,passage,title,"Wes Welker Wanted Dinner With Tom Brady, But P...","Wes Welker Wanted Dinner With Tom Brady, But P..."
2,passage,paragraph,"Wes Welker Wanted Dinner With Tom Brady, But P...",It’ll be just like old times this weekend for ...
4,phrase,paragraph,NASA sets date for full recovery of ozone hole,That's when NASA scientists are predicting the...
5,phrase,paragraph,NASA sets date for full recovery of ozone hole,"Instead, the scientists believe the most recen..."
7,phrase,paragraph,This is what makes employees happy -- and it's...,Researchers developed a three-part formula for...
8,phrase,paragraph,This is what makes employees happy -- and it's...,The study was based on surveys of nearly 600 U...
14,multi,paragraph,Passion is overrated — 7 work habits you need ...,Here are seven habits you need instead.
15,multi,title,Passion is overrated — 7 work habits you need ...,"‘Follow your passion’ is wrong, here are 7 hab..."
17,phrase,title,The perfect way to cook rice so that it's perf...,Revealed: The perfect way to cook rice so that...
18,phrase,paragraph,The perfect way to cook rice so that it's perf...,But now the perfect way to cook the rice has b...


In [18]:
## saving this tables

train_qa_df.to_pickle(
    os.path.join(OUTPUT_DIR, "train_qa_df.pkl")
)

train_positive_qa.to_pickle(
    os.path.join(OUTPUT_DIR, "train_positive_qa.pkl")
)

val_positive_qa.to_pickle(
    os.path.join(OUTPUT_DIR, "val_positive_qa.pkl")
)

print("Corrected QA tables saved to:")
print(OUTPUT_DIR)

Corrected QA tables saved to:
/content/drive/MyDrive/Task2_FinalShot/outputs


In [19]:
## load tokenizer and create the dataset

from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "deepset/roberta-base-squad2"

MAX_LENGTH = 384
DOC_STRIDE = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

train_dataset_raw = Dataset.from_pandas(
    train_qa_df.reset_index(drop=True),
    preserve_index=False
)

print("Model:", MODEL_NAME)
print("Fast tokenizer:", tokenizer.is_fast)
print("Raw training examples:", len(train_dataset_raw))
print("CLS token:", tokenizer.cls_token)
print("CLS token ID:", tokenizer.cls_token_id)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Model: deepset/roberta-base-squad2
Fast tokenizer: True
Raw training examples: 10773
CLS token: <s>
CLS token ID: 0


In [22]:
## tokenize the align answer positions

def prepare_train_features(examples, example_indices):
    """
    Tokenize question-context pairs using sliding windows.

    example_indices preserves the original global dataframe row
    corresponding to every tokenized feature.
    """
    questions = [
        str(question).lstrip()
        for question in examples["question"]
    ]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    offset_mapping = tokenized.pop(
        "offset_mapping"
    )

    start_positions = []
    end_positions = []
    source_example_indices = []

    for feature_index, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][feature_index]

        try:
            cls_index = input_ids.index(
                tokenizer.cls_token_id
            )
        except ValueError:
            cls_index = 0

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        # Index inside the current map batch.
        batch_sample_index = sample_mapping[feature_index]

        # Original global example index.
        global_sample_index = example_indices[
            batch_sample_index
        ]

        answers = examples["answers"][
            batch_sample_index
        ]

        source_example_indices.append(
            global_sample_index
        )

        # True no-answer example.
        if (
            len(answers["answer_start"]) == 0
            or len(answers["text"]) == 0
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        answer_start_character = int(
            answers["answer_start"][0]
        )

        answer_text = str(
            answers["text"][0]
        )

        answer_end_character = (
            answer_start_character + len(answer_text)
        )

        context_start_token = 0

        while (
            context_start_token < len(sequence_ids)
            and sequence_ids[context_start_token] != 1
        ):
            context_start_token += 1

        context_end_token = len(sequence_ids) - 1

        while (
            context_end_token >= 0
            and sequence_ids[context_end_token] != 1
        ):
            context_end_token -= 1

        if (
            context_start_token >= len(sequence_ids)
            or context_end_token < context_start_token
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Answer lies outside this sliding window.
        if (
            offsets[context_start_token][0]
            > answer_start_character
            or offsets[context_end_token][1]
            < answer_end_character
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        token_start_index = context_start_token

        while (
            token_start_index <= context_end_token
            and offsets[token_start_index][1]
            <= answer_start_character
        ):
            token_start_index += 1

        token_end_index = context_end_token

        while (
            token_end_index >= context_start_token
            and offsets[token_end_index][0]
            >= answer_end_character
        ):
            token_end_index -= 1

        if (
            token_start_index > context_end_token
            or token_end_index < context_start_token
            or token_end_index < token_start_index
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_positions.append(token_start_index)
        end_positions.append(token_end_index)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized["source_example_index"] = (
        source_example_indices
    )

    return tokenized


train_tokenized = train_dataset_raw.map(
    prepare_train_features,
    batched=True,
    with_indices=True,
    remove_columns=train_dataset_raw.column_names,
    desc="Tokenizing corrected QA training data"
)

print("Original QA examples:", len(train_dataset_raw))
print("Tokenized sliding-window features:", len(train_tokenized))
print("Tokenized columns:", train_tokenized.column_names)

Tokenizing corrected QA training data:   0%|          | 0/10773 [00:00<?, ? examples/s]

Original QA examples: 10773
Tokenized sliding-window features: 10796
Tokenized columns: ['input_ids', 'attention_mask', 'start_positions', 'end_positions', 'source_example_index']


In [23]:
cls_token_id = tokenizer.cls_token_id

answer_feature_count = 0
cls_feature_count = 0
checked_examples = []
mismatch_examples = []

for feature_index in range(len(train_tokenized)):
    feature = train_tokenized[feature_index]

    start_position = int(feature["start_positions"])
    end_position = int(feature["end_positions"])
    input_ids = feature["input_ids"]

    cls_index = (
        input_ids.index(cls_token_id)
        if cls_token_id in input_ids
        else 0
    )

    if (
        start_position == cls_index
        and end_position == cls_index
    ):
        cls_feature_count += 1
        continue

    answer_feature_count += 1

    tokenized_answer = tokenizer.decode(
        input_ids[start_position:end_position + 1],
        skip_special_tokens=True
    ).strip()

    source_index = int(
        feature["source_example_index"]
    )

    expected_answer = str(
        train_qa_df.iloc[source_index]["answer_text"]
    ).strip()

    tokenized_normalized = normalize_for_matching(
        tokenized_answer
    )

    expected_normalized = normalize_for_matching(
        expected_answer
    )

    is_match = (
        tokenized_normalized == expected_normalized
        or tokenized_normalized in expected_normalized
        or expected_normalized in tokenized_normalized
    )

    record = {
        "feature_index": feature_index,
        "source_example_index": source_index,
        "expected": expected_answer,
        "tokenized_answer": tokenized_answer,
        "match": is_match
    }

    if len(checked_examples) < 15:
        checked_examples.append(record)

    if not is_match:
        mismatch_examples.append(record)


print("Answer-containing features:", answer_feature_count)
print("CLS/no-answer features:", cls_feature_count)
print("Total features:", len(train_tokenized))
print("Alignment mismatches:", len(mismatch_examples))

print("\nSample recovered answers:")
display(pd.DataFrame(checked_examples))

print("\nFirst alignment mismatches:")
display(pd.DataFrame(mismatch_examples[:20]))

Answer-containing features: 4543
CLS/no-answer features: 6253
Total features: 10796
Alignment mismatches: 0

Sample recovered answers:


,feature_index,source_example_index,expected,tokenized_answer,match
0,0,0,how about that morning we go throw?,how about that morning we go throw?,True
1,3,3,2070,2070,True
2,6,6,intellectual stimulation,intellectual stimulation,True
3,9,9,Purpose connects us to something bigger and in...,Purpose connects us to something bigger and in...,True
4,10,10,"be ruthless with your ""No’s.""","be ruthless with your ""No’s.""",True
5,11,11,Practice means greatness is doable ... one tin...,Practice means greatness is doable ... one tin...,True
6,12,12,planning of the SMART goal and number-crunchin...,planning of the SMART goal and number-crunchin...,True
7,13,13,Objectivity — the ability to see the world as ...,Objectivity — the ability to see the world as ...,True
8,16,16,in a rice cooker,in a rice cooker,True
9,19,19,"Apple says that if AirPods are lost or stolen,...","Apple says that if AirPods are lost or stolen,...",True



First alignment mismatches:


""


## find and remove the three unusable positives

In [24]:
positive_source_indices = set(
    train_qa_df.index[
        train_qa_df["is_answerable"] == 1
    ].tolist()
)

answer_feature_source_indices = set()

for feature in train_tokenized:
    input_ids = feature["input_ids"]

    cls_index = (
        input_ids.index(tokenizer.cls_token_id)
        if tokenizer.cls_token_id in input_ids
        else 0
    )

    start_position = int(feature["start_positions"])
    end_position = int(feature["end_positions"])

    if not (
        start_position == cls_index
        and end_position == cls_index
    ):
        answer_feature_source_indices.add(
            int(feature["source_example_index"])
        )

missing_positive_indices = sorted(
    positive_source_indices
    - answer_feature_source_indices
)

print(
    "Positive examples without an answer-containing window:",
    len(missing_positive_indices)
)

display(
    train_qa_df.iloc[missing_positive_indices][
        [
            "article_id",
            "spoiler_type",
            "source_kind",
            "question",
            "answer_text",
            "context"
        ]
    ]
)

Positive examples without an answer-containing window: 3


,article_id,spoiler_type,source_kind,question,answer_text,context
3567,49b507af-9153-451f-8ec2-f73ee6f7861b,phrase,paragraph,An internet troll is helping to pick your next...,Charles Chuck Johnso,Charles Chuck Johnson speaks to YouTube follo...
4333,51c3ba24-f61b-4d9c-a048-eb1ed5843770,multi,paragraph,I'm a Teacher Who Loves Quizzing: But Where Sh...,"but if you care about the forgetting rate, an...","but if you care about the forgetting rate, an..."
8856,4287926f-5fa2-46d7-a831-d2dfe1f5c529,multi,paragraph,7 Surprising Reasons Your Dog Should Sleep On ...,#1: They Give You Comfor,#1: They Give You Comfort


In [25]:
for source_index in missing_positive_indices:
    row = train_qa_df.iloc[source_index]

    context = str(row["context"])
    answer_text = str(row["answer_text"])
    answer_start = int(row["answer_start"])
    answer_end = answer_start + len(answer_text)

    related_features = [
        feature
        for feature in train_tokenized
        if int(feature["source_example_index"]) == source_index
    ]

    print("=" * 100)
    print("SOURCE INDEX:", source_index)
    print("TYPE:", row["spoiler_type"])
    print("ANSWER:", repr(answer_text))
    print("ANSWER START:", answer_start)
    print("ANSWER END:", answer_end)
    print("CONTEXT LENGTH (characters):", len(context))
    print(
        "CONTEXT SLICE:",
        repr(context[answer_start:answer_end])
    )
    print(
        "ANSWER AT END OF CONTEXT:",
        answer_end == len(context)
    )
    print(
        "ANSWER TOKEN COUNT:",
        len(
            tokenizer(
                answer_text,
                add_special_tokens=False
            )["input_ids"]
        )
    )
    print(
        "CONTEXT TOKEN COUNT:",
        len(
            tokenizer(
                context,
                add_special_tokens=False
            )["input_ids"]
        )
    )
    print("NUMBER OF TOKENIZED FEATURES:", len(related_features))

    for feature_number, feature in enumerate(related_features):
        input_ids = feature["input_ids"]

        cls_index = (
            input_ids.index(tokenizer.cls_token_id)
            if tokenizer.cls_token_id in input_ids
            else 0
        )

        print(
            f"Feature {feature_number}:",
            "start =", feature["start_positions"],
            "| end =", feature["end_positions"],
            "| assigned CLS =",
            (
                int(feature["start_positions"]) == cls_index
                and int(feature["end_positions"]) == cls_index
            )
        )

SOURCE INDEX: 3567
TYPE: phrase
ANSWER: ' Charles Chuck Johnso'
ANSWER START: 0
ANSWER END: 21
CONTEXT LENGTH (characters): 286
CONTEXT SLICE: ' Charles Chuck Johnso'
ANSWER AT END OF CONTEXT: False
ANSWER TOKEN COUNT: 4
CONTEXT TOKEN COUNT: 53
NUMBER OF TOKENIZED FEATURES: 1
Feature 0: start = 0 | end = 0 | assigned CLS = True
SOURCE INDEX: 4333
TYPE: multi
ANSWER: ' but if you care about the forgetting rate, and maybe want something that’s easier for you to implement, then just stick some questions at the end of the lectur'
ANSWER START: 0
ANSWER END: 160
CONTEXT LENGTH (characters): 306
CONTEXT SLICE: ' but if you care about the forgetting rate, and maybe want something that’s easier for you to implement, then just stick some questions at the end of the lectur'
ANSWER AT END OF CONTEXT: False
ANSWER TOKEN COUNT: 35
CONTEXT TOKEN COUNT: 68
NUMBER OF TOKENIZED FEATURES: 1
Feature 0: start = 0 | end = 0 | assigned CLS = True
SOURCE INDEX: 8856
TYPE: multi
ANSWER: ' #1: They Give You Co

### all three annotated answers begin with a leading space at character 0. RoBERTa’s offset mapping begins at the first visible character, so the current alignment logic treats them as outside the context.

In [26]:
def trim_answer_boundaries(dataframe, table_name):
    """
    Remove leading/trailing whitespace from positive answer spans
    and shift answer_start by the number of removed leading spaces.
    """
    dataframe = dataframe.copy()

    changed_count = 0
    failed_repairs = []

    if "is_answerable" in dataframe.columns:
        positive_indices = dataframe.index[
            dataframe["is_answerable"] == 1
        ]
    else:
        positive_indices = dataframe.index

    for index in positive_indices:
        original_answer = str(
            dataframe.at[index, "answer_text"]
        )

        original_start = int(
            dataframe.at[index, "answer_start"]
        )

        context = str(
            dataframe.at[index, "context"]
        )

        leading_whitespace = (
            len(original_answer)
            - len(original_answer.lstrip())
        )

        repaired_answer = original_answer.strip()
        repaired_start = (
            original_start + leading_whitespace
        )

        if not repaired_answer:
            failed_repairs.append({
                "index": index,
                "reason": "empty_after_trimming"
            })
            continue

        recovered_text = context[
            repaired_start:
            repaired_start + len(repaired_answer)
        ]

        if recovered_text != repaired_answer:
            failed_repairs.append({
                "index": index,
                "expected": repaired_answer,
                "recovered": recovered_text
            })
            continue

        if (
            repaired_answer != original_answer
            or repaired_start != original_start
        ):
            changed_count += 1

            dataframe.at[
                index,
                "answer_text"
            ] = repaired_answer

            dataframe.at[
                index,
                "answer_start"
            ] = repaired_start

            dataframe.at[
                index,
                "answers"
            ] = {
                "text": [repaired_answer],
                "answer_start": [repaired_start]
            }

    print(f"{table_name} repaired rows:", changed_count)
    print(
        f"{table_name} failed repairs:",
        len(failed_repairs)
    )

    return dataframe, failed_repairs


train_qa_df, train_repair_failures = (
    trim_answer_boundaries(
        train_qa_df,
        "train_qa_df"
    )
)

train_positive_qa, train_positive_failures = (
    trim_answer_boundaries(
        train_positive_qa,
        "train_positive_qa"
    )
)

val_positive_qa, val_positive_failures = (
    trim_answer_boundaries(
        val_positive_qa,
        "val_positive_qa"
    )
)

print("\nPreviously missing examples after repair:")

display(
    train_qa_df.iloc[missing_positive_indices][
        [
            "answer_text",
            "answer_start",
            "context"
        ]
    ]
)

train_qa_df repaired rows: 9
train_qa_df failed repairs: 0
train_positive_qa repaired rows: 9
train_positive_qa failed repairs: 0
val_positive_qa repaired rows: 1
val_positive_qa failed repairs: 0

Previously missing examples after repair:


,answer_text,answer_start,context
3567,Charles Chuck Johnso,1,Charles Chuck Johnson speaks to YouTube follo...
4333,"but if you care about the forgetting rate, and...",1,"but if you care about the forgetting rate, an..."
8856,#1: They Give You Comfor,1,#1: They Give You Comfort


In [27]:
from datasets import Dataset

# Recreate the raw dataset from the repaired dataframe.
train_dataset_raw = Dataset.from_pandas(
    train_qa_df.reset_index(drop=True),
    preserve_index=False
)

# Retokenize using the corrected answer offsets.
train_tokenized = train_dataset_raw.map(
    prepare_train_features,
    batched=True,
    with_indices=True,
    remove_columns=train_dataset_raw.column_names,
    desc="Retokenizing repaired QA training data"
)

positive_source_indices = set(
    train_qa_df.index[
        train_qa_df["is_answerable"] == 1
    ].tolist()
)

answer_feature_source_indices = set()

for feature in train_tokenized:
    input_ids = feature["input_ids"]

    cls_index = (
        input_ids.index(tokenizer.cls_token_id)
        if tokenizer.cls_token_id in input_ids
        else 0
    )

    start_position = int(feature["start_positions"])
    end_position = int(feature["end_positions"])

    if not (
        start_position == cls_index
        and end_position == cls_index
    ):
        answer_feature_source_indices.add(
            int(feature["source_example_index"])
        )

missing_positive_indices = sorted(
    positive_source_indices
    - answer_feature_source_indices
)

print("Tokenized features:", len(train_tokenized))
print(
    "Positive examples with an answer-containing feature:",
    len(answer_feature_source_indices)
)
print(
    "Positive examples still missing:",
    len(missing_positive_indices)
)
print("Missing indices:", missing_positive_indices)

Retokenizing repaired QA training data:   0%|          | 0/10773 [00:00<?, ? examples/s]

Tokenized features: 10796
Positive examples with an answer-containing feature: 4546
Positive examples still missing: 0
Missing indices: []


In [28]:
REPAIRED_TOKENIZED_DIR = os.path.join(
    OUTPUT_DIR,
    "train_tokenized_repaired"
)

train_qa_df.to_pickle(
    os.path.join(OUTPUT_DIR, "train_qa_df_repaired.pkl")
)

train_positive_qa.to_pickle(
    os.path.join(OUTPUT_DIR, "train_positive_qa_repaired.pkl")
)

val_positive_qa.to_pickle(
    os.path.join(OUTPUT_DIR, "val_positive_qa_repaired.pkl")
)

train_tokenized.save_to_disk(
    REPAIRED_TOKENIZED_DIR
)

print("Repaired training table saved:", os.path.exists(
    os.path.join(OUTPUT_DIR, "train_qa_df_repaired.pkl")
))

print("Tokenized dataset saved:", os.path.exists(
    REPAIRED_TOKENIZED_DIR
))

print("Saved tokenized features:", len(train_tokenized))
print("Location:", REPAIRED_TOKENIZED_DIR)

Saving the dataset (0/1 shards):   0%|          | 0/10796 [00:00<?, ? examples/s]

Repaired training table saved: True
Tokenized dataset saved: True
Saved tokenized features: 10796
Location: /content/drive/MyDrive/Task2_FinalShot/outputs/train_tokenized_repaired


## Load the QA model

In [29]:
from transformers import AutoModelForQuestionAnswering

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME
)

model.config.use_cache = False

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model loaded:", MODEL_NAME)
print("Trainable parameters:", f"{trainable_parameters:,}")

CUDA available: True
GPU: Tesla T4


model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: deepset/roberta-base-squad2
Trainable parameters: 124,056,578


## configure training

In [31]:
import inspect
import math
import os
import shutil

from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator
)

# Remove the diagnostic-only column.
train_tokenized_model = train_tokenized.remove_columns(
    ["source_example_index"]
)

LOCAL_TRAINING_DIR = "/content/qa_training_output"

# Replace the unsupported overwrite_output_dir behavior.
if os.path.exists(LOCAL_TRAINING_DIR):
    shutil.rmtree(LOCAL_TRAINING_DIR)

os.makedirs(LOCAL_TRAINING_DIR, exist_ok=True)

requested_training_kwargs = {
    "output_dir": LOCAL_TRAINING_DIR,
    "num_train_epochs": 1,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,

    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,

    "fp16": True,
    "bf16": False,

    "logging_strategy": "steps",
    "logging_steps": 50,

    "save_strategy": "no",
    "report_to": "none",

    "dataloader_num_workers": 2,
    "remove_unused_columns": True,
    "seed": SEED
}

training_signature = inspect.signature(
    TrainingArguments.__init__
)

supported_parameters = set(
    training_signature.parameters.keys()
)

# Add the correct no-evaluation argument for this installed version.
if "eval_strategy" in supported_parameters:
    requested_training_kwargs["eval_strategy"] = "no"
elif "evaluation_strategy" in supported_parameters:
    requested_training_kwargs["evaluation_strategy"] = "no"

training_kwargs = {
    name: value
    for name, value in requested_training_kwargs.items()
    if name in supported_parameters
}

removed_arguments = sorted(
    set(requested_training_kwargs) - set(training_kwargs)
)

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tokenized_model,
    "data_collator": default_data_collator
}

trainer_signature = inspect.signature(Trainer.__init__)

if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

batch_size = training_args.per_device_train_batch_size
accumulation = training_args.gradient_accumulation_steps

batches_per_epoch = math.ceil(
    len(train_tokenized_model) / batch_size
)

optimizer_steps = math.ceil(
    batches_per_epoch / accumulation
)

print("Unsupported arguments removed:", removed_arguments)
print("Training features:", len(train_tokenized_model))
print("Training columns:", train_tokenized_model.column_names)
print("Epochs:", training_args.num_train_epochs)
print("Per-device batch size:", batch_size)
print("Gradient accumulation:", accumulation)
print("Effective batch size:", batch_size * accumulation)
print("Estimated optimizer steps:", optimizer_steps)
print("FP16 enabled:", training_args.fp16)
print("Trainer created successfully:", trainer is not None)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsupported arguments removed: []
Training features: 10796
Training columns: ['input_ids', 'attention_mask', 'start_positions', 'end_positions']
Epochs: 1
Per-device batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Estimated optimizer steps: 675
FP16 enabled: True
Trainer created successfully: True


In [32]:
import time
import torch

torch.cuda.empty_cache()

start_time = time.time()

train_result = trainer.train()

elapsed_minutes = (time.time() - start_time) / 60

print("\nTraining completed.")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")
print("Training metrics:")

for metric_name, metric_value in train_result.metrics.items():
    print(f"{metric_name}: {metric_value}")

Step,Training Loss
50,1.472678
100,1.019363
150,1.063111
200,1.013334
250,1.037029
300,0.915189
350,0.964908
400,0.960444
450,1.034085
500,1.022225



Training completed.
Elapsed time: 3.76 minutes
Training metrics:
train_runtime: 224.8349
train_samples_per_second: 48.017
train_steps_per_second: 3.002
total_flos: 2115719839291392.0
train_loss: 1.0237477987783927
epoch: 1.0


In [33]:
import json
import os

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

metrics_path = os.path.join(
    OUTPUT_DIR,
    "training_metrics.json"
)

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            key: float(value)
            for key, value in train_result.metrics.items()
        },
        file,
        indent=2
    )

print("Model directory exists:", os.path.exists(MODEL_DIR))
print(
    "Model weights saved:",
    os.path.exists(os.path.join(MODEL_DIR, "model.safetensors"))
    or os.path.exists(os.path.join(MODEL_DIR, "pytorch_model.bin"))
)
print(
    "Tokenizer saved:",
    os.path.exists(os.path.join(MODEL_DIR, "tokenizer_config.json"))
)
print("Metrics saved:", os.path.exists(metrics_path))
print("Model location:", MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model directory exists: True
Model weights saved: True
Tokenizer saved: True
Metrics saved: True
Model location: /content/drive/MyDrive/Task2_FinalShot/paragraph_qa_model


In [34]:
from collections import defaultdict

gold_sources_by_row = {
    int(row_number): set(group["source_index"].astype(int))
    for row_number, group in val_positive_qa.groupby("row_number")
}

retrieval_records = []

for row_number, row in tqdm(
    val_df.iterrows(),
    total=len(val_df),
    desc="Evaluating validation retrieval"
):
    question = clean_text(row.get("postText", ""))
    candidate_contexts = get_article_contexts(row)

    if not candidate_contexts:
        continue

    context_texts = [
        clean_text(candidate["context"])
        for candidate in candidate_contexts
    ]

    try:
        vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            stop_words="english"
        )

        matrix = vectorizer.fit_transform(
            [question] + context_texts
        )

        similarities = cosine_similarity(
            matrix[0:1],
            matrix[1:]
        ).ravel()

    except ValueError:
        similarities = np.zeros(
            len(candidate_contexts),
            dtype=float
        )

    ranked_positions = np.argsort(-similarities)

    ranked_candidates = [
        {
            **candidate_contexts[position],
            "retrieval_score": float(similarities[position]),
            "retrieval_rank": rank + 1
        }
        for rank, position in enumerate(ranked_positions)
    ]

    gold_sources = gold_sources_by_row.get(
        row_number,
        set()
    )

    for candidate in ranked_candidates:
        retrieval_records.append({
            "row_number": row_number,
            "article_id": row["id"],
            "question": question,
            "gold_sources": sorted(gold_sources),
            **candidate
        })

val_retrieval_df = pd.DataFrame(retrieval_records)

rows_with_gold = sorted(gold_sources_by_row.keys())

print("Validation articles:", len(val_df))
print(
    "Articles with at least one usable gold source:",
    len(rows_with_gold)
)
print(
    "Articles without a usable gold source:",
    len(val_df) - len(rows_with_gold)
)

for top_k in [1, 3, 5]:
    any_hits = 0
    all_hits = 0

    for row_number in rows_with_gold:
        gold_sources = gold_sources_by_row[row_number]

        retrieved_sources = set(
            val_retrieval_df[
                (val_retrieval_df["row_number"] == row_number)
                & (val_retrieval_df["retrieval_rank"] <= top_k)
            ]["source_index"].astype(int)
        )

        if gold_sources & retrieved_sources:
            any_hits += 1

        if gold_sources.issubset(retrieved_sources):
            all_hits += 1

    print(
        f"\nTop-{top_k} any-gold coverage:",
        f"{any_hits}/{len(rows_with_gold)}",
        f"({100 * any_hits / len(rows_with_gold):.2f}%)"
    )

    print(
        f"Top-{top_k} all-gold coverage:",
        f"{all_hits}/{len(rows_with_gold)}",
        f"({100 * all_hits / len(rows_with_gold):.2f}%)"
    )

print(
    "\nAverage candidate contexts per article:",
    round(
        val_retrieval_df.groupby("row_number").size().mean(),
        2
    )
)

Evaluating validation retrieval:   0%|          | 0/400 [00:00<?, ?it/s]

Validation articles: 400
Articles with at least one usable gold source: 397
Articles without a usable gold source: 3

Top-1 any-gold coverage: 25/397 (6.30%)
Top-1 all-gold coverage: 23/397 (5.79%)

Top-3 any-gold coverage: 187/397 (47.10%)
Top-3 all-gold coverage: 166/397 (41.81%)

Top-5 any-gold coverage: 280/397 (70.53%)
Top-5 all-gold coverage: 244/397 (61.46%)

Average candidate contexts per article: 15.8


In [35]:
validation_candidate_records = []

for row_number, row in tqdm(
    val_df.iterrows(),
    total=len(val_df),
    desc="Building validation QA candidates"
):
    question = clean_text(row.get("postText", ""))
    candidate_contexts = get_article_contexts(row)

    gold_sources = gold_sources_by_row.get(
        row_number,
        set()
    )

    for candidate_number, candidate in enumerate(
        candidate_contexts
    ):
        validation_candidate_records.append({
            "candidate_id": (
                f"validation_{row_number}"
                f"_candidate_{candidate_number}"
            ),
            "row_number": row_number,
            "article_id": row["id"],
            "question": question,
            "context": candidate["context"],
            "source_kind": candidate["source_kind"],
            "source_index": int(
                candidate["source_index"]
            ),
            "is_gold_source": (
                int(candidate["source_index"])
                in gold_sources
            )
        })

val_candidate_df = pd.DataFrame(
    validation_candidate_records
)

print(
    "Total validation candidate contexts:",
    len(val_candidate_df)
)

print(
    "Average candidates per article:",
    round(
        val_candidate_df.groupby(
            "row_number"
        ).size().mean(),
        2
    )
)

print(
    "Gold-source candidate contexts:",
    int(val_candidate_df["is_gold_source"].sum())
)

print(
    "Validation articles represented:",
    val_candidate_df["row_number"].nunique()
)

display(
    val_candidate_df[
        [
            "row_number",
            "source_kind",
            "source_index",
            "is_gold_source",
            "question",
            "context"
        ]
    ].head(10)
)

Building validation QA candidates:   0%|          | 0/400 [00:00<?, ?it/s]

Total validation candidate contexts: 6318
Average candidates per article: 15.8
Gold-source candidate contexts: 585
Validation articles represented: 400


,row_number,source_kind,source_index,is_gold_source,question,context
0,0,title,-1,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Five Nights at Freddy’s Sequel Delayed for Wei...
1,0,paragraph,0,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Five Nights at Freddy’s creator Scott Cawthon ...
2,0,paragraph,1,False,Five Nights at Freddy’s Sequel Delayed for Wei...,"For the past couple of years, horror gaming fa..."
3,0,paragraph,2,True,Five Nights at Freddy’s Sequel Delayed for Wei...,According to a post by Cawthon on the Five Nig...
4,0,paragraph,3,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Delays happen in the gaming industry all the t...
5,0,paragraph,4,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Cawthon’s reason for suddenly delaying Five Ni...
6,0,paragraph,5,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Fans should also consider the possibility that...
7,0,paragraph,6,False,Five Nights at Freddy’s Sequel Delayed for Wei...,"With October 7th just a few days away, fans wi..."
8,0,paragraph,7,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Five Nights at Freddy’s: Sister Location is sc...
9,0,paragraph,8,False,Five Nights at Freddy’s Sequel Delayed for Wei...,Source: Scott Cawthon


In [36]:
from datasets import Dataset

val_candidate_dataset_raw = Dataset.from_pandas(
    val_candidate_df.reset_index(drop=True),
    preserve_index=False
)


def prepare_validation_features(examples, example_indices):
    """
    Tokenize validation question-context pairs with sliding windows.

    Each tokenized feature keeps:
      - its original candidate dataframe index;
      - character offsets for context tokens;
      - [-1, -1] for question and special tokens.
    """
    questions = [
        str(question).lstrip()
        for question in examples["question"]
    ]

    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    candidate_indices = []
    cleaned_offset_mappings = []

    for feature_index, offsets in enumerate(
        tokenized["offset_mapping"]
    ):
        batch_sample_index = sample_mapping[feature_index]

        global_candidate_index = int(
            example_indices[batch_sample_index]
        )

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        context_offsets = []

        for token_index, offset in enumerate(offsets):
            if sequence_ids[token_index] == 1:
                context_offsets.append([
                    int(offset[0]),
                    int(offset[1])
                ])
            else:
                context_offsets.append([-1, -1])

        candidate_indices.append(
            global_candidate_index
        )

        cleaned_offset_mappings.append(
            context_offsets
        )

    tokenized["offset_mapping"] = (
        cleaned_offset_mappings
    )

    tokenized["candidate_index"] = (
        candidate_indices
    )

    return tokenized


val_tokenized = val_candidate_dataset_raw.map(
    prepare_validation_features,
    batched=True,
    with_indices=True,
    remove_columns=val_candidate_dataset_raw.column_names,
    desc="Tokenizing validation QA candidates"
)

candidate_indices_present = set(
    int(value)
    for value in val_tokenized["candidate_index"]
)

features_per_candidate = pd.Series(
    val_tokenized["candidate_index"]
).value_counts()

print(
    "Original candidate contexts:",
    len(val_candidate_dataset_raw)
)

print(
    "Tokenized validation features:",
    len(val_tokenized)
)

print(
    "Candidate contexts represented:",
    len(candidate_indices_present)
)

print(
    "Candidate contexts missing:",
    len(val_candidate_df) - len(candidate_indices_present)
)

print(
    "Candidates requiring multiple windows:",
    int((features_per_candidate > 1).sum())
)

print(
    "Maximum windows for one candidate:",
    int(features_per_candidate.max())
)

print(
    "Validation tokenized columns:",
    val_tokenized.column_names
)

Tokenizing validation QA candidates:   0%|          | 0/6318 [00:00<?, ? examples/s]

Original candidate contexts: 6318
Tokenized validation features: 6323
Candidate contexts represented: 6318
Candidate contexts missing: 0
Candidates requiring multiple windows: 4
Maximum windows for one candidate: 3
Validation tokenized columns: ['input_ids', 'attention_mask', 'offset_mapping', 'candidate_index']


In [37]:
# Keep metadata separately; send only model inputs to Trainer.
val_model_inputs = val_tokenized.remove_columns(
    ["offset_mapping", "candidate_index"]
)

print("Inference features:", len(val_model_inputs))
print("Model input columns:", val_model_inputs.column_names)

prediction_output = trainer.predict(val_model_inputs)

start_logits, end_logits = prediction_output.predictions

print("\nInference completed.")
print("Start-logit shape:", start_logits.shape)
print("End-logit shape:", end_logits.shape)
print("Prediction runtime:", prediction_output.metrics.get("test_runtime"))
print(
    "Features per second:",
    prediction_output.metrics.get("test_samples_per_second")
)

Inference features: 6323
Model input columns: ['input_ids', 'attention_mask']



Inference completed.
Start-logit shape: (6323, 384)
End-logit shape: (6323, 384)
Prediction runtime: 34.9647
Features per second: 180.84


In [39]:
MAX_ANSWER_LENGTH = 100
N_BEST_START_END = 20

candidate_best_predictions = {}

for feature_index in tqdm(
    range(len(val_tokenized)),
    desc="Extracting validation spans"
):
    feature = val_tokenized[feature_index]

    candidate_index = int(
        feature["candidate_index"]
    )

    input_ids = feature["input_ids"]
    offsets = feature["offset_mapping"]

    feature_start_logits = start_logits[feature_index]
    feature_end_logits = end_logits[feature_index]

    cls_index = (
        input_ids.index(tokenizer.cls_token_id)
        if tokenizer.cls_token_id in input_ids
        else 0
    )

    null_score = float(
        feature_start_logits[cls_index]
        + feature_end_logits[cls_index]
    )

    top_start_indices = np.argsort(
        feature_start_logits
    )[-N_BEST_START_END:][::-1]

    top_end_indices = np.argsort(
        feature_end_logits
    )[-N_BEST_START_END:][::-1]

    best_span_score = -float("inf")
    best_start_character = None
    best_end_character = None

    for start_index in top_start_indices:
        start_offset = offsets[start_index]

        if (
            start_offset[0] < 0
            or start_offset[1] <= start_offset[0]
        ):
            continue

        for end_index in top_end_indices:
            end_offset = offsets[end_index]

            if (
                end_offset[0] < 0
                or end_offset[1] <= end_offset[0]
            ):
                continue

            if end_index < start_index:
                continue

            token_length = (
                end_index - start_index + 1
            )

            if token_length > MAX_ANSWER_LENGTH:
                continue

            start_character = int(start_offset[0])
            end_character = int(end_offset[1])

            if end_character <= start_character:
                continue

            span_score = float(
                feature_start_logits[start_index]
                + feature_end_logits[end_index]
            )

            if span_score > best_span_score:
                best_span_score = span_score
                best_start_character = start_character
                best_end_character = end_character

    candidate_row = val_candidate_df.iloc[
        candidate_index
    ]

    context = str(candidate_row["context"])

    if (
        best_start_character is None
        or best_end_character is None
    ):
        predicted_text = ""
        answerability_margin = -float("inf")
    else:
        predicted_text = context[
            best_start_character:best_end_character
        ].strip()

        answerability_margin = (
            best_span_score - null_score
        )

    prediction_record = {
        "candidate_index": candidate_index,
        "row_number": int(candidate_row["row_number"]),
        "article_id": candidate_row["article_id"],
        "source_kind": candidate_row["source_kind"],
        "source_index": int(candidate_row["source_index"]),
        "is_gold_source": bool(
            candidate_row["is_gold_source"]
        ),
        "predicted_text": predicted_text,
        "best_span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": answerability_margin,
        "feature_index": feature_index
    }

    # For contexts split into multiple windows, retain the
    # window with the strongest answerability margin.
    previous_prediction = candidate_best_predictions.get(
        candidate_index
    )

    if (
        previous_prediction is None
        or answerability_margin
        > previous_prediction["answerability_margin"]
    ):
        candidate_best_predictions[
            candidate_index
        ] = prediction_record


val_candidate_predictions_df = pd.DataFrame(
    candidate_best_predictions.values()
).sort_values(
    ["row_number", "answerability_margin"],
    ascending=[True, False]
).reset_index(drop=True)

print(
    "Candidate predictions:",
    len(val_candidate_predictions_df)
)

print(
    "Validation articles represented:",
    val_candidate_predictions_df[
        "row_number"
    ].nunique()
)

print(
    "Empty predicted spans:",
    int(
        (
            val_candidate_predictions_df[
                "predicted_text"
            ].str.len() == 0
        ).sum()
    )
)

print("\nAnswerability-margin summary:")
print(
    val_candidate_predictions_df[
        "answerability_margin"
    ].describe()
)

print("\nHighest-scoring candidate examples:")
display(
    val_candidate_predictions_df[
        [
            "row_number",
            "source_kind",
            "source_index",
            "is_gold_source",
            "predicted_text",
            "answerability_margin"
        ]
    ].head(15)
)

Extracting validation spans:   0%|          | 0/6323 [00:00<?, ?it/s]

Candidate predictions: 6318
Validation articles represented: 400
Empty predicted spans: 1

Answerability-margin summary:
count    6318.000000
mean       -2.367725
std         6.248394
min       -23.601562
25%        -5.440552
50%        -2.094238
75%         1.225098
max        11.750000
Name: answerability_margin, dtype: float64

Highest-scoring candidate examples:


,row_number,source_kind,source_index,is_gold_source,predicted_text,answerability_margin
0,0,paragraph,2,True,it’s too dark,2.076172
1,0,paragraph,3,False,too dark to release,0.619141
2,0,paragraph,0,False,Five Nights at Freddy’s: Sister Location,-1.294922
3,0,paragraph,5,False,Cawthon is just trolling in an attempt to thro...,-3.133789
4,0,paragraph,1,False,"Five Nights at Freddy’s: Sister Location, was ...",-3.168945
5,0,paragraph,4,False,this is just a weird publicity stunt meant to ...,-3.412109
6,0,paragraph,7,False,Five Nights at Freddy’s: Sister Location,-4.541016
7,0,paragraph,6,False,Sister Location,-5.968018
8,0,paragraph,8,False,Source: Scott Cawthon,-9.536133
9,0,title,-1,False,Five Nights at Freddy’s Sequel Delayed for Wei...,-22.457031


In [40]:
model_ranking_results = []

for row_number in sorted(
    val_candidate_predictions_df["row_number"].unique()
):
    article_predictions = (
        val_candidate_predictions_df[
            val_candidate_predictions_df["row_number"] == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    gold_sources = gold_sources_by_row.get(
        row_number,
        set()
    )

    if not gold_sources:
        continue

    ranked_sources = (
        article_predictions["source_index"]
        .astype(int)
        .tolist()
    )

    result = {
        "row_number": row_number,
        "gold_sources": gold_sources,
        "top1_any_gold": bool(
            set(ranked_sources[:1]) & gold_sources
        ),
        "top3_any_gold": bool(
            set(ranked_sources[:3]) & gold_sources
        ),
        "top5_any_gold": bool(
            set(ranked_sources[:5]) & gold_sources
        ),
        "top1_all_gold": gold_sources.issubset(
            set(ranked_sources[:1])
        ),
        "top3_all_gold": gold_sources.issubset(
            set(ranked_sources[:3])
        ),
        "top5_all_gold": gold_sources.issubset(
            set(ranked_sources[:5])
        )
    }

    model_ranking_results.append(result)


model_ranking_df = pd.DataFrame(
    model_ranking_results
)

print(
    "Validation articles evaluated:",
    len(model_ranking_df)
)

for top_k in [1, 3, 5]:
    any_count = int(
        model_ranking_df[
            f"top{top_k}_any_gold"
        ].sum()
    )

    all_count = int(
        model_ranking_df[
            f"top{top_k}_all_gold"
        ].sum()
    )

    print(
        f"\nModel top-{top_k} any-gold coverage:",
        f"{any_count}/{len(model_ranking_df)}",
        f"({100 * any_count / len(model_ranking_df):.2f}%)"
    )

    print(
        f"Model top-{top_k} all-gold coverage:",
        f"{all_count}/{len(model_ranking_df)}",
        f"({100 * all_count / len(model_ranking_df):.2f}%)"
    )


gold_margin_summary = (
    val_candidate_predictions_df[
        val_candidate_predictions_df["is_gold_source"]
    ]["answerability_margin"]
    .describe()
)

nongold_margin_summary = (
    val_candidate_predictions_df[
        ~val_candidate_predictions_df["is_gold_source"]
    ]["answerability_margin"]
    .describe()
)

print("\nGold-source margin summary:")
print(gold_margin_summary)

print("\nNon-gold-source margin summary:")
print(nongold_margin_summary)

Validation articles evaluated: 397

Model top-1 any-gold coverage: 179/397 (45.09%)
Model top-1 all-gold coverage: 139/397 (35.01%)

Model top-3 any-gold coverage: 279/397 (70.28%)
Model top-3 all-gold coverage: 232/397 (58.44%)

Model top-5 any-gold coverage: 325/397 (81.86%)
Model top-5 all-gold coverage: 282/397 (71.03%)

Gold-source margin summary:
count    585.000000
mean       2.695513
std        4.676465
min      -10.617188
25%       -0.636719
50%        2.474609
75%        6.068359
max       11.750000
Name: answerability_margin, dtype: float64

Non-gold-source margin summary:
count    5733.000000
mean       -2.884382
std         6.157557
min       -23.601562
25%        -5.741211
50%        -2.482422
75%         0.671875
max        11.464844
Name: answerability_margin, dtype: float64


In [41]:
import nltk
import re
import pandas as pd

from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


def normalize_prediction_text(text):
    text = str(text or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def join_unique_predictions(article_predictions, maximum_answers):
    """
    Join the highest-ranked unique non-empty predicted spans.
    """
    selected = []
    seen = set()

    for _, candidate in article_predictions.iterrows():
        prediction = normalize_prediction_text(
            candidate["predicted_text"]
        )

        normalized = prediction.lower()

        if not prediction or normalized in seen:
            continue

        selected.append(prediction)
        seen.add(normalized)

        if len(selected) >= maximum_answers:
            break

    return " ".join(selected)


validation_prediction_records = []

for row_number, row in val_df.iterrows():
    article_predictions = (
        val_candidate_predictions_df[
            val_candidate_predictions_df["row_number"] == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    gold_spoilers = row["spoiler"]

    if not isinstance(gold_spoilers, list):
        gold_spoilers = [gold_spoilers]

    gold_text = normalize_prediction_text(
        " ".join(str(value) for value in gold_spoilers)
    )

    # Strategy 1: strongest candidate only.
    prediction_top1 = join_unique_predictions(
        article_predictions,
        maximum_answers=1
    )

    # Strategy 2: strongest two distinct candidates.
    prediction_top2 = join_unique_predictions(
        article_predictions,
        maximum_answers=2
    )

    # Strategy 3: strongest three distinct candidates.
    prediction_top3 = join_unique_predictions(
        article_predictions,
        maximum_answers=3
    )

    # Strategy 4: use up to three candidates with positive margins.
    positive_candidates = article_predictions[
        article_predictions["answerability_margin"] > 0
    ]

    if len(positive_candidates) == 0:
        positive_candidates = article_predictions.head(1)

    prediction_positive3 = join_unique_predictions(
        positive_candidates,
        maximum_answers=3
    )

    # Strategy 5: include candidates close to the best model score.
    if len(article_predictions) > 0:
        best_margin = float(
            article_predictions.iloc[0]["answerability_margin"]
        )

        close_candidates = article_predictions[
            article_predictions["answerability_margin"]
            >= best_margin - 2.0
        ]
    else:
        close_candidates = article_predictions

    prediction_close3 = join_unique_predictions(
        close_candidates,
        maximum_answers=3
    )

    record = {
        "row_number": row_number,
        "article_id": row["id"],
        "spoiler_type": get_spoiler_type(row["tags"]),
        "gold_text": gold_text,
        "top1": prediction_top1,
        "top2": prediction_top2,
        "top3": prediction_top3,
        "positive3": prediction_positive3,
        "close3": prediction_close3
    }

    validation_prediction_records.append(record)


validation_predictions_df = pd.DataFrame(
    validation_prediction_records
)

strategy_names = [
    "top1",
    "top2",
    "top3",
    "positive3",
    "close3"
]

score_records = []

for strategy in strategy_names:
    row_scores = []

    for _, row in validation_predictions_df.iterrows():
        reference_tokens = row["gold_text"].split()
        prediction_tokens = row[strategy].split()

        if len(prediction_tokens) == 0:
            score = 0.0
        else:
            score = meteor_score(
                [reference_tokens],
                prediction_tokens
            )

        row_scores.append(score)

    validation_predictions_df[
        f"{strategy}_meteor"
    ] = row_scores

    score_records.append({
        "strategy": strategy,
        "mean_meteor": float(
            validation_predictions_df[
                f"{strategy}_meteor"
            ].mean()
        ),
        "median_meteor": float(
            validation_predictions_df[
                f"{strategy}_meteor"
            ].median()
        ),
        "average_prediction_words": float(
            validation_predictions_df[
                strategy
            ].str.split().str.len().mean()
        )
    })


strategy_results_df = (
    pd.DataFrame(score_records)
    .sort_values("mean_meteor", ascending=False)
    .reset_index(drop=True)
)

print("Overall validation results:")
display(strategy_results_df)

best_strategy = strategy_results_df.iloc[0]["strategy"]

print("Best strategy:", best_strategy)
print(
    "Best mean METEOR:",
    strategy_results_df.iloc[0]["mean_meteor"]
)

print("\nBest-strategy METEOR by spoiler type:")
display(
    validation_predictions_df.groupby(
        "spoiler_type"
    )[f"{best_strategy}_meteor"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

print("\nSample predictions from the best strategy:")
display(
    validation_predictions_df[
        [
            "spoiler_type",
            "gold_text",
            best_strategy,
            f"{best_strategy}_meteor"
        ]
    ].head(15)
)

Overall validation results:


,strategy,mean_meteor,median_meteor,average_prediction_words
0,close3,0.357278,0.251757,15.9275
1,positive3,0.352005,0.261865,15.1950
2,top3,0.339304,0.294118,26.1950
3,top2,0.334649,0.247363,15.7500
4,top1,0.282456,0.109596,6.9075


Best strategy: close3
Best mean METEOR: 0.35727848080608454

Best-strategy METEOR by spoiler type:


,count,mean,median
spoiler_type,,,
phrase,162,0.467707,0.477273
multi,84,0.313145,0.265178
passage,154,0.265186,0.127213



Sample predictions from the best strategy:


,spoiler_type,gold_text,close3,close3_meteor
0,passage,some of the plot elements are so disturbing th...,it’s too dark too dark to release,0.000000
1,multi,"""intentionally"" could transform a court case a...","""The Defendants’ unfair, partial, and inequita...",0.051813
2,phrase,20%,$3 to $5 between $5 and $20 20%,0.294118
3,multi,Alan Rickman & Rupert Grint CBGB,candy corn CBGB The Dead Boys,0.083333
4,passage,a man who swallowed a 64GB microSD card and th...,"he couldn't puke it back up, and therefore had...",0.416667
5,phrase,Sprite,Sprite,0.500000
6,phrase,Smoky Paprika-Baked Garbanzo Beans,Smoky Paprika-Baked Garbanzo Beans 1 Tbsp. smo...,0.826823
7,passage,McGonagall was appointed as Dumbledore’s assis...,It’s made of Fir with a core of Dragon Heartst...,0.000000
8,passage,All the scenes are actually in the movie,"No, there’s not. All the scenes are actually i...",0.842144
9,passage,"""I had fake relationships, fake fights. I don'...","""I had fake relationships, fake fights ""What [...",0.330836


In [43]:
type_strategy_records = []

for spoiler_type in ["phrase", "passage", "multi"]:
    type_rows = validation_predictions_df[
        validation_predictions_df["spoiler_type"] == spoiler_type
    ]

    for strategy in strategy_names:
        meteor_column = f"{strategy}_meteor"

        type_strategy_records.append({
            "spoiler_type": spoiler_type,
            "strategy": strategy,
            "count": len(type_rows),
            "mean_meteor": float(
                type_rows[meteor_column].mean()
            ),
            "median_meteor": float(
                type_rows[meteor_column].median()
            ),
            "average_words": float(
                type_rows[strategy]
                .str.split()
                .str.len()
                .mean()
            )
        })

type_strategy_df = pd.DataFrame(
    type_strategy_records
)

print("Strategy results by spoiler type:")

display(
    type_strategy_df.sort_values(
        ["spoiler_type", "mean_meteor"],
        ascending=[True, False]
    )
)

best_strategy_by_type = (
    type_strategy_df
    .sort_values(
        ["spoiler_type", "mean_meteor"],
        ascending=[True, False]
    )
    .groupby("spoiler_type")
    .first()
    .reset_index()
)

print("\nBest existing strategy for each type:")
display(best_strategy_by_type)

best_strategy_lookup = dict(
    zip(
        best_strategy_by_type["spoiler_type"],
        best_strategy_by_type["strategy"]
    )
)

validation_predictions_df[
    "type_specific_prediction"
] = validation_predictions_df.apply(
    lambda row: row[
        best_strategy_lookup[row["spoiler_type"]]
    ],
    axis=1
)

type_specific_scores = []

for _, row in validation_predictions_df.iterrows():
    reference_tokens = row["gold_text"].split()
    prediction_tokens = row[
        "type_specific_prediction"
    ].split()

    if not prediction_tokens:
        score = 0.0
    else:
        score = meteor_score(
            [reference_tokens],
            prediction_tokens
        )

    type_specific_scores.append(score)

validation_predictions_df[
    "type_specific_meteor"
] = type_specific_scores

print(
    "\nType-specific overall METEOR:",
    validation_predictions_df[
        "type_specific_meteor"
    ].mean()
)

print(
    "Type-specific median METEOR:",
    validation_predictions_df[
        "type_specific_meteor"
    ].median()
)

Strategy results by spoiler type:


,spoiler_type,strategy,count,mean_meteor,median_meteor,average_words
12,multi,top3,84,0.347759,0.289025,23.630952
13,multi,positive3,84,0.337944,0.299588,17.797619
14,multi,close3,84,0.313145,0.265178,15.404762
11,multi,top2,84,0.262740,0.195091,13.880952
10,multi,top1,84,0.175698,0.146334,6.285714
7,passage,top3,154,0.314600,0.174336,37.603896
6,passage,top2,154,0.275997,0.124754,24.123377
8,passage,positive3,154,0.267734,0.116162,22.123377
9,passage,close3,154,0.265186,0.127213,25.987013
5,passage,top1,154,0.200922,0.041500,10.642857



Best existing strategy for each type:


,spoiler_type,strategy,count,mean_meteor,median_meteor,average_words
0,multi,top3,84,0.347759,0.289025,23.630952
1,passage,top3,154,0.314600,0.174336,37.603896
2,phrase,close3,162,0.467707,0.477273,6.635802



Type-specific overall METEOR: 0.38357213456645484
Type-specific median METEOR: 0.30845360141759093


## The type-specific strategy improved validation METEOR to 0.3836, but passage generation remains the main weakness. The QA model often finds the right part of the paragraph but returns only a short fragment.Let’s test whether expanding each predicted passage span to its surrounding sentence improves the passage score.

In [44]:
def sentence_spans(text):
    """
    Split text into approximate sentence spans while preserving
    character positions in the original context.
    """
    text = str(text)

    matches = list(
        re.finditer(
            r'[^.!?]+(?:[.!?]+["”’\']*|$)',
            text
        )
    )

    spans = []

    for match in matches:
        start = match.start()
        end = match.end()

        sentence = text[start:end].strip()

        if sentence:
            spans.append({
                "start": start,
                "end": end,
                "text": sentence
            })

    if not spans and text.strip():
        spans.append({
            "start": 0,
            "end": len(text),
            "text": text.strip()
        })

    return spans


def expand_prediction_to_sentence(context, predicted_text):
    """
    Find the predicted QA span inside its source context and return
    the complete sentence containing that span.
    """
    context = str(context)
    predicted_text = normalize_prediction_text(predicted_text)

    if not predicted_text:
        return ""

    start_position = context.find(predicted_text)

    # Case-insensitive fallback.
    if start_position < 0:
        start_position = context.lower().find(
            predicted_text.lower()
        )

    if start_position < 0:
        return predicted_text

    end_position = start_position + len(predicted_text)

    for sentence in sentence_spans(context):
        if (
            sentence["start"] <= start_position
            and sentence["end"] >= end_position
        ):
            return sentence["text"]

    return predicted_text


# Add the original source context to each candidate prediction.
passage_candidate_predictions = (
    val_candidate_predictions_df
    .merge(
        val_candidate_df[
            ["context"]
        ],
        left_on="candidate_index",
        right_index=True,
        how="left"
    )
)

passage_strategy_records = []

for row_number, validation_row in val_df.iterrows():
    spoiler_type = get_spoiler_type(
        validation_row["tags"]
    )

    if spoiler_type != "passage":
        continue

    article_predictions = (
        passage_candidate_predictions[
            passage_candidate_predictions["row_number"]
            == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    gold_values = validation_row["spoiler"]

    if not isinstance(gold_values, list):
        gold_values = [gold_values]

    gold_text = normalize_prediction_text(
        " ".join(str(value) for value in gold_values)
    )

    expanded_sentences = []
    seen_sentences = set()

    for _, candidate in article_predictions.iterrows():
        sentence = expand_prediction_to_sentence(
            candidate["context"],
            candidate["predicted_text"]
        )

        normalized_sentence = sentence.lower().strip()

        if (
            sentence
            and normalized_sentence not in seen_sentences
        ):
            expanded_sentences.append(sentence)
            seen_sentences.add(normalized_sentence)

    passage_strategy_records.append({
        "row_number": row_number,
        "gold_text": gold_text,
        "qa_top3_spans": join_unique_predictions(
            article_predictions,
            maximum_answers=3
        ),
        "sentence_top1": " ".join(
            expanded_sentences[:1]
        ),
        "sentence_top2": " ".join(
            expanded_sentences[:2]
        ),
        "sentence_top3": " ".join(
            expanded_sentences[:3]
        ),
        "full_context_top1": normalize_prediction_text(
            article_predictions.iloc[0]["context"]
        ) if len(article_predictions) else ""
    })


passage_expansion_df = pd.DataFrame(
    passage_strategy_records
)

passage_strategies = [
    "qa_top3_spans",
    "sentence_top1",
    "sentence_top2",
    "sentence_top3",
    "full_context_top1"
]

passage_results = []

for strategy in passage_strategies:
    scores = []

    for _, row in passage_expansion_df.iterrows():
        reference_tokens = row["gold_text"].split()
        prediction_tokens = row[strategy].split()

        score = (
            meteor_score(
                [reference_tokens],
                prediction_tokens
            )
            if prediction_tokens
            else 0.0
        )

        scores.append(score)

    passage_expansion_df[
        f"{strategy}_meteor"
    ] = scores

    passage_results.append({
        "strategy": strategy,
        "mean_meteor": float(np.mean(scores)),
        "median_meteor": float(np.median(scores)),
        "average_words": float(
            passage_expansion_df[
                strategy
            ].str.split().str.len().mean()
        )
    })


passage_results_df = (
    pd.DataFrame(passage_results)
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Passage expansion results:")
display(passage_results_df)

best_passage_strategy = (
    passage_results_df.iloc[0]["strategy"]
)

print("Best passage strategy:", best_passage_strategy)
print(
    "Best passage mean METEOR:",
    passage_results_df.iloc[0]["mean_meteor"]
)

display(
    passage_expansion_df[
        [
            "gold_text",
            best_passage_strategy,
            f"{best_passage_strategy}_meteor"
        ]
    ].head(12)
)

Passage expansion results:


,strategy,mean_meteor,median_meteor,average_words
0,sentence_top3,0.372731,0.291411,62.720779
1,sentence_top2,0.348422,0.182325,41.376623
2,qa_top3_spans,0.314600,0.174336,37.603896
3,sentence_top1,0.301342,0.090430,19.980519
4,full_context_top1,0.295870,0.110804,42.974026


Best passage strategy: sentence_top3
Best passage mean METEOR: 0.37273054572397124


,gold_text,sentence_top3,sentence_top3_meteor
0,some of the plot elements are so disturbing th...,According to a post by Cawthon on the Five Nig...,0.159091
1,a man who swallowed a 64GB microSD card and th...,What’s important is that he couldn't puke it b...,0.635499
2,McGonagall was appointed as Dumbledore’s assis...,It’s made of Fir with a core of Dragon Heartst...,0.155827
3,All the scenes are actually in the movie,"No, there’s not. All the scenes are actually i...",0.471372
4,"""I had fake relationships, fake fights. I don'...","""I had fake relationships, fake fights. ""What ...",0.380130
5,he'd eaten a peanut butter sandwich and wasn't...,"Myriam Ducre-Lemay, 20, died in 2012 after kis...",0.510009
6,kicked her and got into a fight with her curre...,He was escorted out of the hospital and arrest...,0.420833
7,Dunham picked the boy up and took him to a Sub...,Dunham picked the boy up and took him to a Sub...,0.671267
8,"Not only does Aubrey have cerebral palsy, but ...","One day, Lisa’s friend and an orphanage volunt...",0.023697
9,"The bottom line: Unfortunately, there's not en...","The bottom line: Unfortunately, there's not en...",0.952354


In [45]:
# Hybrid decoding using the best validation strategy for each gold type:
# phrase  -> close3
# multi   -> top3
# passage -> sentence_top3

passage_sentence_lookup = dict(
    zip(
        passage_expansion_df["row_number"],
        passage_expansion_df["sentence_top3"]
    )
)

hybrid_predictions = []
hybrid_scores = []

for _, row in validation_predictions_df.iterrows():
    spoiler_type = row["spoiler_type"]
    row_number = int(row["row_number"])

    if spoiler_type == "phrase":
        prediction = row["close3"]

    elif spoiler_type == "multi":
        prediction = row["top3"]

    elif spoiler_type == "passage":
        prediction = passage_sentence_lookup.get(
            row_number,
            row["top3"]
        )

    else:
        prediction = row["close3"]

    prediction = normalize_prediction_text(prediction)

    reference_tokens = row["gold_text"].split()
    prediction_tokens = prediction.split()

    score = (
        meteor_score(
            [reference_tokens],
            prediction_tokens
        )
        if prediction_tokens
        else 0.0
    )

    hybrid_predictions.append(prediction)
    hybrid_scores.append(score)


validation_predictions_df[
    "hybrid_prediction"
] = hybrid_predictions

validation_predictions_df[
    "hybrid_meteor"
] = hybrid_scores

print(
    "Hybrid validation mean METEOR:",
    validation_predictions_df[
        "hybrid_meteor"
    ].mean()
)

print(
    "Hybrid validation median METEOR:",
    validation_predictions_df[
        "hybrid_meteor"
    ].median()
)

print("\nHybrid results by spoiler type:")

display(
    validation_predictions_df.groupby(
        "spoiler_type"
    )["hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

print(
    "\nAverage prediction words:",
    validation_predictions_df[
        "hybrid_prediction"
    ].str.split().str.len().mean()
)

display(
    validation_predictions_df[
        [
            "spoiler_type",
            "gold_text",
            "hybrid_prediction",
            "hybrid_meteor"
        ]
    ].head(15)
)

Hybrid validation mean METEOR: 0.40595220404461996
Hybrid validation median METEOR: 0.40864708739104877

Hybrid results by spoiler type:


,count,mean,median
spoiler_type,,,
multi,84,0.347759,0.289025
passage,154,0.372731,0.291411
phrase,162,0.467707,0.477273



Average prediction words: 31.7975


,spoiler_type,gold_text,hybrid_prediction,hybrid_meteor
0,passage,some of the plot elements are so disturbing th...,According to a post by Cawthon on the Five Nig...,0.159091
1,multi,"""intentionally"" could transform a court case a...","""The Defendants’ unfair, partial, and inequita...",0.051813
2,phrase,20%,$3 to $5 between $5 and $20 20%,0.294118
3,multi,Alan Rickman & Rupert Grint CBGB,candy corn CBGB The Dead Boys,0.083333
4,passage,a man who swallowed a 64GB microSD card and th...,What’s important is that he couldn't puke it b...,0.635499
5,phrase,Sprite,Sprite,0.500000
6,phrase,Smoky Paprika-Baked Garbanzo Beans,Smoky Paprika-Baked Garbanzo Beans 1 Tbsp. smo...,0.826823
7,passage,McGonagall was appointed as Dumbledore’s assis...,It’s made of Fir with a core of Dragon Heartst...,0.155827
8,passage,All the scenes are actually in the movie,"No, there’s not. All the scenes are actually i...",0.471372
9,passage,"""I had fake relationships, fake fights. I don'...","""I had fake relationships, fake fights. ""What ...",0.380130


In [47]:
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


def build_type_classifier_text(row):
    """
    Combine fields available in train, validation, and test.

    The post and title receive the most useful signal, while the
    description adds limited article context.
    """
    post_text = clean_text(row.get("postText", ""))
    title = clean_text(row.get("targetTitle", ""))
    description = clean_text(row.get("targetDescription", ""))

    return (
        f"POST: {post_text} "
        f"TITLE: {title} "
        f"DESCRIPTION: {description}"
    )


train_type_texts = train_df.apply(
    build_type_classifier_text,
    axis=1
)

val_type_texts = val_df.apply(
    build_type_classifier_text,
    axis=1
)

train_type_labels = train_df["tags"].apply(
    get_spoiler_type
)

val_type_labels = val_df["tags"].apply(
    get_spoiler_type
)

type_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        )
    ),
    (
        "char",
        TfidfVectorizer(
            lowercase=True,
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        )
    )
])

type_classifier = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=SEED
)

train_type_matrix = type_features.fit_transform(
    train_type_texts
)

val_type_matrix = type_features.transform(
    val_type_texts
)

type_classifier.fit(
    train_type_matrix,
    train_type_labels
)

val_predicted_types = type_classifier.predict(
    val_type_matrix
)

print(
    "Validation accuracy:",
    accuracy_score(
        val_type_labels,
        val_predicted_types
    )
)

print(
    "Validation macro F1:",
    f1_score(
        val_type_labels,
        val_predicted_types,
        average="macro"
    )
)

print(
    "Validation weighted F1:",
    f1_score(
        val_type_labels,
        val_predicted_types,
        average="weighted"
    )
)

print("\nClassification report:")
print(
    classification_report(
        val_type_labels,
        val_predicted_types,
        digits=4
    )
)

label_order = ["phrase", "passage", "multi"]

print("Confusion matrix:")
display(
    pd.DataFrame(
        confusion_matrix(
            val_type_labels,
            val_predicted_types,
            labels=label_order
        ),
        index=[
            f"true_{label}"
            for label in label_order
        ],
        columns=[
            f"pred_{label}"
            for label in label_order
        ]
    )
)

print("\nPredicted type counts:")
print(pd.Series(val_predicted_types).value_counts())

Validation accuracy: 0.5725
Validation macro F1: 0.544739279644033
Validation weighted F1: 0.5672928170910153

Classification report:
              precision    recall  f1-score   support

       multi     0.5000    0.3690    0.4247        84
     passage     0.5786    0.5974    0.5879       154
      phrase     0.5922    0.6543    0.6217       162

    accuracy                         0.5725       400
   macro avg     0.5569    0.5403    0.5447       400
weighted avg     0.5676    0.5725    0.5673       400

Confusion matrix:


,pred_phrase,pred_passage,pred_multi
true_phrase,106,42,14
true_passage,45,92,17
true_multi,28,25,31



Predicted type counts:
phrase     179
passage    159
multi       62
Name: count, dtype: int64


In [48]:
predicted_type_lookup = {
    row_number: predicted_type
    for row_number, predicted_type
    in enumerate(val_predicted_types)
}

predicted_type_hybrid_predictions = []
predicted_type_hybrid_scores = []

for _, row in validation_predictions_df.iterrows():
    row_number = int(row["row_number"])

    predicted_type = predicted_type_lookup[
        row_number
    ]

    if predicted_type == "phrase":
        prediction = row["close3"]

    elif predicted_type == "multi":
        prediction = row["top3"]

    elif predicted_type == "passage":
        prediction = passage_sentence_lookup.get(
            row_number,
            row["top3"]
        )

    else:
        prediction = row["close3"]

    prediction = normalize_prediction_text(
        prediction
    )

    reference_tokens = row["gold_text"].split()
    prediction_tokens = prediction.split()

    score = (
        meteor_score(
            [reference_tokens],
            prediction_tokens
        )
        if prediction_tokens
        else 0.0
    )

    predicted_type_hybrid_predictions.append(
        prediction
    )

    predicted_type_hybrid_scores.append(
        score
    )


validation_predictions_df[
    "predicted_type"
] = validation_predictions_df[
    "row_number"
].map(predicted_type_lookup)

validation_predictions_df[
    "predicted_type_hybrid_prediction"
] = predicted_type_hybrid_predictions

validation_predictions_df[
    "predicted_type_hybrid_meteor"
] = predicted_type_hybrid_scores


print(
    "Gold-type hybrid METEOR:",
    validation_predictions_df[
        "hybrid_meteor"
    ].mean()
)

print(
    "Predicted-type hybrid METEOR:",
    validation_predictions_df[
        "predicted_type_hybrid_meteor"
    ].mean()
)

print(
    "METEOR loss from type prediction:",
    validation_predictions_df[
        "hybrid_meteor"
    ].mean()
    -
    validation_predictions_df[
        "predicted_type_hybrid_meteor"
    ].mean()
)

print("\nResults by true spoiler type:")

display(
    validation_predictions_df.groupby(
        "spoiler_type"
    )["predicted_type_hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

print("\nResults by predicted spoiler type:")

display(
    validation_predictions_df.groupby(
        "predicted_type"
    )["predicted_type_hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

Gold-type hybrid METEOR: 0.40595220404461996
Predicted-type hybrid METEOR: 0.37780129225250547
METEOR loss from type prediction: 0.028150911792114486

Results by true spoiler type:


,count,mean,median
spoiler_type,,,
multi,84,0.342662,0.291210
passage,154,0.342003,0.207986
phrase,162,0.430052,0.410510



Results by predicted spoiler type:


,count,mean,median
predicted_type,,,
multi,62,0.320149,0.273898
passage,159,0.344474,0.253514
phrase,179,0.427374,0.398936


In [49]:
from datasets import Dataset

TYPE_LABEL_TO_ID = {
    "phrase": 0,
    "passage": 1,
    "multi": 2
}

TYPE_ID_TO_LABEL = {
    value: key
    for key, value in TYPE_LABEL_TO_ID.items()
}

TYPE_MAX_LENGTH = 320


def build_type_article_text(row):
    """
    Build the article-side input using fields available in
    training, validation, and test.
    """
    title = clean_text(row.get("targetTitle", ""))
    description = clean_text(
        row.get("targetDescription", "")
    )

    paragraphs = row.get("targetParagraphs", [])

    if not isinstance(paragraphs, list):
        paragraphs = []

    paragraph_text = " ".join(
        clean_text(paragraph)
        for paragraph in paragraphs
        if clean_text(paragraph)
    )

    return (
        f"TITLE: {title} "
        f"DESCRIPTION: {description} "
        f"ARTICLE: {paragraph_text}"
    )


train_type_model_df = pd.DataFrame({
    "post_text": train_df["postText"].apply(
        clean_text
    ),
    "article_text": train_df.apply(
        build_type_article_text,
        axis=1
    ),
    "labels": train_df["tags"].apply(
        get_spoiler_type
    ).map(TYPE_LABEL_TO_ID)
})

val_type_model_df = pd.DataFrame({
    "post_text": val_df["postText"].apply(
        clean_text
    ),
    "article_text": val_df.apply(
        build_type_article_text,
        axis=1
    ),
    "labels": val_df["tags"].apply(
        get_spoiler_type
    ).map(TYPE_LABEL_TO_ID)
})

type_train_raw = Dataset.from_pandas(
    train_type_model_df,
    preserve_index=False
)

type_val_raw = Dataset.from_pandas(
    val_type_model_df,
    preserve_index=False
)


def tokenize_type_examples(examples):
    """
    Treat the clickbait post and article as a paired input.
    """
    return tokenizer(
        examples["post_text"],
        examples["article_text"],
        truncation="longest_first",
        max_length=TYPE_MAX_LENGTH,
        padding="max_length"
    )


type_train_tokenized = type_train_raw.map(
    tokenize_type_examples,
    batched=True,
    remove_columns=[
        "post_text",
        "article_text"
    ],
    desc="Tokenizing type-classifier training data"
)

type_val_tokenized = type_val_raw.map(
    tokenize_type_examples,
    batched=True,
    remove_columns=[
        "post_text",
        "article_text"
    ],
    desc="Tokenizing type-classifier validation data"
)

print("Training examples:", len(type_train_tokenized))
print("Validation examples:", len(type_val_tokenized))
print(
    "Training columns:",
    type_train_tokenized.column_names
)

print("\nTraining label counts:")
print(
    train_type_model_df["labels"]
    .map(TYPE_ID_TO_LABEL)
    .value_counts()
)

print("\nValidation label counts:")
print(
    val_type_model_df["labels"]
    .map(TYPE_ID_TO_LABEL)
    .value_counts()
)

print(
    "\nMissing training labels:",
    int(train_type_model_df["labels"].isna().sum())
)

print(
    "Missing validation labels:",
    int(val_type_model_df["labels"].isna().sum())
)

Tokenizing type-classifier training data:   0%|          | 0/3200 [00:00<?, ? examples/s]

Tokenizing type-classifier validation data:   0%|          | 0/400 [00:00<?, ? examples/s]

Training examples: 3200
Validation examples: 400
Training columns: ['labels', 'input_ids', 'attention_mask']

Training label counts:
labels
phrase     1367
passage    1274
multi       559
Name: count, dtype: int64

Validation label counts:
labels
phrase     162
passage    154
multi       84
Name: count, dtype: int64

Missing training labels: 0
Missing validation labels: 0


In [50]:
from transformers import AutoModelForSequenceClassification

TYPE_MODEL_NAME = "FacebookAI/roberta-base"

type_model = AutoModelForSequenceClassification.from_pretrained(
    TYPE_MODEL_NAME,
    num_labels=3,
    id2label=TYPE_ID_TO_LABEL,
    label2id=TYPE_LABEL_TO_ID
)

type_model.config.use_cache = False

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Type model loaded:", TYPE_MODEL_NAME)
print("Number of labels:", type_model.config.num_labels)
print("ID-to-label mapping:", type_model.config.id2label)

trainable_parameters = sum(
    parameter.numel()
    for parameter in type_model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", f"{trainable_parameters:,}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


CUDA available: True
GPU: Tesla T4
Type model loaded: FacebookAI/roberta-base
Number of labels: 3
ID-to-label mapping: {0: 'phrase', 1: 'passage', 2: 'multi'}
Trainable parameters: 124,647,939


In [51]:
import gc
import inspect
import math
import os
import shutil

import numpy as np
import torch

from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator
)

# 1. Release the QA model and its Trainer from GPU memory.The QA model is already saved safely in Google Drive.

for variable_name in [
    "trainer",
    "model",
    "prediction_output",
    "start_logits",
    "end_logits",
    "val_model_inputs"
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory after QA cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

# Move the type classifier to CPU until training begins.
type_model.to("cpu")
torch.cuda.empty_cache()

# 2. Calculate class weights. Multi spoilers are less common, so they receive a larger training weight.
label_counts = (
    train_type_model_df["labels"]
    .value_counts()
    .sort_index()
)

number_of_examples = len(train_type_model_df)
number_of_classes = len(TYPE_LABEL_TO_ID)

class_weights = torch.tensor(
    [
        number_of_examples
        / (number_of_classes * label_counts[label_id])
        for label_id in range(number_of_classes)
    ],
    dtype=torch.float32
)

print("\nClass weights:")

for label_id, weight in enumerate(class_weights):
    print(
        f"{TYPE_ID_TO_LABEL[label_id]:8s}:",
        round(float(weight), 4)
    )

# 3. Validation metrics.

def compute_type_metrics(eval_prediction):
    logits, labels = eval_prediction

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted"
        )
    }

# 4. Weighted Trainer.

class WeightedClassificationTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs
    ):
        labels = inputs["labels"]

        model_inputs = {
            name: value
            for name, value in inputs.items()
            if name != "labels"
        }

        outputs = model(**model_inputs)
        logits = outputs.logits

        loss_function = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(
                logits.device
            )
        )

        loss = loss_function(
            logits,
            labels
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

# 5. Training configuration.

TYPE_LOCAL_DIR = "/content/type_training_output"

if os.path.exists(TYPE_LOCAL_DIR):
    shutil.rmtree(TYPE_LOCAL_DIR)

os.makedirs(TYPE_LOCAL_DIR, exist_ok=True)

requested_training_kwargs = {
    "output_dir": TYPE_LOCAL_DIR,

    "num_train_epochs": 3,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_steps": 40,

    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,

    "fp16": True,
    "bf16": False,

    "logging_strategy": "steps",
    "logging_steps": 25,

    "save_strategy": "epoch",
    "save_total_limit": 1,

    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,

    "report_to": "none",
    "dataloader_num_workers": 2,
    "remove_unused_columns": True,
    "seed": SEED
}

training_signature = inspect.signature(
    TrainingArguments.__init__
)

supported_parameters = set(
    training_signature.parameters
)

if "eval_strategy" in supported_parameters:
    requested_training_kwargs[
        "eval_strategy"
    ] = "epoch"

elif "evaluation_strategy" in supported_parameters:
    requested_training_kwargs[
        "evaluation_strategy"
    ] = "epoch"

type_training_kwargs = {
    name: value
    for name, value
    in requested_training_kwargs.items()
    if name in supported_parameters
}

removed_arguments = sorted(
    set(requested_training_kwargs)
    - set(type_training_kwargs)
)

type_training_args = TrainingArguments(
    **type_training_kwargs
)

type_trainer_kwargs = {
    "model": type_model,
    "args": type_training_args,
    "train_dataset": type_train_tokenized,
    "eval_dataset": type_val_tokenized,
    "data_collator": default_data_collator,
    "compute_metrics": compute_type_metrics,
    "class_weights": class_weights
}

trainer_signature = inspect.signature(
    Trainer.__init__
)

if "processing_class" in trainer_signature.parameters:
    type_trainer_kwargs[
        "processing_class"
    ] = tokenizer

elif "tokenizer" in trainer_signature.parameters:
    type_trainer_kwargs[
        "tokenizer"
    ] = tokenizer

type_trainer = WeightedClassificationTrainer(
    **type_trainer_kwargs
)

batches_per_epoch = math.ceil(
    len(type_train_tokenized)
    / type_training_args.per_device_train_batch_size
)

optimizer_steps_per_epoch = math.ceil(
    batches_per_epoch
    / type_training_args.gradient_accumulation_steps
)

print("\nUnsupported arguments removed:", removed_arguments)
print("Epochs:", type_training_args.num_train_epochs)
print(
    "Per-device batch size:",
    type_training_args.per_device_train_batch_size
)
print(
    "Gradient accumulation:",
    type_training_args.gradient_accumulation_steps
)
print(
    "Effective batch size:",
    type_training_args.per_device_train_batch_size
    * type_training_args.gradient_accumulation_steps
)
print(
    "Estimated optimizer steps per epoch:",
    optimizer_steps_per_epoch
)
print(
    "Estimated total optimizer steps:",
    optimizer_steps_per_epoch
    * int(type_training_args.num_train_epochs)
)
print("FP16 enabled:", type_training_args.fp16)
print(
    "Best-model metric:",
    type_training_args.metric_for_best_model
)
print(
    "Type Trainer created:",
    type_trainer is not None
)

GPU memory after QA cleanup: 1.41 GB

Class weights:
phrase  : 0.7803
passage : 0.8373
multi   : 1.9082

Unsupported arguments removed: []
Epochs: 3
Per-device batch size: 8
Gradient accumulation: 2
Effective batch size: 16
Estimated optimizer steps per epoch: 200
Estimated total optimizer steps: 600
FP16 enabled: True
Best-model metric: macro_f1
Type Trainer created: True


In [52]:
## training cell

import time
import torch

torch.cuda.empty_cache()

start_time = time.time()

type_train_result = type_trainer.train()

elapsed_minutes = (time.time() - start_time) / 60

print("\nType-classifier training completed.")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")

print("\nFinal training metrics:")
for metric_name, metric_value in type_train_result.metrics.items():
    print(f"{metric_name}: {metric_value}")

print("\nBest checkpoint:")
print(type_trainer.state.best_model_checkpoint)

print("\nBest validation metric:")
print(type_trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.843695,0.913402,0.657500,0.650105,0.655250
2,1.538078,0.708063,0.740000,0.725765,0.736607
3,1.205146,0.675094,0.755000,0.747129,0.754201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Type-classifier training completed.
Elapsed time: 4.05 minutes

Final training metrics:
train_runtime: 241.433
train_samples_per_second: 39.763
train_steps_per_second: 2.485
total_flos: 1578680506368000.0
train_loss: 1.6043444283803303
epoch: 3.0

Best checkpoint:
/content/type_training_output/checkpoint-600

Best validation metric:
0.7471291365064004


In [53]:
import json
import os

TYPE_MODEL_DIR = (
    "/content/drive/MyDrive/Task2_FinalShot/"
    "type_classifier_model"
)

os.makedirs(TYPE_MODEL_DIR, exist_ok=True)

type_trainer.save_model(TYPE_MODEL_DIR)
tokenizer.save_pretrained(TYPE_MODEL_DIR)

type_metrics_path = os.path.join(
    TYPE_MODEL_DIR,
    "training_summary.json"
)

type_training_summary = {
    "best_checkpoint": type_trainer.state.best_model_checkpoint,
    "best_macro_f1": float(type_trainer.state.best_metric),
    "final_epoch": float(type_train_result.metrics["epoch"]),
    "train_loss": float(type_train_result.metrics["train_loss"]),
    "train_runtime": float(
        type_train_result.metrics["train_runtime"]
    ),
    "label_mapping": TYPE_ID_TO_LABEL
}

with open(
    type_metrics_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        type_training_summary,
        file,
        indent=2
    )

weights_saved = (
    os.path.exists(
        os.path.join(TYPE_MODEL_DIR, "model.safetensors")
    )
    or os.path.exists(
        os.path.join(TYPE_MODEL_DIR, "pytorch_model.bin")
    )
)

print("Type model directory exists:", os.path.exists(TYPE_MODEL_DIR))
print("Model weights saved:", weights_saved)
print(
    "Tokenizer saved:",
    os.path.exists(
        os.path.join(
            TYPE_MODEL_DIR,
            "tokenizer_config.json"
        )
    )
)
print("Training summary saved:", os.path.exists(type_metrics_path))
print("Type model location:", TYPE_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Type model directory exists: True
Model weights saved: True
Tokenizer saved: True
Training summary saved: True
Type model location: /content/drive/MyDrive/Task2_FinalShot/type_classifier_model


## evaluate the result

In [54]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

type_validation_output = type_trainer.predict(
    type_val_tokenized
)

type_val_logits = type_validation_output.predictions

roberta_val_type_ids = np.argmax(
    type_val_logits,
    axis=-1
)

roberta_val_predicted_types = np.array([
    TYPE_ID_TO_LABEL[int(label_id)]
    for label_id in roberta_val_type_ids
])

true_val_type_ids = np.array(
    type_val_tokenized["labels"]
)

true_val_types = np.array([
    TYPE_ID_TO_LABEL[int(label_id)]
    for label_id in true_val_type_ids
])

print(
    "Validation accuracy:",
    accuracy_score(
        true_val_types,
        roberta_val_predicted_types
    )
)

print(
    "Validation macro F1:",
    f1_score(
        true_val_types,
        roberta_val_predicted_types,
        average="macro"
    )
)

print(
    "Validation weighted F1:",
    f1_score(
        true_val_types,
        roberta_val_predicted_types,
        average="weighted"
    )
)

print("\nClassification report:")
print(
    classification_report(
        true_val_types,
        roberta_val_predicted_types,
        digits=4
    )
)

label_order = ["phrase", "passage", "multi"]

print("Confusion matrix:")
display(
    pd.DataFrame(
        confusion_matrix(
            true_val_types,
            roberta_val_predicted_types,
            labels=label_order
        ),
        index=[
            f"true_{label}"
            for label in label_order
        ],
        columns=[
            f"pred_{label}"
            for label in label_order
        ]
    )
)

print("\nPredicted type counts:")
print(
    pd.Series(
        roberta_val_predicted_types
    ).value_counts()
)

Validation accuracy: 0.755
Validation macro F1: 0.7471291365064004
Validation weighted F1: 0.7542010722020239

Classification report:
              precision    recall  f1-score   support

       multi     0.7108    0.7024    0.7066        84
     passage     0.7560    0.8247    0.7888       154
      phrase     0.7785    0.7160    0.7460       162

    accuracy                         0.7550       400
   macro avg     0.7484    0.7477    0.7471       400
weighted avg     0.7556    0.7550    0.7542       400

Confusion matrix:


,pred_phrase,pred_passage,pred_multi
true_phrase,116,32,14
true_passage,17,127,10
true_multi,16,9,59



Predicted type counts:
passage    168
phrase     149
multi       83
Name: count, dtype: int64


In [55]:
roberta_type_lookup = {
    row_number: predicted_type
    for row_number, predicted_type
    in enumerate(roberta_val_predicted_types)
}

roberta_hybrid_predictions = []
roberta_hybrid_scores = []

for _, row in validation_predictions_df.iterrows():
    row_number = int(row["row_number"])
    predicted_type = roberta_type_lookup[row_number]

    if predicted_type == "phrase":
        prediction = row["close3"]

    elif predicted_type == "multi":
        prediction = row["top3"]

    elif predicted_type == "passage":
        prediction = passage_sentence_lookup.get(
            row_number,
            row["top3"]
        )

    else:
        prediction = row["close3"]

    prediction = normalize_prediction_text(prediction)

    reference_tokens = row["gold_text"].split()
    prediction_tokens = prediction.split()

    score = (
        meteor_score(
            [reference_tokens],
            prediction_tokens
        )
        if prediction_tokens
        else 0.0
    )

    roberta_hybrid_predictions.append(prediction)
    roberta_hybrid_scores.append(score)


validation_predictions_df[
    "roberta_predicted_type"
] = validation_predictions_df[
    "row_number"
].map(roberta_type_lookup)

validation_predictions_df[
    "roberta_type_hybrid_prediction"
] = roberta_hybrid_predictions

validation_predictions_df[
    "roberta_type_hybrid_meteor"
] = roberta_hybrid_scores


gold_type_score = validation_predictions_df[
    "hybrid_meteor"
].mean()

tfidf_type_score = validation_predictions_df[
    "predicted_type_hybrid_meteor"
].mean()

roberta_type_score = validation_predictions_df[
    "roberta_type_hybrid_meteor"
].mean()

print("Gold-type hybrid METEOR:", gold_type_score)
print("TF-IDF-type hybrid METEOR:", tfidf_type_score)
print("RoBERTa-type hybrid METEOR:", roberta_type_score)

print(
    "Improvement over TF-IDF type classifier:",
    roberta_type_score - tfidf_type_score
)

print(
    "Remaining gap from gold types:",
    gold_type_score - roberta_type_score
)

print("\nResults by true spoiler type:")

display(
    validation_predictions_df.groupby(
        "spoiler_type"
    )["roberta_type_hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

print("\nResults by predicted spoiler type:")

display(
    validation_predictions_df.groupby(
        "roberta_predicted_type"
    )["roberta_type_hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

Gold-type hybrid METEOR: 0.40595220404461996
TF-IDF-type hybrid METEOR: 0.37780129225250547
RoBERTa-type hybrid METEOR: 0.39717340391511674
Improvement over TF-IDF type classifier: 0.01937211166261127
Remaining gap from gold types: 0.008778800129503217

Results by true spoiler type:


,count,mean,median
spoiler_type,,,
multi,84,0.348654,0.299588
passage,154,0.363147,0.262449
phrase,162,0.454677,0.450487



Results by predicted spoiler type:


,count,mean,median
roberta_predicted_type,,,
multi,83,0.301507,0.260571
passage,168,0.375724,0.322061
phrase,149,0.474649,0.454545


In [56]:
# Build sentence-top3 predictions for every validation article,
# regardless of its gold spoiler type.

all_sentence_top3_lookup = {}

for row_number in sorted(
    passage_candidate_predictions["row_number"].unique()
):
    article_predictions = (
        passage_candidate_predictions[
            passage_candidate_predictions["row_number"] == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    expanded_sentences = []
    seen_sentences = set()

    for _, candidate in article_predictions.iterrows():
        sentence = expand_prediction_to_sentence(
            candidate["context"],
            candidate["predicted_text"]
        )

        normalized_sentence = normalize_prediction_text(
            sentence
        ).lower()

        if (
            sentence
            and normalized_sentence
            and normalized_sentence not in seen_sentences
        ):
            expanded_sentences.append(
                normalize_prediction_text(sentence)
            )
            seen_sentences.add(normalized_sentence)

        if len(expanded_sentences) >= 3:
            break

    all_sentence_top3_lookup[row_number] = " ".join(
        expanded_sentences[:3]
    )


corrected_predictions = []
corrected_scores = []

for _, row in validation_predictions_df.iterrows():
    row_number = int(row["row_number"])
    predicted_type = roberta_type_lookup[row_number]

    if predicted_type == "phrase":
        prediction = row["close3"]

    elif predicted_type == "multi":
        prediction = row["top3"]

    elif predicted_type == "passage":
        prediction = all_sentence_top3_lookup.get(
            row_number,
            row["top3"]
        )

    else:
        prediction = row["close3"]

    prediction = normalize_prediction_text(prediction)

    prediction_tokens = prediction.split()
    reference_tokens = row["gold_text"].split()

    score = (
        meteor_score(
            [reference_tokens],
            prediction_tokens
        )
        if prediction_tokens
        else 0.0
    )

    corrected_predictions.append(prediction)
    corrected_scores.append(score)


validation_predictions_df[
    "corrected_roberta_hybrid_prediction"
] = corrected_predictions

validation_predictions_df[
    "corrected_roberta_hybrid_meteor"
] = corrected_scores


print(
    "Previous RoBERTa-type METEOR:",
    validation_predictions_df[
        "roberta_type_hybrid_meteor"
    ].mean()
)

print(
    "Corrected RoBERTa-type METEOR:",
    validation_predictions_df[
        "corrected_roberta_hybrid_meteor"
    ].mean()
)

print(
    "Change:",
    validation_predictions_df[
        "corrected_roberta_hybrid_meteor"
    ].mean()
    -
    validation_predictions_df[
        "roberta_type_hybrid_meteor"
    ].mean()
)

print("\nCorrected results by true type:")

display(
    validation_predictions_df.groupby(
        "spoiler_type"
    )["corrected_roberta_hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

print("\nCorrected results by predicted type:")

display(
    validation_predictions_df.groupby(
        "roberta_predicted_type"
    )["corrected_roberta_hybrid_meteor"]
    .agg(["count", "mean", "median"])
)

Previous RoBERTa-type METEOR: 0.39717340391511674
Corrected RoBERTa-type METEOR: 0.38804090142842157
Change: -0.009132502486695171

Corrected results by true type:


,count,mean,median
spoiler_type,,,
multi,84,0.357019,0.304689
passage,154,0.363147,0.262449
phrase,162,0.427790,0.416667



Corrected results by predicted type:


,count,mean,median
roberta_predicted_type,,,
multi,83,0.301507,0.260571
passage,168,0.353980,0.278555
phrase,149,0.474649,0.454545


## We should now optimize one practical decision: when the classifier predicts passage, decide whether to use sentence expansion or the shorter top3 QA spans based on the classifier’s confidence.

In [57]:
import numpy as np
import pandas as pd
import torch

# 1. Convert type-classifier logits into probabilities.
type_val_probabilities = torch.softmax(
    torch.tensor(type_val_logits),
    dim=-1
).numpy()

for label_name, label_id in TYPE_LABEL_TO_ID.items():
    validation_predictions_df[
        f"prob_{label_name}"
    ] = type_val_probabilities[:, label_id]


# 2. Build sentence-top1, top2, and top3 for every article.
all_sentence_predictions = {}

for row_number in sorted(
    passage_candidate_predictions["row_number"].unique()
):
    article_predictions = (
        passage_candidate_predictions[
            passage_candidate_predictions["row_number"]
            == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    expanded_sentences = []
    seen_sentences = set()

    for _, candidate in article_predictions.iterrows():
        sentence = expand_prediction_to_sentence(
            candidate["context"],
            candidate["predicted_text"]
        )

        sentence = normalize_prediction_text(sentence)
        normalized_sentence = sentence.lower()

        if (
            sentence
            and normalized_sentence not in seen_sentences
        ):
            expanded_sentences.append(sentence)
            seen_sentences.add(normalized_sentence)

        if len(expanded_sentences) >= 3:
            break

    all_sentence_predictions[row_number] = {
        "sentence_top1": " ".join(
            expanded_sentences[:1]
        ),
        "sentence_top2": " ".join(
            expanded_sentences[:2]
        ),
        "sentence_top3": " ".join(
            expanded_sentences[:3]
        )
    }


for strategy in [
    "sentence_top1",
    "sentence_top2",
    "sentence_top3"
]:
    validation_predictions_df[strategy] = (
        validation_predictions_df["row_number"]
        .map(
            lambda row_number: (
                all_sentence_predictions
                .get(int(row_number), {})
                .get(strategy, "")
            )
        )
    )


# 3. Precompute METEOR for each possible decoding output.

candidate_strategies = [
    "close3",
    "top3",
    "sentence_top1",
    "sentence_top2",
    "sentence_top3"
]

for strategy in candidate_strategies:
    strategy_scores = []

    for _, row in validation_predictions_df.iterrows():
        reference_tokens = row["gold_text"].split()
        prediction_tokens = str(
            row[strategy]
        ).split()

        score = (
            meteor_score(
                [reference_tokens],
                prediction_tokens
            )
            if prediction_tokens
            else 0.0
        )

        strategy_scores.append(score)

    validation_predictions_df[
        f"{strategy}_routing_meteor"
    ] = strategy_scores


# 4. Grid search deployable confidence-based routing.
#
# Predicted phrase:
#   close3 when confidence is high, otherwise top3.
#
# Predicted passage:
#   sentence expansion when confidence is high,
#   otherwise top3.
#
# Predicted multi:
#   always top3.

thresholds = np.arange(0.30, 0.96, 0.05)

routing_results = []

for passage_strategy in [
    "sentence_top1",
    "sentence_top2",
    "sentence_top3"
]:
    for phrase_threshold in thresholds:
        for passage_threshold in thresholds:
            selected_scores = []

            for _, row in validation_predictions_df.iterrows():
                predicted_type = row[
                    "roberta_predicted_type"
                ]

                if predicted_type == "phrase":
                    if (
                        row["prob_phrase"]
                        >= phrase_threshold
                    ):
                        chosen_strategy = "close3"
                    else:
                        chosen_strategy = "top3"

                elif predicted_type == "passage":
                    if (
                        row["prob_passage"]
                        >= passage_threshold
                    ):
                        chosen_strategy = passage_strategy
                    else:
                        chosen_strategy = "top3"

                else:
                    chosen_strategy = "top3"

                selected_scores.append(
                    row[
                        f"{chosen_strategy}_routing_meteor"
                    ]
                )

            routing_results.append({
                "passage_strategy": passage_strategy,
                "phrase_threshold": float(
                    phrase_threshold
                ),
                "passage_threshold": float(
                    passage_threshold
                ),
                "mean_meteor": float(
                    np.mean(selected_scores)
                ),
                "median_meteor": float(
                    np.median(selected_scores)
                )
            })


routing_results_df = (
    pd.DataFrame(routing_results)
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Fully deployable baseline METEOR:")
print(
    validation_predictions_df[
        "corrected_roberta_hybrid_meteor"
    ].mean()
)

print("\nBest confidence-routing configuration:")
display(routing_results_df.head(10))

Fully deployable baseline METEOR:
0.38804090142842157

Best confidence-routing configuration:


,passage_strategy,phrase_threshold,passage_threshold,mean_meteor,median_meteor
0,sentence_top3,0.35,0.50,0.393997,0.350530
1,sentence_top3,0.40,0.50,0.393997,0.350530
2,sentence_top3,0.30,0.50,0.393997,0.350530
3,sentence_top3,0.45,0.50,0.393420,0.343862
4,sentence_top3,0.30,0.55,0.392039,0.339439
5,sentence_top3,0.40,0.55,0.392039,0.339439
6,sentence_top3,0.35,0.55,0.392039,0.339439
7,sentence_top3,0.50,0.50,0.392027,0.339439
8,sentence_top3,0.45,0.55,0.391461,0.336632
9,sentence_top3,0.40,0.65,0.391352,0.334821


In [58]:
test_type_model_df = pd.DataFrame({
    "post_text": test_df["postText"].apply(clean_text),
    "article_text": test_df.apply(
        build_type_article_text,
        axis=1
    )
})

type_test_raw = Dataset.from_pandas(
    test_type_model_df,
    preserve_index=False
)

type_test_tokenized = type_test_raw.map(
    tokenize_type_examples,
    batched=True,
    remove_columns=[
        "post_text",
        "article_text"
    ],
    desc="Tokenizing test type-classifier data"
)

type_test_output = type_trainer.predict(
    type_test_tokenized
)

type_test_logits = type_test_output.predictions

type_test_probabilities = torch.softmax(
    torch.tensor(type_test_logits),
    dim=-1
).numpy()

test_predicted_type_ids = np.argmax(
    type_test_probabilities,
    axis=-1
)

test_predicted_types = np.array([
    TYPE_ID_TO_LABEL[int(label_id)]
    for label_id in test_predicted_type_ids
])

test_type_predictions_df = pd.DataFrame({
    "row_number": np.arange(len(test_df)),
    "id": test_df["id"].values,
    "predicted_type": test_predicted_types,
    "prob_phrase": type_test_probabilities[
        :, TYPE_LABEL_TO_ID["phrase"]
    ],
    "prob_passage": type_test_probabilities[
        :, TYPE_LABEL_TO_ID["passage"]
    ],
    "prob_multi": type_test_probabilities[
        :, TYPE_LABEL_TO_ID["multi"]
    ]
})

test_type_predictions_df["confidence"] = (
    type_test_probabilities.max(axis=1)
)

print("Test examples predicted:", len(test_type_predictions_df))

print("\nPredicted type counts:")
print(
    test_type_predictions_df[
        "predicted_type"
    ].value_counts()
)

print("\nConfidence summary:")
print(
    test_type_predictions_df[
        "confidence"
    ].describe()
)

print(
    "\nPredicted passage rows meeting 0.50 threshold:",
    int(
        (
            (
                test_type_predictions_df[
                    "predicted_type"
                ] == "passage"
            )
            &
            (
                test_type_predictions_df[
                    "prob_passage"
                ] >= 0.50
            )
        ).sum()
    )
)

print(
    "Predicted phrase rows meeting 0.35 threshold:",
    int(
        (
            (
                test_type_predictions_df[
                    "predicted_type"
                ] == "phrase"
            )
            &
            (
                test_type_predictions_df[
                    "prob_phrase"
                ] >= 0.35
            )
        ).sum()
    )
)

display(test_type_predictions_df.head(10))

Tokenizing test type-classifier data:   0%|          | 0/400 [00:00<?, ? examples/s]

Test examples predicted: 400

Predicted type counts:
predicted_type
passage    183
phrase     151
multi       66
Name: count, dtype: int64

Confidence summary:
count    400.000000
mean       0.760663
std        0.158040
min        0.344921
25%        0.639379
50%        0.792390
75%        0.889635
max        0.988384
Name: confidence, dtype: float64

Predicted passage rows meeting 0.50 threshold: 170
Predicted phrase rows meeting 0.35 threshold: 150


,row_number,id,predicted_type,prob_phrase,prob_passage,prob_multi,confidence
0,0,0,phrase,0.604905,0.107925,0.287170,0.604905
1,1,1,passage,0.050703,0.836890,0.112407,0.836890
2,2,2,phrase,0.854692,0.120896,0.024412,0.854692
3,3,3,phrase,0.827504,0.097014,0.075481,0.827504
4,4,4,phrase,0.502479,0.455951,0.041569,0.502479
5,5,5,phrase,0.924291,0.045794,0.029915,0.924291
6,6,6,multi,0.006569,0.005047,0.988384,0.988384
7,7,7,passage,0.079914,0.831500,0.088586,0.831500
8,8,8,phrase,0.903145,0.026459,0.070395,0.903145
9,9,9,multi,0.095519,0.101185,0.803296,0.803296


In [59]:
import gc
import os
import torch

TYPE_PREDICTIONS_PATH = os.path.join(
    OUTPUT_DIR,
    "test_type_predictions.csv"
)

test_type_predictions_df.to_csv(
    TYPE_PREDICTIONS_PATH,
    index=False
)

# The classifier is safely saved in Drive, so release it.
for variable_name in [
    "type_trainer",
    "type_model",
    "type_test_output",
    "type_test_logits",
    "type_test_tokenized",
    "type_test_raw",
    "type_val_logits",
    "type_validation_output"
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

print(
    "Type predictions saved:",
    os.path.exists(TYPE_PREDICTIONS_PATH)
)
print("Saved rows:", len(test_type_predictions_df))
print("Saved location:", TYPE_PREDICTIONS_PATH)
print(
    "GPU memory after type-model cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Type predictions saved: True
Saved rows: 400
Saved location: /content/drive/MyDrive/Task2_FinalShot/outputs/test_type_predictions.csv
GPU memory after type-model cleanup: 1.87 GB


In [61]:
test_candidate_records = []

for row_number, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Building test QA candidates"
):
    question = clean_text(row.get("postText", ""))
    candidate_contexts = get_article_contexts(row)

    for candidate_number, candidate in enumerate(candidate_contexts):
        test_candidate_records.append({
            "candidate_id": (
                f"test_{row_number}"
                f"_candidate_{candidate_number}"
            ),
            "row_number": int(row_number),
            "article_id": row["id"],
            "question": question,
            "context": candidate["context"],
            "source_kind": candidate["source_kind"],
            "source_index": int(candidate["source_index"])
        })

test_candidate_df = pd.DataFrame(test_candidate_records)

print("Total test candidate contexts:", len(test_candidate_df))

print(
    "Average candidates per article:",
    round(
        test_candidate_df.groupby("row_number").size().mean(),
        2
    )
)

print(
    "Test articles represented:",
    test_candidate_df["row_number"].nunique()
)

print(
    "Articles with no candidate context:",
    len(test_df)
    - test_candidate_df["row_number"].nunique()
)

print("\nCandidate source counts:")
print(test_candidate_df["source_kind"].value_counts())

display(
    test_candidate_df[
        [
            "row_number",
            "article_id",
            "source_kind",
            "source_index",
            "question",
            "context"
        ]
    ].head(10)
)

Building test QA candidates:   0%|          | 0/400 [00:00<?, ?it/s]

Total test candidate contexts: 6091
Average candidates per article: 15.23
Test articles represented: 400
Articles with no candidate context: 0

Candidate source counts:
source_kind
paragraph    5691
title         400
Name: count, dtype: int64


,row_number,article_id,source_kind,source_index,question,context
0,0,0,title,-1,He Tackles A Nurse At The Hospital. Then You S...,Male Nurse Breaks Down When His Friend Reveals...
1,0,0,paragraph,0,He Tackles A Nurse At The Hospital. Then You S...,"When you think about your good friends, many t..."
2,0,0,paragraph,1,He Tackles A Nurse At The Hospital. Then You S...,You may even ask yourself the question — what ...
3,0,0,paragraph,2,He Tackles A Nurse At The Hospital. Then You S...,"For 24-year-old Graham McMillan, the answer is..."
4,0,0,paragraph,3,He Tackles A Nurse At The Hospital. Then You S...,"As it turned out, he was a match!"
5,0,0,paragraph,4,He Tackles A Nurse At The Hospital. Then You S...,He decided to surprise his friend with the goo...
6,0,0,paragraph,5,He Tackles A Nurse At The Hospital. Then You S...,"In this emotional clip, McMillan walks into th..."
7,0,0,paragraph,6,He Tackles A Nurse At The Hospital. Then You S...,"The moment Kolzow sees him, he understandably ..."
8,1,1,title,-1,Why you SHOULD be selfish at work,Why you SHOULD be selfish at work: Helping oth...
9,1,1,paragraph,0,Why you SHOULD be selfish at work,We're always being encouraged to help others b...


In [62]:
from datasets import Dataset

test_candidate_dataset_raw = Dataset.from_pandas(
    test_candidate_df.reset_index(drop=True),
    preserve_index=False
)

test_tokenized = test_candidate_dataset_raw.map(
    prepare_validation_features,
    batched=True,
    with_indices=True,
    remove_columns=test_candidate_dataset_raw.column_names,
    desc="Tokenizing test QA candidates"
)

candidate_indices_present = set(
    int(value)
    for value in test_tokenized["candidate_index"]
)

features_per_candidate = pd.Series(
    test_tokenized["candidate_index"]
).value_counts()

print(
    "Original candidate contexts:",
    len(test_candidate_dataset_raw)
)

print(
    "Tokenized test features:",
    len(test_tokenized)
)

print(
    "Candidate contexts represented:",
    len(candidate_indices_present)
)

print(
    "Candidate contexts missing:",
    len(test_candidate_df) - len(candidate_indices_present)
)

print(
    "Candidates requiring multiple windows:",
    int((features_per_candidate > 1).sum())
)

print(
    "Maximum windows for one candidate:",
    int(features_per_candidate.max())
)

print(
    "Test tokenized columns:",
    test_tokenized.column_names
)

Tokenizing test QA candidates:   0%|          | 0/6091 [00:00<?, ? examples/s]

Original candidate contexts: 6091
Tokenized test features: 6103
Candidate contexts represented: 6091
Candidate contexts missing: 0
Candidates requiring multiple windows: 9
Maximum windows for one candidate: 3
Test tokenized columns: ['input_ids', 'attention_mask', 'offset_mapping', 'candidate_index']


In [63]:
import gc
import torch

from transformers import AutoModelForQuestionAnswering

gc.collect()
torch.cuda.empty_cache()

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_DIR
)

qa_model.config.use_cache = False

print("QA model loaded from:", MODEL_DIR)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in qa_model.parameters() if p.requires_grad):,}"
)

print(
    "GPU memory after loading:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

QA model loaded from: /content/drive/MyDrive/Task2_FinalShot/paragraph_qa_model
CUDA available: True
GPU: Tesla T4
Trainable parameters: 124,056,578
GPU memory after loading: 1.87 GB


In [64]:
import inspect
import os

from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator
)

QA_INFERENCE_DIR = "/content/qa_inference_output"
os.makedirs(QA_INFERENCE_DIR, exist_ok=True)

requested_inference_kwargs = {
    "output_dir": QA_INFERENCE_DIR,
    "per_device_eval_batch_size": 16,
    "fp16": True,
    "bf16": False,
    "report_to": "none",
    "dataloader_num_workers": 2,
    "remove_unused_columns": True
}

training_signature = inspect.signature(
    TrainingArguments.__init__
)

supported_parameters = set(
    training_signature.parameters
)

inference_kwargs = {
    name: value
    for name, value in requested_inference_kwargs.items()
    if name in supported_parameters
}

qa_inference_args = TrainingArguments(
    **inference_kwargs
)

qa_inference_trainer_kwargs = {
    "model": qa_model,
    "args": qa_inference_args,
    "data_collator": default_data_collator
}

trainer_signature = inspect.signature(
    Trainer.__init__
)

if "processing_class" in trainer_signature.parameters:
    qa_inference_trainer_kwargs[
        "processing_class"
    ] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    qa_inference_trainer_kwargs[
        "tokenizer"
    ] = tokenizer

qa_inference_trainer = Trainer(
    **qa_inference_trainer_kwargs
)

print(
    "Evaluation batch size:",
    qa_inference_args.per_device_eval_batch_size
)
print("FP16 enabled:", qa_inference_args.fp16)
print(
    "Inference Trainer created:",
    qa_inference_trainer is not None
)

Evaluation batch size: 16
FP16 enabled: True
Inference Trainer created: True


In [65]:
test_model_inputs = test_tokenized.remove_columns(
    ["offset_mapping", "candidate_index"]
)

print("Test inference features:", len(test_model_inputs))
print("Model input columns:", test_model_inputs.column_names)

test_prediction_output = qa_inference_trainer.predict(
    test_model_inputs
)

test_start_logits, test_end_logits = (
    test_prediction_output.predictions
)

print("\nTest inference completed.")
print("Start-logit shape:", test_start_logits.shape)
print("End-logit shape:", test_end_logits.shape)
print(
    "Prediction runtime:",
    test_prediction_output.metrics.get("test_runtime")
)
print(
    "Features per second:",
    test_prediction_output.metrics.get(
        "test_samples_per_second"
    )
)

Test inference features: 6103
Model input columns: ['input_ids', 'attention_mask']



Test inference completed.
Start-logit shape: (6103, 384)
End-logit shape: (6103, 384)
Prediction runtime: 31.6204
Features per second: 193.008


In [66]:
test_candidate_best_predictions = {}

for feature_index in tqdm(
    range(len(test_tokenized)),
    desc="Extracting test QA spans"
):
    feature = test_tokenized[feature_index]

    candidate_index = int(feature["candidate_index"])
    input_ids = feature["input_ids"]
    offsets = feature["offset_mapping"]

    feature_start_logits = test_start_logits[feature_index]
    feature_end_logits = test_end_logits[feature_index]

    cls_index = (
        input_ids.index(tokenizer.cls_token_id)
        if tokenizer.cls_token_id in input_ids
        else 0
    )

    null_score = float(
        feature_start_logits[cls_index]
        + feature_end_logits[cls_index]
    )

    top_start_indices = np.argsort(
        feature_start_logits
    )[-N_BEST_START_END:][::-1]

    top_end_indices = np.argsort(
        feature_end_logits
    )[-N_BEST_START_END:][::-1]

    best_span_score = -float("inf")
    best_start_character = None
    best_end_character = None

    for start_index in top_start_indices:
        start_offset = offsets[start_index]

        if (
            start_offset[0] < 0
            or start_offset[1] <= start_offset[0]
        ):
            continue

        for end_index in top_end_indices:
            end_offset = offsets[end_index]

            if (
                end_offset[0] < 0
                or end_offset[1] <= end_offset[0]
            ):
                continue

            if end_index < start_index:
                continue

            if (
                end_index - start_index + 1
                > MAX_ANSWER_LENGTH
            ):
                continue

            start_character = int(start_offset[0])
            end_character = int(end_offset[1])

            if end_character <= start_character:
                continue

            span_score = float(
                feature_start_logits[start_index]
                + feature_end_logits[end_index]
            )

            if span_score > best_span_score:
                best_span_score = span_score
                best_start_character = start_character
                best_end_character = end_character

    candidate_row = test_candidate_df.iloc[
        candidate_index
    ]

    context = str(candidate_row["context"])

    if (
        best_start_character is None
        or best_end_character is None
    ):
        predicted_text = ""
        answerability_margin = -float("inf")
    else:
        predicted_text = context[
            best_start_character:best_end_character
        ].strip()

        answerability_margin = (
            best_span_score - null_score
        )

    prediction_record = {
        "candidate_index": candidate_index,
        "row_number": int(candidate_row["row_number"]),
        "article_id": candidate_row["article_id"],
        "source_kind": candidate_row["source_kind"],
        "source_index": int(candidate_row["source_index"]),
        "predicted_text": predicted_text,
        "best_span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": answerability_margin,
        "feature_index": feature_index
    }

    previous_prediction = (
        test_candidate_best_predictions.get(
            candidate_index
        )
    )

    if (
        previous_prediction is None
        or answerability_margin
        > previous_prediction["answerability_margin"]
    ):
        test_candidate_best_predictions[
            candidate_index
        ] = prediction_record


test_candidate_predictions_df = pd.DataFrame(
    test_candidate_best_predictions.values()
).sort_values(
    ["row_number", "answerability_margin"],
    ascending=[True, False]
).reset_index(drop=True)

print(
    "Candidate predictions:",
    len(test_candidate_predictions_df)
)

print(
    "Test articles represented:",
    test_candidate_predictions_df[
        "row_number"
    ].nunique()
)

print(
    "Empty predicted spans:",
    int(
        (
            test_candidate_predictions_df[
                "predicted_text"
            ].str.len() == 0
        ).sum()
    )
)

print("\nAnswerability-margin summary:")
print(
    test_candidate_predictions_df[
        "answerability_margin"
    ].describe()
)

print("\nHighest-ranked candidates for the first article:")
display(
    test_candidate_predictions_df[
        test_candidate_predictions_df[
            "row_number"
        ] == 0
    ][
        [
            "source_kind",
            "source_index",
            "predicted_text",
            "answerability_margin"
        ]
    ].head(10)
)

Extracting test QA spans:   0%|          | 0/6103 [00:00<?, ?it/s]

Candidate predictions: 6091
Test articles represented: 400
Empty predicted spans: 0

Answerability-margin summary:
count    6091.000000
mean       -2.821742
std         6.088535
min       -23.363281
25%        -5.628502
50%        -2.293945
75%         0.775391
max        12.173828
Name: answerability_margin, dtype: float64

Highest-ranked candidates for the first article:


,source_kind,source_index,predicted_text,answerability_margin
0,paragraph,5,"balloons and a sign in hand that reads, ""Heard...",0.748047
1,paragraph,2,"Graham McMillan, the answer is simple: there i...",-0.689453
2,title,-1,Kidney,-1.624023
3,paragraph,3,he was a match,-1.734375
4,paragraph,0,good friends,-2.347656
5,paragraph,6,"The moment Kolzow sees him, he understandably ...",-2.508789
6,paragraph,4,He decided to surprise his friend with the goo...,-6.033691
7,paragraph,1,You may even ask yourself the question — what ...,-9.165527


In [67]:
# Attach each candidate's original context for sentence expansion.
test_predictions_with_context = (
    test_candidate_predictions_df
    .merge(
        test_candidate_df[["context"]],
        left_on="candidate_index",
        right_index=True,
        how="left"
    )
)

final_test_records = []

for row_number in range(len(test_df)):
    article_predictions = (
        test_predictions_with_context[
            test_predictions_with_context["row_number"]
            == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    type_row = test_type_predictions_df.iloc[row_number]

    predicted_type = str(type_row["predicted_type"])
    prob_phrase = float(type_row["prob_phrase"])
    prob_passage = float(type_row["prob_passage"])
    prob_multi = float(type_row["prob_multi"])

    # Standard QA outputs.
    top3_prediction = join_unique_predictions(
        article_predictions,
        maximum_answers=3
    )

    if len(article_predictions) > 0:
        best_margin = float(
            article_predictions.iloc[0][
                "answerability_margin"
            ]
        )

        close_candidates = article_predictions[
            article_predictions["answerability_margin"]
            >= best_margin - 2.0
        ]
    else:
        close_candidates = article_predictions

    close3_prediction = join_unique_predictions(
        close_candidates,
        maximum_answers=3
    )

    # Sentence-expanded passage output.
    expanded_sentences = []
    seen_sentences = set()

    for _, candidate in article_predictions.iterrows():
        sentence = expand_prediction_to_sentence(
            candidate["context"],
            candidate["predicted_text"]
        )

        sentence = normalize_prediction_text(sentence)
        normalized_sentence = sentence.lower()

        if (
            sentence
            and normalized_sentence not in seen_sentences
        ):
            expanded_sentences.append(sentence)
            seen_sentences.add(normalized_sentence)

        if len(expanded_sentences) >= 3:
            break

    sentence_top3_prediction = " ".join(
        expanded_sentences[:3]
    )

    # Confidence-based routing selected on validation.
    if predicted_type == "phrase":
        if prob_phrase >= 0.35:
            final_prediction = close3_prediction
            routing_strategy = "phrase_close3"
        else:
            final_prediction = top3_prediction
            routing_strategy = "phrase_top3"

    elif predicted_type == "passage":
        if prob_passage >= 0.50:
            final_prediction = sentence_top3_prediction
            routing_strategy = "passage_sentence_top3"
        else:
            final_prediction = top3_prediction
            routing_strategy = "passage_top3"

    else:
        final_prediction = top3_prediction
        routing_strategy = "multi_top3"

    final_prediction = normalize_prediction_text(
        final_prediction
    )

    # Defensive fallback.
    if not final_prediction and len(article_predictions) > 0:
        final_prediction = normalize_prediction_text(
            article_predictions.iloc[0]["predicted_text"]
        )
        routing_strategy += "_fallback"

    final_test_records.append({
        "row_number": row_number,
        "id": test_df.iloc[row_number]["id"],
        "predicted_type": predicted_type,
        "prob_phrase": prob_phrase,
        "prob_passage": prob_passage,
        "prob_multi": prob_multi,
        "routing_strategy": routing_strategy,
        "spoiler": final_prediction,
        "word_count": len(final_prediction.split())
    })


final_test_predictions_df = pd.DataFrame(
    final_test_records
)

print("Final predictions:", len(final_test_predictions_df))

print(
    "Empty predictions:",
    int(
        (
            final_test_predictions_df["spoiler"]
            .str.len() == 0
        ).sum()
    )
)

print("\nRouting strategy counts:")
print(
    final_test_predictions_df[
        "routing_strategy"
    ].value_counts()
)

print("\nPrediction word-count summary:")
print(
    final_test_predictions_df[
        "word_count"
    ].describe()
)

print("\nAverage words by predicted type:")
print(
    final_test_predictions_df.groupby(
        "predicted_type"
    )["word_count"].mean().round(2)
)

display(
    final_test_predictions_df[
        [
            "id",
            "predicted_type",
            "routing_strategy",
            "spoiler",
            "word_count"
        ]
    ].head(15)
)

Final predictions: 400
Empty predictions: 0

Routing strategy counts:
routing_strategy
passage_sentence_top3    170
phrase_close3            150
multi_top3                66
passage_top3              13
phrase_top3                1
Name: count, dtype: int64

Prediction word-count summary:
count    400.000000
mean      33.490000
std       32.311453
min        1.000000
25%        5.000000
50%       22.500000
75%       59.000000
max      164.000000
Name: word_count, dtype: float64

Average words by predicted type:
predicted_type
multi      20.11
passage    59.31
phrase      8.05
Name: word_count, dtype: float64


,id,predicted_type,routing_strategy,spoiler,word_count
0,0,phrase,phrase_close3,"balloons and a sign in hand that reads, ""Heard...",59
1,1,passage,passage_sentence_top3,1. Prioritise b. Invite Juan to sit in on your...,31
2,2,phrase,phrase_close3,Have a Bunch of Money,5
3,3,phrase,phrase_close3,Braconid,1
4,4,phrase,phrase_close3,3. Remove the egg yolks from the fridge after ...,32
5,5,phrase,phrase_close3,John Williams music. To celebrate the fake hol...,44
6,6,multi,multi_top3,6. Zone yourself out 5. Tell yourself a story ...,16
7,7,passage,passage_sentence_top3,"They're not a piece of equipment,"" Reiss said....",61
8,8,phrase,phrase_close3,Lord Ivar Mountbatten,3
9,9,multi,multi_top3,putting away too much One extra dollar in your...,55


In [68]:
length_checks = {
    "phrase_over_15_words": (
        (final_test_predictions_df["predicted_type"] == "phrase")
        & (final_test_predictions_df["word_count"] > 15)
    ),
    "multi_over_60_words": (
        (final_test_predictions_df["predicted_type"] == "multi")
        & (final_test_predictions_df["word_count"] > 60)
    ),
    "passage_over_100_words": (
        (final_test_predictions_df["predicted_type"] == "passage")
        & (final_test_predictions_df["word_count"] > 100)
    )
}

for check_name, mask in length_checks.items():
    print(check_name, ":", int(mask.sum()))

print("\nLongest predicted phrases:")
display(
    final_test_predictions_df[
        final_test_predictions_df["predicted_type"] == "phrase"
    ][
        [
            "row_number",
            "id",
            "prob_phrase",
            "routing_strategy",
            "spoiler",
            "word_count"
        ]
    ]
    .sort_values("word_count", ascending=False)
    .head(15)
)

print("\nLongest predicted multis:")
display(
    final_test_predictions_df[
        final_test_predictions_df["predicted_type"] == "multi"
    ][
        [
            "row_number",
            "id",
            "prob_multi",
            "spoiler",
            "word_count"
        ]
    ]
    .sort_values("word_count", ascending=False)
    .head(10)
)

print("\nLongest predicted passages:")
display(
    final_test_predictions_df[
        final_test_predictions_df["predicted_type"] == "passage"
    ][
        [
            "row_number",
            "id",
            "prob_passage",
            "spoiler",
            "word_count"
        ]
    ]
    .sort_values("word_count", ascending=False)
    .head(10)
)

phrase_over_15_words : 21
multi_over_60_words : 3
passage_over_100_words : 11

Longest predicted phrases:


,row_number,id,prob_phrase,routing_strategy,spoiler,word_count
276,276,276,0.517616,phrase_close3,Every blue-eyed person on the planet is descen...,103
0,0,0,0.604905,phrase_close3,"balloons and a sign in hand that reads, ""Heard...",59
68,68,68,0.560815,phrase_close3,"Right now, an Amazon Prime subscription costs ...",55
170,170,170,0.869072,phrase_close3,"$765,759 in loose, unclaimed coins. Yet some p...",51
5,5,5,0.924291,phrase_close3,John Williams music. To celebrate the fake hol...,44
339,339,339,0.511165,phrase_close3,"""The human magnitude of climate change looks m...",41
169,169,169,0.860849,phrase_close3,"being a single mom to daughter Jessica ""It’s i...",34
4,4,4,0.502479,phrase_close3,3. Remove the egg yolks from the fridge after ...,32
216,216,216,0.855108,phrase_close3,Manhattan Eatsa Eatsa Instead of the usual fac...,31
222,222,222,0.637678,phrase_close3,Short-term measures High kill rate Taiwan has ...,30



Longest predicted multis:


,row_number,id,prob_multi,spoiler,word_count
111,111,111,0.820600,"""Law enforcement against drugs is completely i...",64
57,57,57,0.984407,Louise the infant koala Disgruntled over the p...,62
115,115,115,0.410544,After getting my DNA report I learned that the...,61
9,9,9,0.803296,putting away too much One extra dollar in your...,55
281,281,281,0.773231,"3. When cats rub their head against you, they’...",54
233,233,233,0.984782,banned the use of 'sky lanterns In Pennsylvani...,50
24,24,24,0.440238,"On the website, people can donate to sexual ab...",48
142,142,142,0.986604,The unemployment rate in November fell to 4.6 ...,46
138,138,138,0.962924,People complaining about the slightest thing t...,40
247,247,247,0.530475,"The ""Venus Holes"" are a sign of good circulati...",37



Longest predicted passages:


,row_number,id,prob_passage,spoiler,word_count
123,123,123,0.807864,While the idea of a Universal Basic Income (UB...,164
343,343,343,0.763193,"Tom Hanks has the scene in ""Captain Phillips,""...",136
342,342,342,0.671634,A sample of 89 women who usually drank diet be...,121
166,166,166,0.621394,And his numbers have been borne out: The Olymp...,118
107,107,107,0.535895,"In a post which has since been made private, D...",117
74,74,74,0.905491,Vegan athletes do need to be diligent about co...,114
129,129,129,0.891870,"Of course, this could be the first step in a P...",111
16,16,16,0.874564,The letter went on to say that none of the pig...,110
176,176,176,0.754720,"So if you see anyone with a golden ribbon, you...",107
361,361,361,0.707527,"""If you are dropping off your son’s forgotten ...",103


In [69]:
def word_count(text):
    return len(normalize_prediction_text(text).split())


def choose_capped_prediction(
    preferred_prediction,
    fallback_predictions,
    maximum_words
):
    """
    Keep the preferred output when it fits the limit.

    Otherwise, use the first non-empty fallback that fits.
    If none fit, use the shortest non-empty option.
    """
    preferred_prediction = normalize_prediction_text(
        preferred_prediction
    )

    fallback_predictions = [
        normalize_prediction_text(prediction)
        for prediction in fallback_predictions
        if normalize_prediction_text(prediction)
    ]

    if (
        preferred_prediction
        and word_count(preferred_prediction) <= maximum_words
    ):
        return preferred_prediction

    for prediction in fallback_predictions:
        if word_count(prediction) <= maximum_words:
            return prediction

    available_predictions = [
        prediction
        for prediction in (
            [preferred_prediction] + fallback_predictions
        )
        if prediction
    ]

    if not available_predictions:
        return ""

    return min(
        available_predictions,
        key=word_count
    )


length_safeguard_results = []

phrase_caps = [8, 10, 12, 15, 20, 30, 999]
multi_caps = [30, 40, 50, 60, 80, 999]
passage_caps = [60, 80, 100, 120, 150, 999]

for phrase_cap in phrase_caps:
    for multi_cap in multi_caps:
        for passage_cap in passage_caps:
            row_scores = []
            changed_predictions = 0

            for _, row in validation_predictions_df.iterrows():
                predicted_type = row[
                    "roberta_predicted_type"
                ]

                # Confidence-based routing selected previously.
                if predicted_type == "phrase":
                    if row["prob_phrase"] >= 0.35:
                        original_prediction = row["close3"]
                    else:
                        original_prediction = row["top3"]

                    safeguarded_prediction = choose_capped_prediction(
                        preferred_prediction=original_prediction,
                        fallback_predictions=[
                            row["top1"],
                            row["top2"],
                            row["top3"]
                        ],
                        maximum_words=phrase_cap
                    )

                elif predicted_type == "passage":
                    if row["prob_passage"] >= 0.50:
                        original_prediction = row["sentence_top3"]
                    else:
                        original_prediction = row["top3"]

                    safeguarded_prediction = choose_capped_prediction(
                        preferred_prediction=original_prediction,
                        fallback_predictions=[
                            row["sentence_top2"],
                            row["sentence_top1"],
                            row["top3"],
                            row["top1"]
                        ],
                        maximum_words=passage_cap
                    )

                else:
                    original_prediction = row["top3"]

                    safeguarded_prediction = choose_capped_prediction(
                        preferred_prediction=original_prediction,
                        fallback_predictions=[
                            row["top2"],
                            row["top1"]
                        ],
                        maximum_words=multi_cap
                    )

                if (
                    normalize_prediction_text(original_prediction)
                    != safeguarded_prediction
                ):
                    changed_predictions += 1

                reference_tokens = row["gold_text"].split()
                prediction_tokens = safeguarded_prediction.split()

                score = (
                    meteor_score(
                        [reference_tokens],
                        prediction_tokens
                    )
                    if prediction_tokens
                    else 0.0
                )

                row_scores.append(score)

            length_safeguard_results.append({
                "phrase_cap": phrase_cap,
                "multi_cap": multi_cap,
                "passage_cap": passage_cap,
                "mean_meteor": float(np.mean(row_scores)),
                "median_meteor": float(np.median(row_scores)),
                "changed_predictions": changed_predictions
            })


length_safeguard_results_df = (
    pd.DataFrame(length_safeguard_results)
    .sort_values(
        ["mean_meteor", "changed_predictions"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("Uncapped deployable routing METEOR:")
print(0.393997)

print("\nBest validation-tested length safeguards:")
display(length_safeguard_results_df.head(15))

Uncapped deployable routing METEOR:
0.393997

Best validation-tested length safeguards:


,phrase_cap,multi_cap,passage_cap,mean_meteor,median_meteor,changed_predictions
0,999,60,100,0.395041,0.355458,10
1,999,80,100,0.395041,0.355458,10
2,999,999,100,0.395041,0.355458,10
3,30,60,100,0.394634,0.355458,16
4,30,80,100,0.394634,0.355458,16
5,30,999,100,0.394634,0.355458,16
6,999,60,120,0.394612,0.350530,3
7,999,80,120,0.394612,0.350530,3
8,999,999,120,0.394612,0.350530,3
9,30,60,120,0.394205,0.350530,9


In [70]:
PHRASE_CAP = 999
MULTI_CAP = 60
PASSAGE_CAP = 100

safeguarded_test_records = []

for row_number in range(len(test_df)):
    article_predictions = (
        test_predictions_with_context[
            test_predictions_with_context["row_number"]
            == row_number
        ]
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
    )

    type_row = test_type_predictions_df.iloc[row_number]

    predicted_type = str(type_row["predicted_type"])
    prob_phrase = float(type_row["prob_phrase"])
    prob_passage = float(type_row["prob_passage"])
    prob_multi = float(type_row["prob_multi"])

    # QA span alternatives.
    top1_prediction = join_unique_predictions(
        article_predictions,
        maximum_answers=1
    )

    top2_prediction = join_unique_predictions(
        article_predictions,
        maximum_answers=2
    )

    top3_prediction = join_unique_predictions(
        article_predictions,
        maximum_answers=3
    )

    if len(article_predictions) > 0:
        best_margin = float(
            article_predictions.iloc[0][
                "answerability_margin"
            ]
        )

        close_candidates = article_predictions[
            article_predictions["answerability_margin"]
            >= best_margin - 2.0
        ]
    else:
        close_candidates = article_predictions

    close3_prediction = join_unique_predictions(
        close_candidates,
        maximum_answers=3
    )

    # Sentence-expanded alternatives.
    expanded_sentences = []
    seen_sentences = set()

    for _, candidate in article_predictions.iterrows():
        sentence = expand_prediction_to_sentence(
            candidate["context"],
            candidate["predicted_text"]
        )

        sentence = normalize_prediction_text(sentence)
        normalized_sentence = sentence.lower()

        if (
            sentence
            and normalized_sentence not in seen_sentences
        ):
            expanded_sentences.append(sentence)
            seen_sentences.add(normalized_sentence)

        if len(expanded_sentences) >= 3:
            break

    sentence_top1 = " ".join(expanded_sentences[:1])
    sentence_top2 = " ".join(expanded_sentences[:2])
    sentence_top3 = " ".join(expanded_sentences[:3])

    # Apply the confidence routing selected on validation.
    if predicted_type == "phrase":
        if prob_phrase >= 0.35:
            original_prediction = close3_prediction
            original_strategy = "phrase_close3"
        else:
            original_prediction = top3_prediction
            original_strategy = "phrase_top3"

        final_prediction = choose_capped_prediction(
            preferred_prediction=original_prediction,
            fallback_predictions=[
                top1_prediction,
                top2_prediction,
                top3_prediction
            ],
            maximum_words=PHRASE_CAP
        )

    elif predicted_type == "passage":
        if prob_passage >= 0.50:
            original_prediction = sentence_top3
            original_strategy = "passage_sentence_top3"
        else:
            original_prediction = top3_prediction
            original_strategy = "passage_top3"

        final_prediction = choose_capped_prediction(
            preferred_prediction=original_prediction,
            fallback_predictions=[
                sentence_top2,
                sentence_top1,
                top3_prediction,
                top1_prediction
            ],
            maximum_words=PASSAGE_CAP
        )

    else:
        original_prediction = top3_prediction
        original_strategy = "multi_top3"

        final_prediction = choose_capped_prediction(
            preferred_prediction=original_prediction,
            fallback_predictions=[
                top2_prediction,
                top1_prediction
            ],
            maximum_words=MULTI_CAP
        )

    original_prediction = normalize_prediction_text(
        original_prediction
    )

    final_prediction = normalize_prediction_text(
        final_prediction
    )

    # Defensive fallback.
    if not final_prediction and len(article_predictions) > 0:
        final_prediction = normalize_prediction_text(
            article_predictions.iloc[0]["predicted_text"]
        )

    safeguarded_test_records.append({
        "row_number": row_number,
        "id": test_df.iloc[row_number]["id"],
        "predicted_type": predicted_type,
        "routing_strategy": original_strategy,
        "original_prediction": original_prediction,
        "spoiler": final_prediction,
        "original_word_count": word_count(original_prediction),
        "word_count": word_count(final_prediction),
        "changed_by_safeguard": (
            original_prediction != final_prediction
        )
    })


safeguarded_test_predictions_df = pd.DataFrame(
    safeguarded_test_records
)

print(
    "Predictions changed by safeguard:",
    int(
        safeguarded_test_predictions_df[
            "changed_by_safeguard"
        ].sum()
    )
)

print("\nChanges by predicted type:")
print(
    safeguarded_test_predictions_df.groupby(
        "predicted_type"
    )["changed_by_safeguard"].sum()
)

print("\nRemaining length violations:")
print(
    "Phrase over cap:",
    int(
        (
            (
                safeguarded_test_predictions_df[
                    "predicted_type"
                ] == "phrase"
            )
            &
            (
                safeguarded_test_predictions_df[
                    "word_count"
                ] > PHRASE_CAP
            )
        ).sum()
    )
)

print(
    "Multi over cap:",
    int(
        (
            (
                safeguarded_test_predictions_df[
                    "predicted_type"
                ] == "multi"
            )
            &
            (
                safeguarded_test_predictions_df[
                    "word_count"
                ] > MULTI_CAP
            )
        ).sum()
    )
)

print(
    "Passage over cap:",
    int(
        (
            (
                safeguarded_test_predictions_df[
                    "predicted_type"
                ] == "passage"
            )
            &
            (
                safeguarded_test_predictions_df[
                    "word_count"
                ] > PASSAGE_CAP
            )
        ).sum()
    )
)

print("\nFinal word-count summary:")
print(
    safeguarded_test_predictions_df[
        "word_count"
    ].describe()
)

print("\nChanged predictions:")
display(
    safeguarded_test_predictions_df[
        safeguarded_test_predictions_df[
            "changed_by_safeguard"
        ]
    ][
        [
            "row_number",
            "id",
            "predicted_type",
            "original_prediction",
            "spoiler",
            "original_word_count",
            "word_count"
        ]
    ]
)

Predictions changed by safeguard: 14

Changes by predicted type:
predicted_type
multi       3
passage    11
phrase      0
Name: changed_by_safeguard, dtype: int64

Remaining length violations:
Phrase over cap: 0
Multi over cap: 0
Passage over cap: 0

Final word-count summary:
count    400.000000
mean      31.940000
std       29.559445
min        1.000000
25%        5.000000
50%       22.500000
75%       56.000000
max      103.000000
Name: word_count, dtype: float64

Changed predictions:


,row_number,id,predicted_type,original_prediction,spoiler,original_word_count,word_count
16,16,16,passage,The letter went on to say that none of the pig...,The letter went on to say that none of the pig...,110,87
57,57,57,multi,Louise the infant koala Disgruntled over the p...,Louise the infant koala Disgruntled over the p...,62,34
74,74,74,passage,Vegan athletes do need to be diligent about co...,Vegan athletes do need to be diligent about co...,114,83
107,107,107,passage,"In a post which has since been made private, D...","In a post which has since been made private, D...",117,53
111,111,111,multi,"""Law enforcement against drugs is completely i...","""Law enforcement against drugs is completely i...",64,32
115,115,115,multi,After getting my DNA report I learned that the...,After getting my DNA report I learned that the...,61,48
123,123,123,passage,While the idea of a Universal Basic Income (UB...,While the idea of a Universal Basic Income (UB...,164,41
125,125,125,passage,Trump’s role in the porn is relatively benign ...,Trump’s role in the porn is relatively benign ...,102,68
129,129,129,passage,"Of course, this could be the first step in a P...","Of course, this could be the first step in a P...",111,100
166,166,166,passage,And his numbers have been borne out: The Olymp...,And his numbers have been borne out: The Olymp...,118,38


In [71]:
FINAL_SUBMISSION_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_qa_finalshot.csv"
)

FINAL_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_qa_finalshot_audit.csv"
)

# Preserve the exact row and ID order required by sample_solution.csv.
submission_df = (
    sample_df[["id"]]
    .merge(
        safeguarded_test_predictions_df[
            ["id", "spoiler"]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
)

# Save the Kaggle-ready file with exactly two columns.
submission_df.to_csv(
    FINAL_SUBMISSION_PATH,
    index=False
)

# Save a separate detailed file for our records.
safeguarded_test_predictions_df.to_csv(
    FINAL_AUDIT_PATH,
    index=False
)

# Structural validation.
ids_match_sample = submission_df["id"].tolist() == sample_df["id"].tolist()
ids_match_test = submission_df["id"].tolist() == test_df["id"].tolist()

empty_count = int(
    submission_df["spoiler"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Submission file exists:", os.path.exists(FINAL_SUBMISSION_PATH))
print("Submission shape:", submission_df.shape)
print("Submission columns:", submission_df.columns.tolist())
print("IDs match sample order:", ids_match_sample)
print("IDs match test order:", ids_match_test)
print("Unique IDs:", submission_df["id"].nunique())
print("Empty spoilers:", empty_count)
print("Audit file exists:", os.path.exists(FINAL_AUDIT_PATH))
print("Submission location:", FINAL_SUBMISSION_PATH)

print("\nFirst five rows:")
display(submission_df.head())

print("\nLast five rows:")
display(submission_df.tail())

Submission file exists: True
Submission shape: (400, 2)
Submission columns: ['id', 'spoiler']
IDs match sample order: True
IDs match test order: True
Unique IDs: 400
Empty spoilers: 0
Audit file exists: True
Submission location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_qa_finalshot.csv

First five rows:


,id,spoiler
0,0,"balloons and a sign in hand that reads, ""Heard..."
1,1,1. Prioritise b. Invite Juan to sit in on your...
2,2,Have a Bunch of Money
3,3,Braconid
4,4,3. Remove the egg yolks from the fridge after ...



Last five rows:


,id,spoiler
395,395,The video below shows the stunned cleaner init...
396,396,Christopher Suprun
397,397,No Medication. High fat vegan plant based diet...
398,398,WikiLeaks regularly tweets about Assange’s sta...
399,399,Richard Belzer


## run a second epoch experiment since the previous file works and became the highest score file atm

In [75]:
import time
import torch

torch.cuda.empty_cache()

start_time = time.time()

second_epoch_result = second_epoch_trainer.train()

elapsed_minutes = (time.time() - start_time) / 60

print("\nSecond QA epoch completed.")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")

print("\nTraining metrics:")
for metric_name, metric_value in second_epoch_result.metrics.items():
    print(f"{metric_name}: {metric_value}")

Step,Training Loss
50,0.663028
100,0.418228
150,0.419519
200,0.408614
250,0.459682
300,0.399626
350,0.473549
400,0.513496
450,0.692359
500,0.726859



Second QA epoch completed.
Elapsed time: 3.87 minutes

Training metrics:
train_runtime: 231.2699
train_samples_per_second: 46.681
train_steps_per_second: 2.919
total_flos: 2115719839291392.0
train_loss: 0.6012575785319011
epoch: 1.0


### ave loss went down, however near the end the loss seems to be rising again, which could be a sign of overfitting

In [76]:
import json
import os

second_epoch_trainer.save_model(
    SECOND_EPOCH_MODEL_DIR
)

tokenizer.save_pretrained(
    SECOND_EPOCH_MODEL_DIR
)

epoch2_metrics_path = os.path.join(
    SECOND_EPOCH_MODEL_DIR,
    "training_metrics.json"
)

with open(
    epoch2_metrics_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            key: float(value)
            for key, value
            in second_epoch_result.metrics.items()
        },
        file,
        indent=2
    )

weights_saved = (
    os.path.exists(
        os.path.join(
            SECOND_EPOCH_MODEL_DIR,
            "model.safetensors"
        )
    )
    or os.path.exists(
        os.path.join(
            SECOND_EPOCH_MODEL_DIR,
            "pytorch_model.bin"
        )
    )
)

print("Epoch-2 directory exists:", os.path.exists(
    SECOND_EPOCH_MODEL_DIR
))
print("Epoch-2 weights saved:", weights_saved)
print("Tokenizer saved:", os.path.exists(
    os.path.join(
        SECOND_EPOCH_MODEL_DIR,
        "tokenizer_config.json"
    )
))
print("Metrics saved:", os.path.exists(
    epoch2_metrics_path
))
print("Epoch-2 model location:", SECOND_EPOCH_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch-2 directory exists: True
Epoch-2 weights saved: True
Tokenizer saved: True
Metrics saved: True
Epoch-2 model location: /content/drive/MyDrive/Task2_FinalShot/paragraph_qa_model_epoch2


In [77]:
# Prepare validation inputs without metadata columns.
epoch2_val_model_inputs = val_tokenized.remove_columns(
    ["offset_mapping", "candidate_index"]
)

epoch2_validation_output = second_epoch_trainer.predict(
    epoch2_val_model_inputs
)

epoch2_start_logits, epoch2_end_logits = (
    epoch2_validation_output.predictions
)

print("Validation inference completed.")
print("Start-logit shape:", epoch2_start_logits.shape)
print("End-logit shape:", epoch2_end_logits.shape)
print(
    "Prediction runtime:",
    epoch2_validation_output.metrics.get("test_runtime")
)
print(
    "Features per second:",
    epoch2_validation_output.metrics.get(
        "test_samples_per_second"
    )
)

Validation inference completed.
Start-logit shape: (6323, 384)
End-logit shape: (6323, 384)
Prediction runtime: 36.6052
Features per second: 172.735


In [80]:
print("val_tokenized columns:")
print(val_tokenized.column_names)

print("\nval_candidate_predictions_df shape:")
print(val_candidate_predictions_df.shape)

print("\nval_candidate_predictions_df columns:")
print(val_candidate_predictions_df.columns.tolist())

print("\nFirst candidate row:")
display(val_candidate_predictions_df.head(1).T)

print("\nCandidate-related validation objects currently available:")
for variable_name, variable_value in sorted(globals().items()):
    if (
        "val" in variable_name.lower()
        and "candidate" in variable_name.lower()
    ):
        try:
            print(
                f"{variable_name:40s}",
                type(variable_value).__name__,
                getattr(variable_value, "shape", "")
            )
        except Exception:
            pass

val_tokenized columns:
['input_ids', 'attention_mask', 'offset_mapping', 'candidate_index']

val_candidate_predictions_df shape:
(6318, 11)

val_candidate_predictions_df columns:
['candidate_index', 'row_number', 'article_id', 'source_kind', 'source_index', 'is_gold_source', 'predicted_text', 'best_span_score', 'null_score', 'answerability_margin', 'feature_index']

First candidate row:


,0
candidate_index,3
row_number,0
article_id,0
source_kind,paragraph
source_index,2
is_gold_source,True
predicted_text,it’s too dark
best_span_score,8.75
null_score,6.673828
answerability_margin,2.076172



Candidate-related validation objects currently available:
val_candidate_dataset_raw                Dataset (6318, 8)
val_candidate_df                         DataFrame (6318, 8)
val_candidate_predictions_df             DataFrame (6318, 11)
validation_candidate_records             list 


In [81]:
print("val_candidate_df columns:")
print(val_candidate_df.columns.tolist())

print("\nFirst raw validation candidate:")
display(val_candidate_df.head(1).T)

print("\nPossible decoding constants/functions:")
for name in [
    "MAX_ANSWER_LENGTH",
    "MAX_ANSWER_TOKENS",
    "N_BEST_SIZE",
    "decode_qa_predictions",
    "extract_candidate_predictions",
    "get_best_span"
]:
    if name in globals():
        value = globals()[name]
        print(f"{name:30s}", type(value).__name__, value)
    else:
        print(f"{name:30s}", "NOT FOUND")

val_candidate_df columns:
['candidate_id', 'row_number', 'article_id', 'question', 'context', 'source_kind', 'source_index', 'is_gold_source']

First raw validation candidate:


,0
candidate_id,validation_0_candidate_0
row_number,0
article_id,0
question,Five Nights at Freddy’s Sequel Delayed for Wei...
context,Five Nights at Freddy’s Sequel Delayed for Wei...
source_kind,title
source_index,-1
is_gold_source,False



Possible decoding constants/functions:
MAX_ANSWER_LENGTH              int 100
MAX_ANSWER_TOKENS              NOT FOUND
N_BEST_SIZE                    NOT FOUND
decode_qa_predictions          NOT FOUND
extract_candidate_predictions  NOT FOUND
get_best_span                  NOT FOUND


In [82]:
print("Tokenizer CLS token ID:", tokenizer.cls_token_id)

print("\nLogit arrays currently in memory:")
for variable_name, variable_value in sorted(globals().items()):
    lower_name = variable_name.lower()

    if (
        "logit" in lower_name
        and (
            "start" in lower_name
            or "end" in lower_name
        )
    ):
        try:
            print(
                f"{variable_name:40s}",
                type(variable_value).__name__,
                getattr(variable_value, "shape", "")
            )
        except Exception:
            pass

print("\nPrediction-output objects currently in memory:")
for variable_name, variable_value in sorted(globals().items()):
    lower_name = variable_name.lower()

    if (
        "prediction" in lower_name
        and "output" in lower_name
    ):
        try:
            print(
                f"{variable_name:40s}",
                type(variable_value).__name__
            )
        except Exception:
            pass

Tokenizer CLS token ID: 0

Logit arrays currently in memory:
epoch2_end_logits                        ndarray (6323, 384)
epoch2_start_logits                      ndarray (6323, 384)
feature_end_logits                       ndarray (384,)
feature_start_logits                     ndarray (384,)

Prediction-output objects currently in memory:


In [83]:
import numpy as np
import pandas as pd
import time

decode_start_time = time.time()

candidate_best_results = {}

for feature_index in range(len(val_tokenized)):
    feature = val_tokenized[feature_index]

    candidate_index = int(feature["candidate_index"])
    input_ids = feature["input_ids"]
    offsets = feature["offset_mapping"]

    start_logits = epoch2_start_logits[feature_index]
    end_logits = epoch2_end_logits[feature_index]

    # RoBERTa normally places CLS at position 0.
    try:
        cls_index = input_ids.index(tokenizer.cls_token_id)
    except ValueError:
        cls_index = 0

    null_score = float(
        start_logits[cls_index] + end_logits[cls_index]
    )

    valid_mask = np.array([
        offset is not None
        and len(offset) == 2
        and int(offset[1]) > int(offset[0])
        for offset in offsets
    ])

    valid_start_positions = np.flatnonzero(valid_mask)

    best_span_score = -np.inf
    best_start_position = None
    best_end_position = None

    # Exhaustive valid search with a maximum answer length of 100 tokens.
    for start_position in valid_start_positions:
        maximum_end_position = min(
            start_position + MAX_ANSWER_LENGTH - 1,
            len(offsets) - 1
        )

        allowed_end_mask = valid_mask[
            start_position:maximum_end_position + 1
        ]

        if not allowed_end_mask.any():
            continue

        local_end_logits = end_logits[
            start_position:maximum_end_position + 1
        ].copy()

        local_end_logits[~allowed_end_mask] = -np.inf

        relative_end_position = int(
            np.argmax(local_end_logits)
        )

        end_position = (
            start_position + relative_end_position
        )

        span_score = float(
            start_logits[start_position]
            + end_logits[end_position]
        )

        if span_score > best_span_score:
            best_span_score = span_score
            best_start_position = int(start_position)
            best_end_position = int(end_position)

    context = str(
        val_candidate_df.iloc[candidate_index]["context"]
    )

    if (
        best_start_position is not None
        and best_end_position is not None
    ):
        character_start = int(
            offsets[best_start_position][0]
        )
        character_end = int(
            offsets[best_end_position][1]
        )

        predicted_text = normalize_prediction_text(
            context[character_start:character_end]
        )
    else:
        predicted_text = ""
        best_span_score = -np.inf

    answerability_margin = float(
        best_span_score - null_score
    )

    feature_result = {
        "candidate_index": candidate_index,
        "predicted_text": predicted_text,
        "best_span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": answerability_margin,
        "feature_index": feature_index
    }

    # For candidates split across multiple windows,
    # retain the window with the strongest answerability margin.
    previous_result = candidate_best_results.get(
        candidate_index
    )

    if (
        previous_result is None
        or answerability_margin
        > previous_result["answerability_margin"]
    ):
        candidate_best_results[
            candidate_index
        ] = feature_result


epoch2_candidate_records = []

for candidate_index in range(len(val_candidate_df)):
    metadata = val_candidate_df.iloc[candidate_index]

    decoded = candidate_best_results.get(
        candidate_index,
        {
            "predicted_text": "",
            "best_span_score": -np.inf,
            "null_score": np.inf,
            "answerability_margin": -np.inf,
            "feature_index": -1
        }
    )

    epoch2_candidate_records.append({
        "candidate_index": candidate_index,
        "row_number": int(metadata["row_number"]),
        "article_id": metadata["article_id"],
        "source_kind": metadata["source_kind"],
        "source_index": int(metadata["source_index"]),
        "is_gold_source": bool(metadata["is_gold_source"]),
        "predicted_text": decoded["predicted_text"],
        "best_span_score": decoded["best_span_score"],
        "null_score": decoded["null_score"],
        "answerability_margin": decoded[
            "answerability_margin"
        ],
        "feature_index": decoded["feature_index"]
    })


epoch2_val_candidate_predictions_df = pd.DataFrame(
    epoch2_candidate_records
)


def calculate_source_ranking_coverage(
    candidate_predictions_df
):
    article_results = []

    for row_number, group in (
        candidate_predictions_df.groupby("row_number")
    ):
        gold_candidate_indices = set(
            group.loc[
                group["is_gold_source"],
                "candidate_index"
            ].astype(int)
        )

        if not gold_candidate_indices:
            continue

        ranked_candidate_indices = (
            group.sort_values(
                "answerability_margin",
                ascending=False
            )["candidate_index"]
            .astype(int)
            .tolist()
        )

        article_record = {
            "row_number": row_number
        }

        for k in [1, 3, 5]:
            top_k_indices = set(
                ranked_candidate_indices[:k]
            )

            article_record[f"top{k}_any"] = (
                len(
                    top_k_indices
                    & gold_candidate_indices
                ) > 0
            )

            article_record[f"top{k}_all"] = (
                gold_candidate_indices.issubset(
                    top_k_indices
                )
            )

        article_results.append(article_record)

    article_results_df = pd.DataFrame(
        article_results
    )

    return {
        "articles_evaluated": len(article_results_df),
        "top1_any": article_results_df[
            "top1_any"
        ].mean(),
        "top1_all": article_results_df[
            "top1_all"
        ].mean(),
        "top3_any": article_results_df[
            "top3_any"
        ].mean(),
        "top3_all": article_results_df[
            "top3_all"
        ].mean(),
        "top5_any": article_results_df[
            "top5_any"
        ].mean(),
        "top5_all": article_results_df[
            "top5_all"
        ].mean()
    }


epoch1_ranking = calculate_source_ranking_coverage(
    val_candidate_predictions_df
)

epoch2_ranking = calculate_source_ranking_coverage(
    epoch2_val_candidate_predictions_df
)

ranking_comparison_df = pd.DataFrame([
    {
        "model": "epoch_1",
        **epoch1_ranking
    },
    {
        "model": "epoch_2",
        **epoch2_ranking
    }
])

print(
    "Epoch-2 candidates decoded:",
    len(epoch2_val_candidate_predictions_df)
)

print(
    "Empty epoch-2 spans:",
    int(
        epoch2_val_candidate_predictions_df[
            "predicted_text"
        ].eq("").sum()
    )
)

print(
    "Decoding runtime:",
    round(time.time() - decode_start_time, 2),
    "seconds"
)

print("\nSource-ranking comparison:")
display(ranking_comparison_df)

print("\nAnswerability-margin comparison:")

margin_comparison_df = pd.DataFrame([
    {
        "model": "epoch_1",
        "gold_margin_mean": (
            val_candidate_predictions_df.loc[
                val_candidate_predictions_df[
                    "is_gold_source"
                ],
                "answerability_margin"
            ].mean()
        ),
        "nongold_margin_mean": (
            val_candidate_predictions_df.loc[
                ~val_candidate_predictions_df[
                    "is_gold_source"
                ],
                "answerability_margin"
            ].mean()
        )
    },
    {
        "model": "epoch_2",
        "gold_margin_mean": (
            epoch2_val_candidate_predictions_df.loc[
                epoch2_val_candidate_predictions_df[
                    "is_gold_source"
                ],
                "answerability_margin"
            ].mean()
        ),
        "nongold_margin_mean": (
            epoch2_val_candidate_predictions_df.loc[
                ~epoch2_val_candidate_predictions_df[
                    "is_gold_source"
                ],
                "answerability_margin"
            ].mean()
        )
    }
])

display(margin_comparison_df)

Epoch-2 candidates decoded: 6318
Empty epoch-2 spans: 3
Decoding runtime: 24.04 seconds

Source-ranking comparison:


,model,articles_evaluated,top1_any,top1_all,top3_any,top3_all,top5_any,top5_all
0,epoch_1,397,0.450882,0.350126,0.702771,0.584383,0.818640,0.710327
1,epoch_2,397,0.438287,0.329975,0.687657,0.566751,0.811083,0.695214



Answerability-margin comparison:


,model,gold_margin_mean,nongold_margin_mean
0,epoch_1,2.695513,-2.884382
1,epoch_2,4.961227,-2.490260


In [84]:
import numpy as np
import pandas as pd

ensemble_candidate_df = (
    val_candidate_predictions_df[
        [
            "candidate_index",
            "row_number",
            "article_id",
            "source_kind",
            "source_index",
            "is_gold_source",
            "answerability_margin"
        ]
    ]
    .rename(
        columns={
            "answerability_margin": "epoch1_margin"
        }
    )
    .merge(
        epoch2_val_candidate_predictions_df[
            [
                "candidate_index",
                "answerability_margin"
            ]
        ].rename(
            columns={
                "answerability_margin": "epoch2_margin"
            }
        ),
        on="candidate_index",
        how="inner",
        validate="one_to_one"
    )
)


def within_article_zscore(series):
    standard_deviation = series.std(ddof=0)

    if (
        pd.isna(standard_deviation)
        or standard_deviation < 1e-8
    ):
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (
        series - series.mean()
    ) / standard_deviation


ensemble_candidate_df["epoch1_z"] = (
    ensemble_candidate_df
    .groupby("row_number")["epoch1_margin"]
    .transform(within_article_zscore)
)

ensemble_candidate_df["epoch2_z"] = (
    ensemble_candidate_df
    .groupby("row_number")["epoch2_margin"]
    .transform(within_article_zscore)
)


def evaluate_ranking_score(
    candidate_df,
    score_column
):
    article_records = []

    for row_number, group in candidate_df.groupby(
        "row_number"
    ):
        gold_indices = set(
            group.loc[
                group["is_gold_source"],
                "candidate_index"
            ].astype(int)
        )

        if not gold_indices:
            continue

        ranked_indices = (
            group.sort_values(
                score_column,
                ascending=False
            )["candidate_index"]
            .astype(int)
            .tolist()
        )

        record = {"row_number": row_number}

        for k in [1, 3, 5]:
            top_indices = set(ranked_indices[:k])

            record[f"top{k}_any"] = bool(
                top_indices & gold_indices
            )

            record[f"top{k}_all"] = (
                gold_indices.issubset(top_indices)
            )

        article_records.append(record)

    results_df = pd.DataFrame(article_records)

    return {
        "top1_any": results_df["top1_any"].mean(),
        "top1_all": results_df["top1_all"].mean(),
        "top3_any": results_df["top3_any"].mean(),
        "top3_all": results_df["top3_all"].mean(),
        "top5_any": results_df["top5_any"].mean(),
        "top5_all": results_df["top5_all"].mean()
    }


ensemble_results = []

for epoch1_weight in np.arange(
    0.0,
    1.01,
    0.05
):
    epoch1_weight = round(
        float(epoch1_weight),
        2
    )

    epoch2_weight = round(
        1.0 - epoch1_weight,
        2
    )

    score_column = (
        f"ensemble_{epoch1_weight:.2f}"
    )

    ensemble_candidate_df[score_column] = (
        epoch1_weight
        * ensemble_candidate_df["epoch1_z"]
        +
        epoch2_weight
        * ensemble_candidate_df["epoch2_z"]
    )

    metrics = evaluate_ranking_score(
        ensemble_candidate_df,
        score_column
    )

    ensemble_results.append({
        "epoch1_weight": epoch1_weight,
        "epoch2_weight": epoch2_weight,
        **metrics
    })


ensemble_ranking_results_df = (
    pd.DataFrame(ensemble_results)
    .sort_values(
        [
            "top1_any",
            "top3_any",
            "top5_any"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Epoch 1 standalone:")
display(
    ensemble_ranking_results_df[
        ensemble_ranking_results_df[
            "epoch1_weight"
        ] == 1.0
    ]
)

print("\nEpoch 2 standalone:")
display(
    ensemble_ranking_results_df[
        ensemble_ranking_results_df[
            "epoch1_weight"
        ] == 0.0
    ]
)

print("\nBest blended rankings:")
display(
    ensemble_ranking_results_df.head(15)
)

Epoch 1 standalone:


,epoch1_weight,epoch2_weight,top1_any,top1_all,top3_any,top3_all,top5_any,top5_all
0,1.0,0.0,0.450882,0.350126,0.702771,0.584383,0.81864,0.710327



Epoch 2 standalone:


,epoch1_weight,epoch2_weight,top1_any,top1_all,top3_any,top3_all,top5_any,top5_all
15,0.0,1.0,0.438287,0.329975,0.687657,0.566751,0.811083,0.695214



Best blended rankings:


,epoch1_weight,epoch2_weight,top1_any,top1_all,top3_any,top3_all,top5_any,top5_all
0,1.00,0.00,0.450882,0.350126,0.702771,0.584383,0.818640,0.710327
1,0.90,0.10,0.445844,0.342569,0.705290,0.586902,0.813602,0.707809
2,0.95,0.05,0.443325,0.345088,0.705290,0.589421,0.813602,0.705290
3,0.20,0.80,0.443325,0.335013,0.685139,0.561713,0.803526,0.697733
4,0.85,0.15,0.440806,0.337531,0.700252,0.581864,0.813602,0.707809
5,0.70,0.30,0.440806,0.340050,0.697733,0.581864,0.803526,0.697733
6,0.40,0.60,0.440806,0.335013,0.690176,0.569270,0.813602,0.707809
7,0.25,0.75,0.440806,0.335013,0.687657,0.564232,0.808564,0.702771
8,0.30,0.70,0.440806,0.335013,0.687657,0.566751,0.808564,0.702771
9,0.10,0.90,0.440806,0.332494,0.682620,0.561713,0.806045,0.695214


In [85]:
print("Core article dataframes:")

for variable_name in [
    "train_df",
    "val_df",
    "test_df",
    "train_examples_df",
    "validation_examples_df",
    "train_positive_df",
    "validation_positive_df",
    "train_qa_df",
    "validation_qa_df"
]:
    if variable_name in globals():
        value = globals()[variable_name]

        print(
            f"{variable_name:32s}",
            type(value).__name__,
            getattr(value, "shape", "")
        )

        if hasattr(value, "columns"):
            print("  columns:", value.columns.tolist())
    else:
        print(f"{variable_name:32s} NOT FOUND")


print("\nObjects related to repaired spans or positive examples:")

for variable_name, variable_value in sorted(globals().items()):
    lower_name = variable_name.lower()

    if any(
        keyword in lower_name
        for keyword in [
            "positive",
            "repaired",
            "span",
            "annotation"
        ]
    ):
        if isinstance(
            variable_value,
            (pd.DataFrame, list, dict)
        ):
            try:
                size = (
                    variable_value.shape
                    if hasattr(variable_value, "shape")
                    else len(variable_value)
                )

                print(
                    f"{variable_name:45s}",
                    type(variable_value).__name__,
                    size
                )
            except Exception:
                pass


print("\nRaw train columns and first row:")

print(train_df.columns.tolist())
display(train_df.head(1).T)

Core article dataframes:
train_df                         DataFrame (3200, 14)
  columns: ['uuid', 'postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags']
val_df                           DataFrame (400, 14)
  columns: ['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags', 'id']
test_df                          DataFrame (400, 10)
  columns: ['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'id']
train_examples_df                NOT FOUND
validation_examples_df           NOT FOUND
train_positive_df                NOT FOUND
validation_positive_df           NOT FOUND
train_qa_df                      DataFrame (10773, 13)
  colu

,0
uuid,0af11f6b-c889-4520-9372-66ba25cb7657
postId,532quh
postText,"[Wes Welker Wanted Dinner With Tom Brady, But ..."
postPlatform,reddit
targetParagraphs,[It’ll be just like old times this weekend for...
targetTitle,"Wes Welker Wanted Dinner With Tom Brady, But P..."
targetDescription,It'll be just like old times this weekend for ...
targetKeywords,"new england patriots, ricky doyle, top stories,"
targetMedia,"[http://pixel.wp.com/b.gif?v=noscript, http://..."
targetUrl,http://nesn.com/2016/09/wes-welker-wanted-dinn...


In [86]:
import pandas as pd
from collections import Counter


def ensure_text_list(value):
    if isinstance(value, list):
        return value

    if value is None:
        return []

    return [value]


def build_combined_article_context(article_row):
    """
    Combine title and all paragraphs while recording where each
    original source begins inside the combined context.
    """
    pieces = []
    source_offsets = {}
    current_position = 0

    title = clean_text(article_row.get("targetTitle", ""))

    title_prefix = "Title: "
    title_piece = title_prefix + title

    source_offsets[("title", -1)] = (
        current_position + len(title_prefix)
    )

    pieces.append(title_piece)
    current_position += len(title_piece)

    paragraphs = ensure_text_list(
        article_row.get("targetParagraphs", [])
    )

    for paragraph_index, paragraph in enumerate(paragraphs):
        paragraph = clean_text(paragraph)

        separator = "\n\n"
        paragraph_prefix = f"Paragraph {paragraph_index}: "

        current_position += len(separator)
        pieces.append(separator)

        source_offsets[
            ("paragraph", paragraph_index)
        ] = (
            current_position
            + len(paragraph_prefix)
        )

        paragraph_piece = paragraph_prefix + paragraph
        pieces.append(paragraph_piece)

        current_position += len(paragraph_piece)

    combined_context = "".join(pieces)

    return combined_context, source_offsets


def build_article_level_qa(
    article_df,
    positive_qa_df,
    split_name
):
    records = []
    failures = []

    positives_grouped = positive_qa_df.groupby(
        "row_number",
        sort=False
    )

    for row_number, positive_group in positives_grouped:
        row_number = int(row_number)

        article_row = article_df.iloc[row_number]

        combined_context, source_offsets = (
            build_combined_article_context(article_row)
        )

        for positive_index, positive_row in (
            positive_group.iterrows()
        ):
            source_kind = str(
                positive_row["source_kind"]
            )

            source_index = int(
                positive_row["source_index"]
            )

            source_key = (
                source_kind,
                source_index
            )

            if source_key not in source_offsets:
                failures.append({
                    "positive_index": positive_index,
                    "row_number": row_number,
                    "reason": "source_not_found",
                    "source_kind": source_kind,
                    "source_index": source_index
                })
                continue

            local_answer_start = int(
                positive_row["answer_start"]
            )

            answer_text = str(
                positive_row["answer_text"]
            )

            combined_answer_start = (
                source_offsets[source_key]
                + local_answer_start
            )

            combined_answer_end = (
                combined_answer_start
                + len(answer_text)
            )

            extracted_answer = combined_context[
                combined_answer_start:
                combined_answer_end
            ]

            if extracted_answer != answer_text:
                failures.append({
                    "positive_index": positive_index,
                    "row_number": row_number,
                    "reason": "answer_mismatch",
                    "source_kind": source_kind,
                    "source_index": source_index,
                    "expected": answer_text,
                    "extracted": extracted_answer
                })
                continue

            records.append({
                "id": (
                    f"{split_name}_{row_number}_"
                    f"article_answer_{len(records)}"
                ),
                "split": split_name,
                "row_number": row_number,
                "article_id": positive_row[
                    "article_id"
                ],
                "spoiler_type": positive_row[
                    "spoiler_type"
                ],
                "question": positive_row[
                    "question"
                ],
                "context": combined_context,
                "source_kind": source_kind,
                "source_index": source_index,
                "answer_text": answer_text,
                "answer_start": combined_answer_start,
                "answers": {
                    "text": [answer_text],
                    "answer_start": [
                        combined_answer_start
                    ]
                }
            })

    return (
        pd.DataFrame(records),
        pd.DataFrame(failures)
    )


article_train_qa_df, article_train_failures_df = (
    build_article_level_qa(
        article_df=train_df,
        positive_qa_df=train_positive_qa,
        split_name="train"
    )
)

article_val_qa_df, article_val_failures_df = (
    build_article_level_qa(
        article_df=val_df,
        positive_qa_df=val_positive_qa,
        split_name="validation"
    )
)


print("Article-level training examples:")
print(len(article_train_qa_df))

print("\nArticle-level validation examples:")
print(len(article_val_qa_df))

print("\nTraining mapping failures:")
print(len(article_train_failures_df))

print("\nValidation mapping failures:")
print(len(article_val_failures_df))

print("\nUnique training articles:")
print(article_train_qa_df["row_number"].nunique())

print("\nUnique validation articles:")
print(article_val_qa_df["row_number"].nunique())

print("\nTraining examples by spoiler type:")
print(
    article_train_qa_df[
        "spoiler_type"
    ].value_counts()
)

print("\nCombined-context word counts:")
print(
    article_train_qa_df[
        "context"
    ].str.split().str.len().describe()
)

print("\nExample mapped article:")
display(
    article_train_qa_df[
        [
            "row_number",
            "spoiler_type",
            "question",
            "source_kind",
            "source_index",
            "answer_text",
            "answer_start",
            "context"
        ]
    ].head(1).T
)

if len(article_train_failures_df) > 0:
    print("\nFirst training failures:")
    display(article_train_failures_df.head(10))

if len(article_val_failures_df) > 0:
    print("\nFirst validation failures:")
    display(article_val_failures_df.head(10))

Article-level training examples:
4536

Article-level validation examples:
605

Training mapping failures:
10

Validation mapping failures:
2

Unique training articles:
3155

Unique validation articles:
396

Training examples by spoiler type:
spoiler_type
multi      1931
phrase     1363
passage    1242
Name: count, dtype: int64

Combined-context word counts:
count     4536.000000
mean       666.514550
std        730.729553
min         15.000000
25%        284.000000
50%        469.000000
75%        834.000000
max      14087.000000
Name: context, dtype: float64

Example mapped article:


,0
row_number,0
spoiler_type,passage
question,"Wes Welker Wanted Dinner With Tom Brady, But P..."
source_kind,paragraph
source_index,3
answer_text,how about that morning we go throw?
answer_start,833
context,Title: Wes Welker Wanted Dinner With Tom Brady...



First training failures:


,positive_index,row_number,reason,source_kind,source_index,expected,extracted
0,15,9,answer_mismatch,paragraph,31,Cholula Chili Garli,holula Chili Garlic
1,560,407,answer_mismatch,paragraph,5,so their Instagram posts are ruined ̄\_(ツ)_/ ...,so their Instagram posts are ruined ̄\_(ツ)_/ ̄...
2,1505,1061,answer_mismatch,paragraph,0,Charles Chuck Johnso,harles Chuck Johnson
3,1827,1288,answer_mismatch,paragraph,15,"but if you care about the forgetting rate, and...","ut if you care about the forgetting rate, and ..."
4,2554,1783,answer_mismatch,paragraph,11,the Midnight Star,he Midnight Star
5,2661,1858,answer_mismatch,paragraph,1,plastic-cased Disney tapes,"astic-cased Disney tapes,"
6,3108,2179,answer_mismatch,paragraph,0,make the move to being a paid consultan,ake the move to being a paid consultant
7,3646,2591,answer_mismatch,paragraph,1,1. His first ODI century,1. His first ODI century\n
8,3723,2640,answer_mismatch,paragraph,9,#1: They Give You Comfor,1: They Give You Comfort
9,3919,2776,answer_mismatch,paragraph,0,Leg,ego



First validation failures:


,positive_index,row_number,reason,source_kind,source_index,expected,extracted
0,121,84,answer_mismatch,paragraph,0,Yuliana Avalos,uliana Avalos.
1,212,139,answer_mismatch,paragraph,9,Her ticket out of debt and into financial free...,Her ticket out of debt and into financial free...


In [88]:
def build_combined_article_context_v2(
    article_row,
    source_overrides=None
):
    """
    Combine the title and all paragraphs into one article context.

    For any source containing a labeled answer, source_overrides
    supplies the exact repaired QA context so the answer offsets
    remain valid.
    """
    source_overrides = source_overrides or {}

    pieces = []
    source_offsets = {}
    current_position = 0

    # Title
    title_key = ("title", -1)

    title = (
        str(source_overrides[title_key])
        if title_key in source_overrides
        else clean_text(article_row.get("targetTitle", ""))
    )

    title_prefix = "Title: "
    title_piece = title_prefix + title

    source_offsets[title_key] = (
        current_position + len(title_prefix)
    )

    pieces.append(title_piece)
    current_position += len(title_piece)

    # Paragraphs
    paragraphs = ensure_text_list(
        article_row.get("targetParagraphs", [])
    )

    for paragraph_index, raw_paragraph in enumerate(paragraphs):
        source_key = ("paragraph", paragraph_index)

        paragraph = (
            str(source_overrides[source_key])
            if source_key in source_overrides
            else clean_text(raw_paragraph)
        )

        separator = "\n\n"
        paragraph_prefix = f"Paragraph {paragraph_index}: "

        pieces.append(separator)
        current_position += len(separator)

        source_offsets[source_key] = (
            current_position + len(paragraph_prefix)
        )

        paragraph_piece = paragraph_prefix + paragraph

        pieces.append(paragraph_piece)
        current_position += len(paragraph_piece)

    return "".join(pieces), source_offsets


def build_article_level_qa_v2(
    article_df,
    positive_qa_df,
    split_name
):
    records = []
    failures = []
    context_conflicts = []

    for row_number, positive_group in positive_qa_df.groupby(
        "row_number",
        sort=False
    ):
        row_number = int(row_number)
        article_row = article_df.iloc[row_number]

        # Preserve the exact repaired context for every source
        # that contains at least one labeled answer.
        source_overrides = {}

        for _, positive_row in positive_group.iterrows():
            source_key = (
                str(positive_row["source_kind"]),
                int(positive_row["source_index"])
            )

            exact_context = str(positive_row["context"])

            if (
                source_key in source_overrides
                and source_overrides[source_key] != exact_context
            ):
                context_conflicts.append({
                    "row_number": row_number,
                    "source_kind": source_key[0],
                    "source_index": source_key[1]
                })
            else:
                source_overrides[source_key] = exact_context

        combined_context, source_offsets = (
            build_combined_article_context_v2(
                article_row=article_row,
                source_overrides=source_overrides
            )
        )

        for positive_index, positive_row in (
            positive_group.iterrows()
        ):
            source_kind = str(
                positive_row["source_kind"]
            )

            source_index = int(
                positive_row["source_index"]
            )

            source_key = (
                source_kind,
                source_index
            )

            if source_key not in source_offsets:
                failures.append({
                    "positive_index": positive_index,
                    "row_number": row_number,
                    "reason": "source_not_found",
                    "source_kind": source_kind,
                    "source_index": source_index
                })
                continue

            answer_text = str(
                positive_row["answer_text"]
            )

            local_answer_start = int(
                positive_row["answer_start"]
            )

            combined_answer_start = (
                source_offsets[source_key]
                + local_answer_start
            )

            combined_answer_end = (
                combined_answer_start
                + len(answer_text)
            )

            extracted_answer = combined_context[
                combined_answer_start:
                combined_answer_end
            ]

            if extracted_answer != answer_text:
                failures.append({
                    "positive_index": positive_index,
                    "row_number": row_number,
                    "reason": "answer_mismatch",
                    "source_kind": source_kind,
                    "source_index": source_index,
                    "expected": answer_text,
                    "extracted": extracted_answer
                })
                continue

            records.append({
                "id": (
                    f"{split_name}_{row_number}_"
                    f"article_answer_{len(records)}"
                ),
                "split": split_name,
                "row_number": row_number,
                "article_id": positive_row["article_id"],
                "spoiler_type": positive_row["spoiler_type"],
                "question": positive_row["question"],
                "context": combined_context,
                "source_kind": source_kind,
                "source_index": source_index,
                "answer_text": answer_text,
                "answer_start": combined_answer_start,
                "answers": {
                    "text": [answer_text],
                    "answer_start": [combined_answer_start]
                }
            })

    return (
        pd.DataFrame(records),
        pd.DataFrame(failures),
        pd.DataFrame(context_conflicts)
    )


(
    article_train_qa_df,
    article_train_failures_df,
    article_train_conflicts_df
) = build_article_level_qa_v2(
    article_df=train_df,
    positive_qa_df=train_positive_qa,
    split_name="train"
)

(
    article_val_qa_df,
    article_val_failures_df,
    article_val_conflicts_df
) = build_article_level_qa_v2(
    article_df=val_df,
    positive_qa_df=val_positive_qa,
    split_name="validation"
)


print("Training examples:", len(article_train_qa_df))
print("Expected training examples:", len(train_positive_qa))
print("Training mapping failures:", len(article_train_failures_df))
print("Training context conflicts:", len(article_train_conflicts_df))

print("\nValidation examples:", len(article_val_qa_df))
print("Expected validation examples:", len(val_positive_qa))
print("Validation mapping failures:", len(article_val_failures_df))
print("Validation context conflicts:", len(article_val_conflicts_df))

train_exact = all(
    row["context"][
        int(row["answer_start"]):
        int(row["answer_start"])
        + len(str(row["answer_text"]))
    ] == str(row["answer_text"])
    for _, row in article_train_qa_df.iterrows()
)

val_exact = all(
    row["context"][
        int(row["answer_start"]):
        int(row["answer_start"])
        + len(str(row["answer_text"]))
    ] == str(row["answer_text"])
    for _, row in article_val_qa_df.iterrows()
)

print("\nAll training offsets exact:", train_exact)
print("All validation offsets exact:", val_exact)

if len(article_train_failures_df) > 0:
    print("\nTraining failures:")
    display(article_train_failures_df.head(10))

if len(article_val_failures_df) > 0:
    print("\nValidation failures:")
    display(article_val_failures_df.head(10))

Training examples: 4546
Expected training examples: 4546
Training mapping failures: 0
Training context conflicts: 0

Validation examples: 607
Expected validation examples: 607
Validation mapping failures: 0
Validation context conflicts: 0

All training offsets exact: True
All validation offsets exact: True


In [89]:
from datasets import Dataset
import numpy as np
import pandas as pd
import time

ARTICLE_MAX_LENGTH = 384
ARTICLE_DOC_STRIDE = 128

article_train_dataset_raw = Dataset.from_pandas(
    article_train_qa_df,
    preserve_index=False
)

article_val_dataset_raw = Dataset.from_pandas(
    article_val_qa_df,
    preserve_index=False
)


def tokenize_article_qa_batch(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=ARTICLE_MAX_LENGTH,
        stride=ARTICLE_DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    start_positions = []
    end_positions = []
    example_indices = []
    cleaned_offset_mappings = []

    for feature_index, sample_index in enumerate(
        sample_mapping
    ):
        input_ids = tokenized["input_ids"][
            feature_index
        ]

        offsets = tokenized["offset_mapping"][
            feature_index
        ]

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        try:
            cls_index = input_ids.index(
                tokenizer.cls_token_id
            )
        except ValueError:
            cls_index = 0

        answer_start = int(
            examples["answer_start"][sample_index]
        )

        answer_text = str(
            examples["answer_text"][sample_index]
        )

        answer_end = (
            answer_start + len(answer_text)
        )

        # Locate the context-token boundaries.
        context_token_indices = [
            token_index
            for token_index, sequence_id
            in enumerate(sequence_ids)
            if sequence_id == 1
        ]

        if not context_token_indices:
            start_positions.append(cls_index)
            end_positions.append(cls_index)

        else:
            context_start = context_token_indices[0]
            context_end = context_token_indices[-1]

            # This window does not fully contain the answer.
            if (
                offsets[context_start][0] > answer_start
                or offsets[context_end][1] < answer_end
            ):
                start_positions.append(cls_index)
                end_positions.append(cls_index)

            else:
                token_start = context_start

                while (
                    token_start <= context_end
                    and offsets[token_start][0]
                    <= answer_start
                ):
                    token_start += 1

                token_start -= 1

                token_end = context_end

                while (
                    token_end >= context_start
                    and offsets[token_end][1]
                    >= answer_end
                ):
                    token_end -= 1

                token_end += 1

                start_positions.append(token_start)
                end_positions.append(token_end)

        # Keep offsets only for article-context tokens.
        cleaned_offsets = [
            offset if sequence_id == 1 else None
            for offset, sequence_id
            in zip(offsets, sequence_ids)
        ]

        cleaned_offset_mappings.append(
            cleaned_offsets
        )

        example_indices.append(
            int(sample_index)
        )

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized["example_index"] = example_indices
    tokenized["offset_mapping"] = (
        cleaned_offset_mappings
    )

    return tokenized


tokenization_start = time.time()

article_train_tokenized = (
    article_train_dataset_raw.map(
        tokenize_article_qa_batch,
        batched=True,
        batch_size=32,
        remove_columns=(
            article_train_dataset_raw.column_names
        ),
        desc="Tokenizing article-level training data"
    )
)

article_val_tokenized = (
    article_val_dataset_raw.map(
        tokenize_article_qa_batch,
        batched=True,
        batch_size=32,
        remove_columns=(
            article_val_dataset_raw.column_names
        ),
        desc="Tokenizing article-level validation data"
    )
)

tokenization_runtime = (
    time.time() - tokenization_start
)


def summarize_article_features(
    tokenized_dataset,
    number_of_examples,
    split_name
):
    cls_id = tokenizer.cls_token_id

    positive_feature_counts = np.zeros(
        number_of_examples,
        dtype=int
    )

    total_feature_counts = np.zeros(
        number_of_examples,
        dtype=int
    )

    positive_features = 0
    negative_features = 0

    for feature in tokenized_dataset:
        example_index = int(
            feature["example_index"]
        )

        total_feature_counts[example_index] += 1

        input_ids = feature["input_ids"]

        try:
            cls_index = input_ids.index(cls_id)
        except ValueError:
            cls_index = 0

        is_positive = (
            int(feature["start_positions"])
            != cls_index
            or int(feature["end_positions"])
            != cls_index
        )

        if is_positive:
            positive_features += 1
            positive_feature_counts[
                example_index
            ] += 1
        else:
            negative_features += 1

    return {
        "split": split_name,
        "examples": number_of_examples,
        "features": len(tokenized_dataset),
        "positive_features": positive_features,
        "negative_features": negative_features,
        "examples_with_positive_window": int(
            (positive_feature_counts > 0).sum()
        ),
        "examples_missing_positive_window": int(
            (positive_feature_counts == 0).sum()
        ),
        "mean_windows_per_example": float(
            total_feature_counts.mean()
        ),
        "median_windows_per_example": float(
            np.median(total_feature_counts)
        ),
        "max_windows_per_example": int(
            total_feature_counts.max()
        )
    }


article_feature_summary_df = pd.DataFrame([
    summarize_article_features(
        article_train_tokenized,
        len(article_train_qa_df),
        "train"
    ),
    summarize_article_features(
        article_val_tokenized,
        len(article_val_qa_df),
        "validation"
    )
])

print(
    "Tokenization runtime:",
    round(tokenization_runtime, 2),
    "seconds"
)

print("\nArticle-level feature summary:")
display(article_feature_summary_df)

print("\nTokenized training columns:")
print(article_train_tokenized.column_names)

Tokenizing article-level training data:   0%|          | 0/4546 [00:00<?, ? examples/s]

Tokenizing article-level validation data:   0%|          | 0/607 [00:00<?, ? examples/s]

Tokenization runtime: 51.46 seconds

Article-level feature summary:


,split,examples,features,positive_features,negative_features,examples_with_positive_window,examples_missing_positive_window,mean_windows_per_example,median_windows_per_example,max_windows_per_example
0,train,4546,17559,5502,12057,32,4514,3.862516,0.0,673
1,validation,607,2499,738,1761,32,575,4.116969,0.0,146



Tokenized training columns:
['input_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'example_index']


In [90]:
def tokenize_article_qa_batch_v2(
    examples,
    batch_indices
):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=ARTICLE_MAX_LENGTH,
        stride=ARTICLE_DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    start_positions = []
    end_positions = []
    example_indices = []
    cleaned_offset_mappings = []

    for feature_index, local_sample_index in enumerate(
        sample_mapping
    ):
        local_sample_index = int(
            local_sample_index
        )

        # Convert the index within this batch to the
        # true index in the full dataset.
        global_example_index = int(
            batch_indices[local_sample_index]
        )

        input_ids = tokenized["input_ids"][
            feature_index
        ]

        offsets = tokenized["offset_mapping"][
            feature_index
        ]

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        try:
            cls_index = input_ids.index(
                tokenizer.cls_token_id
            )
        except ValueError:
            cls_index = 0

        answer_start = int(
            examples["answer_start"][
                local_sample_index
            ]
        )

        answer_text = str(
            examples["answer_text"][
                local_sample_index
            ]
        )

        answer_end = (
            answer_start + len(answer_text)
        )

        context_token_indices = [
            token_index
            for token_index, sequence_id
            in enumerate(sequence_ids)
            if sequence_id == 1
        ]

        if not context_token_indices:
            start_positions.append(cls_index)
            end_positions.append(cls_index)

        else:
            context_start = context_token_indices[0]
            context_end = context_token_indices[-1]

            window_contains_answer = (
                offsets[context_start][0]
                <= answer_start
                and offsets[context_end][1]
                >= answer_end
            )

            if not window_contains_answer:
                start_positions.append(cls_index)
                end_positions.append(cls_index)

            else:
                token_start = context_start

                while (
                    token_start <= context_end
                    and offsets[token_start][0]
                    <= answer_start
                ):
                    token_start += 1

                token_start -= 1

                token_end = context_end

                while (
                    token_end >= context_start
                    and offsets[token_end][1]
                    >= answer_end
                ):
                    token_end -= 1

                token_end += 1

                start_positions.append(token_start)
                end_positions.append(token_end)

        cleaned_offsets = [
            offset if sequence_id == 1 else None
            for offset, sequence_id
            in zip(offsets, sequence_ids)
        ]

        cleaned_offset_mappings.append(
            cleaned_offsets
        )

        example_indices.append(
            global_example_index
        )

    tokenized["start_positions"] = (
        start_positions
    )

    tokenized["end_positions"] = (
        end_positions
    )

    tokenized["example_index"] = (
        example_indices
    )

    tokenized["offset_mapping"] = (
        cleaned_offset_mappings
    )

    return tokenized


tokenization_start = time.time()

article_train_tokenized = (
    article_train_dataset_raw.map(
        tokenize_article_qa_batch_v2,
        batched=True,
        with_indices=True,
        batch_size=32,
        remove_columns=(
            article_train_dataset_raw.column_names
        ),
        desc="Retokenizing article training data"
    )
)

article_val_tokenized = (
    article_val_dataset_raw.map(
        tokenize_article_qa_batch_v2,
        batched=True,
        with_indices=True,
        batch_size=32,
        remove_columns=(
            article_val_dataset_raw.column_names
        ),
        desc="Retokenizing article validation data"
    )
)

article_feature_summary_df = pd.DataFrame([
    summarize_article_features(
        article_train_tokenized,
        len(article_train_qa_df),
        "train"
    ),
    summarize_article_features(
        article_val_tokenized,
        len(article_val_qa_df),
        "validation"
    )
])

print(
    "Retokenization runtime:",
    round(time.time() - tokenization_start, 2),
    "seconds"
)

print("\nCorrected article-level feature summary:")
display(article_feature_summary_df)

print("\nGlobal example-index ranges:")
print(
    "Train:",
    min(article_train_tokenized["example_index"]),
    "to",
    max(article_train_tokenized["example_index"])
)

print(
    "Validation:",
    min(article_val_tokenized["example_index"]),
    "to",
    max(article_val_tokenized["example_index"])
)

Retokenizing article training data:   0%|          | 0/4546 [00:00<?, ? examples/s]

Retokenizing article validation data:   0%|          | 0/607 [00:00<?, ? examples/s]

Retokenization runtime: 62.12 seconds

Corrected article-level feature summary:


,split,examples,features,positive_features,negative_features,examples_with_positive_window,examples_missing_positive_window,mean_windows_per_example,median_windows_per_example,max_windows_per_example
0,train,4546,17559,5502,12057,4546,0,3.862516,3.0,139
1,validation,607,2499,738,1761,607,0,4.116969,3.0,52



Global example-index ranges:
Train: 0 to 4545
Validation: 0 to 606


In [91]:
from collections import defaultdict
import numpy as np
import pandas as pd

ARTICLE_MAX_NEGATIVE_WINDOWS = 8

features_by_example = defaultdict(list)

for feature_index in range(
    len(article_train_tokenized)
):
    feature = article_train_tokenized[
        feature_index
    ]

    example_index = int(
        feature["example_index"]
    )

    input_ids = feature["input_ids"]

    try:
        cls_index = input_ids.index(
            tokenizer.cls_token_id
        )
    except ValueError:
        cls_index = 0

    is_positive = (
        int(feature["start_positions"])
        != cls_index
        or int(feature["end_positions"])
        != cls_index
    )

    features_by_example[
        example_index
    ].append({
        "feature_index": feature_index,
        "is_positive": is_positive
    })


selected_feature_indices = []

for example_index in sorted(
    features_by_example
):
    example_features = features_by_example[
        example_index
    ]

    positive_features = [
        item
        for item in example_features
        if item["is_positive"]
    ]

    negative_features = [
        item
        for item in example_features
        if not item["is_positive"]
    ]

    # Always retain every answer-containing window.
    selected_feature_indices.extend(
        item["feature_index"]
        for item in positive_features
    )

    if (
        len(negative_features)
        <= ARTICLE_MAX_NEGATIVE_WINDOWS
    ):
        selected_negatives = negative_features

    else:
        positive_positions = [
            example_features.index(item)
            for item in positive_features
        ]

        # Keep half of the negative windows nearest to
        # an answer-containing window.
        nearest_count = (
            ARTICLE_MAX_NEGATIVE_WINDOWS // 2
        )

        negative_with_distance = []

        for negative_item in negative_features:
            negative_position = (
                example_features.index(
                    negative_item
                )
            )

            nearest_distance = min(
                abs(
                    negative_position
                    - positive_position
                )
                for positive_position
                in positive_positions
            )

            negative_with_distance.append(
                (
                    nearest_distance,
                    negative_position,
                    negative_item
                )
            )

        negative_with_distance.sort(
            key=lambda item: (
                item[0],
                item[1]
            )
        )

        selected_negatives = [
            item[2]
            for item in (
                negative_with_distance[
                    :nearest_count
                ]
            )
        ]

        selected_negative_indices = {
            item["feature_index"]
            for item in selected_negatives
        }

        remaining_negatives = [
            item
            for item in negative_features
            if item["feature_index"]
            not in selected_negative_indices
        ]

        remaining_slots = (
            ARTICLE_MAX_NEGATIVE_WINDOWS
            - len(selected_negatives)
        )

        # Fill the remaining slots with windows spread
        # across the complete article.
        spread_positions = np.linspace(
            0,
            len(remaining_negatives) - 1,
            remaining_slots,
            dtype=int
        )

        selected_negatives.extend(
            remaining_negatives[position]
            for position in spread_positions
        )

    selected_feature_indices.extend(
        item["feature_index"]
        for item in selected_negatives
    )


selected_feature_indices = sorted(
    set(selected_feature_indices)
)

article_train_tokenized_capped = (
    article_train_tokenized.select(
        selected_feature_indices
    )
)


def count_article_feature_labels(
    tokenized_dataset
):
    positive_count = 0
    negative_count = 0
    feature_counts_by_example = defaultdict(
        int
    )

    for feature in tokenized_dataset:
        input_ids = feature["input_ids"]

        try:
            cls_index = input_ids.index(
                tokenizer.cls_token_id
            )
        except ValueError:
            cls_index = 0

        is_positive = (
            int(feature["start_positions"])
            != cls_index
            or int(feature["end_positions"])
            != cls_index
        )

        if is_positive:
            positive_count += 1
        else:
            negative_count += 1

        feature_counts_by_example[
            int(feature["example_index"])
        ] += 1

    return {
        "features": len(tokenized_dataset),
        "positive_features": positive_count,
        "negative_features": negative_count,
        "examples": len(
            feature_counts_by_example
        ),
        "mean_features_per_example": float(
            np.mean(
                list(
                    feature_counts_by_example.values()
                )
            )
        ),
        "median_features_per_example": float(
            np.median(
                list(
                    feature_counts_by_example.values()
                )
            )
        ),
        "max_features_per_example": int(
            max(
                feature_counts_by_example.values()
            )
        )
    }


capping_summary_df = pd.DataFrame([
    {
        "dataset": "original",
        **count_article_feature_labels(
            article_train_tokenized
        )
    },
    {
        "dataset": "capped",
        **count_article_feature_labels(
            article_train_tokenized_capped
        )
    }
])

print(
    "Negative-window cap:",
    ARTICLE_MAX_NEGATIVE_WINDOWS
)

print("\nTraining-feature comparison:")
display(capping_summary_df)

original_positive_count = int(
    capping_summary_df.loc[
        capping_summary_df["dataset"]
        == "original",
        "positive_features"
    ].iloc[0]
)

capped_positive_count = int(
    capping_summary_df.loc[
        capping_summary_df["dataset"]
        == "capped",
        "positive_features"
    ].iloc[0]
)

print(
    "\nAll positive windows preserved:",
    original_positive_count
    == capped_positive_count
)

print(
    "All training examples preserved:",
    article_train_tokenized_capped[
        "example_index"
    ]
    and len(
        set(
            article_train_tokenized_capped[
                "example_index"
            ]
        )
    )
    == len(article_train_qa_df)
)

Negative-window cap: 8

Training-feature comparison:


,dataset,features,positive_features,negative_features,examples,mean_features_per_example,median_features_per_example,max_features_per_example
0,original,17559,5502,12057,4546,3.862516,3.0,139
1,capped,15995,5502,10493,4546,3.518478,3.0,10



All positive windows preserved: True
All training examples preserved: True


In [92]:
import gc
import inspect
import math
import os
import shutil
import torch

from transformers import (
    AutoModelForQuestionAnswering,
    Trainer,
    TrainingArguments,
    default_data_collator
)

PARAGRAPH_EPOCH1_MODEL_DIR = (
    "/content/drive/MyDrive/Task2_FinalShot/"
    "paragraph_qa_model"
)

ARTICLE_QA_LOCAL_DIR = (
    "/content/article_qa_training"
)

ARTICLE_QA_MODEL_DIR = (
    "/content/drive/MyDrive/Task2_FinalShot/"
    "article_qa_model"
)

# Remove epoch-2 model and inference objects from GPU memory.
for variable_name in [
    "qa_model",
    "second_epoch_trainer",
    "epoch2_validation_output",
    "epoch2_start_logits",
    "epoch2_end_logits",
    "epoch2_val_model_inputs"
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

assert os.path.exists(
    PARAGRAPH_EPOCH1_MODEL_DIR
), "Epoch-1 paragraph checkpoint was not found."

# Reload the successful epoch-1 checkpoint explicitly.
article_qa_model = (
    AutoModelForQuestionAnswering.from_pretrained(
        PARAGRAPH_EPOCH1_MODEL_DIR
    )
)

article_qa_model.to("cuda")

# Remove metadata that should not be passed into the model.
article_train_model_inputs = (
    article_train_tokenized_capped.remove_columns(
        ["offset_mapping", "example_index"]
    )
)

article_val_model_inputs = (
    article_val_tokenized.remove_columns(
        ["offset_mapping", "example_index"]
    )
)

if os.path.exists(ARTICLE_QA_LOCAL_DIR):
    shutil.rmtree(ARTICLE_QA_LOCAL_DIR)

os.makedirs(ARTICLE_QA_LOCAL_DIR, exist_ok=True)
os.makedirs(ARTICLE_QA_MODEL_DIR, exist_ok=True)

requested_article_kwargs = {
    "output_dir": ARTICLE_QA_LOCAL_DIR,
    "num_train_epochs": 1,
    "learning_rate": 8e-6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,

    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,

    "fp16": True,
    "bf16": False,

    "logging_strategy": "steps",
    "logging_steps": 50,

    "save_strategy": "no",
    "report_to": "none",

    "dataloader_num_workers": 2,
    "remove_unused_columns": True,
    "seed": SEED
}

training_signature = inspect.signature(
    TrainingArguments.__init__
)

supported_parameters = set(
    training_signature.parameters
)

if "eval_strategy" in supported_parameters:
    requested_article_kwargs["eval_strategy"] = "no"
elif "evaluation_strategy" in supported_parameters:
    requested_article_kwargs["evaluation_strategy"] = "no"

article_training_kwargs = {
    name: value
    for name, value in requested_article_kwargs.items()
    if name in supported_parameters
}

article_training_args = TrainingArguments(
    **article_training_kwargs
)

article_trainer_kwargs = {
    "model": article_qa_model,
    "args": article_training_args,
    "train_dataset": article_train_model_inputs,
    "data_collator": default_data_collator
}

trainer_signature = inspect.signature(
    Trainer.__init__
)

if "processing_class" in trainer_signature.parameters:
    article_trainer_kwargs[
        "processing_class"
    ] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    article_trainer_kwargs[
        "tokenizer"
    ] = tokenizer

article_qa_trainer = Trainer(
    **article_trainer_kwargs
)

effective_batch_size = (
    article_training_args.per_device_train_batch_size
    * article_training_args.gradient_accumulation_steps
)

estimated_optimizer_steps = math.ceil(
    len(article_train_model_inputs)
    / effective_batch_size
)

print(
    "Loaded checkpoint:",
    PARAGRAPH_EPOCH1_MODEL_DIR
)
print(
    "Article training features:",
    len(article_train_model_inputs)
)
print(
    "Article validation features:",
    len(article_val_model_inputs)
)
print(
    "Learning rate:",
    article_training_args.learning_rate
)
print(
    "Effective batch size:",
    effective_batch_size
)
print(
    "Estimated optimizer steps:",
    estimated_optimizer_steps
)
print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)
print(
    "Article Trainer created:",
    article_qa_trainer is not None
)
print(
    "Future model location:",
    ARTICLE_QA_MODEL_DIR
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loaded checkpoint: /content/drive/MyDrive/Task2_FinalShot/paragraph_qa_model
Article training features: 15995
Article validation features: 2499
Learning rate: 8e-06
Effective batch size: 16
Estimated optimizer steps: 1000
GPU memory allocated: 2.8 GB
Article Trainer created: True
Future model location: /content/drive/MyDrive/Task2_FinalShot/article_qa_model


In [93]:
import time
import torch

torch.cuda.empty_cache()

article_training_start = time.time()

article_training_result = article_qa_trainer.train()

article_training_minutes = (
    time.time() - article_training_start
) / 60

print("\nArticle-level QA training completed.")
print(
    "Elapsed time:",
    round(article_training_minutes, 2),
    "minutes"
)

print("\nTraining metrics:")
for metric_name, metric_value in (
    article_training_result.metrics.items()
):
    print(
        f"{metric_name}: {metric_value}"
    )

Step,Training Loss
50,1.718224
100,1.477898
150,1.357287
200,1.294594
250,1.302611
300,1.212785
350,1.244283
400,1.305222
450,1.334569
500,1.271539



Article-level QA training completed.
Elapsed time: 5.75 minutes

Training metrics:
train_runtime: 344.2813
train_samples_per_second: 46.459
train_steps_per_second: 2.905
total_flos: 3134581217994240.0
train_loss: 1.2854905548095703
epoch: 1.0


## result looks healthy, ave loss dropped and no late explosion like before

In [94]:
import json
import os

article_qa_trainer.save_model(
    ARTICLE_QA_MODEL_DIR
)

tokenizer.save_pretrained(
    ARTICLE_QA_MODEL_DIR
)

article_metrics_path = os.path.join(
    ARTICLE_QA_MODEL_DIR,
    "training_metrics.json"
)

with open(
    article_metrics_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            key: float(value)
            for key, value
            in article_training_result.metrics.items()
        },
        file,
        indent=2
    )

weights_saved = (
    os.path.exists(
        os.path.join(
            ARTICLE_QA_MODEL_DIR,
            "model.safetensors"
        )
    )
    or os.path.exists(
        os.path.join(
            ARTICLE_QA_MODEL_DIR,
            "pytorch_model.bin"
        )
    )
)

print(
    "Article model directory exists:",
    os.path.exists(ARTICLE_QA_MODEL_DIR)
)
print("Article model weights saved:", weights_saved)
print(
    "Tokenizer saved:",
    os.path.exists(
        os.path.join(
            ARTICLE_QA_MODEL_DIR,
            "tokenizer_config.json"
        )
    )
)
print(
    "Training metrics saved:",
    os.path.exists(article_metrics_path)
)
print(
    "Article model location:",
    ARTICLE_QA_MODEL_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Article model directory exists: True
Article model weights saved: True
Tokenizer saved: True
Training metrics saved: True
Article model location: /content/drive/MyDrive/Task2_FinalShot/article_qa_model


In [95]:
import time
import torch

torch.cuda.empty_cache()

article_validation_start = time.time()

article_validation_output = article_qa_trainer.predict(
    article_val_model_inputs
)

(
    article_val_start_logits,
    article_val_end_logits
) = article_validation_output.predictions

article_validation_seconds = (
    time.time() - article_validation_start
)

print("Article validation inference completed.")
print(
    "Start-logit shape:",
    article_val_start_logits.shape
)
print(
    "End-logit shape:",
    article_val_end_logits.shape
)
print(
    "Runtime:",
    round(article_validation_seconds, 2),
    "seconds"
)
print(
    "Features per second:",
    article_validation_output.metrics.get(
        "test_samples_per_second"
    )
)

Article validation inference completed.
Start-logit shape: (2499, 384)
End-logit shape: (2499, 384)
Runtime: 14.55 seconds
Features per second: 171.877


In [96]:
import re
import time
import numpy as np
import pandas as pd

article_decode_start = time.time()

# Multiple gold answers can create duplicate validation examples for
# the same article. At inference, their inputs are identical, so use
# the first example for each article row.
first_example_by_row = (
    article_val_qa_df
    .reset_index()
    .groupby("row_number")["index"]
    .first()
    .astype(int)
    .to_dict()
)

selected_example_indices = set(
    first_example_by_row.values()
)

window_predictions_by_row = {}

for feature_index in range(
    len(article_val_tokenized)
):
    feature = article_val_tokenized[
        feature_index
    ]

    example_index = int(
        feature["example_index"]
    )

    if example_index not in selected_example_indices:
        continue

    example_row = article_val_qa_df.iloc[
        example_index
    ]

    row_number = int(
        example_row["row_number"]
    )

    context = str(
        example_row["context"]
    )

    input_ids = feature["input_ids"]
    offsets = feature["offset_mapping"]

    start_logits = article_val_start_logits[
        feature_index
    ]

    end_logits = article_val_end_logits[
        feature_index
    ]

    try:
        cls_index = input_ids.index(
            tokenizer.cls_token_id
        )
    except ValueError:
        cls_index = 0

    null_score = float(
        start_logits[cls_index]
        + end_logits[cls_index]
    )

    valid_mask = np.array([
        offset is not None
        and len(offset) == 2
        and int(offset[1]) > int(offset[0])
        for offset in offsets
    ])

    valid_start_positions = np.flatnonzero(
        valid_mask
    )

    best_span_score = -np.inf
    best_start_position = None
    best_end_position = None

    for start_position in valid_start_positions:
        maximum_end_position = min(
            start_position
            + MAX_ANSWER_LENGTH
            - 1,
            len(offsets) - 1
        )

        allowed_end_mask = valid_mask[
            start_position:
            maximum_end_position + 1
        ]

        if not allowed_end_mask.any():
            continue

        possible_end_logits = end_logits[
            start_position:
            maximum_end_position + 1
        ].copy()

        possible_end_logits[
            ~allowed_end_mask
        ] = -np.inf

        relative_end_position = int(
            np.argmax(possible_end_logits)
        )

        end_position = (
            start_position
            + relative_end_position
        )

        span_score = float(
            start_logits[start_position]
            + end_logits[end_position]
        )

        if span_score > best_span_score:
            best_span_score = span_score
            best_start_position = int(
                start_position
            )
            best_end_position = int(
                end_position
            )

    if (
        best_start_position is not None
        and best_end_position is not None
    ):
        character_start = int(
            offsets[best_start_position][0]
        )

        character_end = int(
            offsets[best_end_position][1]
        )

        predicted_text = (
            normalize_prediction_text(
                context[
                    character_start:
                    character_end
                ]
            )
        )
    else:
        predicted_text = ""

    answerability_margin = float(
        best_span_score - null_score
    )

    window_predictions_by_row.setdefault(
        row_number,
        []
    ).append({
        "feature_index": feature_index,
        "context": context,
        "predicted_text": predicted_text,
        "best_span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": (
            answerability_margin
        )
    })


def remove_article_prefix(text):
    text = normalize_prediction_text(text)

    text = re.sub(
        r"^(?:Title|Paragraph\s+\d+):\s*",
        "",
        text
    )

    return text.strip()


article_validation_records = []

for row_number in range(len(val_df)):
    window_records = (
        window_predictions_by_row.get(
            row_number,
            []
        )
    )

    ranked_windows = (
        pd.DataFrame(window_records)
        .sort_values(
            "answerability_margin",
            ascending=False
        )
        .reset_index(drop=True)
        if window_records
        else pd.DataFrame()
    )

    unique_span_records = []
    seen_spans = set()

    if len(ranked_windows) > 0:
        for _, candidate in ranked_windows.iterrows():
            prediction = remove_article_prefix(
                candidate["predicted_text"]
            )

            normalized_prediction = (
                prediction.lower().strip()
            )

            if (
                prediction
                and normalized_prediction
                not in seen_spans
            ):
                unique_span_records.append({
                    "predicted_text": prediction,
                    "context": candidate["context"],
                    "answerability_margin": float(
                        candidate[
                            "answerability_margin"
                        ]
                    )
                })

                seen_spans.add(
                    normalized_prediction
                )

    unique_spans_df = pd.DataFrame(
        unique_span_records
    )

    if len(unique_spans_df) > 0:
        top1 = join_unique_predictions(
            unique_spans_df,
            maximum_answers=1
        )

        top2 = join_unique_predictions(
            unique_spans_df,
            maximum_answers=2
        )

        top3 = join_unique_predictions(
            unique_spans_df,
            maximum_answers=3
        )

        best_margin = float(
            unique_spans_df.iloc[0][
                "answerability_margin"
            ]
        )

        close_candidates = unique_spans_df[
            unique_spans_df[
                "answerability_margin"
            ] >= best_margin - 2.0
        ]

        close3 = join_unique_predictions(
            close_candidates,
            maximum_answers=3
        )
    else:
        top1 = ""
        top2 = ""
        top3 = ""
        close3 = ""

    unique_sentences = []
    seen_sentences = set()

    for candidate in unique_span_records:
        expanded_sentence = (
            expand_prediction_to_sentence(
                candidate["context"],
                candidate["predicted_text"]
            )
        )

        expanded_sentence = remove_article_prefix(
            expanded_sentence
        )

        normalized_sentence = (
            expanded_sentence.lower().strip()
        )

        if (
            expanded_sentence
            and normalized_sentence
            not in seen_sentences
        ):
            unique_sentences.append(
                expanded_sentence
            )

            seen_sentences.add(
                normalized_sentence
            )

        if len(unique_sentences) >= 3:
            break

    sentence_top1 = " ".join(
        unique_sentences[:1]
    )

    sentence_top2 = " ".join(
        unique_sentences[:2]
    )

    sentence_top3 = " ".join(
        unique_sentences[:3]
    )

    article_validation_records.append({
        "row_number": row_number,
        "article_top1": top1,
        "article_top2": top2,
        "article_top3": top3,
        "article_close3": close3,
        "article_sentence_top1": sentence_top1,
        "article_sentence_top2": sentence_top2,
        "article_sentence_top3": sentence_top3,
        "article_window_count": len(
            window_records
        )
    })


article_validation_predictions_df = (
    pd.DataFrame(article_validation_records)
    .merge(
        validation_predictions_df[
            [
                "row_number",
                "spoiler_type",
                "gold_text"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)


article_strategy_columns = [
    "article_top1",
    "article_top2",
    "article_top3",
    "article_close3",
    "article_sentence_top1",
    "article_sentence_top2",
    "article_sentence_top3"
]

article_strategy_scores = []

for strategy_column in article_strategy_columns:
    row_scores = []

    for _, row in (
        article_validation_predictions_df.iterrows()
    ):
        prediction_tokens = str(
            row[strategy_column]
        ).split()

        gold_tokens = str(
            row["gold_text"]
        ).split()

        score = (
            meteor_score(
                [gold_tokens],
                prediction_tokens
            )
            if prediction_tokens
            else 0.0
        )

        row_scores.append(score)

    article_validation_predictions_df[
        f"{strategy_column}_meteor"
    ] = row_scores

    article_strategy_scores.append({
        "strategy": strategy_column,
        "mean_meteor": float(
            np.mean(row_scores)
        ),
        "median_meteor": float(
            np.median(row_scores)
        ),
        "empty_predictions": int(
            article_validation_predictions_df[
                strategy_column
            ].fillna("").str.strip().eq("").sum()
        )
    })


article_strategy_scores_df = (
    pd.DataFrame(article_strategy_scores)
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Decoded validation articles:",
    len(article_validation_predictions_df)
)

print(
    "Decode runtime:",
    round(
        time.time() - article_decode_start,
        2
    ),
    "seconds"
)

print("\nArticle-model validation strategies:")
display(article_strategy_scores_df)

print("\nBest strategy by gold spoiler type:")

type_strategy_records = []

for spoiler_type, group in (
    article_validation_predictions_df.groupby(
        "spoiler_type"
    )
):
    for strategy_column in article_strategy_columns:
        type_strategy_records.append({
            "spoiler_type": spoiler_type,
            "strategy": strategy_column,
            "mean_meteor": float(
                group[
                    f"{strategy_column}_meteor"
                ].mean()
            )
        })

article_type_strategy_df = (
    pd.DataFrame(type_strategy_records)
    .sort_values(
        ["spoiler_type", "mean_meteor"],
        ascending=[True, False]
    )
)

display(
    article_type_strategy_df.groupby(
        "spoiler_type"
    ).head(5)
)

print("\nReference paragraph-pipeline scores:")
print("Deployable routed validation METEOR: 0.393997")
print("Gold-type hybrid validation METEOR: 0.405952")

Decoded validation articles: 400
Decode runtime: 24.1 seconds

Article-model validation strategies:


,strategy,mean_meteor,median_meteor,empty_predictions
0,article_top3,0.366602,0.297202,3
1,article_close3,0.363934,0.238337,3
2,article_top2,0.363475,0.292517,3
3,article_top1,0.348478,0.218850,3
4,article_sentence_top3,0.319048,0.237719,3
5,article_sentence_top2,0.317416,0.235629,3
6,article_sentence_top1,0.304844,0.182983,3



Best strategy by gold spoiler type:


,spoiler_type,strategy,mean_meteor
6,multi,article_sentence_top3,0.367401
2,multi,article_top3,0.342182
5,multi,article_sentence_top2,0.339081
1,multi,article_top2,0.291638
4,multi,article_sentence_top1,0.240334
13,passage,article_sentence_top3,0.415798
12,passage,article_sentence_top2,0.413479
11,passage,article_sentence_top1,0.397618
9,passage,article_top3,0.308176
8,passage,article_top2,0.306286



Reference paragraph-pipeline scores:
Deployable routed validation METEOR: 0.393997
Gold-type hybrid validation METEOR: 0.405952


In [97]:
import itertools
import numpy as np
import pandas as pd


# Merge article-model outputs with the paragraph-model outputs,
# type predictions, probabilities, and gold references.
article_paragraph_comparison_df = (
    article_validation_predictions_df[
        [
            "row_number",
            "spoiler_type",
            "gold_text",
            "article_top1",
            "article_top2",
            "article_top3",
            "article_close3",
            "article_sentence_top1",
            "article_sentence_top2",
            "article_sentence_top3"
        ]
    ]
    .merge(
        validation_predictions_df[
            [
                "row_number",
                "roberta_predicted_type",
                "prob_phrase",
                "prob_passage",
                "prob_multi",
                "top1",
                "top2",
                "top3",
                "close3",
                "sentence_top1",
                "sentence_top2",
                "sentence_top3"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)


def calculate_prediction_meteor(
    dataframe,
    prediction_column
):
    scores = []

    for _, row in dataframe.iterrows():
        prediction = normalize_prediction_text(
            row[prediction_column]
        )

        gold_text = str(row["gold_text"])

        prediction_tokens = prediction.split()
        gold_tokens = gold_text.split()

        score = (
            meteor_score(
                [gold_tokens],
                prediction_tokens
            )
            if prediction_tokens
            else 0.0
        )

        scores.append(score)

    return float(np.mean(scores))


def paragraph_prediction_for_row(row):
    """
    Existing deployable paragraph routing that produced
    approximately 0.393997 validation METEOR.
    """
    predicted_type = str(
        row["roberta_predicted_type"]
    )

    if predicted_type == "phrase":
        if float(row["prob_phrase"]) >= 0.35:
            return normalize_prediction_text(
                row["close3"]
            )
        return normalize_prediction_text(
            row["top3"]
        )

    if predicted_type == "passage":
        if float(row["prob_passage"]) >= 0.50:
            return normalize_prediction_text(
                row["sentence_top3"]
            )
        return normalize_prediction_text(
            row["top3"]
        )

    return normalize_prediction_text(
        row["top3"]
    )


def article_prediction_for_row(row):
    """
    Article strategies selected from the validation results:
    phrase  -> close3
    passage -> sentence_top3
    multi   -> sentence_top3
    """
    predicted_type = str(
        row["roberta_predicted_type"]
    )

    if predicted_type == "phrase":
        prediction = normalize_prediction_text(
            row["article_close3"]
        )

    elif predicted_type == "passage":
        prediction = normalize_prediction_text(
            row["article_sentence_top3"]
        )

    else:
        prediction = normalize_prediction_text(
            row["article_sentence_top3"]
        )

    # Defensive fallback for the three empty article predictions.
    if not prediction:
        prediction = paragraph_prediction_for_row(row)

    return prediction


def gold_type_article_prediction(row):
    gold_type = str(row["spoiler_type"])

    if gold_type == "phrase":
        return normalize_prediction_text(
            row["article_close3"]
        )

    if gold_type == "passage":
        return normalize_prediction_text(
            row["article_sentence_top3"]
        )

    return normalize_prediction_text(
        row["article_sentence_top3"]
    )


def gold_type_paragraph_prediction(row):
    gold_type = str(row["spoiler_type"])

    if gold_type == "phrase":
        return normalize_prediction_text(
            row["close3"]
        )

    if gold_type == "passage":
        return normalize_prediction_text(
            row["sentence_top3"]
        )

    return normalize_prediction_text(
        row["top3"]
    )


article_paragraph_comparison_df[
    "paragraph_deployable"
] = article_paragraph_comparison_df.apply(
    paragraph_prediction_for_row,
    axis=1
)

article_paragraph_comparison_df[
    "article_deployable"
] = article_paragraph_comparison_df.apply(
    article_prediction_for_row,
    axis=1
)

article_paragraph_comparison_df[
    "article_gold_type"
] = article_paragraph_comparison_df.apply(
    gold_type_article_prediction,
    axis=1
)

article_paragraph_comparison_df[
    "paragraph_gold_type"
] = article_paragraph_comparison_df.apply(
    gold_type_paragraph_prediction,
    axis=1
)


baseline_results = []

for prediction_column in [
    "paragraph_deployable",
    "article_deployable",
    "paragraph_gold_type",
    "article_gold_type"
]:
    baseline_results.append({
        "strategy": prediction_column,
        "mean_meteor": calculate_prediction_meteor(
            article_paragraph_comparison_df,
            prediction_column
        )
    })

baseline_results_df = pd.DataFrame(
    baseline_results
).sort_values(
    "mean_meteor",
    ascending=False
)


# Test every article/paragraph source choice by predicted type.
# Example A-P-P means:
# article for predicted phrase,
# paragraph for predicted passage,
# paragraph for predicted multi.
combination_results = []

for (
    phrase_source,
    passage_source,
    multi_source
) in itertools.product(
    ["article", "paragraph"],
    repeat=3
):
    strategy_name = (
        f"{phrase_source[0].upper()}-"
        f"{passage_source[0].upper()}-"
        f"{multi_source[0].upper()}"
    )

    prediction_column = (
        "combination_"
        + strategy_name.replace("-", "_")
    )

    predictions = []

    for _, row in (
        article_paragraph_comparison_df.iterrows()
    ):
        predicted_type = str(
            row["roberta_predicted_type"]
        )

        source_by_type = {
            "phrase": phrase_source,
            "passage": passage_source,
            "multi": multi_source
        }

        selected_source = source_by_type[
            predicted_type
        ]

        if selected_source == "article":
            prediction = article_prediction_for_row(
                row
            )
        else:
            prediction = paragraph_prediction_for_row(
                row
            )

        predictions.append(prediction)

    article_paragraph_comparison_df[
        prediction_column
    ] = predictions

    combination_results.append({
        "strategy": strategy_name,
        "phrase_source": phrase_source,
        "passage_source": passage_source,
        "multi_source": multi_source,
        "mean_meteor": calculate_prediction_meteor(
            article_paragraph_comparison_df,
            prediction_column
        )
    })


combination_results_df = (
    pd.DataFrame(combination_results)
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)


print("Overall baseline comparisons:")
display(baseline_results_df)

print("\nArticle/paragraph choices by predicted type:")
display(combination_results_df)

print("\nType-classifier validation distribution:")
print(
    article_paragraph_comparison_df[
        "roberta_predicted_type"
    ].value_counts()
)

print("\nGold-type validation distribution:")
print(
    article_paragraph_comparison_df[
        "spoiler_type"
    ].value_counts()
)

Overall baseline comparisons:


,strategy,mean_meteor
3,article_gold_type,0.442256
2,paragraph_gold_type,0.405952
1,article_deployable,0.404351
0,paragraph_deployable,0.393997



Article/paragraph choices by predicted type:


,strategy,phrase_source,passage_source,multi_source,mean_meteor
0,P-A-A,paragraph,article,article,0.405470
1,A-A-A,article,article,article,0.404351
2,P-P-A,paragraph,paragraph,article,0.401085
3,A-P-A,article,paragraph,article,0.399965
4,P-A-P,paragraph,article,paragraph,0.398383
5,A-A-P,article,article,paragraph,0.397263
6,P-P-P,paragraph,paragraph,paragraph,0.393997
7,A-P-P,article,paragraph,paragraph,0.392877



Type-classifier validation distribution:
roberta_predicted_type
passage    168
phrase     149
multi       83
Name: count, dtype: int64

Gold-type validation distribution:
spoiler_type
phrase     162
passage    154
multi       84
Name: count, dtype: int64


In [100]:
import itertools
import numpy as np
import pandas as pd

confidence_thresholds = [
    0.00,   # always use article model
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.01    # always use paragraph model
]


def prediction_meteor_score(gold_text, prediction):
    prediction = normalize_prediction_text(prediction)

    prediction_tokens = prediction.split()
    gold_tokens = str(gold_text).split()

    if not prediction_tokens:
        return 0.0

    return meteor_score(
        [gold_tokens],
        prediction_tokens
    )


confidence_routing_results = []

for (
    phrase_threshold,
    passage_threshold,
    multi_threshold
) in itertools.product(
    confidence_thresholds,
    repeat=3
):
    row_scores = []

    article_usage = {
        "phrase": 0,
        "passage": 0,
        "multi": 0
    }

    for _, row in (
        article_paragraph_comparison_df.iterrows()
    ):
        predicted_type = str(
            row["roberta_predicted_type"]
        )

        probability_by_type = {
            "phrase": float(row["prob_phrase"]),
            "passage": float(row["prob_passage"]),
            "multi": float(row["prob_multi"])
        }

        threshold_by_type = {
            "phrase": phrase_threshold,
            "passage": passage_threshold,
            "multi": multi_threshold
        }

        use_article = (
            probability_by_type[predicted_type]
            >= threshold_by_type[predicted_type]
        )

        if use_article:
            prediction = article_prediction_for_row(row)
            article_usage[predicted_type] += 1
        else:
            prediction = paragraph_prediction_for_row(row)

        row_scores.append(
            prediction_meteor_score(
                row["gold_text"],
                prediction
            )
        )

    confidence_routing_results.append({
        "phrase_threshold": phrase_threshold,
        "passage_threshold": passage_threshold,
        "multi_threshold": multi_threshold,
        "mean_meteor": float(np.mean(row_scores)),
        "median_meteor": float(np.median(row_scores)),
        "article_phrase_rows": article_usage["phrase"],
        "article_passage_rows": article_usage["passage"],
        "article_multi_rows": article_usage["multi"],
        "total_article_rows": sum(
            article_usage.values()
        )
    })


confidence_routing_results_df = (
    pd.DataFrame(confidence_routing_results)
    .sort_values(
        [
            "mean_meteor",
            "total_article_rows"
        ],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("Previous paragraph deployable score:")
print(0.393997)

print("\nBest simple P-A-A hybrid:")
print(0.405470)

print("\nBest confidence-routed hybrids:")
display(
    confidence_routing_results_df.head(20)
)

Previous paragraph deployable score:
0.393997

Best simple P-A-A hybrid:
0.40547

Best confidence-routed hybrids:


,phrase_threshold,passage_threshold,multi_threshold,mean_meteor,median_meteor,article_phrase_rows,article_passage_rows,article_multi_rows,total_article_rows
0,0.8,0.5,0.8,0.414097,0.359556,85,155,54,294
1,0.8,0.5,0.9,0.412987,0.357143,85,155,51,291
2,0.8,0.5,0.0,0.412767,0.379075,85,155,83,323
3,0.8,0.5,0.3,0.412767,0.379075,85,155,83,323
4,0.8,0.5,0.4,0.412717,0.379075,85,155,81,321
5,0.8,0.4,0.8,0.412626,0.357143,85,167,54,306
6,0.6,0.5,0.8,0.412210,0.359556,134,155,54,343
7,0.8,0.0,0.8,0.412002,0.355741,85,168,54,307
8,0.8,0.3,0.8,0.412002,0.355741,85,168,54,307
9,0.8,0.5,0.6,0.411853,0.359556,85,155,68,308


In [101]:
from datasets import Dataset
import pandas as pd
import numpy as np
import time

BEST_PHRASE_THRESHOLD = 0.80
BEST_PASSAGE_THRESHOLD = 0.50
BEST_MULTI_THRESHOLD = 0.80

# Reuse exactly the same questions used by the paragraph-QA test data.
test_questions_by_row = (
    test_candidate_df
    .groupby("row_number")["question"]
    .first()
    .to_dict()
)

article_test_records = []

for row_number in range(len(test_df)):
    article_row = test_df.iloc[row_number]

    combined_context, _ = (
        build_combined_article_context_v2(
            article_row=article_row,
            source_overrides={}
        )
    )

    article_test_records.append({
        "row_number": row_number,
        "id": article_row["id"],
        "question": test_questions_by_row[
            row_number
        ],
        "context": combined_context
    })

article_test_df = pd.DataFrame(
    article_test_records
)

article_test_dataset_raw = Dataset.from_pandas(
    article_test_df,
    preserve_index=False
)


def tokenize_article_test_batch(
    examples,
    batch_indices
):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=ARTICLE_MAX_LENGTH,
        stride=ARTICLE_DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop(
        "overflow_to_sample_mapping"
    )

    example_indices = []
    cleaned_offset_mappings = []

    for feature_index, local_sample_index in enumerate(
        sample_mapping
    ):
        local_sample_index = int(
            local_sample_index
        )

        global_example_index = int(
            batch_indices[local_sample_index]
        )

        sequence_ids = tokenized.sequence_ids(
            feature_index
        )

        offsets = tokenized["offset_mapping"][
            feature_index
        ]

        cleaned_offsets = [
            offset if sequence_id == 1 else None
            for offset, sequence_id
            in zip(offsets, sequence_ids)
        ]

        example_indices.append(
            global_example_index
        )

        cleaned_offset_mappings.append(
            cleaned_offsets
        )

    tokenized["example_index"] = (
        example_indices
    )

    tokenized["offset_mapping"] = (
        cleaned_offset_mappings
    )

    return tokenized


test_tokenization_start = time.time()

article_test_tokenized = (
    article_test_dataset_raw.map(
        tokenize_article_test_batch,
        batched=True,
        with_indices=True,
        batch_size=32,
        remove_columns=(
            article_test_dataset_raw.column_names
        ),
        desc="Tokenizing article-level test data"
    )
)

test_feature_counts = pd.Series(
    article_test_tokenized[
        "example_index"
    ]
).value_counts()

print(
    "Test tokenization runtime:",
    round(
        time.time() - test_tokenization_start,
        2
    ),
    "seconds"
)

print("Test articles:", len(article_test_df))
print(
    "Tokenized test features:",
    len(article_test_tokenized)
)
print(
    "Articles represented:",
    test_feature_counts.size
)

print("\nWindows per test article:")
print(test_feature_counts.describe())

print("\nGlobal example-index range:")
print(
    min(article_test_tokenized["example_index"]),
    "to",
    max(article_test_tokenized["example_index"])
)

print("\nEmpty article contexts:")
print(
    int(
        article_test_df["context"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

Tokenizing article-level test data:   0%|          | 0/400 [00:00<?, ? examples/s]

Test tokenization runtime: 6.21 seconds
Test articles: 400
Tokenized test features: 1330
Articles represented: 400

Windows per test article:
count    400.00000
mean       3.32500
std        3.28502
min        1.00000
25%        1.00000
50%        2.00000
75%        4.00000
max       29.00000
Name: count, dtype: float64

Global example-index range:
0 to 399

Empty article contexts:
0


In [102]:
import time
import torch

# Remove metadata columns that the QA model does not accept.
article_test_model_inputs = (
    article_test_tokenized.remove_columns(
        ["offset_mapping", "example_index"]
    )
)

torch.cuda.empty_cache()

article_test_inference_start = time.time()

article_test_output = article_qa_trainer.predict(
    article_test_model_inputs
)

(
    article_test_start_logits,
    article_test_end_logits
) = article_test_output.predictions

article_test_inference_seconds = (
    time.time() - article_test_inference_start
)

print("Article test inference completed.")
print(
    "Start-logit shape:",
    article_test_start_logits.shape
)
print(
    "End-logit shape:",
    article_test_end_logits.shape
)
print(
    "Runtime:",
    round(article_test_inference_seconds, 2),
    "seconds"
)
print(
    "Features per second:",
    article_test_output.metrics.get(
        "test_samples_per_second"
    )
)
print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

Article test inference completed.
Start-logit shape: (1330, 384)
End-logit shape: (1330, 384)
Runtime: 9.03 seconds
Features per second: 147.435
GPU memory allocated: 3.73 GB


## decode the article windows and apply the validation selected routing, phrase if prob >= 0.8; passge if >=0.5; multi is >=0.8

In [104]:
import re
import time
import numpy as np
import pandas as pd

decode_start = time.time()

ARTICLE_PHRASE_THRESHOLD = 0.80
ARTICLE_PASSAGE_THRESHOLD = 0.50
ARTICLE_MULTI_THRESHOLD = 0.80


def strip_article_label(text):
    text = normalize_prediction_text(text)

    text = re.sub(
        r"^(?:Title|Paragraph\s+\d+):\s*",
        "",
        text
    )

    return text.strip()

# 1. Decode the best span from every article window

article_test_windows_by_row = {}

for feature_index in range(len(article_test_tokenized)):
    feature = article_test_tokenized[feature_index]

    example_index = int(feature["example_index"])
    row_number = int(
        article_test_df.iloc[example_index]["row_number"]
    )

    context = str(
        article_test_df.iloc[example_index]["context"]
    )

    input_ids = feature["input_ids"]
    offsets = feature["offset_mapping"]

    start_logits = article_test_start_logits[
        feature_index
    ]
    end_logits = article_test_end_logits[
        feature_index
    ]

    try:
        cls_index = input_ids.index(
            tokenizer.cls_token_id
        )
    except ValueError:
        cls_index = 0

    null_score = float(
        start_logits[cls_index]
        + end_logits[cls_index]
    )

    valid_mask = np.array([
        offset is not None
        and len(offset) == 2
        and int(offset[1]) > int(offset[0])
        for offset in offsets
    ])

    valid_start_positions = np.flatnonzero(
        valid_mask
    )

    best_span_score = -np.inf
    best_start_position = None
    best_end_position = None

    for start_position in valid_start_positions:
        maximum_end_position = min(
            start_position
            + MAX_ANSWER_LENGTH
            - 1,
            len(offsets) - 1
        )

        allowed_end_mask = valid_mask[
            start_position:
            maximum_end_position + 1
        ]

        if not allowed_end_mask.any():
            continue

        possible_end_logits = end_logits[
            start_position:
            maximum_end_position + 1
        ].copy()

        possible_end_logits[
            ~allowed_end_mask
        ] = -np.inf

        relative_end_position = int(
            np.argmax(possible_end_logits)
        )

        end_position = (
            start_position
            + relative_end_position
        )

        span_score = float(
            start_logits[start_position]
            + end_logits[end_position]
        )

        if span_score > best_span_score:
            best_span_score = span_score
            best_start_position = int(
                start_position
            )
            best_end_position = int(
                end_position
            )

    if (
        best_start_position is not None
        and best_end_position is not None
    ):
        character_start = int(
            offsets[best_start_position][0]
        )

        character_end = int(
            offsets[best_end_position][1]
        )

        predicted_text = strip_article_label(
            context[
                character_start:
                character_end
            ]
        )
    else:
        predicted_text = ""

    answerability_margin = float(
        best_span_score - null_score
    )

    article_test_windows_by_row.setdefault(
        row_number,
        []
    ).append({
        "feature_index": feature_index,
        "context": context,
        "predicted_text": predicted_text,
        "best_span_score": best_span_score,
        "null_score": null_score,
        "answerability_margin": answerability_margin
    })

# 2. Create article-level decoding strategies

article_test_prediction_records = []

for row_number in range(len(test_df)):
    window_records = article_test_windows_by_row.get(
        row_number,
        []
    )

    ranked_windows = sorted(
        window_records,
        key=lambda record: record[
            "answerability_margin"
        ],
        reverse=True
    )

    unique_span_records = []
    seen_spans = set()

    for candidate in ranked_windows:
        prediction = strip_article_label(
            candidate["predicted_text"]
        )

        normalized_prediction = prediction.lower()

        if (
            prediction
            and normalized_prediction not in seen_spans
        ):
            unique_span_records.append({
                "predicted_text": prediction,
                "context": candidate["context"],
                "answerability_margin": candidate[
                    "answerability_margin"
                ]
            })

            seen_spans.add(normalized_prediction)

    unique_spans_df = pd.DataFrame(
        unique_span_records
    )

    if len(unique_spans_df) > 0:
        article_top1 = join_unique_predictions(
            unique_spans_df,
            maximum_answers=1
        )

        article_top2 = join_unique_predictions(
            unique_spans_df,
            maximum_answers=2
        )

        article_top3 = join_unique_predictions(
            unique_spans_df,
            maximum_answers=3
        )

        best_margin = float(
            unique_spans_df.iloc[0][
                "answerability_margin"
            ]
        )

        close_candidates = unique_spans_df[
            unique_spans_df[
                "answerability_margin"
            ] >= best_margin - 2.0
        ]

        article_close3 = join_unique_predictions(
            close_candidates,
            maximum_answers=3
        )
    else:
        article_top1 = ""
        article_top2 = ""
        article_top3 = ""
        article_close3 = ""

    expanded_sentences = []
    seen_sentences = set()

    for candidate in unique_span_records:
        sentence = expand_prediction_to_sentence(
            candidate["context"],
            candidate["predicted_text"]
        )

        sentence = strip_article_label(sentence)
        normalized_sentence = sentence.lower()

        if (
            sentence
            and normalized_sentence
            not in seen_sentences
        ):
            expanded_sentences.append(sentence)
            seen_sentences.add(normalized_sentence)

        if len(expanded_sentences) >= 3:
            break

    article_sentence_top1 = " ".join(
        expanded_sentences[:1]
    )
    article_sentence_top2 = " ".join(
        expanded_sentences[:2]
    )
    article_sentence_top3 = " ".join(
        expanded_sentences[:3]
    )

    article_test_prediction_records.append({
        "row_number": row_number,
        "id": test_df.iloc[row_number]["id"],
        "article_top1": article_top1,
        "article_top2": article_top2,
        "article_top3": article_top3,
        "article_close3": article_close3,
        "article_sentence_top1": article_sentence_top1,
        "article_sentence_top2": article_sentence_top2,
        "article_sentence_top3": article_sentence_top3,
        "article_window_count": len(window_records)
    })


article_test_predictions_df = pd.DataFrame(
    article_test_prediction_records
)

# 3. Apply confidence routing between article and paragraph

routed_test_records = []

for row_number in range(len(test_df)):
    type_row = test_type_predictions_df.iloc[
        row_number
    ]

    article_row = article_test_predictions_df.iloc[
        row_number
    ]

    paragraph_row = (
        safeguarded_test_predictions_df[
            safeguarded_test_predictions_df[
                "row_number"
            ] == row_number
        ].iloc[0]
    )

    predicted_type = str(
        type_row["predicted_type"]
    )

    prob_phrase = float(
        type_row["prob_phrase"]
    )
    prob_passage = float(
        type_row["prob_passage"]
    )
    prob_multi = float(
        type_row["prob_multi"]
    )

    paragraph_prediction = normalize_prediction_text(
        paragraph_row["spoiler"]
    )

    if predicted_type == "phrase":
        article_prediction = normalize_prediction_text(
            article_row["article_close3"]
        )

        use_article = (
            prob_phrase
            >= ARTICLE_PHRASE_THRESHOLD
        )

    elif predicted_type == "passage":
        article_prediction = normalize_prediction_text(
            article_row["article_sentence_top3"]
        )

        use_article = (
            prob_passage
            >= ARTICLE_PASSAGE_THRESHOLD
        )

    else:
        article_prediction = normalize_prediction_text(
            article_row["article_sentence_top3"]
        )

        use_article = (
            prob_multi
            >= ARTICLE_MULTI_THRESHOLD
        )

    # Never replace a valid paragraph output with an empty article output.
    if not article_prediction:
        use_article = False

    final_prediction = (
        article_prediction
        if use_article
        else paragraph_prediction
    )

    routed_test_records.append({
        "row_number": row_number,
        "id": test_df.iloc[row_number]["id"],
        "predicted_type": predicted_type,
        "prob_phrase": prob_phrase,
        "prob_passage": prob_passage,
        "prob_multi": prob_multi,
        "selected_model": (
            "article" if use_article else "paragraph"
        ),
        "paragraph_prediction": paragraph_prediction,
        "article_prediction": article_prediction,
        "spoiler": final_prediction,
        "word_count": len(
            final_prediction.split()
        ),
        "changed_from_previous": (
            final_prediction
            != paragraph_prediction
        )
    })


confidence_routed_test_df = pd.DataFrame(
    routed_test_records
)

# 4. Diagnostics

print(
    "Decode and routing runtime:",
    round(time.time() - decode_start, 2),
    "seconds"
)

print("\nSelected model counts:")
print(
    confidence_routed_test_df[
        "selected_model"
    ].value_counts()
)

print("\nArticle usage by predicted type:")
print(
    pd.crosstab(
        confidence_routed_test_df[
            "predicted_type"
        ],
        confidence_routed_test_df[
            "selected_model"
        ]
    )
)

print(
    "\nPredictions changed from 0.41804 submission:",
    int(
        confidence_routed_test_df[
            "changed_from_previous"
        ].sum()
    )
)

print("\nEmpty final predictions:")
print(
    int(
        confidence_routed_test_df[
            "spoiler"
        ].fillna("").str.strip().eq("").sum()
    )
)

print("\nFinal word-count summary:")
print(
    confidence_routed_test_df[
        "word_count"
    ].describe()
)

print("\nSample changed predictions:")
display(
    confidence_routed_test_df[
        confidence_routed_test_df[
            "changed_from_previous"
        ]
    ][
        [
            "row_number",
            "id",
            "predicted_type",
            "selected_model",
            "paragraph_prediction",
            "article_prediction",
            "spoiler",
            "word_count"
        ]
    ].head(20)
)

Decode and routing runtime: 13.2 seconds

Selected model counts:
selected_model
article      302
paragraph     98
Name: count, dtype: int64

Article usage by predicted type:
selected_model  article  paragraph
predicted_type                    
multi                48         18
passage             170         13
phrase               84         67

Predictions changed from 0.41804 submission: 264

Empty final predictions:
0

Final word-count summary:
count    400.000000
mean      35.457500
std       37.600634
min        1.000000
25%        3.000000
50%       24.000000
75%       57.000000
max      248.000000
Name: word_count, dtype: float64

Sample changed predictions:


,row_number,id,predicted_type,selected_model,paragraph_prediction,article_prediction,spoiler,word_count
1,1,1,passage,article,1. Prioritise b. Invite Juan to sit in on your...,Why you SHOULD be selfish at work: Helping oth...,Why you SHOULD be selfish at work: Helping oth...,71
5,5,5,phrase,article,John Williams music. To celebrate the fake hol...,"""Main Title,"" ""The Imperial March,"" ""Princess ...","""Main Title,"" ""The Imperial March,"" ""Princess ...",18
6,6,6,multi,article,6. Zone yourself out 5. Tell yourself a story ...,"1. Take a long, warm shower with sweet-smellin...","1. Take a long, warm shower with sweet-smellin...",61
7,7,7,passage,article,"They're not a piece of equipment,"" Reiss said....",Why You Should Never Pet A Service Dog Paragra...,Why You Should Never Pet A Service Dog Paragra...,73
9,9,9,multi,article,putting away too much One extra dollar in your...,When an audience member asked Brandon and this...,When an audience member asked Brandon and this...,48
10,10,10,passage,article,But what was perhaps the most baffling part of...,But what was perhaps the most baffling part of...,But what was perhaps the most baffling part of...,48
11,11,11,passage,article,"According to Crowley, the key is authenticity....",Foursquare's Dennis Crowley: This Mistake Will...,Foursquare's Dennis Crowley: This Mistake Will...,39
13,13,13,passage,article,"""We can make a great team and I welcome him."" ...","""People talk a lot and they know little,"" De G...","""People talk a lot and they know little,"" De G...",72
14,14,14,multi,article,a man obsesses over a woman who works at a vid...,When he goes to a meet-up he’s planned with a ...,When he goes to a meet-up he’s planned with a ...,67
15,15,15,phrase,article,$117 billion $123.6 billion,$117 billion,$117 billion,2


## The routing itself looks plausible, but the samples reveal a real post-processing problem: synthetic labels such as Paragraph 6: and sometimes the article title are leaking into outputs. The 248-word maximum also needs validation testing later.

In [105]:
import re
import numpy as np
import pandas as pd


def remove_synthetic_article_markers(text):
    """
    Remove labels that we inserted when combining the article:
    'Title:' and 'Paragraph 12:'.

    These labels were not part of the original article and should
    never appear in a Kaggle spoiler.
    """
    text = normalize_prediction_text(text)

    text = re.sub(
        r"(?:^|\s)(?:Title|Paragraph\s+\d+):\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return normalize_prediction_text(text)


def cleaned_article_prediction_for_row(row):
    predicted_type = str(
        row["roberta_predicted_type"]
    )

    if predicted_type == "phrase":
        prediction = row["article_close3"]

    elif predicted_type == "passage":
        prediction = row[
            "article_sentence_top3"
        ]

    else:
        prediction = row[
            "article_sentence_top3"
        ]

    prediction = remove_synthetic_article_markers(
        prediction
    )

    # Preserve the defensive fallback.
    if not prediction:
        prediction = paragraph_prediction_for_row(row)

    return prediction


marker_pattern = (
    r"(?:Title|Paragraph\s+\d+):"
)

validation_cleanup_df = (
    article_paragraph_comparison_df.copy()
)

validation_cleanup_df[
    "article_original"
] = validation_cleanup_df.apply(
    article_prediction_for_row,
    axis=1
)

validation_cleanup_df[
    "article_cleaned"
] = validation_cleanup_df.apply(
    cleaned_article_prediction_for_row,
    axis=1
)

validation_cleanup_df[
    "paragraph_prediction"
] = validation_cleanup_df.apply(
    paragraph_prediction_for_row,
    axis=1
)


def route_validation_prediction(
    row,
    article_column
):
    predicted_type = str(
        row["roberta_predicted_type"]
    )

    if predicted_type == "phrase":
        use_article = (
            float(row["prob_phrase"]) >= 0.80
        )

    elif predicted_type == "passage":
        use_article = (
            float(row["prob_passage"]) >= 0.50
        )

    else:
        use_article = (
            float(row["prob_multi"]) >= 0.80
        )

    article_prediction = normalize_prediction_text(
        row[article_column]
    )

    paragraph_prediction = (
        normalize_prediction_text(
            row["paragraph_prediction"]
        )
    )

    if use_article and article_prediction:
        return article_prediction

    return paragraph_prediction


validation_cleanup_df[
    "routed_original"
] = validation_cleanup_df.apply(
    lambda row: route_validation_prediction(
        row,
        "article_original"
    ),
    axis=1
)

validation_cleanup_df[
    "routed_cleaned"
] = validation_cleanup_df.apply(
    lambda row: route_validation_prediction(
        row,
        "article_cleaned"
    ),
    axis=1
)


def calculate_mean_meteor_for_column(
    dataframe,
    prediction_column
):
    scores = []

    for _, row in dataframe.iterrows():
        prediction_tokens = str(
            row[prediction_column]
        ).split()

        gold_tokens = str(
            row["gold_text"]
        ).split()

        score = (
            meteor_score(
                [gold_tokens],
                prediction_tokens
            )
            if prediction_tokens
            else 0.0
        )

        scores.append(score)

    return float(np.mean(scores))


original_routed_meteor = (
    calculate_mean_meteor_for_column(
        validation_cleanup_df,
        "routed_original"
    )
)

cleaned_routed_meteor = (
    calculate_mean_meteor_for_column(
        validation_cleanup_df,
        "routed_cleaned"
    )
)

original_marker_count = int(
    validation_cleanup_df[
        "article_original"
    ]
    .str.contains(
        marker_pattern,
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

cleaned_marker_count = int(
    validation_cleanup_df[
        "article_cleaned"
    ]
    .str.contains(
        marker_pattern,
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

changed_count = int(
    (
        validation_cleanup_df[
            "article_original"
        ]
        != validation_cleanup_df[
            "article_cleaned"
        ]
    ).sum()
)

print(
    "Original confidence-routed METEOR:",
    original_routed_meteor
)

print(
    "Cleaned confidence-routed METEOR:",
    cleaned_routed_meteor
)

print(
    "Difference:",
    cleaned_routed_meteor
    - original_routed_meteor
)

print(
    "\nArticle outputs containing markers before cleanup:",
    original_marker_count
)

print(
    "Article outputs containing markers after cleanup:",
    cleaned_marker_count
)

print(
    "Article predictions changed by cleanup:",
    changed_count
)

print("\nExamples changed by cleanup:")

display(
    validation_cleanup_df[
        validation_cleanup_df[
            "article_original"
        ]
        != validation_cleanup_df[
            "article_cleaned"
        ]
    ][
        [
            "row_number",
            "roberta_predicted_type",
            "article_original",
            "article_cleaned",
            "gold_text"
        ]
    ].head(15)
)

Original confidence-routed METEOR: 0.4140974119614453
Cleaned confidence-routed METEOR: 0.4151768334763123
Difference: 0.0010794215148670072

Article outputs containing markers before cleanup: 110
Article outputs containing markers after cleanup: 0
Article predictions changed by cleanup: 110

Examples changed by cleanup:


,row_number,roberta_predicted_type,article_original,article_cleaned,gold_text
0,0,passage,According to a post by Cawthon on the Five Nig...,According to a post by Cawthon on the Five Nig...,some of the plot elements are so disturbing th...
10,10,multi,"Elettra Wiedemann, Agent Provocateur Model, Ex...","Elettra Wiedemann, Agent Provocateur Model, Ex...","Elettra Wiedemann extra strength work, so weig..."
12,12,multi,How This 20-Year-Old Died From Kissing Her Boy...,How This 20-Year-Old Died From Kissing Her Boy...,he'd eaten a peanut butter sandwich and wasn't...
16,16,passage,They don’t fart—detectably. In a blog post on ...,They don’t fart—detectably. In a blog post on ...,They don’t fart
17,17,passage,"Angry ex-boyfriend bursts into delivery room, ...","Angry ex-boyfriend bursts into delivery room, ...",kicked her and got into a fight with her curre...
18,18,passage,Officer Steve Dunham responded to the call abo...,Officer Steve Dunham responded to the call abo...,Dunham picked the boy up and took him to a Sub...
22,22,passage,Why There Will Probably Never Be A 'Dawson's C...,Why There Will Probably Never Be A 'Dawson's C...,Kevin Williamson said he didn't want to write it
24,24,passage,When affected devices install iOS 9.3.2 and re...,When affected devices install iOS 9.3.2 and re...,bricking iPad Pros
41,41,passage,A meta-analysis in Perspectives in Psychologic...,A meta-analysis in Perspectives in Psychologic...,Some people are just better at sports than others
44,44,phrase,"""I love you, but ..."" Paragraph 1: It could ha...","""I love you, but ..."" It could have been ""I lo...","""but"""


## we have a better meteor, validate a length-based fallback before applying it to test

In [106]:
import itertools
import numpy as np
import pandas as pd


length_routing_df = validation_cleanup_df.copy()

length_routing_df["article_word_count"] = (
    length_routing_df["article_cleaned"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

# Precompute the two possible scores once per row.
length_routing_df["paragraph_meteor"] = [
    prediction_meteor_score(gold, prediction)
    for gold, prediction in zip(
        length_routing_df["gold_text"],
        length_routing_df["paragraph_prediction"]
    )
]

length_routing_df["article_cleaned_meteor"] = [
    prediction_meteor_score(gold, prediction)
    for gold, prediction in zip(
        length_routing_df["gold_text"],
        length_routing_df["article_cleaned"]
    )
]

predicted_types = (
    length_routing_df["roberta_predicted_type"]
    .astype(str)
    .to_numpy()
)

prob_phrase = (
    length_routing_df["prob_phrase"]
    .astype(float)
    .to_numpy()
)

prob_passage = (
    length_routing_df["prob_passage"]
    .astype(float)
    .to_numpy()
)

prob_multi = (
    length_routing_df["prob_multi"]
    .astype(float)
    .to_numpy()
)

article_word_counts = (
    length_routing_df["article_word_count"]
    .astype(int)
    .to_numpy()
)

paragraph_scores = (
    length_routing_df["paragraph_meteor"]
    .to_numpy()
)

article_scores = (
    length_routing_df["article_cleaned_meteor"]
    .to_numpy()
)

phrase_caps = [15, 20, 30, 50, 80, 999]
passage_caps = [60, 80, 100, 120, 150, 999]
multi_caps = [40, 60, 80, 100, 150, 999]

length_cap_results = []

for phrase_cap, passage_cap, multi_cap in itertools.product(
    phrase_caps,
    passage_caps,
    multi_caps
):
    use_article = np.zeros(
        len(length_routing_df),
        dtype=bool
    )

    phrase_mask = predicted_types == "phrase"
    passage_mask = predicted_types == "passage"
    multi_mask = predicted_types == "multi"

    use_article[phrase_mask] = (
        (prob_phrase[phrase_mask] >= 0.80)
        & (article_word_counts[phrase_mask] <= phrase_cap)
    )

    use_article[passage_mask] = (
        (prob_passage[passage_mask] >= 0.50)
        & (article_word_counts[passage_mask] <= passage_cap)
    )

    use_article[multi_mask] = (
        (prob_multi[multi_mask] >= 0.80)
        & (article_word_counts[multi_mask] <= multi_cap)
    )

    routed_scores = np.where(
        use_article,
        article_scores,
        paragraph_scores
    )

    length_cap_results.append({
        "phrase_cap": phrase_cap,
        "passage_cap": passage_cap,
        "multi_cap": multi_cap,
        "mean_meteor": float(
            routed_scores.mean()
        ),
        "median_meteor": float(
            np.median(routed_scores)
        ),
        "article_phrase_rows": int(
            use_article[phrase_mask].sum()
        ),
        "article_passage_rows": int(
            use_article[passage_mask].sum()
        ),
        "article_multi_rows": int(
            use_article[multi_mask].sum()
        ),
        "total_article_rows": int(
            use_article.sum()
        )
    })


length_cap_results_df = (
    pd.DataFrame(length_cap_results)
    .sort_values(
        ["mean_meteor", "total_article_rows"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("Cleaned routing without length caps:")
print(0.4151768334763123)

print("\nBest validation-tested length caps:")
display(length_cap_results_df.head(20))

print("\nCleaned article word counts by predicted type:")
display(
    length_routing_df.groupby(
        "roberta_predicted_type"
    )["article_word_count"].describe()
)

Cleaned routing without length caps:
0.4151768334763123

Best validation-tested length caps:


,phrase_cap,passage_cap,multi_cap,mean_meteor,median_meteor,article_phrase_rows,article_passage_rows,article_multi_rows,total_article_rows
0,50,100,100,0.416904,0.373857,84,142,50,276
1,50,120,100,0.416688,0.373857,84,149,50,283
2,50,150,100,0.416679,0.373857,84,151,50,285
3,50,100,150,0.416672,0.379404,84,142,54,280
4,50,100,999,0.416672,0.379404,84,142,54,280
5,50,120,150,0.416457,0.379404,84,149,54,287
6,50,120,999,0.416457,0.379404,84,149,54,287
7,50,150,150,0.416447,0.379404,84,151,54,289
8,50,150,999,0.416447,0.379404,84,151,54,289
9,80,100,100,0.416383,0.373857,85,142,50,277



Cleaned article word counts by predicted type:


,count,mean,std,min,25%,50%,75%,max
roberta_predicted_type,,,,,,,,
multi,83.0,54.096386,31.073718,2.0,31.5,51.0,76.50,146.0
passage,168.0,56.357143,33.229070,5.0,30.0,55.0,75.25,178.0
phrase,149.0,4.744966,7.334806,1.0,1.0,2.0,5.00,52.0


In [107]:
FINAL_PHRASE_CAP = 50
FINAL_PASSAGE_CAP = 100
FINAL_MULTI_CAP = 100

final_hybrid_records = []

for _, row in confidence_routed_test_df.iterrows():
    predicted_type = str(row["predicted_type"])

    paragraph_prediction = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    cleaned_article_prediction = (
        remove_synthetic_article_markers(
            row["article_prediction"]
        )
    )

    article_word_count = len(
        cleaned_article_prediction.split()
    )

    if predicted_type == "phrase":
        confidence_passed = (
            float(row["prob_phrase"]) >= 0.80
        )
        length_cap = FINAL_PHRASE_CAP

    elif predicted_type == "passage":
        confidence_passed = (
            float(row["prob_passage"]) >= 0.50
        )
        length_cap = FINAL_PASSAGE_CAP

    else:
        confidence_passed = (
            float(row["prob_multi"]) >= 0.80
        )
        length_cap = FINAL_MULTI_CAP

    length_passed = (
        article_word_count <= length_cap
    )

    use_article = (
        confidence_passed
        and length_passed
        and bool(cleaned_article_prediction)
    )

    final_prediction = (
        cleaned_article_prediction
        if use_article
        else paragraph_prediction
    )

    fallback_reason = ""

    if not use_article:
        if not confidence_passed:
            fallback_reason = "confidence"
        elif not cleaned_article_prediction:
            fallback_reason = "empty_article"
        elif not length_passed:
            fallback_reason = "length_cap"

    final_hybrid_records.append({
        "row_number": int(row["row_number"]),
        "id": row["id"],
        "predicted_type": predicted_type,
        "prob_phrase": float(row["prob_phrase"]),
        "prob_passage": float(row["prob_passage"]),
        "prob_multi": float(row["prob_multi"]),
        "selected_model": (
            "article" if use_article else "paragraph"
        ),
        "fallback_reason": fallback_reason,
        "paragraph_prediction": paragraph_prediction,
        "article_prediction_original": normalize_prediction_text(
            row["article_prediction"]
        ),
        "article_prediction_cleaned": cleaned_article_prediction,
        "article_word_count": article_word_count,
        "spoiler": final_prediction,
        "word_count": len(final_prediction.split()),
        "changed_from_041804": (
            final_prediction != paragraph_prediction
        )
    })


final_hybrid_test_df = pd.DataFrame(
    final_hybrid_records
)

marker_pattern = r"(?:Title|Paragraph\s+\d+):"

print("Final model-selection counts:")
print(
    final_hybrid_test_df[
        "selected_model"
    ].value_counts()
)

print("\nModel selection by predicted type:")
print(
    pd.crosstab(
        final_hybrid_test_df["predicted_type"],
        final_hybrid_test_df["selected_model"]
    )
)

print("\nFallback reasons:")
print(
    final_hybrid_test_df[
        "fallback_reason"
    ].replace("", "article_selected").value_counts()
)

print(
    "\nPredictions changed from 0.41804 submission:",
    int(
        final_hybrid_test_df[
            "changed_from_041804"
        ].sum()
    )
)

print(
    "Empty final predictions:",
    int(
        final_hybrid_test_df["spoiler"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Final predictions containing synthetic markers:",
    int(
        final_hybrid_test_df["spoiler"]
        .str.contains(
            marker_pattern,
            case=False,
            regex=True,
            na=False
        )
        .sum()
    )
)

print("\nFinal word-count summary:")
print(
    final_hybrid_test_df[
        "word_count"
    ].describe()
)

print("\nArticle predictions rejected by length cap:")
display(
    final_hybrid_test_df[
        final_hybrid_test_df[
            "fallback_reason"
        ] == "length_cap"
    ][
        [
            "row_number",
            "id",
            "predicted_type",
            "article_prediction_cleaned",
            "article_word_count",
            "paragraph_prediction",
            "word_count"
        ]
    ]
)

print("\nSample final article-selected predictions:")
display(
    final_hybrid_test_df[
        final_hybrid_test_df[
            "selected_model"
        ] == "article"
    ][
        [
            "row_number",
            "id",
            "predicted_type",
            "article_prediction_cleaned",
            "spoiler",
            "word_count"
        ]
    ].head(15)
)

Final model-selection counts:
selected_model
article      281
paragraph    119
Name: count, dtype: int64

Model selection by predicted type:
selected_model  article  paragraph
predicted_type                    
multi                43         23
passage             154         29
phrase               84         67

Fallback reasons:
fallback_reason
article_selected    281
confidence           98
length_cap           21
Name: count, dtype: int64

Predictions changed from 0.41804 submission: 243
Empty final predictions: 0
Final predictions containing synthetic markers: 0

Final word-count summary:
count    400.000000
mean      30.252500
std       28.468929
min        1.000000
25%        3.000000
50%       22.000000
75%       52.000000
max      103.000000
Name: word_count, dtype: float64

Article predictions rejected by length cap:


,row_number,id,predicted_type,article_prediction_cleaned,article_word_count,paragraph_prediction,word_count
28,28,28,passage,Donald Trump’s stunning win in Florida was a m...,119,Donald Trump’s stunning win in Florida was a m...,96
41,41,41,passage,"if this photo is to be believed, there is fres...",117,"It is a form of apophenia, when people see pat...",49
46,46,46,passage,Harry Potter and the Philosopher's Stone illus...,204,"""Harry pocketed it"" was the exact phrase in th...",47
66,66,66,passage,Trump and Hillary Refuse to Explain Why They B...,144,"(ANTIMEDIA) Wilmington, DE — As it turns out, ...",85
70,70,70,multi,The famously diligent test site ran the iPhone...,112,"New Home Button, New Limitations Secondly you ...",14
101,101,101,multi,"""There are 2,000 products that are going to be...",109,"SECOND WIND LOCATION, LOCATION, LOCATION FRESH...",7
108,108,108,multi,After Barely Recognizing Herself In A Family P...,103,"Height: 5'5"" Delores Curtis Eating what I want...",20
145,145,145,passage,0.2 percentage points ahead of Clinton Polling...,102,The polling average has become the go-to numbe...,60
166,166,166,passage,It upheld the International Association of Ath...,108,And his numbers have been borne out: The Olymp...,38
198,198,198,passage,It simply couldn't make enough toys to satiate...,112,The Danish company scaled back its advertising...,71



Sample final article-selected predictions:


,row_number,id,predicted_type,article_prediction_cleaned,spoiler,word_count
1,1,1,passage,Why you SHOULD be selfish at work: Helping oth...,Why you SHOULD be selfish at work: Helping oth...,67
2,2,2,phrase,Have a Bunch of Money,Have a Bunch of Money,5
3,3,3,phrase,Braconid,Braconid,1
5,5,5,phrase,"""Main Title,"" ""The Imperial March,"" ""Princess ...","""Main Title,"" ""The Imperial March,"" ""Princess ...",18
6,6,6,multi,"1. Take a long, warm shower with sweet-smellin...","1. Take a long, warm shower with sweet-smellin...",57
7,7,7,passage,Why You Should Never Pet A Service Dog Flynn t...,Why You Should Never Pet A Service Dog Flynn t...,69
8,8,8,phrase,Lord Ivar Mountbatten,Lord Ivar Mountbatten,3
9,9,9,multi,When an audience member asked Brandon and this...,When an audience member asked Brandon and this...,48
10,10,10,passage,But what was perhaps the most baffling part of...,But what was perhaps the most baffling part of...,48
11,11,11,passage,Foursquare's Dennis Crowley: This Mistake Will...,Foursquare's Dennis Crowley: This Mistake Will...,37


In [108]:
import os
import pandas as pd

FINAL_HYBRID_SUBMISSION_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_article_paragraph_hybrid.csv"
)

FINAL_HYBRID_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_article_paragraph_hybrid_audit.csv"
)

# Preserve exactly the ID order required by sample_solution.csv.
hybrid_submission_df = (
    sample_df[["id"]]
    .merge(
        final_hybrid_test_df[
            ["id", "spoiler"]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
)

# Kaggle-ready file: exactly id and spoiler.
hybrid_submission_df.to_csv(
    FINAL_HYBRID_SUBMISSION_PATH,
    index=False
)

# Detailed audit file for our records.
final_hybrid_test_df.to_csv(
    FINAL_HYBRID_AUDIT_PATH,
    index=False
)

ids_match_sample = (
    hybrid_submission_df["id"].tolist()
    == sample_df["id"].tolist()
)

ids_match_test = (
    hybrid_submission_df["id"].tolist()
    == test_df["id"].tolist()
)

empty_count = int(
    hybrid_submission_df["spoiler"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

synthetic_marker_count = int(
    hybrid_submission_df["spoiler"]
    .astype(str)
    .str.contains(
        r"(?:Title|Paragraph\s+\d+):",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

print(
    "Submission exists:",
    os.path.exists(FINAL_HYBRID_SUBMISSION_PATH)
)
print("Submission shape:", hybrid_submission_df.shape)
print(
    "Submission columns:",
    hybrid_submission_df.columns.tolist()
)
print("IDs match sample order:", ids_match_sample)
print("IDs match test order:", ids_match_test)
print(
    "Unique IDs:",
    hybrid_submission_df["id"].nunique()
)
print("Empty spoilers:", empty_count)
print("Synthetic markers:", synthetic_marker_count)
print(
    "Audit exists:",
    os.path.exists(FINAL_HYBRID_AUDIT_PATH)
)
print(
    "Submission location:",
    FINAL_HYBRID_SUBMISSION_PATH
)

print("\nFirst five rows:")
display(hybrid_submission_df.head())

print("\nLast five rows:")
display(hybrid_submission_df.tail())

Submission exists: True
Submission shape: (400, 2)
Submission columns: ['id', 'spoiler']
IDs match sample order: True
IDs match test order: True
Unique IDs: 400
Empty spoilers: 0
Synthetic markers: 0
Audit exists: True
Submission location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_article_paragraph_hybrid.csv

First five rows:


,id,spoiler
0,0,"balloons and a sign in hand that reads, ""Heard..."
1,1,Why you SHOULD be selfish at work: Helping oth...
2,2,Have a Bunch of Money
3,3,Braconid
4,4,3. Remove the egg yolks from the fridge after ...



Last five rows:


,id,spoiler
395,395,This Is What Happens When You Leave A Hotel Cl...
396,396,Christopher Suprun
397,397,No Medication. High fat vegan plant based diet...
398,398,WikiLeaks regularly tweets about Assange’s sta...
399,399,Richard Belzer


### great progress! we are reaching 0.43ish, target is to reach 0.45+. So lets run a diagnostic and check how much score we lose bc the router chosses the wrong model

In [110]:
import numpy as np
import pandas as pd

routing_diagnostic_df = length_routing_df.copy()

predicted_type = (
    routing_diagnostic_df["roberta_predicted_type"]
    .astype(str)
)

current_use_article = pd.Series(
    False,
    index=routing_diagnostic_df.index
)

phrase_mask = predicted_type.eq("phrase")
passage_mask = predicted_type.eq("passage")
multi_mask = predicted_type.eq("multi")

current_use_article.loc[phrase_mask] = (
    routing_diagnostic_df.loc[
        phrase_mask,
        "prob_phrase"
    ].astype(float).ge(0.80)
    &
    routing_diagnostic_df.loc[
        phrase_mask,
        "article_word_count"
    ].le(50)
)

current_use_article.loc[passage_mask] = (
    routing_diagnostic_df.loc[
        passage_mask,
        "prob_passage"
    ].astype(float).ge(0.50)
    &
    routing_diagnostic_df.loc[
        passage_mask,
        "article_word_count"
    ].le(100)
)

current_use_article.loc[multi_mask] = (
    routing_diagnostic_df.loc[
        multi_mask,
        "prob_multi"
    ].astype(float).ge(0.80)
    &
    routing_diagnostic_df.loc[
        multi_mask,
        "article_word_count"
    ].le(100)
)

routing_diagnostic_df[
    "current_selected_model"
] = np.where(
    current_use_article,
    "article",
    "paragraph"
)

routing_diagnostic_df[
    "current_meteor"
] = np.where(
    current_use_article,
    routing_diagnostic_df[
        "article_cleaned_meteor"
    ],
    routing_diagnostic_df[
        "paragraph_meteor"
    ]
)

routing_diagnostic_df[
    "oracle_selected_model"
] = np.where(
    routing_diagnostic_df[
        "article_cleaned_meteor"
    ]
    >
    routing_diagnostic_df[
        "paragraph_meteor"
    ],
    "article",
    "paragraph"
)

routing_diagnostic_df[
    "oracle_meteor"
] = routing_diagnostic_df[
    [
        "article_cleaned_meteor",
        "paragraph_meteor"
    ]
].max(axis=1)

routing_diagnostic_df[
    "routing_regret"
] = (
    routing_diagnostic_df["oracle_meteor"]
    - routing_diagnostic_df["current_meteor"]
)

routing_diagnostic_df[
    "article_better"
] = (
    routing_diagnostic_df[
        "article_cleaned_meteor"
    ]
    >
    routing_diagnostic_df[
        "paragraph_meteor"
    ]
)

routing_diagnostic_df[
    "paragraph_better"
] = (
    routing_diagnostic_df[
        "paragraph_meteor"
    ]
    >
    routing_diagnostic_df[
        "article_cleaned_meteor"
    ]
)

print("Current routed METEOR:")
print(
    routing_diagnostic_df[
        "current_meteor"
    ].mean()
)

print("\nOracle article-or-paragraph METEOR:")
print(
    routing_diagnostic_df[
        "oracle_meteor"
    ].mean()
)

print("\nRemaining routing opportunity:")
print(
    routing_diagnostic_df[
        "routing_regret"
    ].mean()
)

print("\nPer-row model comparison:")
print(
    "Article better:",
    int(
        routing_diagnostic_df[
            "article_better"
        ].sum()
    )
)
print(
    "Paragraph better:",
    int(
        routing_diagnostic_df[
            "paragraph_better"
        ].sum()
    )
)
print(
    "Ties:",
    int(
        (
            ~routing_diagnostic_df[
                "article_better"
            ]
            &
            ~routing_diagnostic_df[
                "paragraph_better"
            ]
        ).sum()
    )
)

print("\nResults by predicted type:")
display(
    routing_diagnostic_df.groupby(
        "roberta_predicted_type"
    ).agg(
        rows=("row_number", "size"),
        current_meteor=(
            "current_meteor",
            "mean"
        ),
        oracle_meteor=(
            "oracle_meteor",
            "mean"
        ),
        mean_routing_regret=(
            "routing_regret",
            "mean"
        ),
        article_win_rate=(
            "article_better",
            "mean"
        )
    )
)

print("\nResults by true spoiler type:")
display(
    routing_diagnostic_df.groupby(
        "spoiler_type"
    ).agg(
        rows=("row_number", "size"),
        current_meteor=(
            "current_meteor",
            "mean"
        ),
        oracle_meteor=(
            "oracle_meteor",
            "mean"
        ),
        mean_routing_regret=(
            "routing_regret",
            "mean"
        ),
        article_win_rate=(
            "article_better",
            "mean"
        )
    )
)

Current routed METEOR:
0.4169037738283222

Oracle article-or-paragraph METEOR:
0.493058092534097

Remaining routing opportunity:
0.07615431870577477

Per-row model comparison:
Article better: 187
Paragraph better: 134
Ties: 79

Results by predicted type:


,rows,current_meteor,oracle_meteor,mean_routing_regret,article_win_rate
roberta_predicted_type,,,,,
multi,83,0.345667,0.424721,0.079054,0.542169
passage,168,0.387260,0.471961,0.084701,0.559524
phrase,149,0.490009,0.554912,0.064902,0.322148



Results by true spoiler type:


,rows,current_meteor,oracle_meteor,mean_routing_regret,article_win_rate
spoiler_type,,,,,
multi,84,0.376481,0.446010,0.069529,0.476190
passage,154,0.395823,0.493468,0.097645,0.603896
phrase,162,0.457904,0.517064,0.059160,0.333333


In [111]:
## feature building cell

import re
import numpy as np
import pandas as pd


def normalized_token_set(text):
    text = normalize_prediction_text(text).lower()

    tokens = re.findall(
        r"\b\w+\b",
        text
    )

    return set(tokens)


def token_jaccard(text_a, text_b):
    tokens_a = normalized_token_set(text_a)
    tokens_b = normalized_token_set(text_b)

    if not tokens_a and not tokens_b:
        return 1.0

    if not tokens_a or not tokens_b:
        return 0.0

    return (
        len(tokens_a & tokens_b)
        / len(tokens_a | tokens_b)
    )


def containment_score(text_a, text_b):
    tokens_a = normalized_token_set(text_a)
    tokens_b = normalized_token_set(text_b)

    smaller_size = min(
        len(tokens_a),
        len(tokens_b)
    )

    if smaller_size == 0:
        return 0.0

    return (
        len(tokens_a & tokens_b)
        / smaller_size
    )


# ---------------------------------------------------------
# Paragraph-QA confidence features
# ---------------------------------------------------------

paragraph_confidence_records = []

for row_number, group in (
    val_candidate_predictions_df.groupby(
        "row_number"
    )
):
    ranked = group.sort_values(
        "answerability_margin",
        ascending=False
    ).reset_index(drop=True)

    best_margin = float(
        ranked.iloc[0]["answerability_margin"]
    )

    second_margin = (
        float(
            ranked.iloc[1]["answerability_margin"]
        )
        if len(ranked) > 1
        else best_margin
    )

    third_margin = (
        float(
            ranked.iloc[2]["answerability_margin"]
        )
        if len(ranked) > 2
        else second_margin
    )

    paragraph_confidence_records.append({
        "row_number": int(row_number),
        "paragraph_best_margin": best_margin,
        "paragraph_margin_gap_1_2": (
            best_margin - second_margin
        ),
        "paragraph_margin_gap_1_3": (
            best_margin - third_margin
        ),
        "paragraph_positive_candidates": int(
            (
                ranked["answerability_margin"]
                > 0
            ).sum()
        ),
        "paragraph_candidate_count": len(ranked)
    })


paragraph_confidence_df = pd.DataFrame(
    paragraph_confidence_records
)


# ---------------------------------------------------------
# Article-QA confidence features
# ---------------------------------------------------------

article_confidence_records = []

for row_number in range(len(val_df)):
    window_records = window_predictions_by_row.get(
        row_number,
        []
    )

    ranked = sorted(
        window_records,
        key=lambda item: item[
            "answerability_margin"
        ],
        reverse=True
    )

    if ranked:
        best_margin = float(
            ranked[0]["answerability_margin"]
        )

        second_margin = (
            float(
                ranked[1]["answerability_margin"]
            )
            if len(ranked) > 1
            else best_margin
        )

        third_margin = (
            float(
                ranked[2]["answerability_margin"]
            )
            if len(ranked) > 2
            else second_margin
        )

        positive_windows = sum(
            float(item["answerability_margin"]) > 0
            for item in ranked
        )
    else:
        best_margin = -999.0
        second_margin = -999.0
        third_margin = -999.0
        positive_windows = 0

    article_confidence_records.append({
        "row_number": row_number,
        "article_best_margin": best_margin,
        "article_margin_gap_1_2": (
            best_margin - second_margin
        ),
        "article_margin_gap_1_3": (
            best_margin - third_margin
        ),
        "article_positive_windows": (
            positive_windows
        ),
        "article_window_count": len(ranked)
    })


article_confidence_df = pd.DataFrame(
    article_confidence_records
)


# ---------------------------------------------------------
# Build deployable router feature table
# ---------------------------------------------------------

router_feature_df = (
    routing_diagnostic_df[
        [
            "row_number",
            "spoiler_type",
            "roberta_predicted_type",
            "prob_phrase",
            "prob_passage",
            "prob_multi",
            "paragraph_prediction",
            "article_cleaned",
            "paragraph_meteor",
            "article_cleaned_meteor",
            "article_better",
            "routing_regret"
        ]
    ]
    .merge(
        paragraph_confidence_df,
        on="row_number",
        how="left",
        validate="one_to_one"
    )
    .merge(
        article_confidence_df,
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)

router_feature_df[
    "paragraph_word_count"
] = (
    router_feature_df[
        "paragraph_prediction"
    ]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

router_feature_df[
    "article_word_count"
] = (
    router_feature_df[
        "article_cleaned"
    ]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

router_feature_df[
    "word_count_difference"
] = (
    router_feature_df["article_word_count"]
    - router_feature_df["paragraph_word_count"]
)

router_feature_df[
    "absolute_word_count_difference"
] = (
    router_feature_df[
        "word_count_difference"
    ].abs()
)

router_feature_df[
    "article_to_paragraph_length_ratio"
] = (
    router_feature_df["article_word_count"]
    / router_feature_df[
        "paragraph_word_count"
    ].clip(lower=1)
)

router_feature_df[
    "prediction_token_jaccard"
] = [
    token_jaccard(article, paragraph)
    for article, paragraph in zip(
        router_feature_df["article_cleaned"],
        router_feature_df[
            "paragraph_prediction"
        ]
    )
]

router_feature_df[
    "prediction_containment"
] = [
    containment_score(article, paragraph)
    for article, paragraph in zip(
        router_feature_df["article_cleaned"],
        router_feature_df[
            "paragraph_prediction"
        ]
    )
]

router_feature_df[
    "predictions_exact_match"
] = (
    router_feature_df["article_cleaned"]
    .fillna("")
    .str.lower()
    .str.strip()
    ==
    router_feature_df[
        "paragraph_prediction"
    ]
    .fillna("")
    .str.lower()
    .str.strip()
).astype(int)

router_feature_df[
    "best_margin_difference"
] = (
    router_feature_df["article_best_margin"]
    - router_feature_df[
        "paragraph_best_margin"
    ]
)

# Target used only during validation training.
router_feature_df[
    "article_gain"
] = (
    router_feature_df[
        "article_cleaned_meteor"
    ]
    - router_feature_df[
        "paragraph_meteor"
    ]
)

router_feature_df[
    "article_win_target"
] = (
    router_feature_df[
        "article_gain"
    ] > 0
).astype(int)


deployable_router_features = [
    "prob_phrase",
    "prob_passage",
    "prob_multi",
    "paragraph_word_count",
    "article_word_count",
    "word_count_difference",
    "absolute_word_count_difference",
    "article_to_paragraph_length_ratio",
    "prediction_token_jaccard",
    "prediction_containment",
    "predictions_exact_match",
    "paragraph_best_margin",
    "paragraph_margin_gap_1_2",
    "paragraph_margin_gap_1_3",
    "paragraph_positive_candidates",
    "paragraph_candidate_count",
    "article_best_margin",
    "article_margin_gap_1_2",
    "article_margin_gap_1_3",
    "article_positive_windows",
    "article_window_count",
    "best_margin_difference"
]

print("Router rows:", len(router_feature_df))
print(
    "Deployable numeric features:",
    len(deployable_router_features)
)

print(
    "Missing deployable feature values:",
    int(
        router_feature_df[
            deployable_router_features
        ].isna().sum().sum()
    )
)

print("\nArticle-win target distribution:")
print(
    router_feature_df[
        "article_win_target"
    ].value_counts()
)

print("\nArticle-gain summary:")
print(
    router_feature_df[
        "article_gain"
    ].describe()
)

print("\nFeature summary:")
display(
    router_feature_df[
        deployable_router_features
    ].describe().T
)

print("\nSample router rows:")
display(
    router_feature_df[
        [
            "row_number",
            "roberta_predicted_type",
            "prob_phrase",
            "prob_passage",
            "prob_multi",
            "paragraph_word_count",
            "article_word_count",
            "prediction_token_jaccard",
            "paragraph_best_margin",
            "article_best_margin",
            "article_gain",
            "article_win_target"
        ]
    ].head(10)
)

Router rows: 400
Deployable numeric features: 22
Missing deployable feature values: 0

Article-win target distribution:
article_win_target
0    213
1    187
Name: count, dtype: int64

Article-gain summary:
count    400.000000
mean       0.011863
std        0.291785
min       -0.981481
25%       -0.070588
50%        0.000000
75%        0.120029
max        0.992188
Name: article_gain, dtype: float64

Feature summary:


,count,mean,std,min,25%,50%,75%,max
prob_phrase,400.0,0.386503,0.348702,0.005822,0.069845,0.242918,0.748595,0.959461
prob_passage,400.0,0.354941,0.334149,0.004191,0.028441,0.232835,0.690779,0.910906
prob_multi,400.0,0.258556,0.321066,0.017317,0.046317,0.088569,0.310645,0.988223
paragraph_word_count,400.0,32.125000,31.078434,1.000000,5.000000,22.000000,55.000000,132.000000
article_word_count,400.0,36.662500,35.881923,1.000000,3.000000,28.000000,63.000000,178.000000
word_count_difference,400.0,4.537500,33.775220,-118.000000,-8.000000,0.000000,17.000000,145.000000
absolute_word_count_difference,400.0,22.077500,25.937780,0.000000,2.000000,12.500000,34.000000,145.000000
article_to_paragraph_length_ratio,400.0,1.866255,3.044179,0.049505,0.520565,1.000000,1.539231,32.666667
prediction_token_jaccard,400.0,0.436077,0.324928,0.000000,0.169708,0.342588,0.625000,1.000000
prediction_containment,400.0,0.741925,0.304038,0.000000,0.500000,0.857143,1.000000,1.000000



Sample router rows:


,row_number,roberta_predicted_type,prob_phrase,prob_passage,prob_multi,paragraph_word_count,article_word_count,prediction_token_jaccard,paragraph_best_margin,article_best_margin,article_gain,article_win_target
0,0,passage,0.247788,0.718224,0.033988,85,94,0.945455,2.076172,5.599609,-0.006252,0
1,1,passage,0.056799,0.694663,0.248538,57,63,0.696429,4.376953,7.666016,-0.024989,0
2,2,phrase,0.920324,0.041355,0.038322,8,1,0.166667,2.632812,1.292969,0.205882,1
3,3,phrase,0.601267,0.096821,0.301912,6,49,0.020000,0.671875,2.041016,0.280744,1
4,4,passage,0.057797,0.869886,0.072317,101,5,0.066667,0.914062,1.460938,-0.528356,0
5,5,phrase,0.941393,0.022248,0.036359,1,1,1.000000,5.208984,7.913086,0.000000,0
6,6,phrase,0.945689,0.031364,0.022946,12,4,0.454545,9.666016,2.531250,0.165365,1
7,7,passage,0.242862,0.673351,0.083787,56,46,0.555556,1.988281,-0.463867,0.010119,1
8,8,passage,0.111905,0.863175,0.024921,77,88,0.522727,5.115234,5.431641,-0.032407,0
9,9,passage,0.238007,0.654518,0.107475,56,6,0.121951,-0.804688,2.279297,0.073363,1


## using repeated out-of-fold predictions to learn routers, the models never predict a row they trained on; larger article receive more training weight

In [112]:
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


router_start_time = time.time()

# 1. Build deployable feature matrix

router_model_df = router_feature_df.copy()

# Explicitly flag the few validation rows where article inference
# did not produce a valid window prediction.
router_model_df["article_output_missing"] = (
    router_model_df["article_best_margin"] <= -900
).astype(int)

# Replace artificial -999 placeholders with a conservative value.
margin_columns = [
    "article_best_margin",
    "best_margin_difference"
]

for column in margin_columns:
    router_model_df[column] = (
        router_model_df[column]
        .replace([-999.0, -1001.289062], -20.0)
        .clip(-30, 30)
    )

numeric_features = (
    deployable_router_features
    + ["article_output_missing"]
)

categorical_features = [
    "roberta_predicted_type"
]

X_router = router_model_df[
    numeric_features + categorical_features
].copy()

y_router = router_model_df[
    "article_win_target"
].astype(int).to_numpy()

article_gain = router_model_df[
    "article_gain"
].astype(float).to_numpy()

paragraph_meteor = router_model_df[
    "paragraph_meteor"
].astype(float).to_numpy()

article_meteor = router_model_df[
    "article_cleaned_meteor"
].astype(float).to_numpy()

# Weight costly decisions more heavily, while still retaining
# rows where the two models tie or nearly tie.
sample_weights = (
    np.abs(article_gain) + 0.03
)

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

# 2. Candidate learned routers

router_models = {
    "logistic": Pipeline([
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                C=0.20,
                max_iter=3000,
                class_weight="balanced",
                random_state=SEED
            )
        )
    ]),

    "random_forest": Pipeline([
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=SEED
            )
        )
    ]),

    "extra_trees": Pipeline([
        ("preprocess", preprocessor),
        (
            "model",
            ExtraTreesClassifier(
                n_estimators=300,
                max_depth=7,
                min_samples_leaf=4,
                max_features=0.70,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED
            )
        )
    ]),

    "hist_gradient_boosting": Pipeline([
        ("preprocess", preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.04,
                max_iter=180,
                max_leaf_nodes=12,
                min_samples_leaf=15,
                l2_regularization=2.0,
                random_state=SEED
            )
        )
    ])
}


# Every row is predicted five times, each time by a model that
# did not train on that row.
router_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED
)

oof_probability_by_model = {}

for model_name, base_model in router_models.items():
    probability_sum = np.zeros(
        len(router_model_df),
        dtype=float
    )

    prediction_count = np.zeros(
        len(router_model_df),
        dtype=int
    )

    for fold_number, (
        train_indices,
        validation_indices
    ) in enumerate(
        router_cv.split(X_router, y_router),
        start=1
    ):
        fold_model = clone(base_model)

        fit_kwargs = {}

        # Pipeline parameters use step-name prefixes.
        if model_name in {
            "logistic",
            "random_forest",
            "extra_trees",
            "hist_gradient_boosting"
        }:
            fit_kwargs[
                "model__sample_weight"
            ] = sample_weights[train_indices]

        fold_model.fit(
            X_router.iloc[train_indices],
            y_router[train_indices],
            **fit_kwargs
        )

        fold_probabilities = (
            fold_model.predict_proba(
                X_router.iloc[validation_indices]
            )[:, 1]
        )

        probability_sum[
            validation_indices
        ] += fold_probabilities

        prediction_count[
            validation_indices
        ] += 1

    assert np.all(
        prediction_count == 5
    ), (
        model_name,
        np.unique(prediction_count)
    )

    oof_probability_by_model[
        model_name
    ] = probability_sum / prediction_count

    print(
        f"Completed {model_name}:",
        round(
            time.time() - router_start_time,
            1
        ),
        "seconds"
    )


# Add simple probability ensembles.
oof_probability_by_model[
    "ensemble_all"
] = np.mean(
    np.column_stack(
        list(
            oof_probability_by_model.values()
        )
    ),
    axis=1
)

oof_probability_by_model[
    "ensemble_tree"
] = np.mean(
    np.column_stack([
        oof_probability_by_model[
            "random_forest"
        ],
        oof_probability_by_model[
            "extra_trees"
        ],
        oof_probability_by_model[
            "hist_gradient_boosting"
        ]
    ]),
    axis=1
)

# 3. Evaluate fully out-of-fold routing

threshold_grid = np.round(
    np.arange(0.20, 0.81, 0.02),
    2
)

router_oof_results = []

for model_name, probabilities in (
    oof_probability_by_model.items()
):
    auc = roc_auc_score(
        y_router,
        probabilities,
        sample_weight=sample_weights
    )

    for threshold in threshold_grid:
        use_article = (
            probabilities >= threshold
        )

        routed_scores = np.where(
            use_article,
            article_meteor,
            paragraph_meteor
        )

        classifications = (
            use_article.astype(int)
        )

        router_oof_results.append({
            "model": model_name,
            "threshold": float(threshold),
            "mean_meteor": float(
                routed_scores.mean()
            ),
            "median_meteor": float(
                np.median(routed_scores)
            ),
            "article_rows": int(
                use_article.sum()
            ),
            "accuracy": float(
                accuracy_score(
                    y_router,
                    classifications
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_router,
                    classifications
                )
            ),
            "weighted_auc": float(auc)
        })


router_oof_results_df = (
    pd.DataFrame(router_oof_results)
    .sort_values(
        [
            "mean_meteor",
            "balanced_accuracy"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

best_result_by_model_df = (
    router_oof_results_df
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .groupby(
        "model",
        as_index=False
    )
    .first()
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\nTotal router runtime:",
    round(
        time.time() - router_start_time,
        2
    ),
    "seconds"
)

print("\nCurrent rule-based validation METEOR:")
print(0.4169037738283222)

print("\nArticle-or-paragraph oracle METEOR:")
print(0.493058092534097)

print("\nBest OOF result from each learned router:")
display(best_result_by_model_df)

print("\nTop 20 OOF router configurations:")
display(router_oof_results_df.head(20))

Completed logistic: 1.6 seconds
Completed random_forest: 31.1 seconds
Completed extra_trees: 47.1 seconds
Completed hist_gradient_boosting: 53.4 seconds

Total router runtime: 53.64 seconds

Current rule-based validation METEOR:
0.4169037738283222

Article-or-paragraph oracle METEOR:
0.493058092534097

Best OOF result from each learned router:


,model,threshold,mean_meteor,median_meteor,article_rows,accuracy,balanced_accuracy,weighted_auc
0,logistic,0.54,0.412125,0.385943,169,0.5850,0.580302,0.547210
1,extra_trees,0.46,0.408205,0.359757,277,0.6150,0.628054,0.506015
2,random_forest,0.36,0.406789,0.359757,332,0.5875,0.609412,0.515396
3,ensemble_tree,0.30,0.406331,0.359757,340,0.5925,0.615739,0.503344
4,ensemble_all,0.32,0.406214,0.359757,336,0.5975,0.620107,0.508827
5,hist_gradient_boosting,0.20,0.405006,0.351543,369,0.5150,0.542643,0.483953



Top 20 OOF router configurations:


,model,threshold,mean_meteor,median_meteor,article_rows,accuracy,balanced_accuracy,weighted_auc
0,logistic,0.54,0.412125,0.385943,169,0.5850,0.580302,0.547210
1,extra_trees,0.46,0.408205,0.359757,277,0.6150,0.628054,0.506015
2,logistic,0.56,0.407320,0.384333,150,0.5825,0.574691,0.547210
3,random_forest,0.36,0.406789,0.359757,332,0.5875,0.609412,0.515396
4,random_forest,0.50,0.406429,0.345645,236,0.5875,0.593746,0.515396
5,ensemble_tree,0.30,0.406331,0.359757,340,0.5925,0.615739,0.503344
6,ensemble_all,0.32,0.406214,0.359757,336,0.5975,0.620107,0.508827
7,logistic,0.38,0.406211,0.363224,320,0.6075,0.627539,0.547210
8,ensemble_tree,0.24,0.406053,0.359757,356,0.5725,0.598265,0.503344
9,extra_trees,0.26,0.405956,0.359757,338,0.6175,0.640519,0.506015


## the article router did not beat the current hybrid model, so maybe not worth the effort dive deeper into it. Next target will be the spoiler type classifier, since the current type accuracy is 75.5%, and article model with the correct type reach 0.4422/

In [113]:
import itertools
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)


meta_start_time = time.time()

# 1. Build validation data for type correction

meta_type_df = (
    router_model_df.copy()
    .merge(
        article_paragraph_comparison_df[
            [
                "row_number",
                "top1",
                "top2",
                "top3",
                "close3",
                "sentence_top1",
                "sentence_top2",
                "sentence_top3",
                "article_top1",
                "article_top2",
                "article_top3",
                "article_close3",
                "article_sentence_top1",
                "article_sentence_top2",
                "article_sentence_top3",
                "gold_text"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)

# Clean every article decoding strategy.
article_strategy_columns = [
    "article_top1",
    "article_top2",
    "article_top3",
    "article_close3",
    "article_sentence_top1",
    "article_sentence_top2",
    "article_sentence_top3"
]

for column in article_strategy_columns:
    meta_type_df[
        f"{column}_cleaned"
    ] = meta_type_df[column].apply(
        remove_synthetic_article_markers
    )

# Length features from type-specific strategies.
meta_type_df["paragraph_phrase_length"] = (
    meta_type_df["close3"]
    .fillna("")
    .str.split()
    .str.len()
)

meta_type_df["paragraph_passage_length"] = (
    meta_type_df["sentence_top3"]
    .fillna("")
    .str.split()
    .str.len()
)

meta_type_df["paragraph_multi_length"] = (
    meta_type_df["top3"]
    .fillna("")
    .str.split()
    .str.len()
)

meta_type_df["article_phrase_length"] = (
    meta_type_df["article_close3_cleaned"]
    .fillna("")
    .str.split()
    .str.len()
)

meta_type_df["article_passage_length"] = (
    meta_type_df[
        "article_sentence_top3_cleaned"
    ]
    .fillna("")
    .str.split()
    .str.len()
)

meta_type_df["article_multi_length"] = (
    meta_type_df[
        "article_sentence_top3_cleaned"
    ]
    .fillna("")
    .str.split()
    .str.len()
)

additional_type_features = [
    "paragraph_phrase_length",
    "paragraph_passage_length",
    "paragraph_multi_length",
    "article_phrase_length",
    "article_passage_length",
    "article_multi_length"
]

meta_numeric_features = (
    numeric_features
    + additional_type_features
)

meta_categorical_features = [
    "roberta_predicted_type"
]

X_meta = meta_type_df[
    meta_numeric_features
    + meta_categorical_features
].copy()

type_labels = [
    "phrase",
    "passage",
    "multi"
]

type_to_id = {
    label: index
    for index, label in enumerate(type_labels)
}

id_to_type = {
    index: label
    for label, index in type_to_id.items()
}

y_meta = (
    meta_type_df["spoiler_type"]
    .map(type_to_id)
    .astype(int)
    .to_numpy()
)

# 2. Preprocessing and candidate classifiers

meta_numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

meta_categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

meta_preprocessor = ColumnTransformer([
    (
        "numeric",
        meta_numeric_pipeline,
        meta_numeric_features
    ),
    (
        "categorical",
        meta_categorical_pipeline,
        meta_categorical_features
    )
])

meta_models = {
    "logistic": Pipeline([
        ("preprocess", meta_preprocessor),
        (
            "model",
            LogisticRegression(
                C=0.20,
                max_iter=3000,
                class_weight="balanced",
                random_state=SEED
            )
        )
    ]),

    "random_forest": Pipeline([
        ("preprocess", meta_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=7,
                min_samples_leaf=4,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=SEED
            )
        )
    ]),

    "extra_trees": Pipeline([
        ("preprocess", meta_preprocessor),
        (
            "model",
            ExtraTreesClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=3,
                max_features=0.70,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED
            )
        )
    ]),

    "hist_gradient_boosting": Pipeline([
        ("preprocess", meta_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.04,
                max_iter=180,
                max_leaf_nodes=12,
                min_samples_leaf=15,
                l2_regularization=2.0,
                random_state=SEED
            )
        )
    ])
}

meta_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED
)

meta_oof_probabilities = {}

for model_name, base_model in meta_models.items():
    probability_sum = np.zeros(
        (len(meta_type_df), len(type_labels)),
        dtype=float
    )

    prediction_count = np.zeros(
        len(meta_type_df),
        dtype=int
    )

    for train_indices, validation_indices in (
        meta_cv.split(X_meta, y_meta)
    ):
        fold_model = clone(base_model)

        fold_model.fit(
            X_meta.iloc[train_indices],
            y_meta[train_indices]
        )

        fold_probabilities = (
            fold_model.predict_proba(
                X_meta.iloc[validation_indices]
            )
        )

        probability_sum[
            validation_indices
        ] += fold_probabilities

        prediction_count[
            validation_indices
        ] += 1

    assert np.all(prediction_count == 5)

    meta_oof_probabilities[model_name] = (
        probability_sum
        / prediction_count[:, None]
    )

    print(
        f"Completed {model_name}:",
        round(
            time.time() - meta_start_time,
            1
        ),
        "seconds"
    )


meta_oof_probabilities[
    "ensemble_all"
] = np.mean(
    np.stack(
        list(meta_oof_probabilities.values())
    ),
    axis=0
)

# 3. Evaluate extraction using OOF-predicted spoiler types

def select_prediction_by_type(
    row,
    predicted_spoiler_type,
    phrase_source,
    passage_source,
    multi_source
):
    source_by_type = {
        "phrase": phrase_source,
        "passage": passage_source,
        "multi": multi_source
    }

    selected_source = source_by_type[
        predicted_spoiler_type
    ]

    if predicted_spoiler_type == "phrase":
        paragraph_prediction = (
            normalize_prediction_text(
                row["close3"]
            )
        )

        article_prediction = (
            normalize_prediction_text(
                row["article_close3_cleaned"]
            )
        )

        article_cap = 50

    elif predicted_spoiler_type == "passage":
        paragraph_prediction = (
            normalize_prediction_text(
                row["sentence_top3"]
            )
        )

        article_prediction = (
            normalize_prediction_text(
                row[
                    "article_sentence_top3_cleaned"
                ]
            )
        )

        article_cap = 100

    else:
        paragraph_prediction = (
            normalize_prediction_text(
                row["top3"]
            )
        )

        article_prediction = (
            normalize_prediction_text(
                row[
                    "article_sentence_top3_cleaned"
                ]
            )
        )

        article_cap = 100

    if (
        selected_source == "article"
        and article_prediction
        and len(article_prediction.split())
        <= article_cap
    ):
        return article_prediction

    return paragraph_prediction


meta_results = []

for model_name, probabilities in (
    meta_oof_probabilities.items()
):
    predicted_ids = probabilities.argmax(axis=1)

    predicted_types = np.array([
        id_to_type[int(predicted_id)]
        for predicted_id in predicted_ids
    ])

    type_accuracy = accuracy_score(
        y_meta,
        predicted_ids
    )

    type_macro_f1 = f1_score(
        y_meta,
        predicted_ids,
        average="macro"
    )

    for (
        phrase_source,
        passage_source,
        multi_source
    ) in itertools.product(
        ["article", "paragraph"],
        repeat=3
    ):
        row_scores = []

        for row_position, (_, row) in enumerate(
            meta_type_df.iterrows()
        ):
            prediction = select_prediction_by_type(
                row=row,
                predicted_spoiler_type=(
                    predicted_types[row_position]
                ),
                phrase_source=phrase_source,
                passage_source=passage_source,
                multi_source=multi_source
            )

            row_scores.append(
                prediction_meteor_score(
                    row["gold_text"],
                    prediction
                )
            )

        meta_results.append({
            "model": model_name,
            "phrase_source": phrase_source,
            "passage_source": passage_source,
            "multi_source": multi_source,
            "type_accuracy": float(
                type_accuracy
            ),
            "type_macro_f1": float(
                type_macro_f1
            ),
            "mean_meteor": float(
                np.mean(row_scores)
            ),
            "median_meteor": float(
                np.median(row_scores)
            )
        })


meta_results_df = (
    pd.DataFrame(meta_results)
    .sort_values(
        [
            "mean_meteor",
            "type_macro_f1"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

best_meta_result_by_model_df = (
    meta_results_df
    .groupby(
        "model",
        as_index=False
    )
    .first()
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\nTotal meta-classifier runtime:",
    round(
        time.time() - meta_start_time,
        2
    ),
    "seconds"
)

print("\nCurrent RoBERTa type accuracy:")
print(0.755)

print("\nCurrent deployable hybrid METEOR:")
print(0.4169037738283222)

print("\nBest OOF result for each meta type classifier:")
display(best_meta_result_by_model_df)

print("\nTop 20 OOF type-routing configurations:")
display(meta_results_df.head(20))

Completed logistic: 3.1 seconds
Completed random_forest: 35.5 seconds
Completed extra_trees: 52.1 seconds
Completed hist_gradient_boosting: 68.9 seconds

Total meta-classifier runtime: 90.02 seconds

Current RoBERTa type accuracy:
0.755

Current deployable hybrid METEOR:
0.4169037738283222

Best OOF result for each meta type classifier:


,model,phrase_source,passage_source,multi_source,type_accuracy,type_macro_f1,mean_meteor,median_meteor
0,hist_gradient_boosting,article,article,article,0.7400,0.727684,0.413604,0.385943
1,extra_trees,article,article,article,0.7400,0.729496,0.409756,0.368396
2,random_forest,paragraph,article,article,0.7450,0.734686,0.408803,0.384615
3,ensemble_all,article,article,article,0.7400,0.726744,0.407609,0.368396
4,logistic,article,article,article,0.7425,0.727769,0.404260,0.334821



Top 20 OOF type-routing configurations:


,model,phrase_source,passage_source,multi_source,type_accuracy,type_macro_f1,mean_meteor,median_meteor
0,hist_gradient_boosting,article,article,article,0.7400,0.727684,0.413604,0.385943
1,extra_trees,article,article,article,0.7400,0.729496,0.409756,0.368396
2,random_forest,paragraph,article,article,0.7450,0.734686,0.408803,0.384615
3,random_forest,article,article,article,0.7450,0.734686,0.408769,0.368396
4,extra_trees,paragraph,article,article,0.7400,0.729496,0.408589,0.384615
5,hist_gradient_boosting,paragraph,article,article,0.7400,0.727684,0.407910,0.384212
6,ensemble_all,article,article,article,0.7400,0.726744,0.407609,0.368396
7,ensemble_all,paragraph,article,article,0.7400,0.726744,0.406942,0.378262
8,logistic,article,article,article,0.7425,0.727769,0.404260,0.334821
9,hist_gradient_boosting,article,article,paragraph,0.7400,0.727684,0.404205,0.360610


## again it failed to beat our current highest accuracy and METEOR. Another approach we can try would be the prediction fusion.

In [114]:
import itertools
import re
import numpy as np
import pandas as pd


fusion_df = routing_diagnostic_df.copy()


def comparison_tokens(text):
    return re.findall(
        r"\b[\w’'-]+\b",
        normalize_prediction_text(text).lower()
    )


def simple_token_jaccard(text_a, text_b):
    tokens_a = set(comparison_tokens(text_a))
    tokens_b = set(comparison_tokens(text_b))

    if not tokens_a and not tokens_b:
        return 1.0

    if not tokens_a or not tokens_b:
        return 0.0

    return (
        len(tokens_a & tokens_b)
        / len(tokens_a | tokens_b)
    )


def simple_containment(text_a, text_b):
    tokens_a = set(comparison_tokens(text_a))
    tokens_b = set(comparison_tokens(text_b))

    smaller_size = min(
        len(tokens_a),
        len(tokens_b)
    )

    if smaller_size == 0:
        return 0.0

    return (
        len(tokens_a & tokens_b)
        / smaller_size
    )


def choose_shorter_text(text_a, text_b):
    text_a = normalize_prediction_text(text_a)
    text_b = normalize_prediction_text(text_b)

    if not text_a:
        return text_b

    if not text_b:
        return text_a

    if len(text_a.split()) <= len(text_b.split()):
        return text_a

    return text_b


def choose_longer_text(text_a, text_b):
    text_a = normalize_prediction_text(text_a)
    text_b = normalize_prediction_text(text_b)

    if not text_a:
        return text_b

    if not text_b:
        return text_a

    if len(text_a.split()) >= len(text_b.split()):
        return text_a

    return text_b


def concatenate_without_exact_duplication(
    first_text,
    second_text
):
    first_text = normalize_prediction_text(first_text)
    second_text = normalize_prediction_text(second_text)

    if not first_text:
        return second_text

    if not second_text:
        return first_text

    first_lower = first_text.lower()
    second_lower = second_text.lower()

    if first_lower in second_lower:
        return second_text

    if second_lower in first_lower:
        return first_text

    return normalize_prediction_text(
        first_text + " " + second_text
    )


def longest_common_contiguous_span(
    text_a,
    text_b
):
    tokens_a = comparison_tokens(text_a)
    tokens_b = comparison_tokens(text_b)

    if not tokens_a or not tokens_b:
        return ""

    previous_row = np.zeros(
        len(tokens_b) + 1,
        dtype=int
    )

    best_length = 0
    best_end_position = 0

    for position_a, token_a in enumerate(
        tokens_a,
        start=1
    ):
        current_row = np.zeros(
            len(tokens_b) + 1,
            dtype=int
        )

        for position_b, token_b in enumerate(
            tokens_b,
            start=1
        ):
            if token_a == token_b:
                current_row[position_b] = (
                    previous_row[position_b - 1]
                    + 1
                )

                if (
                    current_row[position_b]
                    > best_length
                ):
                    best_length = int(
                        current_row[position_b]
                    )

                    best_end_position = position_a

        previous_row = current_row

    if best_length == 0:
        return ""

    best_start_position = (
        best_end_position - best_length
    )

    return " ".join(
        tokens_a[
            best_start_position:
            best_end_position
        ]
    )


def type_length_cap(predicted_type):
    if predicted_type == "phrase":
        return 50

    return 100


def cap_or_fallback(
    candidate,
    fallback,
    predicted_type
):
    candidate = normalize_prediction_text(
        candidate
    )

    fallback = normalize_prediction_text(
        fallback
    )

    maximum_words = type_length_cap(
        predicted_type
    )

    if (
        candidate
        and len(candidate.split())
        <= maximum_words
    ):
        return candidate

    return fallback


fusion_outputs = {
    "paragraph": [],
    "article": [],
    "current_rule": [],
    "shorter": [],
    "longer": [],
    "concat_paragraph_article": [],
    "concat_article_paragraph": [],
    "concat_paragraph_article_capped": [],
    "concat_article_paragraph_capped": [],
    "common_span": [],
    "common_or_shorter": [],
    "common_or_current": [],
    "overlap_shorter_else_concat": [],
    "overlap_longer_else_current": []
}


for _, row in fusion_df.iterrows():
    paragraph = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    article = normalize_prediction_text(
        row["article_cleaned"]
    )

    predicted_type = str(
        row["roberta_predicted_type"]
    )

    current_prediction = (
        article
        if row["current_selected_model"]
        == "article"
        else paragraph
    )

    shorter_prediction = choose_shorter_text(
        paragraph,
        article
    )

    longer_prediction = choose_longer_text(
        paragraph,
        article
    )

    paragraph_article_concat = (
        concatenate_without_exact_duplication(
            paragraph,
            article
        )
    )

    article_paragraph_concat = (
        concatenate_without_exact_duplication(
            article,
            paragraph
        )
    )

    common_span = (
        longest_common_contiguous_span(
            paragraph,
            article
        )
    )

    jaccard = simple_token_jaccard(
        paragraph,
        article
    )

    containment = simple_containment(
        paragraph,
        article
    )

    high_overlap = (
        jaccard >= 0.50
        or containment >= 0.85
    )

    overlap_shorter_else_concat = (
        shorter_prediction
        if high_overlap
        else cap_or_fallback(
            paragraph_article_concat,
            current_prediction,
            predicted_type
        )
    )

    overlap_longer_else_current = (
        cap_or_fallback(
            longer_prediction,
            current_prediction,
            predicted_type
        )
        if high_overlap
        else current_prediction
    )

    fusion_outputs["paragraph"].append(
        paragraph
    )

    fusion_outputs["article"].append(
        article
    )

    fusion_outputs["current_rule"].append(
        current_prediction
    )

    fusion_outputs["shorter"].append(
        shorter_prediction
    )

    fusion_outputs["longer"].append(
        longer_prediction
    )

    fusion_outputs[
        "concat_paragraph_article"
    ].append(
        paragraph_article_concat
    )

    fusion_outputs[
        "concat_article_paragraph"
    ].append(
        article_paragraph_concat
    )

    fusion_outputs[
        "concat_paragraph_article_capped"
    ].append(
        cap_or_fallback(
            paragraph_article_concat,
            current_prediction,
            predicted_type
        )
    )

    fusion_outputs[
        "concat_article_paragraph_capped"
    ].append(
        cap_or_fallback(
            article_paragraph_concat,
            current_prediction,
            predicted_type
        )
    )

    fusion_outputs["common_span"].append(
        common_span
    )

    fusion_outputs["common_or_shorter"].append(
        common_span
        if common_span
        else shorter_prediction
    )

    fusion_outputs["common_or_current"].append(
        common_span
        if common_span
        else current_prediction
    )

    fusion_outputs[
        "overlap_shorter_else_concat"
    ].append(
        overlap_shorter_else_concat
    )

    fusion_outputs[
        "overlap_longer_else_current"
    ].append(
        overlap_longer_else_current
    )

# Score every fusion operator

fusion_score_columns = {}

for operator_name, predictions in (
    fusion_outputs.items()
):
    fusion_df[
        f"prediction_{operator_name}"
    ] = predictions

    fusion_df[
        f"meteor_{operator_name}"
    ] = [
        prediction_meteor_score(
            gold_text,
            prediction
        )
        for gold_text, prediction in zip(
            fusion_df["gold_text"],
            predictions
        )
    ]

    fusion_score_columns[
        operator_name
    ] = f"meteor_{operator_name}"


global_fusion_results = []

for operator_name, score_column in (
    fusion_score_columns.items()
):
    global_fusion_results.append({
        "operator": operator_name,
        "mean_meteor": float(
            fusion_df[score_column].mean()
        ),
        "median_meteor": float(
            fusion_df[score_column].median()
        ),
        "mean_words": float(
            fusion_df[
                f"prediction_{operator_name}"
            ]
            .fillna("")
            .str.split()
            .str.len()
            .mean()
        ),
        "max_words": int(
            fusion_df[
                f"prediction_{operator_name}"
            ]
            .fillna("")
            .str.split()
            .str.len()
            .max()
        )
    })


global_fusion_results_df = (
    pd.DataFrame(global_fusion_results)
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

# Best operator separately for each predicted type

per_type_fusion_results = []

for predicted_type, type_group in (
    fusion_df.groupby(
        "roberta_predicted_type"
    )
):
    for operator_name, score_column in (
        fusion_score_columns.items()
    ):
        per_type_fusion_results.append({
            "predicted_type": predicted_type,
            "operator": operator_name,
            "rows": len(type_group),
            "mean_meteor": float(
                type_group[score_column].mean()
            ),
            "median_meteor": float(
                type_group[score_column].median()
            )
        })


per_type_fusion_results_df = (
    pd.DataFrame(per_type_fusion_results)
    .sort_values(
        [
            "predicted_type",
            "mean_meteor"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

# Search type-specific operator combinations

operator_names = list(
    fusion_score_columns.keys()
)

predicted_type_array = (
    fusion_df["roberta_predicted_type"]
    .astype(str)
    .to_numpy()
)

score_arrays = {
    operator_name: (
        fusion_df[
            score_column
        ].to_numpy()
    )
    for operator_name, score_column
    in fusion_score_columns.items()
}

type_specific_results = []

for (
    phrase_operator,
    passage_operator,
    multi_operator
) in itertools.product(
    operator_names,
    repeat=3
):
    selected_scores = np.zeros(
        len(fusion_df),
        dtype=float
    )

    phrase_mask = (
        predicted_type_array == "phrase"
    )

    passage_mask = (
        predicted_type_array == "passage"
    )

    multi_mask = (
        predicted_type_array == "multi"
    )

    selected_scores[phrase_mask] = (
        score_arrays[
            phrase_operator
        ][phrase_mask]
    )

    selected_scores[passage_mask] = (
        score_arrays[
            passage_operator
        ][passage_mask]
    )

    selected_scores[multi_mask] = (
        score_arrays[
            multi_operator
        ][multi_mask]
    )

    type_specific_results.append({
        "phrase_operator": phrase_operator,
        "passage_operator": passage_operator,
        "multi_operator": multi_operator,
        "mean_meteor": float(
            selected_scores.mean()
        ),
        "median_meteor": float(
            np.median(selected_scores)
        )
    })


type_specific_fusion_results_df = (
    pd.DataFrame(type_specific_results)
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)


print("Current deployable hybrid:")
print(0.4169037738283222)

print("\nGlobal fusion operators:")
display(global_fusion_results_df)

print("\nBest five operators for each predicted type:")
display(
    per_type_fusion_results_df.groupby(
        "predicted_type",
        group_keys=False
    ).head(5)
)

print("\nTop 20 type-specific fusion combinations:")
display(
    type_specific_fusion_results_df.head(20)
)

Current deployable hybrid:
0.4169037738283222

Global fusion operators:


,operator,mean_meteor,median_meteor,mean_words,max_words
0,concat_paragraph_article_capped,0.427102,0.416667,43.3475,101
1,concat_article_paragraph_capped,0.425366,0.416667,43.3475,101
2,concat_article_paragraph,0.422452,0.414474,65.4625,249
3,concat_paragraph_article,0.421696,0.412978,65.4625,249
4,overlap_shorter_else_concat,0.418186,0.411013,36.0300,101
5,current_rule,0.416904,0.373857,31.3350,101
6,overlap_longer_else_current,0.416446,0.388326,35.7850,101
7,longer,0.406649,0.384333,45.4325,178
8,article,0.405861,0.359757,36.6625,178
9,paragraph,0.393997,0.350530,32.1250,132



Best five operators for each predicted type:


,predicted_type,operator,rows,mean_meteor,median_meteor
0,multi,concat_article_paragraph,83,0.397504,0.405455
1,multi,concat_paragraph_article,83,0.395009,0.405455
2,multi,concat_paragraph_article_capped,83,0.389803,0.383809
3,multi,concat_article_paragraph_capped,83,0.385337,0.383809
4,multi,overlap_shorter_else_concat,83,0.379410,0.372714
14,passage,current_rule,168,0.387260,0.288078
15,passage,overlap_shorter_else_concat,168,0.381158,0.276111
16,passage,article,168,0.380245,0.272585
17,passage,overlap_longer_else_current,168,0.379668,0.297214
18,passage,shorter,168,0.373691,0.266910



Top 20 type-specific fusion combinations:


,phrase_operator,passage_operator,multi_operator,mean_meteor,median_meteor
0,concat_paragraph_article_capped,current_rule,concat_article_paragraph,0.436008,0.417805
1,concat_paragraph_article_capped,current_rule,concat_paragraph_article,0.435490,0.416667
2,concat_paragraph_article,current_rule,concat_article_paragraph,0.435405,0.417805
3,concat_paragraph_article,current_rule,concat_paragraph_article,0.434888,0.416667
4,concat_article_paragraph_capped,current_rule,concat_article_paragraph,0.434423,0.416667
5,concat_paragraph_article_capped,current_rule,concat_paragraph_article_capped,0.434410,0.416667
6,concat_article_paragraph_capped,current_rule,concat_paragraph_article,0.433905,0.416667
7,concat_article_paragraph,current_rule,concat_article_paragraph,0.433820,0.416667
8,concat_paragraph_article,current_rule,concat_paragraph_article_capped,0.433808,0.416667
9,concat_paragraph_article_capped,current_rule,concat_article_paragraph_capped,0.433483,0.416667


## seems like there are chances that this will beat our current system, run the stability test and check how they preform on unseem validation rows

In [115]:
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold

# 1. Prepare score arrays

fusion_types = (
    fusion_df["roberta_predicted_type"]
    .astype(str)
    .to_numpy()
)

operator_names = list(
    fusion_score_columns.keys()
)

operator_score_arrays = {
    operator_name: fusion_df[
        score_column
    ].to_numpy()
    for operator_name, score_column
    in fusion_score_columns.items()
}

current_scores = operator_score_arrays[
    "current_rule"
]

simple_global_scores = operator_score_arrays[
    "concat_paragraph_article_capped"
]


def scores_for_type_specific_combo(
    phrase_operator,
    passage_operator,
    multi_operator
):
    scores = np.zeros(
        len(fusion_df),
        dtype=float
    )

    operator_by_type = {
        "phrase": phrase_operator,
        "passage": passage_operator,
        "multi": multi_operator
    }

    for spoiler_type, operator_name in (
        operator_by_type.items()
    ):
        mask = fusion_types == spoiler_type

        scores[mask] = (
            operator_score_arrays[
                operator_name
            ][mask]
        )

    return scores


# Full-validation best combination.
fixed_best_scores = (
    scores_for_type_specific_combo(
        phrase_operator=(
            "concat_paragraph_article_capped"
        ),
        passage_operator="current_rule",
        multi_operator=(
            "concat_article_paragraph"
        )
    )
)

# Safer version that caps multi predictions too.
fixed_capped_multi_scores = (
    scores_for_type_specific_combo(
        phrase_operator=(
            "concat_paragraph_article_capped"
        ),
        passage_operator="current_rule",
        multi_operator=(
            "concat_article_paragraph_capped"
        )
    )
)

# 2. Repeated split stability evaluation

stability_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=20,
    random_state=SEED
)

stability_records = []
selected_combo_counter = Counter()

dummy_X = np.zeros(
    (len(fusion_df), 1)
)

for split_number, (
    train_indices,
    validation_indices
) in enumerate(
    stability_cv.split(
        dummy_X,
        fusion_types
    ),
    start=1
):
    selected_operator_by_type = {}

    # Select the best operator separately for each
    # predicted type using training rows only.
    for predicted_type in [
        "phrase",
        "passage",
        "multi"
    ]:
        train_type_mask = (
            fusion_types[train_indices]
            == predicted_type
        )

        train_type_indices = (
            train_indices[
                train_type_mask
            ]
        )

        operator_train_scores = {}

        for operator_name in operator_names:
            operator_train_scores[
                operator_name
            ] = float(
                operator_score_arrays[
                    operator_name
                ][train_type_indices].mean()
            )

        selected_operator = max(
            operator_train_scores,
            key=operator_train_scores.get
        )

        selected_operator_by_type[
            predicted_type
        ] = selected_operator

    selected_combo = (
        selected_operator_by_type["phrase"],
        selected_operator_by_type["passage"],
        selected_operator_by_type["multi"]
    )

    selected_combo_counter[
        selected_combo
    ] += 1

    nested_validation_scores = np.zeros(
        len(validation_indices),
        dtype=float
    )

    for predicted_type in [
        "phrase",
        "passage",
        "multi"
    ]:
        validation_type_mask = (
            fusion_types[
                validation_indices
            ] == predicted_type
        )

        validation_type_positions = (
            np.flatnonzero(
                validation_type_mask
            )
        )

        validation_type_indices = (
            validation_indices[
                validation_type_mask
            ]
        )

        selected_operator = (
            selected_operator_by_type[
                predicted_type
            ]
        )

        nested_validation_scores[
            validation_type_positions
        ] = operator_score_arrays[
            selected_operator
        ][validation_type_indices]

    stability_records.append({
        "split": split_number,

        "current_rule": float(
            current_scores[
                validation_indices
            ].mean()
        ),

        "simple_global_capped_concat": float(
            simple_global_scores[
                validation_indices
            ].mean()
        ),

        "fixed_best_type_specific": float(
            fixed_best_scores[
                validation_indices
            ].mean()
        ),

        "fixed_capped_multi": float(
            fixed_capped_multi_scores[
                validation_indices
            ].mean()
        ),

        "nested_selected_type_specific": float(
            nested_validation_scores.mean()
        ),

        "selected_phrase_operator": (
            selected_operator_by_type[
                "phrase"
            ]
        ),

        "selected_passage_operator": (
            selected_operator_by_type[
                "passage"
            ]
        ),

        "selected_multi_operator": (
            selected_operator_by_type[
                "multi"
            ]
        )
    })


fusion_stability_df = pd.DataFrame(
    stability_records
)

# 3. Summarize stability

strategy_columns = [
    "current_rule",
    "simple_global_capped_concat",
    "fixed_best_type_specific",
    "fixed_capped_multi",
    "nested_selected_type_specific"
]

stability_summary_records = []

for strategy in strategy_columns:
    split_scores = (
        fusion_stability_df[strategy]
    )

    improvement_over_current = (
        split_scores
        - fusion_stability_df[
            "current_rule"
        ]
    )

    stability_summary_records.append({
        "strategy": strategy,
        "mean_meteor": float(
            split_scores.mean()
        ),
        "std_across_splits": float(
            split_scores.std()
        ),
        "fifth_percentile": float(
            split_scores.quantile(0.05)
        ),
        "median": float(
            split_scores.median()
        ),
        "ninety_fifth_percentile": float(
            split_scores.quantile(0.95)
        ),
        "mean_gain_over_current": float(
            improvement_over_current.mean()
        ),
        "win_rate_vs_current": float(
            (
                improvement_over_current > 0
            ).mean()
        )
    })


fusion_stability_summary_df = (
    pd.DataFrame(
        stability_summary_records
    )
    .sort_values(
        "mean_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Full-validation scores:")
print(
    "Current rule:",
    round(current_scores.mean(), 6)
)
print(
    "Simple global capped concat:",
    round(
        simple_global_scores.mean(),
        6
    )
)
print(
    "Fixed best type-specific:",
    round(
        fixed_best_scores.mean(),
        6
    )
)
print(
    "Fixed capped-multi version:",
    round(
        fixed_capped_multi_scores.mean(),
        6
    )
)

print("\nRepeated-split stability summary:")
display(
    fusion_stability_summary_df
)

print(
    "\nMost frequently selected "
    "type-specific combinations:"
)

most_common_combinations = []

for combination, frequency in (
    selected_combo_counter.most_common(15)
):
    most_common_combinations.append({
        "phrase_operator": combination[0],
        "passage_operator": combination[1],
        "multi_operator": combination[2],
        "selected_folds": frequency,
        "selection_rate": (
            frequency
            / len(fusion_stability_df)
        )
    })

display(
    pd.DataFrame(
        most_common_combinations
    )
)

Full-validation scores:
Current rule: 0.416904
Simple global capped concat: 0.427102
Fixed best type-specific: 0.436008
Fixed capped-multi version: 0.433483

Repeated-split stability summary:


,strategy,mean_meteor,std_across_splits,fifth_percentile,median,ninety_fifth_percentile,mean_gain_over_current,win_rate_vs_current
0,fixed_best_type_specific,0.436008,0.035412,0.386730,0.434937,0.489534,0.019104,0.95
1,fixed_capped_multi,0.433483,0.035061,0.385472,0.433809,0.490309,0.016579,0.90
2,simple_global_capped_concat,0.427102,0.030733,0.381330,0.423812,0.480985,0.010199,0.75
3,nested_selected_type_specific,0.426945,0.031414,0.385970,0.426070,0.479089,0.010041,0.78
4,current_rule,0.416904,0.036623,0.357121,0.418910,0.473985,0.000000,0.00



Most frequently selected type-specific combinations:


,phrase_operator,passage_operator,multi_operator,selected_folds,selection_rate
0,concat_paragraph_article_capped,current_rule,concat_article_paragraph,43,0.43
1,concat_paragraph_article_capped,current_rule,concat_paragraph_article,12,0.12
2,concat_paragraph_article,current_rule,concat_article_paragraph,11,0.11
3,concat_paragraph_article_capped,current_rule,concat_paragraph_article_capped,6,0.06
4,concat_paragraph_article_capped,longer,concat_article_paragraph,5,0.05
5,concat_paragraph_article_capped,overlap_longer_else_current,concat_article_paragraph,4,0.04
6,concat_paragraph_article_capped,overlap_shorter_else_concat,concat_article_paragraph,3,0.03
7,concat_paragraph_article,current_rule,concat_paragraph_article,2,0.02
8,concat_paragraph_article_capped,overlap_longer_else_current,concat_paragraph_article,2,0.02
9,concat_paragraph_article_capped,paragraph,concat_article_paragraph,2,0.02


## the result seems pretty promising, we should test the aggressive best version first

In [116]:
import numpy as np
import pandas as pd


fusion_test_records = []

for _, row in final_hybrid_test_df.iterrows():
    row_number = int(row["row_number"])
    predicted_type = str(row["predicted_type"])

    paragraph_prediction = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    article_prediction = remove_synthetic_article_markers(
        row["article_prediction_cleaned"]
    )

    current_hybrid_prediction = normalize_prediction_text(
        row["spoiler"]
    )

    if predicted_type == "phrase":
        # Best phrase operator:
        # paragraph prediction followed by article prediction,
        # capped at 50 words with current hybrid as fallback.
        fused_candidate = (
            concatenate_without_exact_duplication(
                paragraph_prediction,
                article_prediction
            )
        )

        final_prediction = cap_or_fallback(
            candidate=fused_candidate,
            fallback=current_hybrid_prediction,
            predicted_type="phrase"
        )

        fusion_operator = (
            "concat_paragraph_article_capped"
        )

    elif predicted_type == "passage":
        # Passage fusion did not improve validation,
        # so preserve the successful 0.43851 strategy.
        final_prediction = current_hybrid_prediction
        fusion_operator = "current_rule"

    else:
        # Best multi operator:
        # article prediction followed by paragraph prediction.
        final_prediction = (
            concatenate_without_exact_duplication(
                article_prediction,
                paragraph_prediction
            )
        )

        if not final_prediction:
            final_prediction = current_hybrid_prediction

        fusion_operator = (
            "concat_article_paragraph"
        )

    final_prediction = remove_synthetic_article_markers(
        final_prediction
    )

    fusion_test_records.append({
        "row_number": row_number,
        "id": row["id"],
        "predicted_type": predicted_type,
        "fusion_operator": fusion_operator,
        "paragraph_prediction": paragraph_prediction,
        "article_prediction": article_prediction,
        "previous_hybrid_prediction": (
            current_hybrid_prediction
        ),
        "spoiler": final_prediction,
        "word_count": len(
            final_prediction.split()
        ),
        "changed_from_043851": (
            final_prediction
            != current_hybrid_prediction
        )
    })


fusion_test_df = pd.DataFrame(
    fusion_test_records
).sort_values(
    "row_number"
).reset_index(drop=True)


marker_pattern = (
    r"(?:Title|Paragraph\s+\d+):"
)

print("Fusion operator counts:")
print(
    fusion_test_df[
        "fusion_operator"
    ].value_counts()
)

print("\nChanged predictions by predicted type:")
print(
    pd.crosstab(
        fusion_test_df["predicted_type"],
        fusion_test_df[
            "changed_from_043851"
        ]
    )
)

print(
    "\nTotal predictions changed from 0.43851:",
    int(
        fusion_test_df[
            "changed_from_043851"
        ].sum()
    )
)

print(
    "Empty predictions:",
    int(
        fusion_test_df["spoiler"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Predictions containing synthetic markers:",
    int(
        fusion_test_df["spoiler"]
        .str.contains(
            marker_pattern,
            case=False,
            regex=True,
            na=False
        )
        .sum()
    )
)

print("\nWord-count summary:")
print(
    fusion_test_df[
        "word_count"
    ].describe()
)

print("\nWord-count summary by predicted type:")
display(
    fusion_test_df.groupby(
        "predicted_type"
    )["word_count"].describe()
)

print("\nLongest fusion predictions:")
display(
    fusion_test_df.sort_values(
        "word_count",
        ascending=False
    )[
        [
            "row_number",
            "id",
            "predicted_type",
            "fusion_operator",
            "paragraph_prediction",
            "article_prediction",
            "spoiler",
            "word_count"
        ]
    ].head(20)
)

print("\nSample changed predictions:")
display(
    fusion_test_df[
        fusion_test_df[
            "changed_from_043851"
        ]
    ][
        [
            "row_number",
            "id",
            "predicted_type",
            "fusion_operator",
            "previous_hybrid_prediction",
            "spoiler",
            "word_count"
        ]
    ].head(20)
)

Fusion operator counts:
fusion_operator
current_rule                       183
concat_paragraph_article_capped    151
concat_article_paragraph            66
Name: count, dtype: int64

Changed predictions by predicted type:
changed_from_043851  False  True 
predicted_type                   
multi                    1     65
passage                183      0
phrase                  97     54

Total predictions changed from 0.43851: 119
Empty predictions: 0
Predictions containing synthetic markers: 0

Word-count summary:
count    400.000000
mean      37.347500
std       33.982109
min        1.000000
25%        6.000000
50%       31.000000
75%       60.250000
max      190.000000
Name: word_count, dtype: float64

Word-count summary by predicted type:


,count,mean,std,min,25%,50%,75%,max
predicted_type,,,,,,,,
multi,66.0,74.772727,36.371500,14.0,51.0,75.0,99.5,190.0
passage,183.0,46.688525,25.804128,4.0,25.5,43.0,65.5,100.0
phrase,151.0,9.668874,14.022231,1.0,2.0,3.0,12.5,103.0



Longest fusion predictions:


,row_number,id,predicted_type,fusion_operator,paragraph_prediction,article_prediction,spoiler,word_count
384,384,384,multi,concat_article_paragraph,monoclonal antibody monoclonal antibodies The ...,"In doing so he developed, together with George...","In doing so he developed, together with George...",190
115,115,115,multi,concat_article_paragraph,After getting my DNA report I learned that the...,After getting my DNA report I learned that the...,After getting my DNA report I learned that the...,154
388,388,388,multi,concat_article_paragraph,Banking on bad movies Mary-Kate's Kool wedding...,"Banking on bad movies Yeah, Mary-Kate and Ashl...","Banking on bad movies Yeah, Mary-Kate and Ashl...",139
177,177,177,multi,concat_article_paragraph,dropped from $21.75 to $18.56 Valve announced ...,What happened to CS:GO skin prices after Valve...,What happened to CS:GO skin prices after Valve...,127
70,70,70,multi,concat_article_paragraph,"New Home Button, New Limitations Secondly you ...",The famously diligent test site ran the iPhone...,The famously diligent test site ran the iPhone...,126
108,108,108,multi,concat_article_paragraph,"Height: 5'5"" Delores Curtis Eating what I want...",After Barely Recognizing Herself In A Family P...,After Barely Recognizing Herself In A Family P...,123
101,101,101,multi,concat_article_paragraph,"SECOND WIND LOCATION, LOCATION, LOCATION FRESH...","""There are 2,000 products that are going to be...","""There are 2,000 products that are going to be...",116
385,385,385,multi,concat_article_paragraph,AfrikaBurn - Tankwa Karoo BOOM – Portugal Elec...,Here we round up some global options when it c...,Here we round up some global options when it c...,113
57,57,57,multi,concat_article_paragraph,Louise the infant koala Disgruntled over the p...,"Louise the infant koala — a squeaking, wet, gr...","Louise the infant koala — a squeaking, wet, gr...",113
281,281,281,multi,concat_article_paragraph,"3. When cats rub their head against you, they’...","3. When cats rub their head against you, they’...","3. When cats rub their head against you, they’...",110



Sample changed predictions:


,row_number,id,predicted_type,fusion_operator,previous_hybrid_prediction,spoiler,word_count
4,4,4,phrase,concat_paragraph_article_capped,3. Remove the egg yolks from the fridge after ...,3. Remove the egg yolks from the fridge after ...,35
5,5,5,phrase,concat_paragraph_article_capped,"""Main Title,"" ""The Imperial March,"" ""Princess ...",John Williams music. To celebrate the fake hol...,44
6,6,6,multi,concat_article_paragraph,"1. Take a long, warm shower with sweet-smellin...","1. Take a long, warm shower with sweet-smellin...",73
9,9,9,multi,concat_article_paragraph,When an audience member asked Brandon and this...,When an audience member asked Brandon and this...,103
14,14,14,multi,concat_article_paragraph,When he goes to a meet-up he’s planned with a ...,When he goes to a meet-up he’s planned with a ...,87
15,15,15,phrase,concat_paragraph_article_capped,$117 billion,$117 billion $123.6 billion,4
24,24,24,multi,concat_article_paragraph,"On the website, people can donate to sexual ab...",Men who paint one fingernail are helping to ra...,106
29,29,29,multi,concat_article_paragraph,"The program, to be carried out in nearby Oakla...","The program, to be carried out in nearby Oakla...",82
30,30,30,phrase,concat_paragraph_article_capped,3. Political empowerment 4. Sweden,2. Finland 17. South Africa 10. Nicaragua 3. P...,12
32,32,32,phrase,concat_paragraph_article_capped,Tesco,They are retail dinosaurs Tesco,5


In [117]:
import os
import pandas as pd

FUSION_SUBMISSION_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_type_specific_fusion_aggressive.csv"
)

FUSION_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_type_specific_fusion_aggressive_audit.csv"
)

# Match the sample submission order exactly.
fusion_submission_df = (
    sample_df[["id"]]
    .merge(
        fusion_test_df[
            ["id", "spoiler"]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
)

fusion_submission_df.to_csv(
    FUSION_SUBMISSION_PATH,
    index=False
)

fusion_test_df.to_csv(
    FUSION_AUDIT_PATH,
    index=False
)

ids_match_sample = (
    fusion_submission_df["id"].tolist()
    == sample_df["id"].tolist()
)

ids_match_test = (
    fusion_submission_df["id"].tolist()
    == test_df["id"].tolist()
)

empty_count = int(
    fusion_submission_df["spoiler"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

marker_count = int(
    fusion_submission_df["spoiler"]
    .astype(str)
    .str.contains(
        r"(?:Title|Paragraph\s+\d+):",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

word_counts = (
    fusion_submission_df["spoiler"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

print(
    "Submission exists:",
    os.path.exists(FUSION_SUBMISSION_PATH)
)
print("Submission shape:", fusion_submission_df.shape)
print(
    "Submission columns:",
    fusion_submission_df.columns.tolist()
)
print("IDs match sample order:", ids_match_sample)
print("IDs match test order:", ids_match_test)
print(
    "Unique IDs:",
    fusion_submission_df["id"].nunique()
)
print("Empty spoilers:", empty_count)
print("Synthetic markers:", marker_count)
print(
    "Predictions changed from 0.43851:",
    int(
        fusion_test_df[
            "changed_from_043851"
        ].sum()
    )
)
print(
    "Minimum words:",
    int(word_counts.min())
)
print(
    "Maximum words:",
    int(word_counts.max())
)
print(
    "Submission location:",
    FUSION_SUBMISSION_PATH
)
print(
    "Audit location:",
    FUSION_AUDIT_PATH
)

print("\nFirst five rows:")
display(fusion_submission_df.head())

print("\nLast five rows:")
display(fusion_submission_df.tail())

Submission exists: True
Submission shape: (400, 2)
Submission columns: ['id', 'spoiler']
IDs match sample order: True
IDs match test order: True
Unique IDs: 400
Empty spoilers: 0
Synthetic markers: 0
Predictions changed from 0.43851: 119
Minimum words: 1
Maximum words: 190
Submission location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_type_specific_fusion_aggressive.csv
Audit location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_type_specific_fusion_aggressive_audit.csv

First five rows:


,id,spoiler
0,0,"balloons and a sign in hand that reads, ""Heard..."
1,1,Why you SHOULD be selfish at work: Helping oth...
2,2,Have a Bunch of Money
3,3,Braconid
4,4,3. Remove the egg yolks from the fridge after ...



Last five rows:


,id,spoiler
395,395,This Is What Happens When You Leave A Hotel Cl...
396,396,Christopher Suprun
397,397,No Medication. High fat vegan plant based diet...
398,398,WikiLeaks regularly tweets about Assange’s sta...
399,399,Richard Belzer


## Another promising result, we are now at Kaggle score of 0.45+. Next is to try a phrase fusion cleanup to see if we can boost the mark further

In [119]:
import re
import numpy as np
import pandas as pd


def normalized_word_key(token):
    return re.sub(
        r"(^[^\w]+|[^\w]+$)",
        "",
        str(token).lower()
    )


def remove_adjacent_duplicate_tokens(text):
    """
    Conservatively remove directly repeated tokens:
    'blue blue' -> 'blue'
    """
    text = normalize_prediction_text(text)
    tokens = text.split()

    cleaned_tokens = []

    for token in tokens:
        token_key = normalized_word_key(token)

        if cleaned_tokens:
            previous_key = normalized_word_key(
                cleaned_tokens[-1]
            )

            if (
                token_key
                and token_key == previous_key
            ):
                continue

        cleaned_tokens.append(token)

    return normalize_prediction_text(
        " ".join(cleaned_tokens)
    )


def remove_repeated_suffixes(
    text,
    maximum_ngram=8
):
    """
    Remove a repeated ending when the same token sequence
    already occurred earlier.

    Example:
    'Miley Ray Cyrus Miley Cyrus Cyrus'
    can remove repeated suffix material without rewriting
    the rest of the prediction.
    """
    text = remove_adjacent_duplicate_tokens(text)
    tokens = text.split()

    changed = True

    while changed and len(tokens) > 1:
        changed = False

        normalized_tokens = [
            normalized_word_key(token)
            for token in tokens
        ]

        largest_ngram = min(
            maximum_ngram,
            len(tokens) // 2
        )

        for ngram_length in range(
            largest_ngram,
            0,
            -1
        ):
            suffix = normalized_tokens[
                -ngram_length:
            ]

            if not all(suffix):
                continue

            found_earlier = False

            latest_start = (
                len(tokens)
                - ngram_length
            )

            for earlier_start in range(
                0,
                latest_start
                - ngram_length
                + 1
            ):
                earlier_ngram = (
                    normalized_tokens[
                        earlier_start:
                        earlier_start
                        + ngram_length
                    ]
                )

                if earlier_ngram == suffix:
                    found_earlier = True
                    break

            if found_earlier:
                tokens = tokens[
                    :-ngram_length
                ]

                changed = True
                break

    return normalize_prediction_text(
        " ".join(tokens)
    )


def append_only_novel_tokens(
    first_text,
    second_text
):
    """
    Keep the first prediction and append only tokens from
    the second prediction that do not already occur in it.
    This is experimental and evaluated only on validation.
    """
    first_text = normalize_prediction_text(
        first_text
    )

    second_text = normalize_prediction_text(
        second_text
    )

    if not first_text:
        return second_text

    if not second_text:
        return first_text

    first_tokens = first_text.split()
    second_tokens = second_text.split()

    seen_keys = {
        normalized_word_key(token)
        for token in first_tokens
        if normalized_word_key(token)
    }

    novel_tokens = []

    for token in second_tokens:
        token_key = normalized_word_key(token)

        if not token_key:
            continue

        if token_key not in seen_keys:
            novel_tokens.append(token)
            seen_keys.add(token_key)

    return normalize_prediction_text(
        " ".join(
            first_tokens + novel_tokens
        )
    )


def phrase_cap_or_current(
    candidate,
    current_prediction,
    cap
):
    candidate = normalize_prediction_text(
        candidate
    )

    current_prediction = (
        normalize_prediction_text(
            current_prediction
        )
    )

    if (
        candidate
        and len(candidate.split()) <= cap
    ):
        return candidate

    return current_prediction


phrase_candidate_records = []

phrase_caps = [
    8,
    10,
    12,
    15,
    20,
    25,
    30,
    40,
    50,
    60
]

for _, row in fusion_df.iterrows():
    paragraph = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    article = normalize_prediction_text(
        row["article_cleaned"]
    )

    current_prediction = (
        normalize_prediction_text(
            row["prediction_current_rule"]
        )
    )

    raw_concat = (
        concatenate_without_exact_duplication(
            paragraph,
            article
        )
    )

    adjacent_cleaned = (
        remove_adjacent_duplicate_tokens(
            raw_concat
        )
    )

    suffix_cleaned = (
        remove_repeated_suffixes(
            raw_concat
        )
    )

    novel_append = append_only_novel_tokens(
        paragraph,
        article
    )

    phrase_candidate_records.append({
        "row_number": int(row["row_number"]),
        "predicted_type": str(
            row["roberta_predicted_type"]
        ),
        "gold_text": row["gold_text"],
        "paragraph": paragraph,
        "article": article,
        "current_prediction": (
            current_prediction
        ),
        "raw_concat": raw_concat,
        "adjacent_cleaned": adjacent_cleaned,
        "suffix_cleaned": suffix_cleaned,
        "novel_append": novel_append
    })


phrase_candidate_df = pd.DataFrame(
    phrase_candidate_records
)

strategy_builders = {
    "raw_concat": lambda row: row[
        "raw_concat"
    ],

    "adjacent_cleaned": lambda row: row[
        "adjacent_cleaned"
    ],

    "suffix_cleaned": lambda row: row[
        "suffix_cleaned"
    ],

    "novel_append": lambda row: row[
        "novel_append"
    ]
}

phrase_optimization_results = []

for strategy_name, strategy_builder in (
    strategy_builders.items()
):
    for cap in phrase_caps:
        predictions = []

        for _, row in (
            phrase_candidate_df.iterrows()
        ):
            predicted_type = row[
                "predicted_type"
            ]

            if predicted_type == "phrase":
                candidate = strategy_builder(row)

                prediction = (
                    phrase_cap_or_current(
                        candidate=candidate,
                        current_prediction=row[
                            "current_prediction"
                        ],
                        cap=cap
                    )
                )

            elif predicted_type == "passage":
                prediction = normalize_prediction_text(
                    fusion_df.loc[
                        fusion_df["row_number"]
                        == row["row_number"],
                        "prediction_current_rule"
                    ].iloc[0]
                )

            else:
                prediction = normalize_prediction_text(
                    fusion_df.loc[
                        fusion_df["row_number"]
                        == row["row_number"],
                        "prediction_concat_article_paragraph"
                    ].iloc[0]
                )

            predictions.append(prediction)

        scores = [
            prediction_meteor_score(
                gold,
                prediction
            )
            for gold, prediction in zip(
                phrase_candidate_df[
                    "gold_text"
                ],
                predictions
            )
        ]

        phrase_mask = (
            phrase_candidate_df[
                "predicted_type"
            ] == "phrase"
        ).to_numpy()

        phrase_scores = np.array(scores)[
            phrase_mask
        ]

        current_phrase_predictions = (
            phrase_candidate_df.loc[
                phrase_mask,
                "current_prediction"
            ].tolist()
        )

        proposed_phrase_predictions = (
            np.array(
                predictions,
                dtype=object
            )[phrase_mask].tolist()
        )

        phrase_optimization_results.append({
            "strategy": strategy_name,
            "phrase_cap": cap,
            "overall_meteor": float(
                np.mean(scores)
            ),
            "phrase_meteor": float(
                phrase_scores.mean()
            ),
            "median_meteor": float(
                np.median(scores)
            ),
            "changed_phrase_rows": int(
                sum(
                    proposed != current
                    for proposed, current
                    in zip(
                        proposed_phrase_predictions,
                        current_phrase_predictions
                    )
                )
            ),
            "maximum_phrase_words": int(
                max(
                    len(
                        prediction.split()
                    )
                    for prediction in (
                        proposed_phrase_predictions
                    )
                )
            )
        })


phrase_optimization_results_df = (
    pd.DataFrame(
        phrase_optimization_results
    )
    .sort_values(
        [
            "overall_meteor",
            "phrase_meteor"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("Current aggressive fusion validation:")
print(0.436008)

print("\nTop 25 phrase-cleanup configurations:")
display(
    phrase_optimization_results_df.head(25)
)

print("\nBest result for each cleanup strategy:")
display(
    phrase_optimization_results_df
    .sort_values(
        "overall_meteor",
        ascending=False
    )
    .groupby(
        "strategy",
        as_index=False
    )
    .first()
    .sort_values(
        "overall_meteor",
        ascending=False
    )
)

print("\nExamples affected by suffix cleanup:")
display(
    phrase_candidate_df[
        (
            phrase_candidate_df[
                "predicted_type"
            ] == "phrase"
        )
        &
        (
            phrase_candidate_df[
                "raw_concat"
            ]
            != phrase_candidate_df[
                "suffix_cleaned"
            ]
        )
    ][
        [
            "row_number",
            "paragraph",
            "article",
            "raw_concat",
            "suffix_cleaned",
            "gold_text"
        ]
    ].head(20)
)

Current aggressive fusion validation:
0.436008

Top 25 phrase-cleanup configurations:


,strategy,phrase_cap,overall_meteor,phrase_meteor,median_meteor,changed_phrase_rows,maximum_phrase_words
0,suffix_cleaned,50,0.438144,0.518156,0.416667,63,49
1,suffix_cleaned,60,0.437570,0.516613,0.416667,67,59
2,suffix_cleaned,30,0.437554,0.516570,0.416667,59,45
3,suffix_cleaned,25,0.437519,0.516477,0.416667,56,45
4,suffix_cleaned,40,0.437389,0.516129,0.416667,61,45
5,adjacent_cleaned,50,0.436891,0.514790,0.417805,59,49
6,adjacent_cleaned,60,0.436645,0.514129,0.417805,62,59
7,suffix_cleaned,10,0.436348,0.513333,0.416667,41,45
8,adjacent_cleaned,30,0.436182,0.512889,0.417805,53,45
9,adjacent_cleaned,40,0.436136,0.512764,0.417805,57,45



Best result for each cleanup strategy:


,strategy,phrase_cap,overall_meteor,phrase_meteor,median_meteor,changed_phrase_rows,maximum_phrase_words
3,suffix_cleaned,50,0.438144,0.518156,0.416667,63,49
0,adjacent_cleaned,50,0.436891,0.514790,0.417805,59,49
2,raw_concat,50,0.436008,0.512420,0.417805,59,49
1,novel_append,50,0.435219,0.510302,0.416667,62,49



Examples affected by suffix cleanup:


,row_number,paragraph,article,raw_concat,suffix_cleaned,gold_text
2,2,$3 to $5 between $5 and $20 20%,20%,$3 to $5 between $5 and $20 20%,$3 to $5 between $5 and $20,20%
44,44,"""but"" ""but"" with the word ""and."" ""I love you, ...","""I love you, but ..."" It could have been ""I lo...","""but"" ""but"" with the word ""and."" ""I love you, ...","""but"" with the word ""and."" ""I love you, but I ...","""but"""
53,53,Santa Clara University Loyola Marymount Univer...,Loyola Marymount University,Santa Clara University Loyola Marymount Univer...,Santa Clara University Loyola Marymount,Santa Clara University
56,56,"""California"" Edan Lepucki Ms. Lepucki",Ms. Lepucki Edan Lepucki,"""California"" Edan Lepucki Ms. Lepucki Ms. Lepu...","""California"" Edan Lepucki Ms.",Edan Lepucki
67,67,"Ring finger to see if she is taken."" Face and ...",It's your face,"Ring finger to see if she is taken."" Face and ...","Ring finger to see if she is taken."" Face and ...",face
71,71,Chronic wasting disease kuru Creutzfeldt-Jakob...,kuru,Chronic wasting disease kuru Creutzfeldt-Jakob...,Chronic wasting disease kuru Creutzfeldt-Jakob,kuru
75,75,Michael and Kirk,Kirk Douglass,Michael and Kirk Kirk Douglass,Michael and Kirk Douglass,Michael and Kirk
77,77,"gut bacteria M. vaccae ""You Found Wisdom Among...",gut bacteria two 1/4-cup servings daily of yog...,"gut bacteria M. vaccae ""You Found Wisdom Among...","gut bacteria M. vaccae ""You Found Wisdom Among...",gut bacteria
107,107,$900 million $700 million,$700 million,$900 million $700 million,$900 million $700,$900 million
119,119,EasyAuto123 EasyAuto123.com,fair interest rates EasyAuto123.com,EasyAuto123 EasyAuto123.com fair interest rate...,EasyAuto123 EasyAuto123.com fair interest rates,EasyAuto123.com


## still promising however, smaller than previous fusion gain, about 0.0021 gain. But its an improvement anyways, run a stability test before submitting another file

In [120]:
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold

# 1. Prepare phrase rows and candidate score arrays

phrase_stability_df = (
    phrase_candidate_df[
        phrase_candidate_df[
            "predicted_type"
        ] == "phrase"
    ]
    .copy()
    .reset_index(drop=True)
)

# Add true spoiler type only for stratified validation splitting.
phrase_stability_df = (
    phrase_stability_df
    .merge(
        fusion_df[
            [
                "row_number",
                "spoiler_type"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)

cleanup_strategies = [
    "raw_concat",
    "adjacent_cleaned",
    "suffix_cleaned",
    "novel_append"
]

cleanup_caps = [
    8,
    10,
    12,
    15,
    20,
    25,
    30,
    40,
    50,
    60
]

configuration_score_arrays = {}

for strategy in cleanup_strategies:
    for cap in cleanup_caps:
        configuration_name = (
            f"{strategy}_cap_{cap}"
        )

        predictions = []

        for _, row in (
            phrase_stability_df.iterrows()
        ):
            prediction = phrase_cap_or_current(
                candidate=row[strategy],
                current_prediction=row[
                    "current_prediction"
                ],
                cap=cap
            )

            predictions.append(prediction)

        configuration_score_arrays[
            configuration_name
        ] = np.array([
            prediction_meteor_score(
                gold,
                prediction
            )
            for gold, prediction in zip(
                phrase_stability_df[
                    "gold_text"
                ],
                predictions
            )
        ])


raw_50_scores = (
    configuration_score_arrays[
        "raw_concat_cap_50"
    ]
)

suffix_50_scores = (
    configuration_score_arrays[
        "suffix_cleaned_cap_50"
    ]
)

# 2. Repeated unseen-fold evaluation

phrase_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=20,
    random_state=SEED
)

dummy_features = np.zeros(
    (len(phrase_stability_df), 1)
)

stratification_labels = (
    phrase_stability_df[
        "spoiler_type"
    ].astype(str).to_numpy()
)

phrase_stability_records = []
selected_configuration_counter = Counter()

for split_number, (
    train_indices,
    validation_indices
) in enumerate(
    phrase_cv.split(
        dummy_features,
        stratification_labels
    ),
    start=1
):
    training_means = {
        configuration_name: float(
            scores[train_indices].mean()
        )
        for configuration_name, scores
        in configuration_score_arrays.items()
    }

    selected_configuration = max(
        training_means,
        key=training_means.get
    )

    selected_configuration_counter[
        selected_configuration
    ] += 1

    nested_validation_scores = (
        configuration_score_arrays[
            selected_configuration
        ][validation_indices]
    )

    phrase_stability_records.append({
        "split": split_number,

        "raw_concat_cap_50": float(
            raw_50_scores[
                validation_indices
            ].mean()
        ),

        "suffix_cleaned_cap_50": float(
            suffix_50_scores[
                validation_indices
            ].mean()
        ),

        "nested_selected_cleanup": float(
            nested_validation_scores.mean()
        ),

        "selected_configuration": (
            selected_configuration
        )
    })


phrase_cleanup_stability_df = pd.DataFrame(
    phrase_stability_records
)

# 3. Summarize phrase-level and estimated overall gains

summary_records = []

for strategy_column in [
    "raw_concat_cap_50",
    "suffix_cleaned_cap_50",
    "nested_selected_cleanup"
]:
    phrase_scores = (
        phrase_cleanup_stability_df[
            strategy_column
        ]
    )

    phrase_gain = (
        phrase_scores
        - phrase_cleanup_stability_df[
            "raw_concat_cap_50"
        ]
    )

    # Only 149 of the 400 rows are affected.
    estimated_overall_scores = (
        0.436008
        + phrase_gain
        * (
            len(phrase_stability_df)
            / len(fusion_df)
        )
    )

    summary_records.append({
        "strategy": strategy_column,

        "mean_phrase_meteor": float(
            phrase_scores.mean()
        ),

        "mean_phrase_gain_vs_raw": float(
            phrase_gain.mean()
        ),

        "estimated_overall_meteor": float(
            estimated_overall_scores.mean()
        ),

        "estimated_overall_gain": float(
            estimated_overall_scores.mean()
            - 0.436008
        ),

        "win_rate_vs_raw": float(
            (phrase_gain > 0).mean()
        ),

        "tie_rate_vs_raw": float(
            (phrase_gain == 0).mean()
        ),

        "fifth_percentile_phrase_gain": float(
            phrase_gain.quantile(0.05)
        ),

        "median_phrase_gain": float(
            phrase_gain.median()
        ),

        "ninety_fifth_percentile_phrase_gain": float(
            phrase_gain.quantile(0.95)
        )
    })


phrase_cleanup_stability_summary_df = (
    pd.DataFrame(summary_records)
    .sort_values(
        "estimated_overall_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Full-validation comparison:")
print(
    "Raw aggressive fusion:",
    0.436008
)
print(
    "Suffix-cleaned fusion:",
    0.438144
)
print(
    "Full-validation gain:",
    0.438144 - 0.436008
)

print("\nRepeated unseen-fold stability:")
display(
    phrase_cleanup_stability_summary_df
)

print(
    "\nMost frequently selected cleanup configurations:"
)

selection_records = []

for configuration, frequency in (
    selected_configuration_counter
    .most_common(15)
):
    selection_records.append({
        "configuration": configuration,
        "selected_folds": frequency,
        "selection_rate": (
            frequency
            / len(
                phrase_cleanup_stability_df
            )
        )
    })

display(pd.DataFrame(selection_records))

Full-validation comparison:
Raw aggressive fusion: 0.436008
Suffix-cleaned fusion: 0.438144
Full-validation gain: 0.0021359999999999713

Repeated unseen-fold stability:


,strategy,mean_phrase_meteor,mean_phrase_gain_vs_raw,estimated_overall_meteor,estimated_overall_gain,win_rate_vs_raw,tie_rate_vs_raw,fifth_percentile_phrase_gain,median_phrase_gain,ninety_fifth_percentile_phrase_gain
0,suffix_cleaned_cap_50,0.518192,0.005728,0.438142,2.133745e-03,0.68,0.00,-0.013191,0.004513,0.026633
1,raw_concat_cap_50,0.512464,0.000000,0.436008,1.110223e-16,0.00,1.00,0.000000,0.000000,0.000000
2,nested_selected_cleanup,0.511487,-0.000977,0.435644,-3.639174e-04,0.47,0.04,-0.021902,0.000000,0.019784



Most frequently selected cleanup configurations:


,configuration,selected_folds,selection_rate
0,suffix_cleaned_cap_50,61,0.61
1,suffix_cleaned_cap_30,13,0.13
2,suffix_cleaned_cap_60,11,0.11
3,adjacent_cleaned_cap_50,6,0.06
4,suffix_cleaned_cap_25,4,0.04
5,raw_concat_cap_50,2,0.02
6,adjacent_cleaned_cap_60,2,0.02
7,raw_concat_cap_60,1,0.01


In [121]:
import pandas as pd


suffix_cleaned_test_records = []

for _, row in fusion_test_df.iterrows():
    predicted_type = str(row["predicted_type"])

    paragraph_prediction = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    article_prediction = (
        remove_synthetic_article_markers(
            row["article_prediction"]
        )
    )

    aggressive_prediction = (
        normalize_prediction_text(
            row["spoiler"]
        )
    )

    previous_hybrid_prediction = (
        normalize_prediction_text(
            row["previous_hybrid_prediction"]
        )
    )

    if predicted_type == "phrase":
        raw_phrase_fusion = (
            concatenate_without_exact_duplication(
                paragraph_prediction,
                article_prediction
            )
        )

        cleaned_phrase_fusion = (
            remove_repeated_suffixes(
                raw_phrase_fusion
            )
        )

        final_prediction = (
            phrase_cap_or_current(
                candidate=cleaned_phrase_fusion,
                current_prediction=(
                    previous_hybrid_prediction
                ),
                cap=50
            )
        )

        operator = (
            "suffix_cleaned_phrase_cap_50"
        )

    elif predicted_type == "passage":
        # Preserve the successful current-rule passage output.
        final_prediction = (
            previous_hybrid_prediction
        )

        raw_phrase_fusion = ""
        cleaned_phrase_fusion = ""
        operator = "current_rule"

    else:
        # Preserve the aggressive multi fusion that helped
        # reach the 0.45280 public score.
        final_prediction = aggressive_prediction

        raw_phrase_fusion = ""
        cleaned_phrase_fusion = ""
        operator = "concat_article_paragraph"

    final_prediction = (
        remove_synthetic_article_markers(
            final_prediction
        )
    )

    suffix_cleaned_test_records.append({
        "row_number": int(row["row_number"]),
        "id": row["id"],
        "predicted_type": predicted_type,
        "operator": operator,
        "paragraph_prediction": (
            paragraph_prediction
        ),
        "article_prediction": (
            article_prediction
        ),
        "raw_phrase_fusion": (
            raw_phrase_fusion
        ),
        "cleaned_phrase_fusion": (
            cleaned_phrase_fusion
        ),
        "previous_045280_prediction": (
            aggressive_prediction
        ),
        "spoiler": final_prediction,
        "word_count": len(
            final_prediction.split()
        ),
        "changed_from_045280": (
            final_prediction
            != aggressive_prediction
        )
    })


suffix_cleaned_fusion_test_df = (
    pd.DataFrame(
        suffix_cleaned_test_records
    )
    .sort_values("row_number")
    .reset_index(drop=True)
)


marker_pattern = (
    r"(?:Title|Paragraph\s+\d+):"
)

print("Operator counts:")
print(
    suffix_cleaned_fusion_test_df[
        "operator"
    ].value_counts()
)

print("\nChanges from the 0.45280 submission:")
print(
    pd.crosstab(
        suffix_cleaned_fusion_test_df[
            "predicted_type"
        ],
        suffix_cleaned_fusion_test_df[
            "changed_from_045280"
        ]
    )
)

print(
    "\nTotal changed predictions:",
    int(
        suffix_cleaned_fusion_test_df[
            "changed_from_045280"
        ].sum()
    )
)

print(
    "Empty predictions:",
    int(
        suffix_cleaned_fusion_test_df[
            "spoiler"
        ]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Synthetic markers:",
    int(
        suffix_cleaned_fusion_test_df[
            "spoiler"
        ]
        .str.contains(
            marker_pattern,
            case=False,
            regex=True,
            na=False
        )
        .sum()
    )
)

print("\nOverall word-count summary:")
print(
    suffix_cleaned_fusion_test_df[
        "word_count"
    ].describe()
)

print("\nWord counts by predicted type:")
display(
    suffix_cleaned_fusion_test_df.groupby(
        "predicted_type"
    )["word_count"].describe()
)

print("\nChanged phrase examples:")
display(
    suffix_cleaned_fusion_test_df[
        suffix_cleaned_fusion_test_df[
            "changed_from_045280"
        ]
    ][
        [
            "row_number",
            "id",
            "raw_phrase_fusion",
            "cleaned_phrase_fusion",
            "previous_045280_prediction",
            "spoiler",
            "word_count"
        ]
    ].head(25)
)

Operator counts:
operator
current_rule                    183
suffix_cleaned_phrase_cap_50    151
concat_article_paragraph         66
Name: count, dtype: int64

Changes from the 0.45280 submission:
changed_from_045280  False  True 
predicted_type                   
multi                   66      0
passage                183      0
phrase                 124     27

Total changed predictions: 27
Empty predictions: 0
Synthetic markers: 0

Overall word-count summary:
count    400.000000
mean      37.212500
std       34.069402
min        1.000000
25%        6.000000
50%       30.000000
75%       60.250000
max      190.000000
Name: word_count, dtype: float64

Word counts by predicted type:


,count,mean,std,min,25%,50%,75%,max
predicted_type,,,,,,,,
multi,66.0,74.772727,36.371500,14.0,51.0,75.0,99.5,190.0
passage,183.0,46.688525,25.804128,4.0,25.5,43.0,65.5,100.0
phrase,151.0,9.311258,13.871402,1.0,2.0,3.0,11.0,103.0



Changed phrase examples:


,row_number,id,raw_phrase_fusion,cleaned_phrase_fusion,previous_045280_prediction,spoiler,word_count
4,4,4,3. Remove the egg yolks from the fridge after ...,3. Remove the egg yolks from the fridge after ...,3. Remove the egg yolks from the fridge after ...,3. Remove the egg yolks from the fridge after ...,33
15,15,15,$117 billion $123.6 billion,$117 billion $123.6,$117 billion $123.6 billion,$117 billion $123.6,3
39,39,39,"YInMn blue."" blue","YInMn blue.""","YInMn blue."" blue","YInMn blue.""",2
40,40,40,low-illumination night lights blue blue wavele...,low-illumination night lights blue wavelength ...,low-illumination night lights blue blue wavele...,low-illumination night lights blue wavelength ...,30
59,59,59,Miley Ray Cyrus Miley Cyrus Cyrus,Miley Ray Cyrus,Miley Ray Cyrus Miley Cyrus Cyrus,Miley Ray Cyrus,3
71,71,71,Vermont University of Vermont,Vermont University of,Vermont University of Vermont,Vermont University of,3
78,78,78,The cosmetics company has revealed that they'l...,The cosmetics company has revealed that they'l...,The cosmetics company has revealed that they'l...,The cosmetics company has revealed that they'l...,28
80,80,80,MS Dhoni M S Dhoni,MS Dhoni M S,MS Dhoni M S Dhoni,MS Dhoni M S,4
89,89,89,World War I is trench warfare DICE showed him ...,World War I is trench warfare DICE showed him ...,World War I is trench warfare DICE showed him ...,World War I is trench warfare DICE showed him ...,18
140,140,140,José José,José,José José,José,1


In [122]:
import re
import numpy as np
import pandas as pd


DANGLING_END_WORDS = {
    "a", "an", "the",
    "and", "or", "but",
    "of", "to", "for", "from",
    "in", "on", "at", "by",
    "with", "without", "as",
    "than", "between", "into",
    "over", "under", "through",
    "his", "her", "their", "your",
    "this", "that", "these", "those"
}

NUMBER_UNIT_WORDS = {
    "percent", "percentage",
    "million", "billion", "trillion",
    "dollar", "dollars",
    "pound", "pounds",
    "euro", "euros",
    "year", "years",
    "month", "months",
    "week", "weeks",
    "day", "days",
    "hour", "hours",
    "minute", "minutes",
    "second", "seconds",
    "percent", "points"
}


def is_number_like_token(token):
    token = str(token).strip()

    token = re.sub(
        r"^[\$£€¥]",
        "",
        token
    )

    token = re.sub(
        r"[%.,:;!?\"')\]]+$",
        "",
        token
    )

    return bool(
        re.fullmatch(
            r"[+-]?\d+(?:\.\d+)?",
            token
        )
    )


def conservative_suffix_cleanup(
    text,
    mode="basic"
):
    """
    Start with suffix cleanup, but reject structurally
    suspicious reductions.

    basic:
        Blocks dangling connectors, single-letter endings,
        and incomplete number-unit expressions.

    medium:
        Also rejects removal of more than 5 tokens or
        more than 45% of the original output.

    strict:
        Also rejects removal of more than 3 tokens or
        more than 30% of the original output.
    """
    raw_text = normalize_prediction_text(text)

    adjacent_version = (
        remove_adjacent_duplicate_tokens(
            raw_text
        )
    )

    suffix_version = (
        remove_repeated_suffixes(
            raw_text
        )
    )

    if suffix_version == raw_text:
        return suffix_version

    raw_tokens = raw_text.split()
    cleaned_tokens = suffix_version.split()

    if not cleaned_tokens:
        return adjacent_version

    removed_token_count = max(
        len(raw_tokens) - len(cleaned_tokens),
        0
    )

    removed_fraction = (
        removed_token_count
        / max(len(raw_tokens), 1)
    )

    final_token_key = normalized_word_key(
        cleaned_tokens[-1]
    )

    raw_final_token_key = normalized_word_key(
        raw_tokens[-1]
    )

    structurally_invalid = False

    # Example: "University of"
    if final_token_key in DANGLING_END_WORDS:
        structurally_invalid = True

    # Example: "MS Dhoni M S"
    if (
        len(final_token_key) == 1
        and final_token_key.isalpha()
    ):
        structurally_invalid = True

    # Example: "$123.6 billion" becoming "$123.6"
    if (
        is_number_like_token(
            cleaned_tokens[-1]
        )
        and raw_final_token_key
        in NUMBER_UNIT_WORDS
    ):
        structurally_invalid = True

    if mode == "medium":
        if (
            removed_token_count > 5
            or removed_fraction > 0.45
        ):
            structurally_invalid = True

    elif mode == "strict":
        if (
            removed_token_count > 3
            or removed_fraction > 0.30
        ):
            structurally_invalid = True

    if structurally_invalid:
        return adjacent_version

    return suffix_version

# Build validation candidates

validation_fusion_lookup = (
    fusion_df
    .set_index("row_number")
)

safe_cleanup_strategy_functions = {
    "raw_concat": (
        lambda text:
        normalize_prediction_text(text)
    ),

    "adjacent_cleaned": (
        lambda text:
        remove_adjacent_duplicate_tokens(text)
    ),

    "suffix_cleaned_original": (
        lambda text:
        remove_repeated_suffixes(text)
    ),

    "suffix_safe_basic": (
        lambda text:
        conservative_suffix_cleanup(
            text,
            mode="basic"
        )
    ),

    "suffix_safe_medium": (
        lambda text:
        conservative_suffix_cleanup(
            text,
            mode="medium"
        )
    ),

    "suffix_safe_strict": (
        lambda text:
        conservative_suffix_cleanup(
            text,
            mode="strict"
        )
    )
}


safe_cleanup_results = []
safe_cleanup_predictions = {}

for strategy_name, cleanup_function in (
    safe_cleanup_strategy_functions.items()
):
    predictions = []

    for _, row in phrase_candidate_df.iterrows():
        row_number = int(row["row_number"])

        validation_row = (
            validation_fusion_lookup.loc[
                row_number
            ]
        )

        predicted_type = str(
            row["predicted_type"]
        )

        if predicted_type == "phrase":
            candidate = cleanup_function(
                row["raw_concat"]
            )

            prediction = phrase_cap_or_current(
                candidate=candidate,
                current_prediction=row[
                    "current_prediction"
                ],
                cap=50
            )

        elif predicted_type == "passage":
            prediction = (
                normalize_prediction_text(
                    validation_row[
                        "prediction_current_rule"
                    ]
                )
            )

        else:
            prediction = (
                normalize_prediction_text(
                    validation_row[
                        "prediction_concat_article_paragraph"
                    ]
                )
            )

        predictions.append(prediction)

    scores = np.array([
        prediction_meteor_score(
            gold,
            prediction
        )
        for gold, prediction in zip(
            phrase_candidate_df["gold_text"],
            predictions
        )
    ])

    phrase_mask = (
        phrase_candidate_df[
            "predicted_type"
        ].eq("phrase")
        .to_numpy()
    )

    phrase_predictions = np.array(
        predictions,
        dtype=object
    )[phrase_mask]

    raw_phrase_predictions = (
        phrase_candidate_df.loc[
            phrase_mask,
            "raw_concat"
        ]
        .apply(
            lambda prediction:
            phrase_cap_or_current(
                candidate=prediction,
                current_prediction="",
                cap=50
            )
        )
        .to_numpy()
    )

    safe_cleanup_predictions[
        strategy_name
    ] = predictions

    safe_cleanup_results.append({
        "strategy": strategy_name,
        "overall_meteor": float(
            scores.mean()
        ),
        "phrase_meteor": float(
            scores[phrase_mask].mean()
        ),
        "median_meteor": float(
            np.median(scores)
        ),
        "changed_phrase_rows": int(
            np.sum(
                phrase_predictions
                != raw_phrase_predictions
            )
        ),
        "mean_phrase_words": float(
            np.mean([
                len(
                    str(prediction).split()
                )
                for prediction
                in phrase_predictions
            ])
        ),
        "maximum_phrase_words": int(
            max(
                len(
                    str(prediction).split()
                )
                for prediction
                in phrase_predictions
            )
        )
    })


safe_cleanup_results_df = (
    pd.DataFrame(safe_cleanup_results)
    .sort_values(
        "overall_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Current public-best aggressive validation:")
print(0.436008)

print("\nOriginal suffix-cleaned validation:")
print(0.438144)

print("\nConservative cleanup comparison:")
display(safe_cleanup_results_df)

# Show cases where conservative rules reject the original suffix cleanup

phrase_only_df = (
    phrase_candidate_df[
        phrase_candidate_df[
            "predicted_type"
        ] == "phrase"
    ]
    .copy()
)

phrase_only_df[
    "suffix_original"
] = phrase_only_df[
    "raw_concat"
].apply(
    remove_repeated_suffixes
)

phrase_only_df[
    "suffix_safe_basic"
] = phrase_only_df[
    "raw_concat"
].apply(
    lambda text:
    conservative_suffix_cleanup(
        text,
        mode="basic"
    )
)

phrase_only_df[
    "suffix_safe_medium"
] = phrase_only_df[
    "raw_concat"
].apply(
    lambda text:
    conservative_suffix_cleanup(
        text,
        mode="medium"
    )
)

print(
    "\nOriginal cleanups rejected by "
    "the basic safety rules:"
)

display(
    phrase_only_df[
        phrase_only_df[
            "suffix_original"
        ]
        != phrase_only_df[
            "suffix_safe_basic"
        ]
    ][
        [
            "row_number",
            "raw_concat",
            "suffix_original",
            "suffix_safe_basic",
            "gold_text"
        ]
    ].head(30)
)

Current public-best aggressive validation:
0.436008

Original suffix-cleaned validation:
0.438144

Conservative cleanup comparison:


,strategy,overall_meteor,phrase_meteor,median_meteor,changed_phrase_rows,mean_phrase_words,maximum_phrase_words
0,suffix_cleaned_original,0.438144,0.518156,0.416667,29,7.657718,49
1,adjacent_cleaned,0.436891,0.514790,0.417805,10,7.986577,49
2,suffix_safe_strict,0.436839,0.514653,0.416667,20,7.885906,49
3,suffix_safe_basic,0.436816,0.514590,0.416667,25,7.758389,49
4,raw_concat,0.436008,0.512420,0.417805,4,8.026846,49
5,suffix_safe_medium,0.435848,0.511991,0.416667,23,7.825503,49



Original cleanups rejected by the basic safety rules:


,row_number,raw_concat,suffix_original,suffix_safe_basic,gold_text
67,67,"Ring finger to see if she is taken."" Face and ...","Ring finger to see if she is taken."" Face and ...","Ring finger to see if she is taken."" Face and ...",face
107,107,$900 million $700 million,$900 million $700,$900 million $700 million,$900 million
238,238,Salman Aamir and Salman,Salman Aamir and,Salman Aamir and Salman,Salman
327,327,2. Unbroken $31.7 million 10. Wild $5.4 millio...,2. Unbroken $31.7 million 10. Wild $5.4 millio...,2. Unbroken $31.7 million 10. Wild $5.4 millio...,$15 million


In [123]:
## boundary aware strategy

import re
import numpy as np
import pandas as pd


def original_tokens(text):
    return normalize_prediction_text(
        text
    ).split()


def token_keys(text):
    return [
        normalized_word_key(token)
        for token in original_tokens(text)
        if normalized_word_key(token)
    ]


def contains_contiguous_sequence(
    container_keys,
    contained_keys
):
    if not contained_keys:
        return False

    if len(contained_keys) > len(container_keys):
        return False

    maximum_start = (
        len(container_keys)
        - len(contained_keys)
    )

    for start in range(
        maximum_start + 1
    ):
        if (
            container_keys[
                start:
                start + len(contained_keys)
            ]
            == contained_keys
        ):
            return True

    return False


def merge_suffix_prefix(
    first_text,
    second_text
):
    """
    Merge only when the ending of the first prediction
    exactly overlaps the beginning of the second.
    """
    first_text = normalize_prediction_text(
        first_text
    )

    second_text = normalize_prediction_text(
        second_text
    )

    if not first_text:
        return second_text

    if not second_text:
        return first_text

    first_original = original_tokens(
        first_text
    )

    second_original = original_tokens(
        second_text
    )

    first_keys = [
        normalized_word_key(token)
        for token in first_original
    ]

    second_keys = [
        normalized_word_key(token)
        for token in second_original
    ]

    maximum_overlap = min(
        len(first_keys),
        len(second_keys)
    )

    overlap_length = 0

    for candidate_length in range(
        maximum_overlap,
        0,
        -1
    ):
        if (
            first_keys[-candidate_length:]
            == second_keys[:candidate_length]
        ):
            overlap_length = candidate_length
            break

    merged_tokens = (
        first_original
        + second_original[overlap_length:]
    )

    return remove_adjacent_duplicate_tokens(
        " ".join(merged_tokens)
    )


def token_containment_score(
    first_text,
    second_text
):
    first_set = set(
        token_keys(first_text)
    )

    second_set = set(
        token_keys(second_text)
    )

    minimum_size = min(
        len(first_set),
        len(second_set)
    )

    if minimum_size == 0:
        return 0.0

    return (
        len(first_set & second_set)
        / minimum_size
    )


def boundary_aware_phrase_merge(
    first_text,
    second_text,
    containment_threshold=None,
    containment_choice="longer"
):
    first_text = normalize_prediction_text(
        first_text
    )

    second_text = normalize_prediction_text(
        second_text
    )

    if not first_text:
        return second_text

    if not second_text:
        return first_text

    first_keys = token_keys(first_text)
    second_keys = token_keys(second_text)

    # Preserve whichever complete prediction contains the entire other prediction.
    if contains_contiguous_sequence(
        first_keys,
        second_keys
    ):
        return first_text

    if contains_contiguous_sequence(
        second_keys,
        first_keys
    ):
        return second_text

    containment = token_containment_score(
        first_text,
        second_text
    )

    if (
        containment_threshold is not None
        and containment
        >= containment_threshold
    ):
        first_length = len(
            original_tokens(first_text)
        )

        second_length = len(
            original_tokens(second_text)
        )

        if containment_choice == "shorter":
            return (
                first_text
                if first_length <= second_length
                else second_text
            )

        return (
            first_text
            if first_length >= second_length
            else second_text
        )

    return merge_suffix_prefix(
        first_text,
        second_text
    )


boundary_strategy_functions = {
    "raw_concat": (
        lambda paragraph, article:
        concatenate_without_exact_duplication(
            paragraph,
            article
        )
    ),

    "boundary_merge": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            paragraph,
            article
        )
    ),

    "contain_longer_075": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            paragraph,
            article,
            containment_threshold=0.75,
            containment_choice="longer"
        )
    ),

    "contain_longer_085": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            paragraph,
            article,
            containment_threshold=0.85,
            containment_choice="longer"
        )
    ),

    "contain_longer_095": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            paragraph,
            article,
            containment_threshold=0.95,
            containment_choice="longer"
        )
    ),

    "contain_shorter_075": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            paragraph,
            article,
            containment_threshold=0.75,
            containment_choice="shorter"
        )
    ),

    "contain_shorter_085": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            paragraph,
            article,
            containment_threshold=0.85,
            containment_choice="shorter"
        )
    ),

    "article_first_contain_longer_085": (
        lambda paragraph, article:
        boundary_aware_phrase_merge(
            article,
            paragraph,
            containment_threshold=0.85,
            containment_choice="longer"
        )
    )
}


validation_fusion_lookup = (
    fusion_df.set_index("row_number")
)

boundary_merge_results = []
boundary_prediction_columns = {}

for strategy_name, strategy_function in (
    boundary_strategy_functions.items()
):
    predictions = []

    for _, row in phrase_candidate_df.iterrows():
        row_number = int(row["row_number"])

        predicted_type = str(
            row["predicted_type"]
        )

        validation_row = (
            validation_fusion_lookup.loc[
                row_number
            ]
        )

        if predicted_type == "phrase":
            candidate = strategy_function(
                row["paragraph"],
                row["article"]
            )

            prediction = phrase_cap_or_current(
                candidate=candidate,
                current_prediction=row[
                    "current_prediction"
                ],
                cap=50
            )

        elif predicted_type == "passage":
            prediction = normalize_prediction_text(
                validation_row[
                    "prediction_current_rule"
                ]
            )

        else:
            prediction = normalize_prediction_text(
                validation_row[
                    "prediction_concat_article_paragraph"
                ]
            )

        predictions.append(prediction)

    scores = np.array([
        prediction_meteor_score(
            gold,
            prediction
        )
        for gold, prediction in zip(
            phrase_candidate_df["gold_text"],
            predictions
        )
    ])

    phrase_mask = (
        phrase_candidate_df[
            "predicted_type"
        ].eq("phrase").to_numpy()
    )

    phrase_predictions = np.array(
        predictions,
        dtype=object
    )[phrase_mask]

    boundary_prediction_columns[
        strategy_name
    ] = predictions

    boundary_merge_results.append({
        "strategy": strategy_name,
        "overall_meteor": float(
            scores.mean()
        ),
        "phrase_meteor": float(
            scores[phrase_mask].mean()
        ),
        "median_meteor": float(
            np.median(scores)
        ),
        "mean_phrase_words": float(
            np.mean([
                len(str(value).split())
                for value in phrase_predictions
            ])
        ),
        "maximum_phrase_words": int(
            max(
                len(str(value).split())
                for value in phrase_predictions
            )
        )
    })


boundary_merge_results_df = (
    pd.DataFrame(boundary_merge_results)
    .sort_values(
        "overall_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Current public-best aggressive validation:")
print(0.436008)

print("\nUnsafe suffix-cleaned validation:")
print(0.438144)

print("\nBoundary-aware phrase strategies:")
display(boundary_merge_results_df)


best_boundary_strategy = (
    boundary_merge_results_df.iloc[0][
        "strategy"
    ]
)

phrase_comparison_df = (
    phrase_candidate_df[
        phrase_candidate_df[
            "predicted_type"
        ] == "phrase"
    ][
        [
            "row_number",
            "paragraph",
            "article",
            "raw_concat",
            "gold_text"
        ]
    ]
    .copy()
)

phrase_mask = (
    phrase_candidate_df[
        "predicted_type"
    ].eq("phrase").to_numpy()
)

phrase_comparison_df[
    "best_boundary_prediction"
] = np.array(
    boundary_prediction_columns[
        best_boundary_strategy
    ],
    dtype=object
)[phrase_mask]

print(
    "\nBest boundary-aware strategy:",
    best_boundary_strategy
)

print("\nChanged phrase examples:")
display(
    phrase_comparison_df[
        phrase_comparison_df[
            "best_boundary_prediction"
        ]
        != phrase_comparison_df[
            "raw_concat"
        ]
    ].head(30)
)

Current public-best aggressive validation:
0.436008

Unsafe suffix-cleaned validation:
0.438144

Boundary-aware phrase strategies:


,strategy,overall_meteor,phrase_meteor,median_meteor,mean_phrase_words,maximum_phrase_words
0,contain_shorter_085,0.437666,0.516871,0.417805,7.664430,49
1,contain_shorter_075,0.437666,0.516871,0.417805,7.664430,49
2,boundary_merge,0.436699,0.514276,0.417805,7.966443,49
3,raw_concat,0.436008,0.512420,0.417805,8.026846,49
4,contain_longer_095,0.435246,0.510374,0.416667,7.885906,49
5,contain_longer_075,0.435084,0.509939,0.416667,7.798658,49
6,contain_longer_085,0.435084,0.509939,0.416667,7.798658,49
7,article_first_contain_longer_085,0.434924,0.509509,0.416667,7.798658,49



Best boundary-aware strategy: contain_shorter_085

Changed phrase examples:


,row_number,paragraph,article,raw_concat,gold_text,best_boundary_prediction
3,3,candy corn CBGB The Dead Boys,Alan Rickman as bemused owner Hilly Kristal. I...,candy corn CBGB The Dead Boys Alan Rickman as ...,Alan Rickman & Rupert Grint CBGB,candy corn CBGB The Dead Boys
44,44,"""but"" ""but"" with the word ""and."" ""I love you, ...","""I love you, but ..."" It could have been ""I lo...","""but"" ""but"" with the word ""and."" ""I love you, ...","""but""","""but"" ""but"" with the word ""and."" ""I love you, ..."
56,56,"""California"" Edan Lepucki Ms. Lepucki",Ms. Lepucki Edan Lepucki,"""California"" Edan Lepucki Ms. Lepucki Ms. Lepu...",Edan Lepucki,Ms. Lepucki Edan Lepucki
75,75,Michael and Kirk,Kirk Douglass,Michael and Kirk Kirk Douglass,Michael and Kirk,Michael and Kirk Douglass
120,120,Modern Family,"""Modern Family""","""Modern Family""","On September 10, flash sale site Joss & Main w...",Modern Family
169,169,Let Hillary live,Let Hillary live!,Let Hillary live!,Fashion Advice,Let Hillary live
189,189,"""OMG same! I keep my closet perfectly color-co...","""OMG same! I keep my closet perfectly color-co...","""OMG same! I keep my closet perfectly color-co...","""OMG same! I keep my closet perfectly color-co...","""OMG same! I keep my closet perfectly color-co..."
229,229,The Micro Bit is entering what is now quite a ...,"The BBC Micro Bit, the tiny computing device d...",The Micro Bit is entering what is now quite a ...,£12.99,The Micro Bit is entering what is now quite a ...
258,258,Optimists and pessimists Realistic views Optim...,realistic optimists,Optimists and pessimists Realistic views Optim...,realistic optimists,realistic optimists
265,265,lead Flint,"Flint, Mich., revealed that the number of chil...","lead Flint Flint, Mich., revealed that the num...",direct result of the city using its own river ...,lead Flint


In [124]:
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold

# 1. Prepare phrase-only validation data

boundary_stability_df = (
    phrase_candidate_df[
        phrase_candidate_df[
            "predicted_type"
        ] == "phrase"
    ]
    .copy()
    .reset_index(drop=True)
    .merge(
        fusion_df[
            [
                "row_number",
                "spoiler_type"
            ]
        ],
        on="row_number",
        how="left",
        validate="one_to_one"
    )
)


def build_phrase_strategy_predictions(
    dataframe,
    strategy_name
):
    predictions = []

    for _, row in dataframe.iterrows():
        paragraph = normalize_prediction_text(
            row["paragraph"]
        )

        article = normalize_prediction_text(
            row["article"]
        )

        current_prediction = (
            normalize_prediction_text(
                row["current_prediction"]
            )
        )

        if strategy_name == "raw_concat":
            candidate = (
                concatenate_without_exact_duplication(
                    paragraph,
                    article
                )
            )

        elif strategy_name == "boundary_merge":
            candidate = (
                boundary_aware_phrase_merge(
                    paragraph,
                    article
                )
            )

        elif strategy_name == "contain_shorter_075":
            candidate = (
                boundary_aware_phrase_merge(
                    paragraph,
                    article,
                    containment_threshold=0.75,
                    containment_choice="shorter"
                )
            )

        elif strategy_name == "contain_shorter_085":
            candidate = (
                boundary_aware_phrase_merge(
                    paragraph,
                    article,
                    containment_threshold=0.85,
                    containment_choice="shorter"
                )
            )

        else:
            raise ValueError(
                f"Unknown strategy: {strategy_name}"
            )

        prediction = phrase_cap_or_current(
            candidate=candidate,
            current_prediction=current_prediction,
            cap=50
        )

        predictions.append(prediction)

    return predictions


boundary_strategy_names = [
    "raw_concat",
    "boundary_merge",
    "contain_shorter_075",
    "contain_shorter_085"
]

boundary_score_arrays = {}

for strategy_name in boundary_strategy_names:
    predictions = (
        build_phrase_strategy_predictions(
            boundary_stability_df,
            strategy_name
        )
    )

    boundary_score_arrays[
        strategy_name
    ] = np.array([
        prediction_meteor_score(
            gold,
            prediction
        )
        for gold, prediction in zip(
            boundary_stability_df[
                "gold_text"
            ],
            predictions
        )
    ])

# 2. Repeated unseen-fold stability evaluation

boundary_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=20,
    random_state=SEED
)

dummy_X = np.zeros(
    (len(boundary_stability_df), 1)
)

stratification_labels = (
    boundary_stability_df[
        "spoiler_type"
    ]
    .astype(str)
    .to_numpy()
)

boundary_stability_records = []
selected_strategy_counter = Counter()

for split_number, (
    train_indices,
    validation_indices
) in enumerate(
    boundary_cv.split(
        dummy_X,
        stratification_labels
    ),
    start=1
):
    training_means = {
        strategy_name: float(
            scores[train_indices].mean()
        )
        for strategy_name, scores
        in boundary_score_arrays.items()
    }

    selected_strategy = max(
        training_means,
        key=training_means.get
    )

    selected_strategy_counter[
        selected_strategy
    ] += 1

    boundary_stability_records.append({
        "split": split_number,

        "raw_concat": float(
            boundary_score_arrays[
                "raw_concat"
            ][validation_indices].mean()
        ),

        "contain_shorter_085": float(
            boundary_score_arrays[
                "contain_shorter_085"
            ][validation_indices].mean()
        ),

        "boundary_merge": float(
            boundary_score_arrays[
                "boundary_merge"
            ][validation_indices].mean()
        ),

        "nested_selected": float(
            boundary_score_arrays[
                selected_strategy
            ][validation_indices].mean()
        ),

        "selected_strategy": (
            selected_strategy
        )
    })


boundary_stability_results_df = (
    pd.DataFrame(
        boundary_stability_records
    )
)

# 3. Convert phrase-level gains into overall estimates

summary_records = []

for strategy_column in [
    "raw_concat",
    "contain_shorter_085",
    "boundary_merge",
    "nested_selected"
]:
    phrase_scores = (
        boundary_stability_results_df[
            strategy_column
        ]
    )

    phrase_gain = (
        phrase_scores
        - boundary_stability_results_df[
            "raw_concat"
        ]
    )

    estimated_overall_scores = (
        0.436008
        + phrase_gain
        * (
            len(boundary_stability_df)
            / len(fusion_df)
        )
    )

    summary_records.append({
        "strategy": strategy_column,

        "mean_phrase_meteor": float(
            phrase_scores.mean()
        ),

        "mean_phrase_gain_vs_raw": float(
            phrase_gain.mean()
        ),

        "estimated_overall_meteor": float(
            estimated_overall_scores.mean()
        ),

        "estimated_overall_gain": float(
            estimated_overall_scores.mean()
            - 0.436008
        ),

        "win_rate_vs_raw": float(
            (phrase_gain > 0).mean()
        ),

        "tie_rate_vs_raw": float(
            (phrase_gain == 0).mean()
        ),

        "fifth_percentile_phrase_gain": float(
            phrase_gain.quantile(0.05)
        ),

        "median_phrase_gain": float(
            phrase_gain.median()
        ),

        "ninety_fifth_percentile_phrase_gain": float(
            phrase_gain.quantile(0.95)
        )
    })


boundary_stability_summary_df = (
    pd.DataFrame(summary_records)
    .sort_values(
        "estimated_overall_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Full-validation comparison:")
print("Raw aggressive fusion:", 0.436008)
print(
    "Contain-shorter-085:",
    0.437666
)
print(
    "Validation gain:",
    0.437666 - 0.436008
)

print("\nRepeated unseen-fold stability:")
display(boundary_stability_summary_df)

print("\nStrategy selection frequency:")

selection_records = []

for strategy_name, frequency in (
    selected_strategy_counter.most_common()
):
    selection_records.append({
        "strategy": strategy_name,
        "selected_folds": frequency,
        "selection_rate": (
            frequency
            / len(
                boundary_stability_results_df
            )
        )
    })

display(pd.DataFrame(selection_records))

Full-validation comparison:
Raw aggressive fusion: 0.436008
Contain-shorter-085: 0.437666
Validation gain: 0.0016579999999999928

Repeated unseen-fold stability:


,strategy,mean_phrase_meteor,mean_phrase_gain_vs_raw,estimated_overall_meteor,estimated_overall_gain,win_rate_vs_raw,tie_rate_vs_raw,fifth_percentile_phrase_gain,median_phrase_gain,ninety_fifth_percentile_phrase_gain
0,contain_shorter_085,0.516904,0.004440,0.437662,1.654033e-03,0.68,0.23,-0.00365,0.003966,0.013213
1,nested_selected,0.516298,0.003834,0.437436,1.428166e-03,0.68,0.23,-0.00365,0.003691,0.012472
2,boundary_merge,0.514322,0.001858,0.436700,6.921204e-04,0.71,0.29,0.00000,0.001852,0.006894
3,raw_concat,0.512464,0.000000,0.436008,1.110223e-16,0.00,1.00,0.00000,0.000000,0.000000



Strategy selection frequency:


,strategy,selected_folds,selection_rate
0,contain_shorter_075,96,0.96
1,boundary_merge,4,0.04


In [125]:
import pandas as pd


BOUNDARY_CONTAINMENT_THRESHOLD = 0.75

boundary_phrase_test_records = []

for _, row in fusion_test_df.iterrows():
    predicted_type = str(row["predicted_type"])

    paragraph_prediction = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    article_prediction = remove_synthetic_article_markers(
        row["article_prediction"]
    )

    aggressive_prediction = normalize_prediction_text(
        row["spoiler"]
    )

    previous_hybrid_prediction = normalize_prediction_text(
        row["previous_hybrid_prediction"]
    )

    if predicted_type == "phrase":
        boundary_candidate = boundary_aware_phrase_merge(
            first_text=paragraph_prediction,
            second_text=article_prediction,
            containment_threshold=(
                BOUNDARY_CONTAINMENT_THRESHOLD
            ),
            containment_choice="shorter"
        )

        final_prediction = phrase_cap_or_current(
            candidate=boundary_candidate,
            current_prediction=previous_hybrid_prediction,
            cap=50
        )

        operator = "contain_shorter_075"

    elif predicted_type == "passage":
        # Preserve the successful passage routing.
        final_prediction = previous_hybrid_prediction
        boundary_candidate = ""
        operator = "current_rule"

    else:
        # Preserve the uncapped multi fusion that produced 0.45280.
        final_prediction = aggressive_prediction
        boundary_candidate = ""
        operator = "concat_article_paragraph"

    final_prediction = remove_synthetic_article_markers(
        final_prediction
    )

    boundary_phrase_test_records.append({
        "row_number": int(row["row_number"]),
        "id": row["id"],
        "predicted_type": predicted_type,
        "operator": operator,
        "paragraph_prediction": paragraph_prediction,
        "article_prediction": article_prediction,
        "boundary_candidate": boundary_candidate,
        "previous_045280_prediction": (
            aggressive_prediction
        ),
        "spoiler": final_prediction,
        "word_count": len(final_prediction.split()),
        "changed_from_045280": (
            final_prediction
            != aggressive_prediction
        )
    })


boundary_phrase_test_df = (
    pd.DataFrame(boundary_phrase_test_records)
    .sort_values("row_number")
    .reset_index(drop=True)
)

marker_pattern = r"(?:Title|Paragraph\s+\d+):"

print("Operator counts:")
print(
    boundary_phrase_test_df[
        "operator"
    ].value_counts()
)

print("\nChanges from 0.45280 by predicted type:")
print(
    pd.crosstab(
        boundary_phrase_test_df[
            "predicted_type"
        ],
        boundary_phrase_test_df[
            "changed_from_045280"
        ]
    )
)

print(
    "\nTotal changed predictions:",
    int(
        boundary_phrase_test_df[
            "changed_from_045280"
        ].sum()
    )
)

print(
    "Empty predictions:",
    int(
        boundary_phrase_test_df["spoiler"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Synthetic markers:",
    int(
        boundary_phrase_test_df["spoiler"]
        .str.contains(
            marker_pattern,
            case=False,
            regex=True,
            na=False
        )
        .sum()
    )
)

print("\nOverall word-count summary:")
print(
    boundary_phrase_test_df[
        "word_count"
    ].describe()
)

print("\nWord counts by predicted type:")
display(
    boundary_phrase_test_df.groupby(
        "predicted_type"
    )["word_count"].describe()
)

print("\nChanged phrase examples:")
display(
    boundary_phrase_test_df[
        boundary_phrase_test_df[
            "changed_from_045280"
        ]
    ][
        [
            "row_number",
            "id",
            "paragraph_prediction",
            "article_prediction",
            "previous_045280_prediction",
            "boundary_candidate",
            "spoiler",
            "word_count"
        ]
    ].head(30)
)

Operator counts:
operator
current_rule                183
contain_shorter_075         151
concat_article_paragraph     66
Name: count, dtype: int64

Changes from 0.45280 by predicted type:
changed_from_045280  False  True 
predicted_type                   
multi                   66      0
passage                183      0
phrase                 144      7

Total changed predictions: 7
Empty predictions: 0
Synthetic markers: 0

Overall word-count summary:
count    400.000000
mean      37.207500
std       34.057293
min        1.000000
25%        6.000000
50%       30.500000
75%       60.250000
max      190.000000
Name: word_count, dtype: float64

Word counts by predicted type:


,count,mean,std,min,25%,50%,75%,max
predicted_type,,,,,,,,
multi,66.0,74.772727,36.371500,14.0,51.0,75.0,99.5,190.0
passage,183.0,46.688525,25.804128,4.0,25.5,43.0,65.5,100.0
phrase,151.0,9.298013,13.765074,1.0,2.0,3.0,10.5,103.0



Changed phrase examples:


,row_number,id,paragraph_prediction,article_prediction,previous_045280_prediction,boundary_candidate,spoiler,word_count
40,40,40,low-illumination night lights blue blue wavele...,artificial lens implants can help improve slee...,low-illumination night lights blue blue wavele...,low-illumination night lights blue wavelength ...,low-illumination night lights blue wavelength ...,31
42,42,42,Parenthood,'Parenthood','Parenthood',Parenthood,Parenthood,1
209,209,209,sexual objectification being a sex object is e...,"being a sex object is empowering.""",sexual objectification being a sex object is e...,sexual objectification being a sex object is e...,sexual objectification being a sex object is e...,8
218,218,218,"cell phone 1-999-367-3767, you’re automaticall...","1-999-367-3767, you’re automatically connected...","cell phone 1-999-367-3767, you’re automaticall...","cell phone 1-999-367-3767, you’re automaticall...","cell phone 1-999-367-3767, you’re automaticall...",9
263,263,263,Comedy Central has found its replacement for L...,His untitled Daily Show companion series,Comedy Central has found its replacement for L...,Comedy Central has found its replacement for L...,Comedy Central has found its replacement for L...,19
291,291,291,Chocolate pizza,crack cocaine pizza Chocolate,Chocolate pizza crack cocaine pizza Chocolate,Chocolate pizza,Chocolate pizza,2
315,315,315,more than seven hours seven hours,seven hours spent in deep sleep is the perfect...,more than seven hours seven hours seven hours ...,more than seven hours seven hours,more than seven hours seven hours,6


In [126]:
import os
import pandas as pd

BOUNDARY_FUSION_SUBMISSION_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_boundary_phrase_fusion.csv"
)

BOUNDARY_FUSION_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_boundary_phrase_fusion_audit.csv"
)

boundary_fusion_submission_df = (
    sample_df[["id"]]
    .merge(
        boundary_phrase_test_df[
            ["id", "spoiler"]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
)

boundary_fusion_submission_df.to_csv(
    BOUNDARY_FUSION_SUBMISSION_PATH,
    index=False
)

boundary_phrase_test_df.to_csv(
    BOUNDARY_FUSION_AUDIT_PATH,
    index=False
)

ids_match_sample = (
    boundary_fusion_submission_df[
        "id"
    ].tolist()
    == sample_df["id"].tolist()
)

ids_match_test = (
    boundary_fusion_submission_df[
        "id"
    ].tolist()
    == test_df["id"].tolist()
)

empty_count = int(
    boundary_fusion_submission_df[
        "spoiler"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

marker_count = int(
    boundary_fusion_submission_df[
        "spoiler"
    ]
    .astype(str)
    .str.contains(
        r"(?:Title|Paragraph\s+\d+):",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

print(
    "Submission exists:",
    os.path.exists(
        BOUNDARY_FUSION_SUBMISSION_PATH
    )
)

print(
    "Submission shape:",
    boundary_fusion_submission_df.shape
)

print(
    "Submission columns:",
    boundary_fusion_submission_df
    .columns.tolist()
)

print(
    "IDs match sample order:",
    ids_match_sample
)

print(
    "IDs match test order:",
    ids_match_test
)

print(
    "Unique IDs:",
    boundary_fusion_submission_df[
        "id"
    ].nunique()
)

print("Empty spoilers:", empty_count)
print("Synthetic markers:", marker_count)

print(
    "Changed from 0.45280:",
    int(
        boundary_phrase_test_df[
            "changed_from_045280"
        ].sum()
    )
)

print(
    "Submission location:",
    BOUNDARY_FUSION_SUBMISSION_PATH
)

print(
    "Audit location:",
    BOUNDARY_FUSION_AUDIT_PATH
)

print("\nThe seven changed rows:")

display(
    boundary_phrase_test_df[
        boundary_phrase_test_df[
            "changed_from_045280"
        ]
    ][
        [
            "row_number",
            "id",
            "previous_045280_prediction",
            "spoiler",
            "word_count"
        ]
    ]
)

Submission exists: True
Submission shape: (400, 2)
Submission columns: ['id', 'spoiler']
IDs match sample order: True
IDs match test order: True
Unique IDs: 400
Empty spoilers: 0
Synthetic markers: 0
Changed from 0.45280: 7
Submission location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_boundary_phrase_fusion.csv
Audit location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_boundary_phrase_fusion_audit.csv

The seven changed rows:


,row_number,id,previous_045280_prediction,spoiler,word_count
40,40,40,low-illumination night lights blue blue wavele...,low-illumination night lights blue wavelength ...,31
42,42,42,'Parenthood',Parenthood,1
209,209,209,sexual objectification being a sex object is e...,sexual objectification being a sex object is e...,8
218,218,218,"cell phone 1-999-367-3767, you’re automaticall...","cell phone 1-999-367-3767, you’re automaticall...",9
263,263,263,Comedy Central has found its replacement for L...,Comedy Central has found its replacement for L...,19
291,291,291,Chocolate pizza crack cocaine pizza Chocolate,Chocolate pizza,2
315,315,315,more than seven hours seven hours seven hours ...,more than seven hours seven hours,6


## another small but positive improvement on Kaggle - 0.45436. Next approach we can try would be multi-spoilter fusion.

In [128]:
import re
import numpy as np
import pandas as pd

# Safe sentence-level utilities

def split_sentence_units(text):
    text = normalize_prediction_text(text)

    if not text:
        return []

    units = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        normalize_prediction_text(unit)
        for unit in units
        if normalize_prediction_text(unit)
    ]


def normalized_sentence_key(text):
    return " ".join(
        token_keys(text)
    )


def concatenate_without_duplicate_sentences(
    first_text,
    second_text
):
    """
    Concatenate complete sentence units while removing
    only exact normalized sentence duplicates.
    """
    first_units = split_sentence_units(
        first_text
    )

    second_units = split_sentence_units(
        second_text
    )

    combined_units = []
    seen_keys = set()

    for unit in first_units + second_units:
        key = normalized_sentence_key(unit)

        if not key:
            continue

        if key in seen_keys:
            continue

        combined_units.append(unit)
        seen_keys.add(key)

    return normalize_prediction_text(
        " ".join(combined_units)
    )


def multi_cap_or_current(
    candidate,
    current_prediction,
    cap
):
    candidate = normalize_prediction_text(
        candidate
    )

    current_prediction = normalize_prediction_text(
        current_prediction
    )

    if not candidate:
        return current_prediction

    if cap is None:
        return candidate

    if len(candidate.split()) <= cap:
        return candidate

    return current_prediction

# Multi-spoiler candidate strategies

multi_strategy_functions = {
    "raw_article_paragraph": (
        lambda article, paragraph:
        concatenate_without_exact_duplication(
            article,
            paragraph
        )
    ),

    "raw_paragraph_article": (
        lambda article, paragraph:
        concatenate_without_exact_duplication(
            paragraph,
            article
        )
    ),

    "adjacent_cleaned": (
        lambda article, paragraph:
        remove_adjacent_duplicate_tokens(
            concatenate_without_exact_duplication(
                article,
                paragraph
            )
        )
    ),

    "boundary_merge": (
        lambda article, paragraph:
        boundary_aware_phrase_merge(
            article,
            paragraph
        )
    ),

    "contain_longer_075": (
        lambda article, paragraph:
        boundary_aware_phrase_merge(
            article,
            paragraph,
            containment_threshold=0.75,
            containment_choice="longer"
        )
    ),

    "contain_longer_085": (
        lambda article, paragraph:
        boundary_aware_phrase_merge(
            article,
            paragraph,
            containment_threshold=0.85,
            containment_choice="longer"
        )
    ),

    "contain_shorter_085": (
        lambda article, paragraph:
        boundary_aware_phrase_merge(
            article,
            paragraph,
            containment_threshold=0.85,
            containment_choice="shorter"
        )
    ),

    "sentence_dedup_article_paragraph": (
        lambda article, paragraph:
        concatenate_without_duplicate_sentences(
            article,
            paragraph
        )
    )
}


multi_caps = [
    None,
    100,
    125,
    150,
    175,
    200
]


validation_fusion_lookup = (
    fusion_df.set_index("row_number")
)

multi_optimization_records = []
multi_configuration_predictions = {}


for strategy_name, strategy_function in (
    multi_strategy_functions.items()
):
    for cap in multi_caps:
        predictions = []

        for _, row in phrase_candidate_df.iterrows():
            row_number = int(row["row_number"])

            predicted_type = str(
                row["predicted_type"]
            )

            validation_row = (
                validation_fusion_lookup.loc[
                    row_number
                ]
            )

            paragraph_prediction = (
                normalize_prediction_text(
                    row["paragraph"]
                )
            )

            article_prediction = (
                remove_synthetic_article_markers(
                    row["article"]
                )
            )

            current_prediction = (
                normalize_prediction_text(
                    row["current_prediction"]
                )
            )

            if predicted_type == "phrase":
                phrase_candidate = (
                    boundary_aware_phrase_merge(
                        first_text=paragraph_prediction,
                        second_text=article_prediction,
                        containment_threshold=0.75,
                        containment_choice="shorter"
                    )
                )

                prediction = phrase_cap_or_current(
                    candidate=phrase_candidate,
                    current_prediction=current_prediction,
                    cap=50
                )

            elif predicted_type == "passage":
                prediction = normalize_prediction_text(
                    validation_row[
                        "prediction_current_rule"
                    ]
                )

            else:
                multi_candidate = strategy_function(
                    article_prediction,
                    paragraph_prediction
                )

                prediction = multi_cap_or_current(
                    candidate=multi_candidate,
                    current_prediction=current_prediction,
                    cap=cap
                )

            prediction = (
                remove_synthetic_article_markers(
                    prediction
                )
            )

            predictions.append(prediction)

        scores = np.array([
            prediction_meteor_score(
                gold,
                prediction
            )
            for gold, prediction in zip(
                phrase_candidate_df[
                    "gold_text"
                ],
                predictions
            )
        ])

        multi_mask = (
            phrase_candidate_df[
                "predicted_type"
            ].eq("multi").to_numpy()
        )

        multi_predictions = np.array(
            predictions,
            dtype=object
        )[multi_mask]

        cap_name = (
            "uncapped"
            if cap is None
            else str(cap)
        )

        configuration_name = (
            f"{strategy_name}_cap_{cap_name}"
        )

        multi_configuration_predictions[
            configuration_name
        ] = predictions

        multi_optimization_records.append({
            "strategy": strategy_name,
            "multi_cap": cap_name,

            "overall_meteor": float(
                scores.mean()
            ),

            "multi_meteor": float(
                scores[multi_mask].mean()
            ),

            "median_meteor": float(
                np.median(scores)
            ),

            "mean_multi_words": float(
                np.mean([
                    len(
                        str(prediction).split()
                    )
                    for prediction
                    in multi_predictions
                ])
            ),

            "maximum_multi_words": int(
                max(
                    len(
                        str(prediction).split()
                    )
                    for prediction
                    in multi_predictions
                )
            )
        })


multi_optimization_results_df = (
    pd.DataFrame(
        multi_optimization_records
    )
    .sort_values(
        [
            "overall_meteor",
            "multi_meteor"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


print(
    "Current boundary-phrase fusion validation:",
    0.437666
)

print("\nTop 25 multi-fusion configurations:")

display(
    multi_optimization_results_df.head(25)
)

print("\nBest configuration for each strategy:")

display(
    multi_optimization_results_df
    .sort_values(
        "overall_meteor",
        ascending=False
    )
    .groupby(
        "strategy",
        as_index=False
    )
    .first()
    .sort_values(
        "overall_meteor",
        ascending=False
    )
)

Current boundary-phrase fusion validation: 0.437666

Top 25 multi-fusion configurations:


,strategy,multi_cap,overall_meteor,multi_meteor,median_meteor,mean_multi_words,maximum_multi_words
0,boundary_merge,150,0.438527,0.401655,0.421014,72.481928,141
1,boundary_merge,175,0.438527,0.401655,0.421014,72.481928,141
2,sentence_dedup_article_paragraph,125,0.438322,0.400669,0.420799,67.554217,123
3,boundary_merge,uncapped,0.438226,0.400206,0.417805,74.240964,183
4,boundary_merge,200,0.438226,0.400206,0.417805,74.240964,183
5,adjacent_cleaned,150,0.437969,0.398964,0.419003,73.457831,141
6,adjacent_cleaned,175,0.437969,0.398964,0.419003,73.457831,141
7,raw_article_paragraph,150,0.437966,0.398953,0.419003,73.530120,141
8,raw_article_paragraph,175,0.437966,0.398953,0.419003,73.530120,141
9,adjacent_cleaned,uncapped,0.437668,0.397514,0.417805,75.216867,183



Best configuration for each strategy:


,strategy,multi_cap,overall_meteor,multi_meteor,median_meteor,mean_multi_words,maximum_multi_words
1,boundary_merge,150,0.438527,0.401655,0.421014,72.481928,141
7,sentence_dedup_article_paragraph,125,0.438322,0.400669,0.420799,67.554217,123
0,adjacent_cleaned,150,0.437969,0.398964,0.419003,73.457831,141
5,raw_article_paragraph,150,0.437966,0.398953,0.419003,73.530120,141
6,raw_paragraph_article,150,0.437626,0.397312,0.417805,73.530120,141
3,contain_longer_085,150,0.437174,0.395133,0.416667,69.313253,140
4,contain_shorter_085,150,0.435721,0.388132,0.416667,63.072289,140
2,contain_longer_075,uncapped,0.434428,0.381899,0.416667,68.385542,146


## verify if this boundary_merge and 150 cap is the best configuration or not, run a stability test

In [129]:
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold

# 1. Select the meaningful multi configurations

multi_candidate_names = [
    "raw_article_paragraph_cap_uncapped",
    "raw_article_paragraph_cap_150",
    "boundary_merge_cap_uncapped",
    "boundary_merge_cap_150",
    "sentence_dedup_article_paragraph_cap_125",
    "adjacent_cleaned_cap_150"
]

missing_configurations = [
    configuration
    for configuration in multi_candidate_names
    if configuration
    not in multi_configuration_predictions
]

if missing_configurations:
    raise KeyError(
        "Missing multi configurations: "
        + ", ".join(missing_configurations)
    )

# 2. Prepare predicted-multi validation rows

multi_mask = (
    phrase_candidate_df[
        "predicted_type"
    ]
    .astype(str)
    .eq("multi")
    .to_numpy()
)

multi_row_numbers = (
    phrase_candidate_df.loc[
        multi_mask,
        "row_number"
    ]
    .astype(int)
    .to_numpy()
)

multi_gold_texts = (
    phrase_candidate_df.loc[
        multi_mask,
        "gold_text"
    ]
    .tolist()
)

spoiler_type_lookup = (
    fusion_df
    .set_index("row_number")[
        "spoiler_type"
    ]
    .astype(str)
)

multi_true_types = np.array([
    spoiler_type_lookup.loc[
        row_number
    ]
    for row_number in multi_row_numbers
])

print(
    "Predicted-multi validation rows:",
    len(multi_row_numbers)
)

print("\nTrue-type distribution:")
print(
    pd.Series(
        multi_true_types
    ).value_counts()
)

# 3. Calculate per-row METEOR for every configuration

multi_score_arrays = {}

for configuration_name in multi_candidate_names:
    all_predictions = np.array(
        multi_configuration_predictions[
            configuration_name
        ],
        dtype=object
    )

    multi_predictions = all_predictions[
        multi_mask
    ]

    multi_score_arrays[
        configuration_name
    ] = np.array([
        prediction_meteor_score(
            gold_text,
            prediction
        )
        for gold_text, prediction in zip(
            multi_gold_texts,
            multi_predictions
        )
    ])


baseline_name = (
    "raw_article_paragraph_cap_uncapped"
)

best_fixed_name = (
    "boundary_merge_cap_150"
)

sentence_dedup_name = (
    "sentence_dedup_article_paragraph_cap_125"
)

# 4. Repeated stratified unseen-fold evaluation

minimum_class_count = int(
    pd.Series(
        multi_true_types
    )
    .value_counts()
    .min()
)

number_of_splits = min(
    5,
    minimum_class_count
)

if number_of_splits < 2:
    raise ValueError(
        "Not enough examples per true type "
        "for repeated stratified validation."
    )

multi_cv = RepeatedStratifiedKFold(
    n_splits=number_of_splits,
    n_repeats=20,
    random_state=SEED
)

dummy_features = np.zeros(
    (len(multi_row_numbers), 1)
)

multi_stability_records = []
selected_configuration_counter = Counter()

for split_number, (
    train_indices,
    validation_indices
) in enumerate(
    multi_cv.split(
        dummy_features,
        multi_true_types
    ),
    start=1
):
    training_means = {
        configuration_name: float(
            score_array[
                train_indices
            ].mean()
        )
        for configuration_name, score_array
        in multi_score_arrays.items()
    }

    selected_configuration = max(
        training_means,
        key=training_means.get
    )

    selected_configuration_counter[
        selected_configuration
    ] += 1

    multi_stability_records.append({
        "split": split_number,

        "baseline_raw_uncapped": float(
            multi_score_arrays[
                baseline_name
            ][validation_indices].mean()
        ),

        "boundary_merge_cap_150": float(
            multi_score_arrays[
                best_fixed_name
            ][validation_indices].mean()
        ),

        "sentence_dedup_cap_125": float(
            multi_score_arrays[
                sentence_dedup_name
            ][validation_indices].mean()
        ),

        "nested_selected": float(
            multi_score_arrays[
                selected_configuration
            ][validation_indices].mean()
        ),

        "selected_configuration": (
            selected_configuration
        )
    })


multi_stability_df = pd.DataFrame(
    multi_stability_records
)

# 5. Estimate impact on the full 400-row validation set

summary_records = []

comparison_columns = [
    "baseline_raw_uncapped",
    "boundary_merge_cap_150",
    "sentence_dedup_cap_125",
    "nested_selected"
]

baseline_split_scores = (
    multi_stability_df[
        "baseline_raw_uncapped"
    ]
)

multi_fraction = (
    len(multi_row_numbers)
    / len(phrase_candidate_df)
)

for strategy_column in comparison_columns:
    multi_scores = (
        multi_stability_df[
            strategy_column
        ]
    )

    multi_gain = (
        multi_scores
        - baseline_split_scores
    )

    estimated_overall_scores = (
        0.437666
        + multi_gain
        * multi_fraction
    )

    summary_records.append({
        "strategy": strategy_column,

        "mean_multi_meteor": float(
            multi_scores.mean()
        ),

        "mean_multi_gain_vs_baseline": float(
            multi_gain.mean()
        ),

        "estimated_overall_meteor": float(
            estimated_overall_scores.mean()
        ),

        "estimated_overall_gain": float(
            estimated_overall_scores.mean()
            - 0.437666
        ),

        "win_rate_vs_baseline": float(
            (multi_gain > 0).mean()
        ),

        "tie_rate_vs_baseline": float(
            (multi_gain == 0).mean()
        ),

        "fifth_percentile_multi_gain": float(
            multi_gain.quantile(0.05)
        ),

        "median_multi_gain": float(
            multi_gain.median()
        ),

        "ninety_fifth_percentile_multi_gain": float(
            multi_gain.quantile(0.95)
        )
    })


multi_stability_summary_df = (
    pd.DataFrame(
        summary_records
    )
    .sort_values(
        "estimated_overall_meteor",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nFull-validation comparison:")
print(
    "Raw article-paragraph uncapped:",
    0.437666
)

print(
    "Boundary merge cap 150:",
    0.438527
)

print(
    "Full-validation gain:",
    0.438527 - 0.437666
)

print("\nRepeated unseen-fold stability:")
display(
    multi_stability_summary_df
)

print("\nConfiguration-selection frequency:")

selection_records = []

for configuration_name, frequency in (
    selected_configuration_counter
    .most_common()
):
    selection_records.append({
        "configuration": configuration_name,
        "selected_folds": frequency,
        "selection_rate": (
            frequency
            / len(multi_stability_df)
        )
    })

display(
    pd.DataFrame(
        selection_records
    )
)

Predicted-multi validation rows: 83

True-type distribution:
multi      59
phrase     14
passage    10
Name: count, dtype: int64

Full-validation comparison:
Raw article-paragraph uncapped: 0.437666
Boundary merge cap 150: 0.438527
Full-validation gain: 0.0008610000000000007

Repeated unseen-fold stability:


,strategy,mean_multi_meteor,mean_multi_gain_vs_baseline,estimated_overall_meteor,estimated_overall_gain,win_rate_vs_baseline,tie_rate_vs_baseline,fifth_percentile_multi_gain,median_multi_gain,ninety_fifth_percentile_multi_gain
0,boundary_merge_cap_150,0.401694,0.004157,0.438529,8.625461e-04,0.81,0.02,-0.004978,0.002257,0.014950
1,sentence_dedup_cap_125,0.400717,0.003180,0.438326,6.597690e-04,0.50,0.00,-0.010266,0.000072,0.025772
2,baseline_raw_uncapped,0.397537,0.000000,0.437666,5.551115e-17,0.00,1.00,0.000000,0.000000,0.000000
3,nested_selected,0.396294,-0.001243,0.437408,-2.579450e-04,0.48,0.02,-0.010266,-0.000075,0.007958



Configuration-selection frequency:


,configuration,selected_folds,selection_rate
0,boundary_merge_cap_150,49,0.49
1,sentence_dedup_article_paragraph_cap_125,35,0.35
2,boundary_merge_cap_uncapped,16,0.16


In [130]:
import pandas as pd


fusion_test_lookup = (
    fusion_test_df
    .set_index("row_number")
)

multi_boundary_test_records = []

for _, row in boundary_phrase_test_df.iterrows():
    row_number = int(row["row_number"])
    predicted_type = str(row["predicted_type"])

    paragraph_prediction = normalize_prediction_text(
        row["paragraph_prediction"]
    )

    article_prediction = remove_synthetic_article_markers(
        row["article_prediction"]
    )

    current_045436_prediction = normalize_prediction_text(
        row["spoiler"]
    )

    # This is the original current-rule prediction used as
    # the validation fallback when a multi candidate exceeded
    # the selected 150-word cap.
    current_rule_fallback = normalize_prediction_text(
        fusion_test_lookup.loc[
            row_number,
            "previous_hybrid_prediction"
        ]
    )

    multi_candidate = ""
    exceeded_multi_cap = False

    if predicted_type == "multi":
        multi_candidate = boundary_aware_phrase_merge(
            first_text=article_prediction,
            second_text=paragraph_prediction
        )

        if (
            multi_candidate
            and len(multi_candidate.split()) <= 150
        ):
            final_prediction = multi_candidate
            operator = "boundary_merge_cap_150"

        else:
            final_prediction = current_rule_fallback
            exceeded_multi_cap = True
            operator = "current_rule_fallback_over_150"

    else:
        # Preserve every phrase and passage prediction from
        # the current 0.45436 submission.
        final_prediction = current_045436_prediction

        operator = (
            "contain_shorter_075"
            if predicted_type == "phrase"
            else "current_rule"
        )

    final_prediction = remove_synthetic_article_markers(
        final_prediction
    )

    multi_boundary_test_records.append({
        "row_number": row_number,
        "id": row["id"],
        "predicted_type": predicted_type,
        "operator": operator,
        "paragraph_prediction": paragraph_prediction,
        "article_prediction": article_prediction,
        "multi_candidate": multi_candidate,
        "current_rule_fallback": current_rule_fallback,
        "previous_045436_prediction": current_045436_prediction,
        "spoiler": final_prediction,
        "word_count": len(final_prediction.split()),
        "exceeded_multi_cap": exceeded_multi_cap,
        "changed_from_045436": (
            final_prediction
            != current_045436_prediction
        )
    })


multi_boundary_test_df = (
    pd.DataFrame(multi_boundary_test_records)
    .sort_values("row_number")
    .reset_index(drop=True)
)

marker_pattern = r"(?:Title|Paragraph\s+\d+):"

print("Operator counts:")
print(
    multi_boundary_test_df[
        "operator"
    ].value_counts()
)

print("\nChanges from the 0.45436 submission:")
print(
    pd.crosstab(
        multi_boundary_test_df[
            "predicted_type"
        ],
        multi_boundary_test_df[
            "changed_from_045436"
        ]
    )
)

print(
    "\nTotal changed predictions:",
    int(
        multi_boundary_test_df[
            "changed_from_045436"
        ].sum()
    )
)

print(
    "Multi rows exceeding the 150-word cap:",
    int(
        multi_boundary_test_df[
            "exceeded_multi_cap"
        ].sum()
    )
)

print(
    "Empty predictions:",
    int(
        multi_boundary_test_df[
            "spoiler"
        ]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
)

print(
    "Synthetic markers:",
    int(
        multi_boundary_test_df[
            "spoiler"
        ]
        .str.contains(
            marker_pattern,
            case=False,
            regex=True,
            na=False
        )
        .sum()
    )
)

print("\nOverall word-count summary:")
print(
    multi_boundary_test_df[
        "word_count"
    ].describe()
)

print("\nWord counts by predicted type:")
display(
    multi_boundary_test_df.groupby(
        "predicted_type"
    )["word_count"].describe()
)

print("\nChanged multi predictions:")
display(
    multi_boundary_test_df[
        multi_boundary_test_df[
            "changed_from_045436"
        ]
    ][
        [
            "row_number",
            "id",
            "operator",
            "previous_045436_prediction",
            "multi_candidate",
            "current_rule_fallback",
            "spoiler",
            "word_count"
        ]
    ].head(30)
)

print("\nLongest resulting multi predictions:")
display(
    multi_boundary_test_df[
        multi_boundary_test_df[
            "predicted_type"
        ] == "multi"
    ]
    .sort_values(
        "word_count",
        ascending=False
    )[
        [
            "row_number",
            "id",
            "operator",
            "spoiler",
            "word_count"
        ]
    ]
    .head(15)
)

Operator counts:
operator
current_rule                      183
contain_shorter_075               151
boundary_merge_cap_150             64
current_rule_fallback_over_150      2
Name: count, dtype: int64

Changes from the 0.45436 submission:
changed_from_045436  False  True 
predicted_type                   
multi                   51     15
passage                183      0
phrase                 151      0

Total changed predictions: 15
Multi rows exceeding the 150-word cap: 2
Empty predictions: 0
Synthetic markers: 0

Overall word-count summary:
count    400.000000
mean      36.370000
std       32.477223
min        1.000000
25%        6.000000
50%       30.000000
75%       60.000000
max      139.000000
Name: word_count, dtype: float64

Word counts by predicted type:


,count,mean,std,min,25%,50%,75%,max
predicted_type,,,,,,,,
multi,66.0,69.696970,32.294305,14.0,45.75,71.5,96.75,139.0
passage,183.0,46.688525,25.804128,4.0,25.50,43.0,65.50,100.0
phrase,151.0,9.298013,13.765074,1.0,2.00,3.0,10.50,103.0



Changed multi predictions:


,row_number,id,operator,previous_045436_prediction,multi_candidate,current_rule_fallback,spoiler,word_count
6,6,6,boundary_merge_cap_150,"1. Take a long, warm shower with sweet-smellin...","1. Take a long, warm shower with sweet-smellin...","1. Take a long, warm shower with sweet-smellin...","1. Take a long, warm shower with sweet-smellin...",69
9,9,9,boundary_merge_cap_150,When an audience member asked Brandon and this...,When an audience member asked Brandon and this...,When an audience member asked Brandon and this...,When an audience member asked Brandon and this...,99
24,24,24,boundary_merge_cap_150,Men who paint one fingernail are helping to ra...,Men who paint one fingernail are helping to ra...,"On the website, people can donate to sexual ab...",Men who paint one fingernail are helping to ra...,93
38,38,38,boundary_merge_cap_150,An expert on Lou Gehrig's disease explains wha...,An expert on Lou Gehrig's disease explains wha...,less than a few percent I don't believe that a...,An expert on Lou Gehrig's disease explains wha...,74
101,101,101,boundary_merge_cap_150,"""There are 2,000 products that are going to be...","""There are 2,000 products that are going to be...","SECOND WIND LOCATION, LOCATION, LOCATION FRESH...","""There are 2,000 products that are going to be...",112
115,115,115,current_rule_fallback_over_150,After getting my DNA report I learned that the...,After getting my DNA report I learned that the...,After getting my DNA report I learned that the...,After getting my DNA report I learned that the...,48
153,153,153,boundary_merge_cap_150,2. The Road to Serfdom by F.A. Hayek 3. The Cl...,2. The Road to Serfdom by F.A. Hayek 3. The Cl...,2. The Road to Serfdom by F.A. Hayek 3. The Cl...,2. The Road to Serfdom by F.A. Hayek 3. The Cl...,38
188,188,188,boundary_merge_cap_150,5. Beyond Good & Evil 2 3. Final Fantasy XV 2....,5. Beyond Good & Evil 2 3. Final Fantasy XV 2....,5. Beyond Good & Evil 2 3. Final Fantasy XV 2....,5. Beyond Good & Evil 2 3. Final Fantasy XV 2....,24
233,233,233,boundary_merge_cap_150,it is now legal to hunt for catfish with pitch...,it is now legal to hunt for catfish with pitch...,it is now legal to hunt for catfish with pitch...,it is now legal to hunt for catfish with pitch...,99
241,241,241,boundary_merge_cap_150,negotiations with the European Union over the ...,negotiations with the European Union over the ...,the vote has exposed polarization Brexit’ Vote...,negotiations with the European Union over the ...,36



Longest resulting multi predictions:


,row_number,id,operator,spoiler,word_count
388,388,388,boundary_merge_cap_150,"Banking on bad movies Yeah, Mary-Kate and Ashl...",139
177,177,177,boundary_merge_cap_150,What happened to CS:GO skin prices after Valve...,127
70,70,70,boundary_merge_cap_150,The famously diligent test site ran the iPhone...,126
108,108,108,boundary_merge_cap_150,After Barely Recognizing Herself In A Family P...,123
385,385,385,boundary_merge_cap_150,Here we round up some global options when it c...,113
57,57,57,boundary_merge_cap_150,"Louise the infant koala — a squeaking, wet, gr...",113
101,101,101,boundary_merge_cap_150,"""There are 2,000 products that are going to be...",112
281,281,281,boundary_merge_cap_150,"3. When cats rub their head against you, they’...",110
142,142,142,boundary_merge_cap_150,The unemployment rate in November fell to 4.6 ...,107
51,51,51,boundary_merge_cap_150,How Free DLC Helped Turn A Small Steam Game In...,104


In [131]:
import os
import pandas as pd

MULTI_BOUNDARY_SUBMISSION_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_boundary_phrase_multi_cap150.csv"
)

MULTI_BOUNDARY_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_boundary_phrase_multi_cap150_audit.csv"
)

multi_boundary_submission_df = (
    sample_df[["id"]]
    .merge(
        multi_boundary_test_df[
            ["id", "spoiler"]
        ],
        on="id",
        how="left",
        validate="one_to_one"
    )
)

multi_boundary_submission_df.to_csv(
    MULTI_BOUNDARY_SUBMISSION_PATH,
    index=False
)

multi_boundary_test_df.to_csv(
    MULTI_BOUNDARY_AUDIT_PATH,
    index=False
)

ids_match_sample = (
    multi_boundary_submission_df["id"].tolist()
    == sample_df["id"].tolist()
)

ids_match_test = (
    multi_boundary_submission_df["id"].tolist()
    == test_df["id"].tolist()
)

empty_count = int(
    multi_boundary_submission_df["spoiler"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

marker_count = int(
    multi_boundary_submission_df["spoiler"]
    .astype(str)
    .str.contains(
        r"(?:Title|Paragraph\s+\d+):",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

word_counts = (
    multi_boundary_submission_df["spoiler"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

print(
    "Submission exists:",
    os.path.exists(
        MULTI_BOUNDARY_SUBMISSION_PATH
    )
)

print(
    "Submission shape:",
    multi_boundary_submission_df.shape
)

print(
    "Submission columns:",
    multi_boundary_submission_df.columns.tolist()
)

print("IDs match sample order:", ids_match_sample)
print("IDs match test order:", ids_match_test)

print(
    "Unique IDs:",
    multi_boundary_submission_df["id"].nunique()
)

print("Empty spoilers:", empty_count)
print("Synthetic markers:", marker_count)

print(
    "Changed from 0.45436:",
    int(
        multi_boundary_test_df[
            "changed_from_045436"
        ].sum()
    )
)

print(
    "150-word fallbacks:",
    int(
        multi_boundary_test_df[
            "exceeded_multi_cap"
        ].sum()
    )
)

print("Minimum words:", int(word_counts.min()))
print("Maximum words:", int(word_counts.max()))

print(
    "Submission location:",
    MULTI_BOUNDARY_SUBMISSION_PATH
)

print(
    "Audit location:",
    MULTI_BOUNDARY_AUDIT_PATH
)

print("\nFirst five rows:")
display(multi_boundary_submission_df.head())

print("\nLast five rows:")
display(multi_boundary_submission_df.tail())

Submission exists: True
Submission shape: (400, 2)
Submission columns: ['id', 'spoiler']
IDs match sample order: True
IDs match test order: True
Unique IDs: 400
Empty spoilers: 0
Synthetic markers: 0
Changed from 0.45436: 15
150-word fallbacks: 2
Minimum words: 1
Maximum words: 139
Submission location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_boundary_phrase_multi_cap150.csv
Audit location: /content/drive/MyDrive/Task2_FinalShot/outputs/submission_boundary_phrase_multi_cap150_audit.csv

First five rows:


,id,spoiler
0,0,"balloons and a sign in hand that reads, ""Heard..."
1,1,Why you SHOULD be selfish at work: Helping oth...
2,2,Have a Bunch of Money
3,3,Braconid
4,4,3. Remove the egg yolks from the fridge after ...



Last five rows:


,id,spoiler
395,395,This Is What Happens When You Leave A Hotel Cl...
396,396,Christopher Suprun
397,397,No Medication. High fat vegan plant based diet...
398,398,WikiLeaks regularly tweets about Assange’s sta...
399,399,Richard Belzer


## again a small improvement, to see if we can drive to a noticeable improvement, next step is to target passage spoilers, since they cover 183/400 test rows, and had the largest routing regret during validation

In [133]:
import pandas as pd


dataframes_to_inspect = {
    "fusion_df": fusion_df,
    "article_paragraph_comparison_df": (
        article_paragraph_comparison_df
    ),
    "routing_diagnostic_df": (
        routing_diagnostic_df
    ),
    "length_routing_df": (
        length_routing_df
    )
}

for dataframe_name, dataframe in (
    dataframes_to_inspect.items()
):
    print("\n" + "=" * 80)
    print(dataframe_name)
    print("Shape:", dataframe.shape)

    prediction_columns = [
        column
        for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in [
                "prediction",
                "article",
                "paragraph",
                "current",
                "concat",
                "sentence",
                "top",
                "close"
            ]
        )
    ]

    score_columns = [
        column
        for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in [
                "meteor",
                "score",
                "regret",
                "oracle"
            ]
        )
    ]

    print("\nPrediction-related columns:")
    print(prediction_columns)

    print("\nScore-related columns:")
    print(score_columns)


print("\n" + "=" * 80)
print("fusion_score_columns object:")
print(fusion_score_columns)

# Inspect the highest-regret predicted-passage examples

passage_diagnostic_df = (
    routing_diagnostic_df.copy()
)

predicted_type_column = next(
    (
        column
        for column in passage_diagnostic_df.columns
        if column.lower() in {
            "predicted_type",
            "roberta_predicted_type"
        }
    ),
    None
)

regret_column = next(
    (
        column
        for column in passage_diagnostic_df.columns
        if "regret" in column.lower()
    ),
    None
)

if predicted_type_column is not None:
    passage_diagnostic_df = (
        passage_diagnostic_df[
            passage_diagnostic_df[
                predicted_type_column
            ].astype(str).eq("passage")
        ]
        .copy()
    )

display_columns = [
    column
    for column in passage_diagnostic_df.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "row_number",
            "id",
            "type",
            "gold",
            "prediction",
            "article",
            "paragraph",
            "current",
            "score",
            "meteor",
            "oracle",
            "regret",
            "word"
        ]
    )
]

if regret_column is not None:
    passage_diagnostic_df = (
        passage_diagnostic_df
        .sort_values(
            regret_column,
            ascending=False
        )
    )

print("\nHighest-regret predicted-passage rows:")

display(
    passage_diagnostic_df[
        display_columns
    ].head(20)
)


fusion_df
Shape: (400, 76)

Prediction-related columns:
['article_top1', 'article_top2', 'article_top3', 'article_close3', 'article_sentence_top1', 'article_sentence_top2', 'article_sentence_top3', 'top1', 'top2', 'top3', 'close3', 'sentence_top1', 'sentence_top2', 'sentence_top3', 'paragraph_deployable', 'article_deployable', 'article_gold_type', 'paragraph_gold_type', 'article_original', 'article_cleaned', 'paragraph_prediction', 'article_word_count', 'paragraph_meteor', 'article_cleaned_meteor', 'current_selected_model', 'current_meteor', 'article_better', 'paragraph_better', 'prediction_paragraph', 'meteor_paragraph', 'prediction_article', 'meteor_article', 'prediction_current_rule', 'meteor_current_rule', 'prediction_shorter', 'prediction_longer', 'prediction_concat_paragraph_article', 'meteor_concat_paragraph_article', 'prediction_concat_article_paragraph', 'meteor_concat_article_paragraph', 'prediction_concat_paragraph_article_capped', 'meteor_concat_paragraph_article_capped', 

,row_number,spoiler_type,gold_text,article_top1,article_top2,article_top3,article_close3,article_sentence_top1,article_sentence_top2,article_sentence_top3,...,article_word_count,paragraph_meteor,article_cleaned_meteor,current_selected_model,current_meteor,oracle_selected_model,oracle_meteor,routing_regret,article_better,paragraph_better
225,225,passage,Martin says this is perfect for modeling agenc...,"Now, in exchange for legal tender (U.S. dollar...","Now, in exchange for legal tender (U.S. dollar...","Now, in exchange for legal tender (U.S. dollar...","Now, in exchange for legal tender (U.S. dollar...","Now, in exchange for legal tender (U.S. dollar...","Now, in exchange for legal tender (U.S. dollar...","Now, in exchange for legal tender (U.S. dollar...",...,14,0.817621,0.013055,article,0.013055,paragraph,0.817621,0.804566,False,True
391,391,passage,Uber let us get behind the wheel and experienc...,Uber is letting people experience its self-dri...,Uber is letting people experience its self-dri...,Uber is letting people experience its self-dri...,Uber is letting people experience its self-dri...,Uber is letting people experience its self-dri...,Uber is letting people experience its self-dri...,Uber is letting people experience its self-dri...,...,35,0.878222,0.111524,article,0.111524,paragraph,0.878222,0.766698,False,True
307,307,passage,A recent study proves that cheating is not a o...,Because some people just love to cheat,Because some people just love to cheat The bra...,Because some people just love to cheat The bra...,Because some people just love to cheat,Because some people just love to cheat and it ...,Because some people just love to cheat and it ...,Because some people just love to cheat and it ...,...,47,0.818906,0.088106,article,0.088106,paragraph,0.818906,0.730800,False,True
216,216,passage,You can get infections in your fingers from th...,dermatophagia,dermatophagia,dermatophagia,dermatophagia,A condition called dermatophagia can be develo...,A condition called dermatophagia can be develo...,A condition called dermatophagia can be develo...,...,23,0.826431,0.099338,article,0.099338,paragraph,0.826431,0.727093,False,True
235,235,passage,Resistance to antibiotics is growing at such a...,Antibiotic-resistant infections spread through...,Antibiotic-resistant infections spread through...,Antibiotic-resistant infections spread through...,Antibiotic-resistant infections spread through...,Antibiotic-resistant infections spread through...,Antibiotic-resistant infections spread through...,Antibiotic-resistant infections spread through...,...,81,0.786774,0.088889,article,0.088889,paragraph,0.786774,0.697885,False,True
228,228,passage,Kit Harington admitted during filming he may h...,Ramsay Bolton,Ramsay Bolton beating Ramsay’s face into a fin...,Ramsay Bolton beating Ramsay’s face into a fin...,Ramsay Bolton,"HBO Paragraph 2: Finally, after teasing the co...","HBO Paragraph 2: Finally, after teasing the co...","HBO Paragraph 2: Finally, after teasing the co...",...,45,0.802647,0.177627,article,0.177627,paragraph,0.802647,0.625020,False,True
82,82,passage,Republican voters would enthusiastically welco...,a black real estate developer and reality tele...,a black real estate developer and reality tele...,a black real estate developer and reality tele...,a black real estate developer and reality tele...,"Last summer, a black real estate developer and...","Last summer, a black real estate developer and...","Last summer, a black real estate developer and...",...,104,0.197949,0.739519,paragraph,0.197949,article,0.739519,0.541570,True,False
311,311,passage,"""It's both sides playing equally bad football....","""It's both sides playing equally bad football.""","""It's both sides playing equally bad football....","""It's both sides playing equally bad football....","""It's both sides playing equally bad football.""","Stoops sums up UK's second half pretty well: ""...","Stoops sums up UK's second half pretty well: ""...","Stoops su

In [7]:
import os
import json
import platform
import pandas as pd
import torch

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Task2_FinalShot"
)

DATA_DIR = os.path.join(
    PROJECT_DIR,
    "data"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs"
)

PARAGRAPH_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "paragraph_qa_model"
)

ARTICLE_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "article_qa_model"
)

TYPE_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "type_classifier_model"
)

VAL_PATH = os.path.join(
    DATA_DIR,
    "val.jsonl"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "test.jsonl"
)

BEST_AUDIT_PATH = os.path.join(
    OUTPUT_DIR,
    "submission_boundary_phrase_multi_cap150_audit.csv"
)


def load_jsonl(filepath):
    records = []

    with open(
        filepath,
        "r",
        encoding="utf-8"
    ) as file:
        for line in file:
            line = line.strip()

            if line:
                records.append(
                    json.loads(line)
                )

    return pd.DataFrame(records)


val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

best_test_audit_df = pd.read_csv(
    BEST_AUDIT_PATH
)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device: CPU")

print("\nValidation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print(
    "Current-best audit shape:",
    best_test_audit_df.shape
)

print("\nSaved models:")
print(
    "Paragraph:",
    os.path.exists(
        os.path.join(
            PARAGRAPH_MODEL_DIR,
            "model.safetensors"
        )
    )
)
print(
    "Article:",
    os.path.exists(
        os.path.join(
            ARTICLE_MODEL_DIR,
            "model.safetensors"
        )
    )
)
print(
    "Type classifier:",
    os.path.exists(
        os.path.join(
            TYPE_MODEL_DIR,
            "model.safetensors"
        )
    )
)

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nCurrent-best audit columns:")
print(best_test_audit_df.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13
PyTorch: 2.11.0+cpu
CUDA available: False
Selected device: CPU

Validation shape: (400, 14)
Test shape: (400, 10)
Current-best audit shape: (400, 13)

Saved models:
Paragraph: True
Article: True
Type classifier: True

Validation columns:
['postId', 'postText', 'postPlatform', 'targetParagraphs', 'targetTitle', 'targetDescription', 'targetKeywords', 'targetMedia', 'targetUrl', 'provenance', 'spoiler', 'spoilerPositions', 'tags', 'id']

Current-best audit columns:
['row_number', 'id', 'predicted_type', 'operator', 'paragraph_prediction', 'article_prediction', 'multi_candidate', 'current_rule_fallback', 'previous_045436_prediction', 'spoiler', 'word_count', 'exceeded_multi_cap', 'changed_from_045436']


In [8]:
import os
import json
import pandas as pd

TYPE_SUMMARY_PATH = os.path.join(
    TYPE_MODEL_DIR,
    "training_summary.json"
)

TYPE_CONFIG_PATH = os.path.join(
    TYPE_MODEL_DIR,
    "config.json"
)

TEST_TYPE_PREDICTIONS_PATH = os.path.join(
    OUTPUT_DIR,
    "test_type_predictions.csv"
)

with open(
    TYPE_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:
    type_config = json.load(file)

if os.path.exists(TYPE_SUMMARY_PATH):
    with open(
        TYPE_SUMMARY_PATH,
        "r",
        encoding="utf-8"
    ) as file:
        type_training_summary = json.load(file)
else:
    type_training_summary = {}

test_type_saved_df = pd.read_csv(
    TEST_TYPE_PREDICTIONS_PATH
)

print("Classifier label mapping:")
print("id2label:", type_config.get("id2label"))
print("label2id:", type_config.get("label2id"))

print("\nTraining summary:")
print(
    json.dumps(
        type_training_summary,
        indent=2
    )
)

print("\nSaved test-type predictions:")
print("Shape:", test_type_saved_df.shape)
print("Columns:", test_type_saved_df.columns.tolist())

print("\nPredicted-type counts:")
if "predicted_type" in test_type_saved_df.columns:
    print(
        test_type_saved_df[
            "predicted_type"
        ].value_counts()
    )

print("\nFirst five saved predictions:")
display(test_type_saved_df.head())

print("\nFirst validation example fields:")
print("postText:", val_df.loc[0, "postText"])
print("targetTitle:", val_df.loc[0, "targetTitle"])
print(
    "First target paragraph:",
    val_df.loc[0, "targetParagraphs"][0]
)
print("Gold tags:", val_df.loc[0, "tags"])

Classifier label mapping:
id2label: {'0': 'phrase', '1': 'passage', '2': 'multi'}
label2id: {'multi': 2, 'passage': 1, 'phrase': 0}

Training summary:
{
  "best_checkpoint": "/content/type_training_output/checkpoint-600",
  "best_macro_f1": 0.7471291365064004,
  "final_epoch": 3.0,
  "train_loss": 1.6043444283803303,
  "train_runtime": 241.433,
  "label_mapping": {
    "0": "phrase",
    "1": "passage",
    "2": "multi"
  }
}

Saved test-type predictions:
Shape: (400, 7)
Columns: ['row_number', 'id', 'predicted_type', 'prob_phrase', 'prob_passage', 'prob_multi', 'confidence']

Predicted-type counts:
predicted_type
passage    183
phrase     151
multi       66
Name: count, dtype: int64

First five saved predictions:


,row_number,id,predicted_type,prob_phrase,prob_passage,prob_multi,confidence
0,0,0,phrase,0.604905,0.107925,0.287170,0.604905
1,1,1,passage,0.050703,0.836890,0.112407,0.836890
2,2,2,phrase,0.854692,0.120896,0.024412,0.854692
3,3,3,phrase,0.827504,0.097014,0.075481,0.827504
4,4,4,phrase,0.502479,0.455951,0.041569,0.502479



First validation example fields:
postText: ['Five Nights at Freddy’s Sequel Delayed for Weird Reason']
targetTitle: Five Nights at Freddy’s Sequel Delayed for Weird Reason
First target paragraph: Five Nights at Freddy’s creator Scott Cawthon takes to Steam to tease a possible delay for Five Nights at Freddy’s: Sister Location, the fifth game in the series.
Gold tags: ['passage']


In [9]:
import os
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Reload saved test predictions if needed.
TEST_TYPE_PREDICTIONS_PATH = os.path.join(
    OUTPUT_DIR,
    "test_type_predictions.csv"
)

if "test_type_saved_df" not in globals():
    test_type_saved_df = pd.read_csv(
        TEST_TYPE_PREDICTIONS_PATH
    )

print("Loading saved type classifier on CPU...")

type_tokenizer = AutoTokenizer.from_pretrained(
    TYPE_MODEL_DIR,
    local_files_only=True
)

type_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        TYPE_MODEL_DIR,
        local_files_only=True
    )
    .to("cpu")
)

type_model.eval()

print("Model loaded.")
print("Model labels:", type_model.config.id2label)


def join_list_field(value):
    if isinstance(value, list):
        return " ".join(
            str(item)
            for item in value
            if str(item).strip()
        )

    if pd.isna(value):
        return ""

    return str(value)


def post_text(row):
    return join_list_field(
        row["postText"]
    )


def title_text(row):
    return str(
        row.get("targetTitle", "")
        or ""
    )


def paragraph_text(row):
    return join_list_field(
        row.get("targetParagraphs", [])
    )


# Each builder returns:
# (main_text, optional_text_pair)
candidate_builders = {
    "post_only": lambda row: (
        post_text(row),
        None
    ),

    "title_only": lambda row: (
        title_text(row),
        None
    ),

    "post_plus_title": lambda row: (
        post_text(row)
        + " "
        + title_text(row),
        None
    ),

    "post_plus_paragraphs": lambda row: (
        post_text(row)
        + " "
        + paragraph_text(row),
        None
    ),

    "post_title_paragraphs": lambda row: (
        post_text(row)
        + " "
        + title_text(row)
        + " "
        + paragraph_text(row),
        None
    ),

    "labeled_all_fields": lambda row: (
        "Post: "
        + post_text(row)
        + " Title: "
        + title_text(row)
        + " Article: "
        + paragraph_text(row),
        None
    ),

    "post_pair_title": lambda row: (
        post_text(row),
        title_text(row)
    ),

    "post_pair_title_paragraphs": lambda row: (
        post_text(row),
        title_text(row)
        + " "
        + paragraph_text(row)
    ),

    "post_title_pair_paragraphs": lambda row: (
        post_text(row)
        + " "
        + title_text(row),
        paragraph_text(row)
    )
}


def predict_probabilities(
    dataframe,
    builder,
    batch_size=4
):
    all_probabilities = []

    for start_index in range(
        0,
        len(dataframe),
        batch_size
    ):
        batch_df = dataframe.iloc[
            start_index:
            start_index + batch_size
        ]

        main_texts = []
        text_pairs = []

        for _, row in batch_df.iterrows():
            main_text, text_pair = builder(row)

            main_texts.append(
                main_text.strip()
            )

            text_pairs.append(
                None
                if text_pair is None
                else text_pair.strip()
            )

        use_text_pairs = any(
            pair is not None
            for pair in text_pairs
        )

        tokenizer_arguments = {
            "text": main_texts,
            "padding": True,
            "truncation": True,
            "max_length": 512,
            "return_tensors": "pt"
        }

        if use_text_pairs:
            tokenizer_arguments[
                "text_pair"
            ] = text_pairs

        encoded = type_tokenizer(
            **tokenizer_arguments
        )

        with torch.no_grad():
            logits = type_model(
                **encoded
            ).logits

        probabilities = torch.softmax(
            logits,
            dim=-1
        ).cpu().numpy()

        all_probabilities.append(
            probabilities
        )

    return np.vstack(
        all_probabilities
    )


# A small sample is enough to identify the exact format.
comparison_rows = 24

test_subset_df = (
    test_df
    .iloc[:comparison_rows]
    .copy()
    .reset_index(drop=True)
)

saved_subset_df = (
    test_type_saved_df
    .sort_values("row_number")
    .iloc[:comparison_rows]
    .reset_index(drop=True)
)

saved_probabilities = saved_subset_df[
    [
        "prob_phrase",
        "prob_passage",
        "prob_multi"
    ]
].to_numpy()

saved_types = saved_subset_df[
    "predicted_type"
].astype(str).to_numpy()

comparison_records = []
candidate_probability_results = {}

for candidate_name, builder in (
    candidate_builders.items()
):
    probabilities = predict_probabilities(
        test_subset_df,
        builder
    )

    candidate_probability_results[
        candidate_name
    ] = probabilities

    predicted_indices = probabilities.argmax(
        axis=1
    )

    predicted_types = np.array([
        type_model.config.id2label[
            int(index)
        ]
        for index in predicted_indices
    ])

    absolute_difference = np.abs(
        probabilities
        - saved_probabilities
    )

    comparison_records.append({
        "input_format": candidate_name,
        "mean_absolute_probability_error": float(
            absolute_difference.mean()
        ),
        "maximum_probability_error": float(
            absolute_difference.max()
        ),
        "predicted_type_agreement": float(
            (
                predicted_types
                == saved_types
            ).mean()
        )
    })


classifier_input_comparison_df = (
    pd.DataFrame(
        comparison_records
    )
    .sort_values(
        [
            "mean_absolute_probability_error",
            "predicted_type_agreement"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("\nClassifier input-format comparison:")
display(
    classifier_input_comparison_df
)

best_input_format = (
    classifier_input_comparison_df
    .iloc[0]["input_format"]
)

print(
    "\nBest matching input format:",
    best_input_format
)

print(
    "Best mean probability error:",
    classifier_input_comparison_df
    .iloc[0][
        "mean_absolute_probability_error"
    ]
)

Loading saved type classifier on CPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.
Model labels: {0: 'phrase', 1: 'passage', 2: 'multi'}

Classifier input-format comparison:


,input_format,mean_absolute_probability_error,maximum_probability_error,predicted_type_agreement
0,labeled_all_fields,0.065455,0.622595,0.916667
1,post_pair_title_paragraphs,0.083089,0.782608,0.916667
2,post_title_paragraphs,0.101030,0.833396,0.875000
3,post_pair_title,0.101804,0.632772,0.916667
4,post_title_pair_paragraphs,0.104305,0.780398,0.875000
5,post_plus_paragraphs,0.105668,0.837226,0.875000
6,post_plus_title,0.141806,0.693206,0.791667
7,post_only,0.147266,0.605621,0.833333
8,title_only,0.194815,0.889476,0.625000



Best matching input format: labeled_all_fields
Best mean probability error: 0.06545530676614958


In [14]:
import os
import json
import shutil
import platform
from datetime import datetime

import numpy as np
import pandas as pd
import torch

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Task2_FinalShot"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs"
)

RESUME_DIR = os.path.join(
    OUTPUT_DIR,
    "resume_checkpoint"
)

os.makedirs(
    RESUME_DIR,
    exist_ok=True
)

saved_files = []
failed_files = []

# ---------------------------------------------------------
# 1. Save tonight's classifier-format comparison
# ---------------------------------------------------------

if (
    "classifier_input_comparison_df" in globals()
    and isinstance(
        classifier_input_comparison_df,
        pd.DataFrame
    )
):
    comparison_path = os.path.join(
        RESUME_DIR,
        "classifier_input_comparison.csv"
    )

    classifier_input_comparison_df.to_csv(
        comparison_path,
        index=False
    )

    saved_files.append(comparison_path)


# ---------------------------------------------------------
# 2. Save the small candidate probability arrays
# ---------------------------------------------------------

if (
    "candidate_probability_results" in globals()
    and isinstance(
        candidate_probability_results,
        dict
    )
    and candidate_probability_results
):
    probability_path = os.path.join(
        RESUME_DIR,
        "candidate_probability_results.npz"
    )

    probability_arrays = {
        str(name): np.asarray(values)
        for name, values
        in candidate_probability_results.items()
    }

    np.savez_compressed(
        probability_path,
        **probability_arrays
    )

    saved_files.append(probability_path)


# ---------------------------------------------------------
# 3. Create clearly named backups of the current best files
# ---------------------------------------------------------

important_files = {
    "best_public_0.45453_submission.csv": (
        "submission_boundary_phrase_multi_cap150.csv"
    ),
    "best_public_0.45453_audit.csv": (
        "submission_boundary_phrase_multi_cap150_audit.csv"
    ),
    "test_type_predictions_backup.csv": (
        "test_type_predictions.csv"
    )
}

for backup_name, original_name in important_files.items():
    source_path = os.path.join(
        OUTPUT_DIR,
        original_name
    )

    destination_path = os.path.join(
        RESUME_DIR,
        backup_name
    )

    if os.path.exists(source_path):
        shutil.copy2(
            source_path,
            destination_path
        )

        saved_files.append(destination_path)
    else:
        failed_files.append(source_path)


# ---------------------------------------------------------
# 4. Save an exact progress record
# ---------------------------------------------------------

resume_state = {
    "saved_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "current_public_best": 0.45453,
    "current_best_submission": (
        "outputs/"
        "submission_boundary_phrase_multi_cap150.csv"
    ),
    "current_best_audit": (
        "outputs/"
        "submission_boundary_phrase_multi_cap150_audit.csv"
    ),
    "models_safely_saved": {
        "paragraph_qa_model": os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "paragraph_qa_model",
                "model.safetensors"
            )
        ),
        "article_qa_model": os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "article_qa_model",
                "model.safetensors"
            )
        ),
        "type_classifier_model": os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "type_classifier_model",
                "model.safetensors"
            )
        )
    },
    "raw_data_safely_saved": {
        "train": os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "data",
                "train.jsonl"
            )
        ),
        "validation": os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "data",
                "val.jsonl"
            )
        ),
        "test": os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "data",
                "test.jsonl"
            )
        )
    },
    "exact_type_classifier_input": {
        "first_sequence": (
            "cleaned postText"
        ),
        "second_sequence": (
            "TITLE: {targetTitle} "
            "DESCRIPTION: {targetDescription} "
            "ARTICLE: {joined targetParagraphs}"
        ),
        "truncation": "longest_first",
        "max_length": 320,
        "padding": "max_length"
    },
    "next_step": (
        "Generate and save validation type predictions "
        "using the exact paired-input classifier format. "
        "Then rerun paragraph and article QA inference for "
        "passage-strategy validation. No retraining is needed."
    ),
    "runtime": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available()
    },
    "notebook_path": (
        "/content/drive/MyDrive/Colab Notebooks/"
        "task_2_model.ipynb"
    ),
    "notebook_exists": os.path.exists(
        "/content/drive/MyDrive/Colab Notebooks/"
        "task_2_model.ipynb"
    ),
    "best_input_format_from_incomplete_test": (
        str(best_input_format)
        if "best_input_format" in globals()
        else None
    ),
    "important_note": (
        "The earlier nine-format comparison did not include "
        "the exact training format, so labeled_all_fields "
        "must not be treated as the correct format."
    )
}

resume_state_path = os.path.join(
    RESUME_DIR,
    "resume_state.json"
)

with open(
    resume_state_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        resume_state,
        file,
        indent=2,
        ensure_ascii=False
    )

saved_files.append(resume_state_path)


# ---------------------------------------------------------
# 5. Save a readable restart note
# ---------------------------------------------------------

restart_note = """
TASK 2 RESTART POINT

Current public best:
0.45453

Best submission:
outputs/submission_boundary_phrase_multi_cap150.csv

Models:
- paragraph_qa_model
- article_qa_model
- type_classifier_model

All models and important CSVs are already stored in Google Drive.

Exact type-classifier preprocessing:
1. First sequence: cleaned postText.
2. Second sequence:
   TITLE: {targetTitle}
   DESCRIPTION: {targetDescription}
   ARTICLE: {all targetParagraphs joined}
3. truncation="longest_first"
4. max_length=320
5. padding="max_length"

Next step:
Generate val_type_predictions.csv with the exact classifier format.
After that, rerun only the validation inference needed for the
passage strategy. Do not retrain any saved model.

The nine-format experiment from the CPU session did not include the
exact original classifier format. Its labeled_all_fields result should
not be used.
""".strip()

restart_note_path = os.path.join(
    RESUME_DIR,
    "READ_ME_NEXT_SESSION.txt"
)

with open(
    restart_note_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(restart_note)

saved_files.append(restart_note_path)


print("Resume checkpoint directory:")
print(RESUME_DIR)

print("\nSaved files:")
for filepath in saved_files:
    print(
        "✓",
        os.path.relpath(
            filepath,
            PROJECT_DIR
        )
    )

print("\nMissing or failed files:")
if failed_files:
    for filepath in failed_files:
        print("✗", filepath)
else:
    print("None")

print(
    "\nCheckpoint complete:",
    os.path.exists(resume_state_path)
    and os.path.exists(restart_note_path)
)

print(
    "Notebook stored in Drive:",
    resume_state["notebook_exists"]
)

print(
    "Current-best backup exists:",
    os.path.exists(
        os.path.join(
            RESUME_DIR,
            "best_public_0.45453_submission.csv"
        )
    )
)

Resume checkpoint directory:
/content/drive/MyDrive/Task2_FinalShot/outputs/resume_checkpoint

Saved files:
✓ outputs/resume_checkpoint/classifier_input_comparison.csv
✓ outputs/resume_checkpoint/candidate_probability_results.npz
✓ outputs/resume_checkpoint/best_public_0.45453_submission.csv
✓ outputs/resume_checkpoint/best_public_0.45453_audit.csv
✓ outputs/resume_checkpoint/test_type_predictions_backup.csv
✓ outputs/resume_checkpoint/resume_state.json
✓ outputs/resume_checkpoint/READ_ME_NEXT_SESSION.txt

Missing or failed files:
None

Checkpoint complete: True
Notebook stored in Drive: True
Current-best backup exists: True
